# Ricostruzione della domanda di mobilità e localizzazione ottimale delle infrastrutture di ricarica per veicoli elettrici: sviluppo di un framework integrato per la rete stradale del Friuli Venezia Giulia

## Documento unico di riferimento del progetto di tesi

**Stato del documento:** riferimento unico e autorevole; architettura consolidata, dati e formulazioni operative ancora in sviluppo  
**Versione documento:** v0.29  
**Ultimo aggiornamento:** 1 settembre 2026
**Progetto:** *Ricostruzione della domanda di mobilità e localizzazione ottimale delle infrastrutture di ricarica per veicoli elettrici: sviluppo di un framework integrato per la rete stradale del Friuli Venezia Giulia.*  
**Fase corrente:** le **Fasi 5.6, 5.7, 5.8A, 5.8R2 e 5.8D restano CLOSED/FROZEN**. La **Fase 5.8E è CLOSED / STOP / NOT_READY FOR PARAMETER FREEZE**: la calibrazione FVG-only ha prodotto un candidato exponential numericamente identificabile, ma la generalizzazione spaziale è risultata insufficiente e sono emersi sia un mismatch del dominio di domanda rispetto ai conteggi ANAS sia una limitazione section-specific della rappresentazione deterministic all-or-nothing su `920022`. Di conseguenza **$Q$ e $\beta$ non sono frozen**; la deterrenza exponential resta `INCUMBENT REFERENCE ONLY`, mentre le varianti `ORIGIN-CONSTRAINED` e `COMBINED/TANNER` sono state testate e respinte come baseline. Non è emerso alcun `GENERAL_TIME_B5_FAILURE`: `G_OSM_operativo`, `Gamma_OSM`, `EXP_REL_300`, `OD_PATH_SYSTEM_OSM`, `TIME_B5` e `PRODUCT_LAMBDA` restano frozen e non vengono riaperti. `ANAS 2025` resta `FINAL TEMPORAL HOLDOUT / NOT USED`.  
**Prossima fase operativa:** **EXTERNAL / TRANSBORDER DEMAND COMPONENT**. Il prossimo blocco deve progettare e materializzare, in modo **separato, versionato e additivo**, la domanda `INTERNAL→EXTERNAL`, `EXTERNAL→INTERNAL` ed `EXTERNAL→EXTERNAL / THROUGH`, con almeno resto d'Italia, Austria e Slovenia e una `GATEWAY / EXTERNAL ZONE ARCHITECTURE` collegata al network interno frozen senza rigenerarlo. Solo dopo l'ampliamento del dominio di domanda si riapriranno il disegno di calibrazione Gravity, l'identificazione di $Q,\beta$, il gate di generalizzazione spaziale e infine il holdout temporale ANAS 2025.

---

## 0. Autorità, scopo e rapporto con i documenti precedenti

Il presente notebook è il **documento unico di riferimento** per lo stato corrente del progetto di tesi. Le affermazioni qui classificate come decisioni consolidate, attività completate, ipotesi, proposte o questioni aperte prevalgono sulle formulazioni incompatibili presenti nelle bozze precedenti.

`Modello_FRLM_Tesi.md` resta una bozza storica utile per:

- l'adozione di un **Capacitated Flow-Refueling Location Model** nel solco del FRLM originario e delle sue estensioni capacitate;[^kuby2005][^saadati2022]
- la presenza di costi fissi di apertura, costi variabili per i punti di ricarica e costi di adeguamento della rete elettrica;
- il pre-calcolo delle strategie di ricarica ammissibili;
- le future estensioni relative a deviazioni, incertezza e rete elettrica.

Questo documento integra in un'unica struttura lo stato dei dati e delle elaborazioni GIS, la ricostruzione della domanda light, la separazione metodologica della componente heavy, la rete stradale, la calibrazione con conteggi ANAS e il successivo ingresso dei flussi di percorso nel FRLM. Definisce inoltre la **struttura topologica del problema**, evita sovrapposizioni tra comuni, nodi stradali e siti di ricarica e stabilisce una notazione coerente per gli sviluppi matematici e informatici.

La modifica concettuale principale rispetto al file originario è la seguente:

> i comuni non sono interpretati come punti fisici nei quali installare direttamente una stazione, ma come **zone territoriali di origine, destinazione e rendicontazione**. La localizzazione operativa avviene su nodi candidati reali o virtuali appartenenti alla rete stradale, mentre il risultato finale può essere nuovamente aggregato a livello comunale.

## 0.1 Mappa rapida del documento e regole di consultazione

Questa sezione è la **porta di ingresso al documento**. Il notebook svolge contemporaneamente tre funzioni:

1. **manuale metodologico**, nel quale sono spiegati concetti, dati, formule e procedure;
2. **registro decisionale**, nel quale è possibile recuperare rapidamente ciò che è stato congelato, ciò che è ancora ipotetico e ciò che deve essere deciso;
3. **audit trail del progetto**, nel quale rimane traccia del percorso con cui si è arrivati alle scelte correnti.

Per questo motivo non è necessario leggere il documento sempre in sequenza. Quando si riprende il lavoro dopo una pausa, la consultazione consigliata è:

$$
\boxed{
\text{stato iniziale}
\rightarrow
\text{§20 decisioni consolidate}
\rightarrow
\text{§22 questioni aperte}
\rightarrow
\text{§23 roadmap}
\rightarrow
\text{sezione tecnica interessata}
}
$$

Il **Registro aggiornamenti (§24)** serve invece a ricostruire *quando* e *come* si è arrivati a una decisione, ma non sostituisce la formulazione metodologica canonica contenuta nelle sezioni tecniche.

### 0.1.1 Dove trovare rapidamente gli argomenti

| Se si cerca... | Sezione di riferimento | Contenuto |
|---|---:|---|
| stato corrente del progetto | intestazione iniziale | fase corrente e prossimo gate |
| autorità del documento e rapporto con le bozze | §0 | cosa è autorevole e cosa è storico |
| principi generali del modello | §1 | separazione zona–rete–candidati e light/heavy |
| simboli e convenzioni | §2 | notazione matematica e regole anti-ambiguità |
| comuni, zone esterne e dati territoriali | §3 | zonizzazione, centroidi, OD concettuali e stato GIS |
| baseline storica GSFVG | §4.1.1 e §4.4.5 | costruzione, direzioni, accessi e audit che hanno preceduto lo switch |
| backbone operativo OSM | §4.4.6--§4.4.7 | `SWITCH_OSM`, pipeline B1--B5, Final Functional Gate e freeze definitivo |
| sistema canonico dei percorsi OD interni OSM | §4.4.8 | Fase 5.7, `TIME_B5`, `PRODUCT-LAMBDA`, 46.010 OD e 414.090 access-pair path frozen |
| accessi GSFVG $\Gamma_o^{\mathrm L}$ | §4.2 | baseline storica chiusa, non più sorgente operativa downstream |
| accessi OSM $\Gamma_o^{L,OSM}$ | §4.4.7 | structural nodes, topological segments, E0--E3, 645 accessi e `EXP_REL_300` frozen |
| ruoli di QGIS e Python | §4.3 | separazione delle responsabilità operative |
| validazione funzionale del grafo | §4.4 | provenance GSFVG, Gate shadow, Final Functional Gate OSM e `NETWORK AUDIT = STOP` |
| shortest path metrici SP0/SP1 GSFVG | §4.4.5 | benchmark storico computazionalmente superato; aggregazione comunale non congelata |
| corridoi e direzioni di marcia | §5 | struttura dei corridoi del modello |
| candidati alla ricarica | §§6–7 | definizione e generazione dei siti candidati |
| vincoli normativi | §8 | AFIR e parametri normativi da trasferire nel modello |
| matrici OD e domanda light | §9 | ISTAT, seed gravitazionale, ANAS, matrix estimation e assegnazione |
| strategie di ricarica | §10 | strategie ammissibili, energia e tempi di occupazione |
| variabili e formulazione FRLM | §§11–14 | variabili decisionali, parametri, funzione obiettivo e vincoli |
| output del modello | §15 | risultati a livello comunale |
| light vs heavy | §16 | specificità dei due segmenti |
| validazione finale | §17 | rete, domanda, localizzazione e gate verso il modello combinato |
| modello combinato futuro | §18 | possibile evoluzione successiva |
| corrispondenza con la bozza FRLM storica | §19 | cosa viene ereditato da `Modello_FRLM_Tesi.md` |
| **decisioni congelate** | **§20** | elenco sintetico delle decisioni metodologiche consolidate |
| ipotesi non ancora consolidate | §21 | proposte operative ancora modificabili |
| **problemi ancora da risolvere** | **§22** | questioni aperte organizzate per tema |
| **sequenza operativa corrente** | **§23** | roadmap e gate successivi |
| storia delle modifiche | §24 | registro cronologico degli aggiornamenti |
| fonti principali | §25 | riferimenti bibliografici, dataset e fonti operative |

### 0.1.2 Mappa dei gate tecnici principali

La sequenza seguente non sostituisce la roadmap completa della §23, ma permette di capire immediatamente quali blocchi metodologici sono già consolidati.

| Blocco | Stato corrente | Riferimento canonico |
|---|---|---|
| base territoriale e zonizzazione interna | consolidata | §3 |
| scelta iniziale backbone GSFVG | **STORICA / SUPERATA OPERATIVAMENTE** | §4.1.1 |
| consolidamento direzionale GSFVG | **CHIUSO — baseline storica** | §4.1.1.4 |
| reachability P3/P4 GSFVG | **CHIUSO / CONVERGENTE — storico** | §4.4.4 |
| accessi GSFVG `Gamma_L` | **CHIUSO — baseline storica** | §4.2 |
| SP0/SP1 GSFVG | **SUPERATI COMPUTAZIONALMENTE — storici** | §4.4.5 |
| aggregazione GSFVG delle 9 combinazioni | **NON CONGELATA / SUPERATA DALLO SWITCH** | §4.4.5 |
| Gate preliminare OSM shadow | **CHIUSO — `SWITCH_OSM`** | §4.4.6 |
| backbone OSM B1--B5 | **COSTRUITO E VALIDATO** | §4.4.7 |
| E0 structural/cardinality `Gamma_OSM` | **PASS** | §4.4.7 |
| E1 materializzazione `Gamma_OSM` | **PASS** | §4.4.7 |
| E2 full 645-source Dijkstra | **PASS** | §4.4.7 |
| E3 freeze/promozione canonica | **PASS** | §4.4.7 |
| `Gamma_OSM` | **FROZEN** | §4.4.7 |
| `EXP_REL_300` | **FROZEN** | §4.4.7 |
| F1 B5 shadow regression | **PASS** | §4.4.7 |
| F2 build fidelity/directional audit | **PASS** | §4.4.7 |
| F3 regional time-optimal metric audit | **PASS_NO_SYSTEMIC_SIGNAL** | §4.4.7 |
| `G_OSM_operativo` | **FROZEN** | §4.4.7 |
| `NETWORK AUDIT` | **STOP** | §4.4.7 |
| Fase 5.7 — OD path system interno FVG | **CLOSED / FROZEN** | §4.4.8 |
| impedenza canonica di routing | **`TIME_B5` FROZEN** | §4.4.8 |
| distanza | **`PATH_ATTRIBUTE`** | §4.4.8 |
| `PRODUCT_LAMBDA_PATH_WEIGHTS` | **FROZEN** | §4.4.8 |
| 46.010 OD comunali / 414.090 access-pair path | **100% FINITE — 0 UNREACHABLE** | §4.4.8 |
| Fase 5.8A — input territoriali Gravity v0 | **CLOSED / FROZEN** | §9.1.2, §20, §24 |
| `Gravity_v0_territorial_inputs_raw.xlsx` | **CANONICAL / FROZEN** | §9.1.2 |
| Fase 5.8B — Gravity Model v0 FVG-only | **NEXT** | §§9, 22, 23 |
| gateway / rete e domanda esterna | **DOWNSTREAM EXTENSION** | §§3, 22, 23 |
| ANAS 2024 reference year + quality eligibility | **RULES FROZEN — SECTION MEMBERSHIP OPEN / 5.8C** | §9.1.5.1, §§20, 22, 23 |
| assignment + calibrazione ANAS | successivo | §§9.1.5--9.1.6, 23 |
| path flows | successivo | §§9, 23 |
| FRLM | successivo | §§10--17, 23 |
| componente heavy-duty | successiva | §16.2 |

### 0.1.3 Gerarchia dell'informazione

Quando due parti del documento sembrano ripetere lo stesso tema, utilizzare questa gerarchia:

1. **sezione metodologica canonica** — contiene la formulazione da usare nella tesi;
2. **§20 Decisioni metodologiche consolidate** — contiene il verdetto sintetico;
3. **§23 Roadmap operativa** — indica quando il tema entra nella sequenza di lavoro;
4. **§24 Registro aggiornamenti** — conserva la storia e le evidenze che hanno portato alla decisione;
5. **§21–§22** — contengono rispettivamente ipotesi ancora modificabili e problemi ancora aperti.

In caso di apparente conflitto, la formulazione più recente e consolidata della **sezione metodologica canonica** prevale sul registro storico.

### 0.1.4 Regola editoriale per i futuri aggiornamenti

Ogni nuovo blocco significativo deve essere inserito in modo da lasciare sempre recuperabili cinque informazioni:

1. **perché** il passaggio è stato necessario;
2. **che cosa** è stato fatto;
3. **con quali input e regole**;
4. **che risultato** è stato ottenuto;
5. **che conseguenza metodologica** produce sul gate successivo.

Quando un gate viene chiuso o cambia stato, devono essere aggiornati almeno:

- intestazione iniziale;
- sezione tecnica canonica;
- §20 Decisioni metodologiche consolidate;
- §22 Questioni ancora aperte;
- §23 Roadmap operativa;
- §24 Registro aggiornamenti.

Questa regola serve a evitare che lo stato corrente del progetto rimanga disperso in un singolo log o in una singola sezione tecnica.

---

### 0.1.5 Contratto permanente di authoring, export e compilazione

Per evitare che gli output derivati tornino a diventare non compilabili nelle versioni future, il documento adotta un **contratto tecnico permanente** distinto dalle decisioni metodologiche della tesi.

1. **Sorgente unica.** `Tesi_FRLM_FVG.ipynb` resta l'unico documento modificabile. `Tesi_FRLM_FVG.md`, `Tesi_FRLM_FVG.tex` e il PDF sono esclusivamente output generati.
2. **Sintassi matematica portabile.** Nel testo Markdown autorevole gli inline math devono usare `$...$` e i display math `$$...$$`. I delimitatori MathJax `\(...\)` e `\[...\]` non devono essere introdotti come sintassi di authoring perché non sono trasformati in modo affidabile dalla catena `nbconvert/Pandoc → LaTeX`. Sequenze LaTeX interne alle formule, come `\\[4pt]` in un ambiente `cases`, restano invece ammesse.
3. **Nessuna correzione manuale degli output.** Il `.tex` non deve essere riparato a mano dopo la generazione: una correzione necessaria deve risalire al notebook autorevole oppure, se riguarda esclusivamente il processo di build, al relativo build contract.
4. **Compiler canonico.** L'output LaTeX deve essere compilato con **XeLaTeX**. `pdfLaTeX` non è il compiler canonico perché il documento contiene caratteri Unicode e simboli che XeLaTeX gestisce nativamente.
5. **Build gate obbligatorio.** Una nuova versione del notebook è considerata completa dal punto di vista editoriale solo se la pipeline automatica `build_tesi.py` restituisce `STATUS = PASS`. La pipeline esegue: authoring lint → esecuzione notebook → rigenerazione `.md/.tex` → compilazione XeLaTeX → controllo degli errori bloccanti → produzione del bundle Overleaf.
6. **Fail closed.** Se il lint trova delimitatori incompatibili o se XeLaTeX produce un errore bloccante, il build termina con errore: gli output di quella versione **non devono essere considerati canonici** finché il problema non viene corretto alla fonte.
7. **Overleaf.** Il bundle prodotto include `latexmkrc`, che forza XeLaTeX tramite `latexmk`; in questo modo il compiler non deve essere riselezionato manualmente ad ogni nuova versione del `.tex`.

Questo contratto è una regola di **manutenzione documentale e riproducibilità**, non una decisione scientifica della tesi. Il suo scopo è impedire che una successiva integrazione metodologica reintroduca silenziosamente gli stessi problemi di compilazione già risolti.

# 1. Principi metodologici

## 1.1 Separazione dei livelli di rappresentazione

Il progetto utilizza tre livelli distinti:

1. **livello territoriale o zonale**, nel quale i comuni rappresentano le unità della matrice Origine-Destinazione;
2. **livello stradale fisico**, nel quale i flussi OD sono assegnati a strade realmente percorribili;
3. **livello dei candidati alla ricarica**, ottenuto integrando la rete fisica con nodi reali e nodi virtuali sui quali il modello può collocare capacità di ricarica.

Questi livelli non devono essere confusi.

In particolare:

- una relazione OD tra due comuni non è un arco stradale;
- un comune non coincide con il municipio, con il centroide geometrico o con un unico punto fisico;
- un nodo candidato non è necessariamente una particella catastale o un sito già progettato;
- l'output comunale non implica che ogni punto del comune sia equivalente dal punto di vista dell'accessibilità stradale.

## 1.2 Separazione tra mobilità leggera e pesante

La mobilità leggera e la mobilità pesante vengono trattate come due problemi distinti.

La pari rilevanza metodologica non implica sviluppo simultaneo: la fase corrente è concentrata esclusivamente sulla matrice light-duty; la componente heavy-duty sarà affrontata in una fase successiva della roadmap.

Si definisce l'insieme dei segmenti:

$$
\mathcal S=\{\mathrm L,\mathrm H\}
$$

con:

- $s=\mathrm L$: mobilità leggera;
- $s=\mathrm H$: mobilità pesante.

Per ciascun segmento vengono costruiti separatamente:

- la matrice OD;
- i flussi assegnati alla rete;
- i percorsi;
- i parametri energetici;
- le strategie di ricarica;
- il modello di localizzazione e dimensionamento;
- la validazione.

Non si definisce inizialmente una matrice unica:

$$
T_{od}^{\mathrm{tot}}=T_{od}^{\mathrm L}+T_{od}^{\mathrm H}
$$

poiché tale somma nasconderebbe differenze essenziali di comportamento, autonomia, potenza richiesta, distribuzione temporale e scelta dei percorsi.

Un eventuale modello combinato verrà valutato solo dopo che i due modelli indipendenti avranno raggiunto un livello soddisfacente di validazione. L'integrazione dovrà riguardare soprattutto la condivisione di:

- sito;
- cabina primaria;
- potenza disponibile;
- opere civili;
- costo fisso di apertura.

La domanda e le tecnologie di ricarica rimarranno comunque distinte.

## 1.3 Livello di precisione dell'output

Il modello è uno strumento di **pianificazione strategica regionale**, non di progettazione esecutiva.

Un risultato ammissibile può essere espresso nella forma:

> nel comune $m$, lungo il corridoio $r$, è raccomandata una capacità pari a $y$ punti di ricarica e $P$ kW complessivi.

Non è necessario che il modello individui già:

- la particella catastale;
- il parcheggio preciso;
- il distributore specifico;
- la posizione finale della cabina secondaria;
- la disposizione fisica degli stalli.

Il modello deve però conservare una sufficiente coerenza topologica. Per questo motivo l'unità operativa interna non sarà semplicemente il comune, ma una **finestra territoriale comune-corridoio**, eventualmente distinta per direzione di marcia.

---

# 2. Indici e convenzioni di notazione

La seguente tabella costituisce il contratto di notazione del documento.

| Indice | Insieme | Significato |
|---|---|---|
| $s$ | $\mathcal S$ | segmento di mobilità: leggero $\mathrm L$ o pesante $\mathrm H$ |
| $m,n$ | $\mathcal M$ | comuni del Friuli Venezia Giulia |
| $e$ | $\mathcal E$ | zone esterne alla regione |
| $o,d$ | $\mathcal Z$ | zona di origine e zona di destinazione |
| $u,v$ | $\mathcal V_s$ | nodi della rete stradale fisica utilizzabile dal segmento $s$ |
| $a$ | $\mathcal A_s$ | arco stradale fisico diretto |
| $r$ | $\mathcal R$ | corridoio o asse stradale |
| $c$ | $\mathcal C_s$ | nodo candidato alla ricarica per il segmento $s$ |
| $q$ | $\mathcal Q_{od}^s$ | percorso ammesso tra $o$ e $d$ per il segmento $s$ |
| $k$ | $\mathcal K_q^s$ | strategia di ricarica ammissibile per il percorso $q$ |
| $b$ | $\mathcal B$ | cabina primaria o nodo equivalente della rete elettrica |
| $\tau$ | $\mathcal T$ | intervallo temporale, se il dimensionamento è temporale |
| $\omega$ | $\Omega_s$ | scenario di domanda, autonomia o stato di carica |

## 2.1 Regole per evitare ambiguità

1. Gli indici $o$ e $d$ indicano sempre **zone OD**, non nodi stradali.
2. Gli indici $u$ e $v$ indicano sempre **nodi della rete fisica**.
3. L'indice $c$ indica sempre un **candidato alla ricarica**.
4. L'indice $q$ indica un **percorso**, non una coppia OD generica.
5. L'indice $k$ indica una **strategia di ricarica lungo un percorso**.
6. La lettera $s$ in apice identifica il segmento leggero o pesante.
7. Il simbolo $T_{od}^s$ è riservato ai flussi OD; il simbolo $f_q^s$ è riservato ai flussi sui percorsi.
8. La lettera $\beta$ non viene utilizzata per indicare una cabina elettrica, poiché può essere impiegata come parametro di deterrenza nei modelli gravitazionali.

---

# 3. Insiemi territoriali

## 3.1 Comuni del Friuli Venezia Giulia

Si definisce:

$$
\mathcal M=\{m_1,m_2,\ldots,m_{|\mathcal M|}\}
$$

come l'insieme dei comuni del Friuli Venezia Giulia considerati nello studio. Nell'applicazione corrente:

$$
|\mathcal M|=215.
$$

I comuni costituiscono:

- zone di origine e destinazione;
- unità di disaggregazione della domanda;
- unità di rendicontazione dei risultati;
- contenitori territoriali dei nodi candidati.

Un comune non è interpretato come un singolo punto della rete. La zona comunale resta un'entità areale e concettuale anche quando, per alcune operazioni GIS, le viene associato un punto rappresentativo.

### 3.1.1 Punto rappresentativo delle origini interne

Per rappresentare operativamente l'origine dei flussi pendolari residenti nei comuni del Friuli Venezia Giulia si utilizza un **centroide ponderato per la popolazione residente**.

Sia $\mathcal H_m$ l'insieme delle sezioni di censimento appartenenti al comune $m$, sia $P_h$ la popolazione residente nella sezione $h$ e sia $(x_h,y_h)$ il punto rappresentativo della sezione. Il punto associato al comune è:

$$
\mathbf p_m^{\mathrm{pop}}
=
\left(
x_m^{\mathrm{pop}},
y_m^{\mathrm{pop}}
\right)
$$

con:

$$
x_m^{\mathrm{pop}}
=
\frac{\sum_{h\in\mathcal H_m}P_hx_h}
{\sum_{h\in\mathcal H_m}P_h},
\qquad
y_m^{\mathrm{pop}}
=
\frac{\sum_{h\in\mathcal H_m}P_hy_h}
{\sum_{h\in\mathcal H_m}P_h}.
$$

Le coordinate sono calcolate nel sistema metrico di progetto EPSG:32632.

Questa scelta è coerente con l'uso del punto come origine della domanda pendolare: la distribuzione della popolazione residente rappresenta meglio la localizzazione della generazione dei viaggi rispetto al centroide geometrico del territorio comunale. Il centroide di popolazione:

- non identifica l'abitazione del singolo pendolare;
- non coincide necessariamente con il centro amministrativo;
- non sostituisce i nodi fisici di accesso alla rete;
- costituisce soltanto un riferimento sintetico per la costruzione preliminare delle relazioni OD.

Nell'applicazione GIS è stato inoltre verificato che ciascuno dei 215 centroidi di popolazione ricadesse all'interno del relativo comune.

## 3.2 Zone esterne

Si definisce:

$$
\mathcal E=\{e_1,e_2,\ldots,e_{|\mathcal E|}\}
$$

come l'insieme delle zone esterne necessarie per rappresentare:

- traffico in ingresso;
- traffico in uscita;
- traffico di attraversamento;
- connessioni con Veneto, Austria, Slovenia e altri sistemi territoriali rilevanti.

Nello stato GIS corrente, ai comuni italiani esterni presenti nelle relazioni pendolari è associato provvisoriamente il **centroide geometrico** del poligono comunale:

$$
\mathbf p_e^{\mathrm{geo}}
=
\operatorname{centroid}(A_e),
$$

dove $A_e$ è la geometria areale del comune esterno $e$.

Il centroide geometrico non è interpretato come localizzazione puntuale del posto di lavoro né come estremo operativo automatico del percorso. Serve alla georeferenziazione e alla visualizzazione direzionale. Dopo il caricamento del grafo si valuteranno, per i comuni vicini mantenuti singolarmente, il punto comunale ISTAT, un nodo del centro abitato o uno o più accessi alla rete principale; per destinazioni lontane o aggregate si valuteranno centroidi macrozonali e gateway di confine.

Non viene utilizzato un centroide di popolazione per le destinazioni esterne perché la distribuzione della popolazione residente non rappresenta necessariamente la distribuzione dei luoghi di lavoro. Le destinazioni possono infatti ricadere in:

- aree industriali;
- poli logistici;
- zone direzionali;
- aree commerciali;
- servizi pubblici o altri attrattori non correlati alla residenza.

In assenza di dati spaziali omogenei e affidabili sugli addetti o sui luoghi di lavoro, il centroide geometrico costituisce una scelta minimale, neutrale e riproducibile. La scelta evita di introdurre una precisione apparente non supportata dai dati disponibili.

La scelta definitiva sarà guidata dalla corretta rappresentazione del corridoio di attraversamento del confine regionale, evitando precisione spaziale non supportata dai dati. Per zone non riconducibili a comuni italiani, punto e gateway saranno definiti separatamente dopo la validazione della rete.

## 3.3 Distinzione tra zona e punto rappresentativo

Per ogni zona $z\in\mathcal Z$ può essere associato, quando necessario, un punto operativo:

$$
\mathbf p_z^{\mathrm{rep}}
=
\begin{cases}
\mathbf p_z^{\mathrm{pop}}, & z\in\mathcal M,\\[4pt]
\mathbf p_z^{\mathrm{geo}}, & z\in\mathcal E
\text{ e }z\text{ è un comune italiano esterno}.
\end{cases}
$$

L'associazione non modifica la natura della zona OD. In particolare:

1. la zona resta un'unità territoriale areale;
2. il punto rappresentativo può servire a controlli direzionali, alla stima dei costi e al collegamento preliminare con il grafo, ma le eventuali linee di desiderio non sono percorsi stradali;
3. il punto non coincide automaticamente con un nodo della rete stradale;
4. l'assegnazione alla rete fisica avviene attraverso i nodi di accesso definiti nella Sezione 4.2;
5. per i flussi diretti fuori regione, il tratto rilevante ai fini del modello termina alla porta di confine o al punto di uscita dalla rete regionale.

## 3.4 Zonizzazione complessiva

La zonizzazione complessiva è:

$$
\mathcal Z=\mathcal M\cup\mathcal E
$$

con:

$$
\mathcal M\cap\mathcal E=\varnothing.
$$

## 3.5 Relazioni OD concettuali

Per ogni segmento $s$, si definisce il grafo zonale:

$$
G_s^{Z}=(\mathcal Z,\mathcal E_s^{Z})
$$

in cui un arco zonale $(o,d)\in\mathcal E_s^{Z}$ rappresenta l'esistenza di una relazione di mobilità tra la zona $o$ e la zona $d$.

L'arco zonale:

$$
(o,d)
$$

non rappresenta una strada fisica. Esso è soltanto il supporto logico della cella della matrice OD:

$$
T_{od}^s\geq 0.
$$

La base temporale della matrice light è il **giorno medio annuo**, in coerenza con il TGMA ANAS. L'unità adottata è:

$$
T_{od}^{\mathrm L}=\text{veicoli/giorno medio annuo}.
$$

I dati ISTAT di pendolarismo non sono flussi veicolari giornalieri e richiedono una conversione coerente con tale base temporale. Eventuali matrici subgiornaliere appartengono a raffinamenti successivi e non modificano l'unità della versione base.

## 3.6 Dati disponibili e loro ruolo

### Dati territoriali e censuari

- limiti dei 215 comuni del Friuli Venezia Giulia;
- sezioni censuarie FVG 2021 e popolazione residente utilizzata per i centroidi ponderati;
- layer nazionale corretto dei comuni italiani e relativi centroidi geometrici;
- dati ISTAT 2021 di pendolarismo preparati per le relazioni interne ed extra-regione, con semantica e conversione minima in veicoli/giorno medio annuo consolidate nella Sezione 9.1.1;
- proxy territoriali per produzione e attrazione della mobilità non pendolare: da selezionare e preparare.

### Dati di traffico e rete

- conteggi ANAS 2017--2025, principale fonte osservata disponibile sui link ma caratterizzata da copertura incompleta, discontinuità temporali e qualità eterogenea tra sezioni e anni;
- grafo ufficiale regionale GSFVG, selezionato come backbone topologico della rete light; il file originale `GSFVG_IRDAT.shp` è conservato immutato e le elaborazioni riguardano esclusivamente copie operative versionate;
- estratto OpenStreetMap Geofabrik `nord-est_2026-08-03.osm.pbf`, oggi **fonte congelata del backbone operativo light OSM**; lo snapshot è conservato immutato e alimenta la pipeline routing-aware B1--B5. La precedente estrazione GIS a 20 km resta provenance della fase preliminare; Austria e Slovenia non costituiscono ancora una rete transnazionale completa;
- **Overture Maps Transportation**, valutato durante la precedente selezione del backbone come rete alternativa e benchmark aggiuntivo; non fu adottato in quella fase e resta oggi una fonte di controllo storica/diagnostica per topologia, connettività, svincoli, rampe e continuità transfrontaliera;[^audit-overture-20260811]
- layer OSM preliminare validato `osm_rete_light_fvg_20km_routing_statico_validato`; classi stradali, velocità di modello, lunghezze ellissoidali, tempi per verso e costo temporale statico restano **provenance metodologica** della fase precedente. Il routing canonico corrente è invece la pipeline OSM B2 + B4 + B5 congelata nella §4.4.7;
- componenti esterne, internazionali e di attraversamento: richieste dal modello ma non ancora costruite in forma operativa.

La disponibilità di un dato non implica che sia già semanticamente armonizzato, trasformato nell'unità finale o validato per l'uso nel modello.

## 3.7 Stato delle elaborazioni QGIS

**Ambiente documentato:** QGIS 3.40.0 Bratislava, CRS EPSG:32632, progetto principale **Tesi_FVG.qgz**, GeoPackage principale **base_territoriale_fvg.gpkg**.

### Attività effettivamente completate

- estrazione dei 215 comuni FVG;
- preparazione delle sezioni censuarie FVG 2021 e correzione delle geometrie problematiche;
- creazione dei punti rappresentativi delle sezioni;
- creazione dei centroidi geometrici e dei centroidi di popolazione dei comuni FVG;
- verifica che tutti i centroidi di popolazione ricadano nel rispettivo comune;
- preparazione del layer nazionale corretto dei comuni italiani e dei centroidi geometrici comunali;
- preparazione e tipizzazione dei dati di pendolarismo extra-regione;
- creazione delle chiavi di join e gestione del caso storico Alano di Piave--Setteville;
- join delle origini e delle destinazioni, con controllo finale di zero relazioni non abbinate;
- creazione del confine FVG dissolto e delle aree di estrazione della rete con buffer di 20 km, anche in WGS84;
- selezione del grafo ufficiale GSFVG come backbone topologico della rete light e audit del file originale `GSFVG_IRDAT.shp`, con 76.326 parti lineari nella componente connessa principale su 76.350 e stabilità del risultato tra 0,10 e 1 metro di tolleranza;
- audit attributivo delle informazioni direzionali GSFVG: `TRIM_USAGE` identificato come campo ufficiale di percorribilità dell'arco; `DIR`, `ENTEXT` e `GEOMETRY_R` mantenuti come attributi LRS non sufficienti, da soli, a ricostruire il senso legale di marcia; individuati 616 archi con `TRIM_USAGE` nullo;
- revisione manuale completa dei 616 archi con `TRIM_USAGE = NULL`, articolata in 54 SC, 363 AS mainline e 199 AS residui, con esito complessivo 557 `FWD_ONLY`, 1 `BWD_ONLY`, 58 `BIDIRECTIONAL` e 0 `UNRESOLVED`;
- consolidamento della revisione in tre CSV e audit globale diretto rispetto a `GSFVG_IRDAT_FULL`, con copertura 616/616, zero sovrapposizioni, zero mancanti, zero estranei e zero casi irrisolti;
- audit semantico di `TRIM_USAGE`, `DBPRIOR_ST`, `DBPRIOR_TI` e `DATA_FINE`: per `TRIM_USAGE=0` risultano 30.418 casi `ST=1/TI=1`, 20 `ST=1/TI=3`, 46 `ST=1/TI=NULL` e 33.993 `ST=NULL/TI=NULL`; nell'intero dataset non risultano elementi `DBPRIOR_ST=2`, `DBPRIOR_TI=2` o `DATA_FINE` valorizzato;
- audit direzionale mirato di `TRIM_USAGE=0` rispetto ai segnali one-way OSM: individuati 3.349 match `HIGH/STRICT`, di cui 212 riconducibili a rotatorie e 588 a carreggiate o rami separati; il residuo di 2.549 casi è stato stratificato in P1=136, P2=1.067, P3=363 e P4=983;
- revisione manuale esaustiva del gruppo P1 completata 136/136: 102 `OSM_ONEWAY_CONFIRMED`, 13 `GSFVG_OK_BIDIRECTIONAL`, 20 `MODELLING_DIFFERENCE_GSFVG_OK`, 1 `ONEWAY_CONFIRMED_OSM_OPPOSITE`, 0 `UNRESOLVED`; direzioni finali 45 `FWD_ONLY`, 58 `BWD_ONLY` e 33 `BIDIRECTIONAL`;
- clustering diagnostico P2 completato sui 1.067 casi `STRICT` con `DBPRIOR_ST` non compilato: 775 cluster fisici, di cui 600 singleton e 175 multipli; il clustering è mantenuto come supporto organizzativo e non come unità decisionale;
- benchmark Overpass API completato sull'intera ground truth P1 (136 casi): 43 `AUTO_ONEWAY_CANDIDATE`, 77 `AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE`, 16 `MANUAL_REVIEW_REQUIRED`; nonostante una copertura teoricamente automatizzabile dell'88,24%, l'accuratezza delle classificazioni automatiche è risultata pari al 55,00%, con 54 errori complessivi; Overpass è quindi mantenuto esclusivamente come strumento diagnostico topologico;
- disegno di revisione P2 consolidato in 127 casi: 120 `INFERENTIAL_SAMPLE` e 7 `STRATEGIC_SUPPLEMENT` motorway/trunk; revisione manuale chiusa con **65 `FWD_ONLY`**, **46 `BWD_ONLY`**, **16 `BIDIRECTIONAL`** e 0 `UNRESOLVED`;
- complessivamente consolidati **879 override manuali distinti per `ID1`**: 616 NULL + 136 P1 + 127 P2; i 940 P2 fuori campione e i 1.346 archi P3/P4 mantengono il default conservativo `TRIM_USAGE=0 → BIDIRECTIONAL`;
- costruito il GeoPackage persistente `GSFVG_operativo_direzionale_v01.gpkg` in EPSG:25833, con layer `GSFVG_operativo_direzionale_v01`, tabella `direction_overrides_v01` e tabella `build_metadata`; SHA-256 finale `24a699f71f5ad115afe2f9fa96c0f2b51334e022f18e5d3f47d3bc873cd0036a`;
- audit source-output della build direzionale completato su 76.349 feature: 76.349 `ID1` univoci, 76.349 FID sorgente materializzati, 76.349 geometrie presenti, zero modifiche geometriche, zero mismatch sui 26 attributi originali e zero incoerenze tra `dir_final`, `dir_fwd_ok` e `dir_bwd_ok`;
- distribuzione finale del layer direzionale: 11.256 `GSFVG_EXPLICIT`, 64.214 `GSFVG_DEFAULT`, 879 `MANUAL_REVIEW`; 9.371 `FWD_ONLY`, 2.657 `BWD_ONLY`, 64.321 `BIDIRECTIONAL`, zero `UNRESOLVED`;
- materializzato anche il layer P2 persistente `GSFVG_P2_review_display_127.gpkg`, verificato sui 127 `ID1` senza mismatch; progetto QGIS consolidato con `02_P2_CHIUSO` in `04_REVISIONE_DIREZIONI` e `GSFVG_operativo_direzionale_v01` in `05_GRAFO_OPERATIVO`;
- audit indipendente OSM ↔ GSFVG completato sui 11.256 archi con `TRIM_USAGE=1/2`: 7.456 `AGREEMENT_HIGH`, 3.009 `CONFLICT_HIGH`, 52 `AMBIGUOUS_MATCH`, 128 `AMBIGUOUS_DIRECTION` e 611 `INSUFFICIENT_MATCH`; tra i conflitti HIGH risultano 2.190 `OPPOSITE_ONEWAY`, 738 `GSFVG_ONEWAY_OSM_BIDIRECTIONAL` e 81 `GSFVG_ONEWAY_OSM_CLOSED`;
- validazione diagnostica del matching nel regime geometrico più robusto mediante 24 casi bilanciati (`6 U1_AGREEMENT`, `6 U1_OPPOSITE`, `6 U2_AGREEMENT`, `6 U2_OPPOSITE`), con esito 24/24 `MATCH_VALID`, 0 `MATCH_INVALID` e 0 `UNCERTAIN`; nel regime validato ricadono 1.459 dei 3.009 conflitti HIGH, inclusi 1.143 `OPPOSITE_ONEWAY`;
- estrazione in `rete_stradale_osm_fvg.gpkg` del layer sorgente `osm_strade_fvg_20km_raw`, contenente le geometrie OSM con `highway` valorizzato;
- separazione tra rete light principale e viabilità ausiliaria `service`;
- conversione della rete principale in 112.466 geometrie `LineString` singole in EPSG:32632;
- estrazione in colonne dedicate dei tag `oneway`, `maxspeed`, `lanes`, `ref`, `surface`, `junction`, `bridge`, `tunnel`, `access`, `motor_vehicle` e `motorcar`, mantenendo `other_tags` come attributo sorgente;
- censimento completo di `other_tags` sulle 112.466 geometrie, con zero errori di parsing, 446 chiavi residue, 312.530 occorrenze e 4.597 combinazioni distinte chiave--valore;
- integrazione delle sette chiavi mancanti rilevanti per accesso e direzione e ricostruzione gerarchica dell'accesso automobilistico;
- conservazione e controllo sintattico delle restrizioni condizionali, escluse per decisione dal primo grafo statico;
- classificazione dell'accesso in cinque classi e combinazione per verso di accesso e direzione mediante i codici 0, 1 e 2; `open` è associato a 1, `local_restricted` a 2 e `permission_restricted`, `authorized_only` e `prohibited` a 0, conservando le classi originarie nei campi `access_fwd_class` e `access_bwd_class`;
- audit mirato delle 118 geometrie speciali e consolidamento della direzionalità in 31.614 geometrie percorribili soltanto nel verso della geometria, 1 soltanto nel verso opposto e 80.851 bidirezionali o trattate staticamente come tali;
- verifica manuale del caso `reversible` di Via dei Bagni Nuova come ponte a senso alternato regolato da semaforo e riclassificazione dei 22 `junction=circular` privi di `oneway` come monodirezionali nel verso della geometria;
- verifica finale dei risultati operativi per verso: 110.325 accessi ordinari, 500 locali, 1.640 chiusure per accesso e 1 per direzione nel verso della geometria; 78.961 accessi ordinari, 424 locali, 1.467 chiusure per accesso e 31.614 per direzione nel verso opposto;
- verifica che le 59 occorrenze condizionali di accesso appartengano a 59 geometrie distinte, senza sovrapposizioni fra le tre famiglie condizionali;
- registrazione dei campi `routing_fwd_status`, `routing_fwd_code`, `routing_bwd_status`, `routing_bwd_code`, `routing_fwd_closure_cause` e `routing_bwd_closure_cause` e del layer validato `osm_rete_light_fvg_20km_routing_statico_validato` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg`;
- controllo finale della normalizzazione statica senza valori nulli, associazioni incoerenti tra stato e codice o categorie non previste;
- lettura, pulizia e interpretazione dei tag OSM di velocità separatamente per verso, con produzione di `speed_fwd_obs_kmh` e `speed_bwd_obs_kmh` nel layer `osm_rete_light_fvg_20km_velocita_osm_normalizzate`;
- costruzione del layer `osm_rete_light_fvg_20km_velocita_modello_base`, con `speed_class_default_kmh`, `speed_fwd_model_kmh`, `speed_bwd_model_kmh`, `speed_fwd_model_source` e `speed_bwd_model_source` valorizzati per tutte le 112.466 geometrie;
- controlli automatici sulle velocità OSM e di modello conclusi senza valori nulli o anomalie complessive;
- calcolo e validazione delle lunghezze ellissoidali della rete light;
- calcolo e validazione dei tempi statici forward e backward a partire dalle velocità di modello;
- definizione del costo della rete ordinaria uguale al tempo statico, con esclusione dei versi `routing_*_code=0`, utilizzo ordinario dei versi con codice 1 e conservazione dei versi con codice 2 esclusivamente per i collegamenti iniziali o finali delle zone.
- benchmark Overture Maps Transportation completato come rete alternativa: 246.223 feature automobilistiche, componente principale pari al 96,954% dei 453.683 routing edge dotati di costo, 11/11 itinerari campione con cammino valido e nessun uso contromano; Overture è escluso come sostituto del GSFVG e mantenuto come terzo controllo mirato.[^audit-overture-20260811]
- analisi del precedente MRS di Tufaro completata come benchmark metodologico: confermati l'uso del GSFVG come base topologica, la soglia di 0,1 m per le coincidenze degli estremi e la riduzione a 29.695 nodi; la reciprocità del grafo MRS è mantenuta come differenza sostanziale rispetto al routing diretto richiesto dalla presente tesi.[^tufaro2022]

Il layer definitivo **pendolari_extra_regione_fvg_2021_od_xy** contiene 2.895 relazioni esclusivamente in uscita dal FVG: origine in un comune regionale e destinazione lavorativa esterna. I flussi interni FVG--FVG sono in una tabella separata e non sono ancora integrati. I flussi esterno--FVG restano da reperire o modellizzare; la componente esterno--esterno in transito è esclusa dalla prima implementazione.

Il layer include comune di origine e destinazione, campo **Pendolari**, coordinate e chiavi territoriali storiche e correnti. I campi **KM_TOT**, **TEP_TOT** e **TTP_TOT** provengono dalla matrice ISTAT delle distanze temporali tra comuni, calcolata sul grafo TomTom Multinet 2020 mediante ArcGIS Network Analyst. **KM_TOT** è la distanza stradale dell'itinerario selezionato, **TEP_TOT** il tempo con impedenze di traffico e barriere, **TTP_TOT** il tempo teorico sul medesimo itinerario in condizioni ideali. Per le OD interne, la Fase 5.7 ha già materializzato sul backbone OSM frozen tempi `TIME_B5` e distanze ex-post dei path canonici; i campi ISTAT/TomTom restano riferimenti esterni di analisi e controllo. Per le relazioni esterne la costruzione dei percorsi appartiene invece al gate gateway.

### Attività downstream non ancora eseguite o completate — stato post-Fase 5.7

Le attività di costruzione del backbone, degli accessi e dei path OD interni non appartengono più a questa lista: sono frozen nelle Fasi 5.6–5.7. Restano invece da completare:

- definizione e versionamento di gateway e rappresentazione della domanda esterna;
- eventuale estensione fisica della rete oltre confine soltanto dove necessaria, separata dalla semplice attestazione della domanda ai gateway;
- materializzazione/freeze della conversione ISTAT `Pendolari → veicoli/giorno`;
- costruzione della seed matrix light $C^{ISTAT}+N^0+E^0$;
- assignment della seed sul sistema di path frozen, senza rigenerare i 414.090 path interni;
- associazione sistematica tra sezioni ANAS e archi/corridoi osservabili;
- calibrazione e, se necessaria, matrix estimation regolarizzata;
- produzione della matrice light finale e dei path flows destinati al FRLM;
- filone heavy-duty separato;
- localizzazione/dimensionamento, integrazione della rete elettrica e valutazione economica.

Le attività GSFVG, OSM shadow, B1--B5, `Gamma_OSM`, sensitivity degli accessi, impedenza `TIME_B5` e shortest path interni restano nella provenance ma **non sono attività aperte**.

## 3.8 Zonizzazione interna ed esterna

L'architettura generale del progetto distingue le zone interne FVG dalle future condizioni al contorno esterne:

$$
\mathcal Z^{full}=\mathcal Z_{\mathrm{FVG}}\cup\mathcal Z_{\mathrm{esterne}}.
$$

Tuttavia, la **baseline LIGHT v0 corrente è esplicitamente FVG-only**:

$$
\mathcal Z^{v0}=\mathcal Z_{\mathrm{FVG}}=\mathcal M,
\qquad |\mathcal M|=215,
$$

con:

```text
EXTERNAL_OD = NOT_IMPLEMENTED / DOWNSTREAM EXTENSION
```

Le future zone esterne rappresenteranno condizioni al contorno per flussi in ingresso, uscita ed eventuale attraversamento, ma non appartengono alla v0 e non modificano il sistema OD/path interno frozen.

Dopo il freeze delle Fasi 5.6, 5.7 e 5.8A, il `NEXT` operativo è la **Fase 5.8B — Gravity Model v0 FVG-only**. Gateway e zonizzazione esterna sono rinviati a una estensione downstream separata e versionata; quando verranno aperti, dovranno definire punti di ingresso/uscita, corridoi e granularità esterna senza modificare il package OD interno della Fase 5.7.

Il perimetro territoriale, infrastrutturale e decisionale della localizzazione resta il confine amministrativo del FVG. Il modello localizza e dimensiona soltanto infrastrutture interne e considera, per copertura e domanda energetica, le porzioni di percorso regionali. Quando l'estensione esterna sarà implementata, le informazioni esterne serviranno come condizioni al contorno e non come rilancio del routing interno.

## 3.9 Decisione operativa sulle linee di desiderio

È consolidata la decisione di non creare ora linee rette origine--destinazione e relative intersezioni con il confine. Una retta non rappresenta il percorso stradale; il punto di attraversamento deve derivare dal percorso sul grafo; inoltre la zonizzazione esterna non è ancora definita. Le linee di desiderio potranno essere usate soltanto per visualizzazione o controllo, non come base metodologica dell'assegnazione.

---

# 4. Rete stradale fisica

## 4.1 Grafo fisico per segmento

Per ogni segmento $s$ si definisce:

$$
G_s^{R}=(\mathcal V_s,\mathcal A_s)
$$

dove:

- $\mathcal V_s$ è l'insieme dei nodi stradali fisici utilizzabili dal segmento $s$;
- $\mathcal A_s$ è l'insieme degli archi stradali diretti utilizzabili dal segmento $s$.

La geometria di base può essere comune, ma i due segmenti possono avere grafi operativi differenti. Per esempio:

- alcuni archi possono essere inadatti ai mezzi pesanti;
- possono esistere limiti di massa, altezza o sagoma;
- i costi generalizzati possono essere diversi;
- i pedaggi possono incidere in modo diverso;
- la velocità rappresentativa può essere diversa.

Per ogni arco $a\in\mathcal A_s$ si definiscono almeno:

| Parametro | Significato | Unità |
|---|---|---|
| $\ell_a$ | lunghezza dell'arco | km |
| $t_a^s$ | tempo di percorrenza per il segmento $s$ | h |
| $g_a^s$ | costo generalizzato | unità di costo |
| $r(a)$ | corridoio al quale appartiene l'arco | categoria |
| $m(a)$ | comune o comuni attraversati dall'arco | categoria |

### 4.1.1 Scelta e validazione del grafo stradale operativo GSFVG

> **Stato dopo il 20 agosto 2026.** Questa sezione documenta una scelta metodologica realmente adottata e consolidata nella prima fase del progetto, ma **non descrive più il backbone operativo corrente**. Gli audit qui riportati restano validi come provenance, benchmark e conoscenza empirica del GSFVG. Il Gate OSM shadow ha successivamente portato alla decisione `SWITCH_OSM`, documentata nelle §§4.4.6--4.4.7.

Il grafo ufficiale **GSFVG** è stato selezionato come principale candidato per il backbone topologico della rete light-duty. Il dataset, pubblicato dalla Regione Autonoma Friuli Venezia Giulia, rappresenta la rete viaria regionale mediante un modello **Linear Reference System (LRS)** e contiene informazioni relative a classificazione amministrativa, progressive, gestione, validità e direzione d'uso degli archi.[^gsfvg-open-data]

La scelta è stata supportata da un audit strutturale e topologico eseguito in QGIS sul file originale `GSFVG_IRDAT.shp`. Delle 76.350 parti lineari analizzate, 76.326 appartengono alla componente connessa principale, pari al **99,969%** della rete. Il risultato è stabile adottando tolleranze comprese tra 0,10 e 1 metro, indicando che la connettività osservata deriva dalla struttura originaria del dataset e non da uno snapping generalizzato.[^audit-gsfvg]

Le eccezioni saranno trattate mediante verifiche locali, evitando operazioni automatiche estese che potrebbero creare collegamenti fittizi tra cavalcavia, sottopassi o carreggiate separate. Le 24 feature esterne alla componente principale saranno confrontate con la cartografia e con OpenStreetMap. I 32 auto-anelli, tutti collegati alla rete nel proprio nodo di chiusura e privi di intersezioni intermedie, saranno conservati come viabilità locale ma non considerati essenziali per il traffico di attraversamento.[^audit-gsfvg]

La rete OSM precedentemente elaborata viene quindi mantenuta come **fonte complementare** per controllare collegamenti mancanti e trasferire, previa validazione, informazioni relative a velocità, sensi unici e restrizioni di accesso. I dati utilizzati derivano dall'estratto OpenStreetMap dell'Italia nord-orientale distribuito da Geofabrik; l'impiego dei dati deve conservarne l'attribuzione agli autori OpenStreetMap e rispettare la licenza ODbL.[^geofabrik-nord-est][^osm-odbl]

Lo shapefile regionale originale non verrà modificato. Eventuali correzioni topologiche, riproiezioni e nuovi attributi saranno applicati esclusivamente a una copia operativa versionata. La rete fisica rimane distinta dalle successive procedure di costruzione degli accessi zonali, dei percorsi OD e dell'assegnazione dei flussi.

La scelta di GSFVG come backbone non annulla il lavoro svolto sulla rete OSM: i layer OSM validati costituiscono un patrimonio informativo complementare e un prototipo delle regole di accessibilità, direzionalità, velocità e costo da trasferire sul grafo regionale mediante procedure di matching e controlli mirati. La chiusura metodologica dei punti 4 e 5 riguarda pertanto la scelta del backbone e la definizione delle regole operative; la loro applicazione definitiva alla copia operativa GSFVG richiede un consolidamento tecnico tracciato, senza modificare le sorgenti originali.

#### 4.1.1.1 Integrazione delle informazioni direzionali

Il grafo regionale **GSFVG** costituisce la base topologica della rete stradale operativa e gli attributi originali sono mantenuti distinti dalle successive elaborazioni. In particolare, `TRIM_USAGE` è il campo ufficiale deputato a indicare la percorribilità direzionale dell'arco; `DIR`, `ENTEXT` e `GEOMETRY_R` descrivono invece aspetti dell'organizzazione LRS, delle entrate e uscite e del rapporto tra geometria e progressive chilometriche, ma **non costituiscono da soli una codifica affidabile della direzione legale di percorrenza**.[^gsfvg-open-data]

##### 4.1.1.1.1 Diagnosi iniziale dei valori nulli

Sul layer completo `GSFVG_IRDAT.shp`, composto da 76.349 feature, sono stati individuati **616 archi con `TRIM_USAGE = NULL`**.[^audit-trim-usage-gsfvg] È stata pertanto esclusa una ricostruzione automatica del verso basata soltanto su `DIR`, `ENTEXT`, `GEOMETRY_R`, orientamento geometrico o denominazione della direzione.

La gerarchia delle fonti adottata è permanente:

> **Durante la fase GSFVG, GSFVG è la fonte autorevole e OSM è una fonte di supporto e controllo.**

Questa gerarchia resta valida per interpretare gli audit storici della baseline GSFVG: una discordanza OSM non costituiva una correzione automatica. **Non deve però essere letta come gerarchia operativa corrente**, perché il successivo Gate shadow ha portato allo switch del backbone verso OSM.

##### 4.1.1.1.2 Caratterizzazione dei 616 archi

L'analisi attributiva ha mostrato la seguente struttura:

```text
616 TRIM_USAGE = NULL
│
├── 54 SC
│
└── 562 AS
    │
    ├── 363 assi autostradali principali
    │   A4 / A23 / A28 / A34
    │
    └── 199 elementi autostradali residui
        svincoli, bretelle, aree di servizio, raccordi...
```

Tutti i 616 archi appartengono alla componente principale del grafo e non sono quindi eliminabili come elementi isolati o topologicamente irrilevanti. È stato inoltre verificato che il fenomeno non fosse spiegabile, nella grande maggioranza dei casi, con semplici duplicazioni geometriche rispetto ad archi già dotati di `TRIM_USAGE`: i valori nulli rappresentano prevalentemente geometrie autonome.

##### 4.1.1.1.3 Tentativo automatico sui 363 archi mainline e sua falsificazione

Per i **363 archi AS delle carreggiate autostradali principali** è stato inizialmente costruito un confronto automatico con la rete OSM validata, basato su prossimità, compatibilità geometrica e direzione OSM. Il risultato suggeriva quasi sistematicamente una classificazione `FWD_ONLY`.

Per verificare indipendentemente tale attribuzione era stato introdotto un controllo campionario stratificato. Il campione ha però individuato **almeno un errore della procedura automatica**. In applicazione della regola di accettazione fissata a priori, il controllo è stato quindi considerato **fallito**: l'ipotesi di attribuzione collettiva dei 363 archi è stata scartata e la revisione è stata riaperta sull'intera popolazione.

Il campionamento non è stato corretto o ripetuto per ottenere un esito favorevole. Il metodo è stato cambiato, passando alla verifica manuale completa.

##### 4.1.1.1.4 Revisione manuale completa dei 616 archi

La ricostruzione è stata effettuata arco per arco in QGIS, visualizzando contemporaneamente:

- geometria GSFVG e relativo verso di digitalizzazione;
- rete OSM;
- frecce OSM costruite sui `routing_fwd_code` e `routing_bwd_code` già validati;
- cartografia OSM come supporto visuale.

Le classi decisionali ammesse sono state:

```text
FWD_ONLY
BWD_ONLY
BIDIRECTIONAL
UNRESOLVED
```

Il principio di fondo è rimasto invariato: **OSM supporta la decisione, ma non sostituisce automaticamente GSFVG**.

La revisione ha prodotto tre popolazioni indipendenti:

| Blocco | Archi | FWD | BWD | BIDIR | UNRES. |
|---|---:|---:|---:|---:|---:|
| SC | 54 | 51 | 0 | 3 | 0 |
| AS mainline | 363 | 362 | 0 | 1 | 0 |
| AS residui | 199 | 144 | 1 | 54 | 0 |
| **Totale** | **616** | **557** | **1** | **58** | **0** |

##### 4.1.1.1.5 Consolidamento in CSV minimali e ripetibili

Durante la prima revisione dei 199 archi AS residui il file di decisione era diventato eccessivamente complesso, con troppe colonne descrittive, note contenenti `;`, duplicazioni accidentali e una feature mancante. La revisione dei 199 è stata quindi ripetuta integralmente adottando uno schema minimale:

```text
review_order;
gsfvg_fid_original;
ID1;
TRIM_STR_C;
DIR;
ENTEXT;
decisione
```

Il CSV ricostruito contiene:

- 199 righe;
- 199 `review_order` distinti;
- 199 `ID1` distinti;
- 199 FID originali distinti;
- corrispondenza esatta con `GSFVG_IRDAT_FULL`.

Questa struttura conserva nel file di decisione soltanto gli attributi necessari alla tracciabilità e rende la procedura più semplice da controllare e ripetere.

##### 4.1.1.1.6 Audit globale dei tre CSV

I tre CSV di revisione sono stati confrontati **direttamente con il layer sorgente completo**, senza dipendere dai memory layer QGIS. L'audit finale ha restituito:

```text
SC             54 / 54
AS mainline   363 / 363
AS residui    199 / 199

UNIONE        616 / 616
```

e contemporaneamente:

```text
Sovrapposizioni: 0
Mancanti:       0
Estranei:       0
UNRESOLVED:     0
```

La ricostruzione dei 616 `TRIM_USAGE = NULL` è pertanto considerata **completa e validata sull'intera popolazione**, non su un campione.[^audit-direzionale-completo-gsfvg]

Gli attributi regionali originali non vengono sovrascritti. Le decisioni ricostruite sono conservate in output separati e dovranno alimentare campi derivati della copia operativa, mantenendo tracciati esito, fonte, confidenza e note di validazione.

##### 4.1.1.1.7 Passaggio agli archi con `TRIM_USAGE` valorizzato

La chiusura dei 616 valori nulli non implica che tutti gli altri archi siano automaticamente pronti per il routing. La distribuzione complessiva dell'attributo ufficiale è:

```text
TRIM_USAGE = 0    64.477
TRIM_USAGE = 1     8.704
TRIM_USAGE = 2     2.552
TRIM_USAGE = NULL    616
```

Per l'interpretazione degli attributi viene mantenuto come riferimento permanente il dizionario ufficiale del dataset Regione FVG.[^gsfvg-open-data] In particolare:

- `TRIM_USAGE=1`: monodirezionale nel verso di digitalizzazione;
- `TRIM_USAGE=2`: monodirezionale nel verso opposto;
- `TRIM_USAGE=0`: richiede maggiore attenzione, poiché la definizione ufficiale comprende la bidirezionalità ma anche l'assenza di direzione nel caso di strada chiusa;
- `DBPRIOR_ST`: stato di esercizio;
- `DBPRIOR_TI`: tipologia dell'elemento stradale.

##### 4.1.1.1.8 Audit semantico degli attributi ufficiali

L'incrocio tra `TRIM_USAGE`, `DBPRIOR_ST` e `DBPRIOR_TI` ha prodotto, per i **64.477 archi con `TRIM_USAGE=0`**, la seguente distribuzione:

```text
30.418   ST=1     TI=1
    20   ST=1     TI=3
    46   ST=1     TI=NULL
33.993   ST=NULL  TI=NULL
```

Nel dataset completo sono inoltre risultati:

```text
DBPRIOR_ST=2  in costruzione     0
DBPRIOR_TI=2  traghetto          0
DBPRIOR_TI=3  speciale          20
DATA_FINE valorizzato            0
```

Non emerge quindi una popolazione esplicitamente classificata come «in costruzione» o terminata. Questo risultato **non autorizza tuttavia a convertire automaticamente tutti i 64.477 `TRIM_USAGE=0` in archi bidirezionali**: i 33.993 casi senza `DBPRIOR_ST` e `DBPRIOR_TI` compilati richiedono una verifica ulteriore, mentre i 20 elementi `DBPRIOR_TI=3` devono essere trattati separatamente rispetto alla normale rete automobilistica.

##### 4.1.1.1.9 Audit indipendente OSM ↔ GSFVG sugli archi monodirezionali: esito e consolidamento

Per gli archi con direzione esplicita nel GSFVG viene consolidata la seguente regola metodologica:

```text
TRIM_USAGE = 1  → direzione operativa = FWD_ONLY
TRIM_USAGE = 2  → direzione operativa = BWD_ONLY
```

La **fonte primaria resta GSFVG**. OSM viene utilizzato esclusivamente come controllo indipendente di coerenza direzionale e **non sovrascrive automaticamente** la direzione ufficiale.

L'audit OSM ↔ GSFVG sull'intera popolazione dei **11.256 archi monodirezionali espliciti** ha prodotto:

```text
AGREEMENT_HIGH       7.456
CONFLICT_HIGH        3.009
AMBIGUOUS_MATCH         52
AMBIGUOUS_DIRECTION    128
INSUFFICIENT_MATCH     611
---------------------------
TOTALE              11.256
```

Tra i **3.009 conflitti HIGH** la classificazione diagnostica è:

```text
OPPOSITE_ONEWAY                       2.190
GSFVG_ONEWAY_OSM_BIDIRECTIONAL          738
GSFVG_ONEWAY_OSM_CLOSED                  81
```

La forte concentrazione iniziale dei conflitti sugli archi `SC` ha richiesto una verifica specifica dell'affidabilità del matching. È stato quindi costruito un campione diagnostico di **24 casi**, bilanciato tra:

```text
6 U1_AGREEMENT
6 U1_OPPOSITE
6 U2_AGREEMENT
6 U2_OPPOSITE
```

Il campione è stato selezionato nel regime geometrico più robusto:

```text
CLASSE = SC
ENTEXT = nd
3 campioni validi
1 solo osm_id
distanza media <= 2 m
allineamento <= 5°
```

La verifica manuale ha restituito:

```text
MATCH_VALID     24 / 24
MATCH_INVALID    0 / 24
UNCERTAIN        0 / 24
```

Il matching fisico è pertanto **empiricamente validato in questo specifico regime geometrico**. Applicando le medesime condizioni all'intera popolazione dei conflitti HIGH si ottiene:

```text
VALIDATED_MATCH_REGIME       1.459   48,49%
OUTSIDE_VALIDATED_REGIME     1.550   51,51%
```

Per la sola classe `OPPOSITE_ONEWAY`:

```text
OPPOSITE_ONEWAY totali                 2.190
OPPOSITE nel regime validato           1.143
quota                                  52,19%
```

La conclusione metodologica è che le discordanti OSM ↔ GSFVG **non possono essere interpretate semplicemente come errori del matcher**. Una quota consistente, e in particolare 1.143 casi `OPPOSITE_ONEWAY`, ricade in condizioni geometriche nelle quali il matching è stato verificato manualmente con esito 24/24 positivo. Ciò dimostra l'esistenza di effettive discordanti direzionali tra le due fonti, ma **non stabilisce automaticamente quale fonte rappresenti correttamente la situazione corrente**.

La regola operativa da conservare per `TRIM_USAGE=1/2` è quindi:

```text
routing direction
    ← GSFVG

OSM
    ← controllo QA indipendente

discordanza OSM
    ≠ correzione automatica GSFVG
```

Nel futuro layer operativo dovranno essere conservati almeno i seguenti campi derivati:

```text
direction_source
    GSFVG
    MANUAL_REVIEW

direction_conflict_osm
    0 / 1

direction_conflict_type
    OPPOSITE_ONEWAY
    GSFVG_ONEWAY_OSM_BIDIRECTIONAL
    GSFVG_ONEWAY_OSM_CLOSED
    NULL

osm_match_confidence
    VALIDATED_MATCH_REGIME
    HIGH_OUTSIDE_VALIDATED_REGIME
    AMBIGUOUS
    INSUFFICIENT
```

I **616 archi originariamente con `TRIM_USAGE=NULL` restano un caso distinto**, già ricostruito e validato manualmente. Non viene avviata un'ulteriore revisione manuale massiva dei 3.009 conflitti `CONFLICT_HIGH`: l'audit per `TRIM_USAGE=1/2` è considerato **metodologicamente congelato**. L'analisi prosegue separatamente sulla popolazione `TRIM_USAGE=0`, secondo la procedura P1--P4 descritta nel paragrafo successivo.

Il percorso metodologico consolidato diventa pertanto:

> **diagnosi → tentativo automatico sui NULL → falsificazione tramite controllo manuale → revisione completa dei 616 NULL → audit globale → audit semantico degli attributi ufficiali → audit indipendente OSM ↔ GSFVG sui `TRIM_USAGE=1/2` → validazione diagnostica del matching → congelamento della regola GSFVG-primary / OSM-QA → analisi stratificata e proporzionata dei `TRIM_USAGE=0`**.

##### 4.1.1.1.10 Audit direzionale degli archi `TRIM_USAGE = 0`: consolidamento P1 e trattamento del residuo

A seguito dell'audit sistematico delle direzioni del grafo stradale regionale GSFVG è stata approfondita la popolazione di archi con `TRIM_USAGE = 0` per i quali OpenStreetMap propone una percorrenza monodirezionale. L'obiettivo dell'analisi non è sostituire automaticamente l'informazione ufficiale GSFVG con quella OSM, ma utilizzare OSM come **fonte indipendente di controllo** per identificare eventuali anomalie direzionali rilevanti ai fini del routing.

La popolazione iniziale comprende **3.349 archi GSFVG con `TRIM_USAGE = 0` associati a una direzione one-way OSM con match geometrico di qualità `HIGH` o `STRICT`**. Un successivo controllo strutturale ha permesso di riconoscere:

```text
212 casi riconducibili a rotatorie
588 casi compatibili con carreggiate o rami monodirezionali separati
---------------------------------------------------------------
2.549 discordanti one-way residue
```

I **2.549 casi** non spiegati automaticamente da differenze evidenti di modellizzazione sono stati suddivisi per qualità geometrica del match e disponibilità del campo ufficiale `DBPRIOR_ST`:

| Priorità | Regime | Archi |
|---|---|---:|
| **P1** | `STRICT` + `DBPRIOR_ST = 1` | **136** |
| **P2** | `STRICT` + `DBPRIOR_ST` non compilato | **1.067** |
| **P3** | `HIGH` + `DBPRIOR_ST = 1` | **363** |
| **P4** | `HIGH` + `DBPRIOR_ST` non compilato | **983** |
| **Totale** |  | **2.549** |

Il gruppo **P1**, caratterizzato contemporaneamente dalla massima criticità e dalla migliore qualità geometrica del confronto, è stato sottoposto a **revisione manuale esaustiva 136/136**. La verifica è stata eseguita mediante confronto cartografico in QGIS e controllo su Google Maps, mantenendo separata la valutazione della direzione reale dalle eventuali differenze di rappresentazione topologica fra GSFVG e OSM.

L'audit finale P1 ha prodotto:

```text
OSM_ONEWAY_CONFIRMED                102
GSFVG_OK_BIDIRECTIONAL               13
MODELLING_DIFFERENCE_GSFVG_OK        20
ONEWAY_CONFIRMED_OSM_OPPOSITE         1
UNRESOLVED                            0
---------------------------------------
TOTALE                              136
```

Le direzioni operative risultanti sono:

```text
FWD_ONLY        45
BWD_ONLY        58
BIDIRECTIONAL   33
------------------
TOTALE         136
```

Pertanto **103/136 casi P1 (75,7%) risultano effettivamente monodirezionali**, mentre **33/136 (24,3%) devono rimanere bidirezionali nel modello GSFVG**. In questi ultimi casi l'informazione ufficiale risulta coerente con la situazione reale oppure la discordanza con OSM deriva da differenti modalità di rappresentazione dell'intersezione, della carreggiata o dei rami. È stato inoltre individuato **un solo caso** nel quale la verifica manuale conferma un senso unico opposto a quello proposto da OSM.

Il risultato dimostra due aspetti complementari. Da un lato, un segnale OSM one-way associato a un match geometrico `STRICT` è **fortemente informativo**. Dall'altro, la sua affidabilità non è sufficiente per trasferire automaticamente la direzione all'arco GSFVG: una quota non trascurabile delle discordanti deriva dalla diversa granularità topologica dei due grafi e una correzione automatica potrebbe eliminare connessioni realmente percorribili, compromettendo la connettività del modello.

Per il gruppo **P2**, costituito da **1.067 archi** con match `STRICT` ma `DBPRIOR_ST` non compilato, sono stati esclusi sia l'aggiornamento automatico sia, in prima battuta, una revisione manuale esaustiva. Un clustering preliminare basato sulla prossimità geometrica e sulla condivisione della strada o dell'elemento OSM ha prodotto:

```text
cluster fisici totali     775
singleton                 600
cluster multipli          175
```

La riduzione del numero di casi è limitata e alcuni cluster concatenano sequenze estese lungo la stessa strada. Il clustering è quindi utilizzato come **strumento organizzativo e diagnostico**, non come unità decisionale unica.

La strategia adottata per P2 è una **validazione campionaria stratificata**, mantenendo come default conservativo `TRIM_USAGE = 0` per gli archi non direttamente verificati. Il campione deve essere controllato rispetto almeno a:

- classe stradale;
- direzione OSM;
- caratteristiche dell'elemento OSM;
- appartenenza a cluster singleton o multipli.

Il disegno di revisione P2 è stato successivamente consolidato in **127 casi**, distinti in **120 `INFERENTIAL_SAMPLE`** e **7 `STRATEGIC_SUPPLEMENT` motorway/trunk**. La revisione manuale del campione è stata quindi **chiusa sui 127 casi predisposti**. Le decisioni finali P2 risultano pari a **65 `FWD_ONLY`**, **46 `BWD_ONLY`** e **16 `BIDIRECTIONAL`**, senza casi `UNRESOLVED`. La distinzione campionaria resta metodologicamente valida: eventuali inferenze sulla popolazione P2 devono utilizzare esclusivamente i 120 casi `INFERENTIAL_SAMPLE`, mentre i 7 casi strategici rimangono separati come controllo censuario delle arterie più rilevanti per il routing.

I **940 archi P2 non revisionati** mantengono nella versione operativa corrente il default conservativo regionale `TRIM_USAGE=0 → BIDIRECTIONAL`. La chiusura P2 non viene inoltre utilizzata per autorizzare un trasferimento automatico delle direzioni OSM a P3 e P4. La revisione esaustiva dei **363 archi P3** e dei **983 archi P4** resta sospesa; anche tali archi mantengono il default regionale finché l'audit funzionale non evidenzi un impatto materiale su raggiungibilità e shortest path.

La priorità metodologica resta la costruzione di un **grafo regionale robusto per l'assegnazione dei flussi e il calcolo dei cammini minimi intercomunali**. Le differenze locali di micro-topologia vengono quindi corrette soltanto quando possono alterare in maniera sostanziale la percorribilità o la connettività della rete, evitando una ricostruzione inutilmente dettagliata della disciplina di circolazione urbana.

##### 4.1.1.1.11 Valutazione di Overpass API per il supporto alla revisione direzionale GSFVG–OSM

Prima di avviare la revisione del campione P2 è stata valutata la possibilità di utilizzare **Overpass Turbo / Overpass API** come strumento semi-automatico per ridurre il numero di verifiche manuali delle discordanti direzionali tra GSFVG e OSM.

Overpass non è considerato una nuova fonte indipendente rispetto a OpenStreetMap: i dati interrogati appartengono allo stesso database OSM già utilizzato mediante estratto Geofabrik. Il possibile vantaggio riguarda esclusivamente l'accesso selettivo alla struttura topologica locale OSM, in particolare:

- recupero diretto delle `way` mediante gli `osm_id` già prodotti dal matcher GSFVG–OSM;
- recupero della sequenza dei nodi;
- individuazione delle highway che condividono i nodi della way corrispondente;
- lettura mirata dei tag `highway`, `oneway`, `junction`, `access`, `vehicle`, `motor_vehicle`, `name`, `ref`, `lanes`, `maxspeed` e dei relativi attributi condizionali;
- riconoscimento di differenze di rappresentazione quali segmentazioni locali, rami di intersezione, carreggiate separate e prosecuzioni dello stesso asse con disciplina differente.

**Benchmark preliminare.** La procedura è stata dapprima verificata su alcuni casi P1 già risolti manualmente, utilizzandoli esclusivamente come ground truth e senza modificarne i verdetti.

Nel caso **P1 review 50 – Via Andervolti**, già classificato come `MODELLING_DIFFERENCE_GSFVG_OK → BIDIRECTIONAL`, Overpass ha mostrato che la way OSM associata all'arco GSFVG era `oneway=yes`, ma risultava collegata a ulteriori segmenti dello stesso asse “Via Leonardo Andervolti” rappresentati come bidirezionali. Il controllo topologico era quindi effettivamente in grado di evidenziare una possibile differenza di granularità tra la rappresentazione OSM e l'arco GSFVG aggregato.

Nel caso **P1 review 91 – Via Giacomo Matteotti**, invece, Overpass restituiva una struttura OSM apparentemente coerente con un asse one-way, mentre il controllo indipendente precedentemente effettuato aveva dimostrato che il verso indicato da OSM era opposto a quello reale. Il caso conferma quindi che Overpass può analizzare la struttura topologica di OSM, ma non può correggere un eventuale errore presente nei dati OSM stessi.

**Benchmark completo sui 136 casi P1.** Per valutare quantitativamente l'utilità del metodo è stato quindi eseguito un benchmark sull'intera popolazione dei **136 casi P1 già revisionati manualmente**. La ground truth comprende:

```text
archi realmente monodirezionali    103
archi da mantenere bidirezionali    33
--------------------------------------
totale                             136
```

È stato costruito un classificatore topologico conservativo basato principalmente sulla presenza di prosecuzioni dello stesso asse stradale, riconosciute mediante `name` o `ref`, e sulla loro disciplina direzionale OSM. Le tre classi preliminari erano:

- `AUTO_ONEWAY_CANDIDATE`;
- `AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE`;
- `MANUAL_REVIEW_REQUIRED`.

Il benchmark ha prodotto:

```text
AUTO_ONEWAY_CANDIDATE                 43
AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE 77
MANUAL_REVIEW_REQUIRED                16
----------------------------------------
totale                               136
```

La copertura teoricamente automatizzabile risultava quindi pari all'**88,24%**. Il confronto con le decisioni manuali consolidate ha però evidenziato prestazioni insufficienti:

```text
precisione AUTO_ONEWAY_CANDIDATE                 93,02%
precisione AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE 33,77%
accuratezza complessiva classificazioni automatiche 55,00%
falsi AUTO_ONEWAY                                     3
falsi AUTO_BIDIRECTIONAL                              51
errori complessivi                                    54
```

Il problema principale riguarda la regola basata sulla presenza di una prosecuzione bidirezionale dello stesso asse. Una way OSM one-way può rappresentare correttamente un segmento con disciplina diversa rispetto ai segmenti immediatamente precedenti o successivi, pur condividendone il nome o il riferimento. La presenza di una prosecuzione bidirezionale non costituisce quindi evidenza sufficiente per concludere che l'intero arco GSFVG debba rimanere bidirezionale.

Anche `AUTO_ONEWAY_CANDIDATE`, pur mostrando una precisione molto superiore, non è sufficientemente affidabile per applicare automaticamente la direzione OSM. Tra i casi classificati come one-way compare infatti anche un caso già noto nel quale la monodirezionalità era corretta ma **il verso OSM risultava opposto a quello verificato mediante fonte indipendente**.

**Decisione metodologica.** Il benchmark porta pertanto a **non adottare Overpass come classificatore automatico delle direzioni GSFVG**:

$$
\text{Overpass} \rightarrow \text{strumento diagnostico topologico}
$$

ma non:

$$
\text{Overpass} \rightarrow \text{fonte sufficiente per correggere automaticamente } TRIM\_USAGE.
$$

Overpass rimane utile nei casi selezionati per comprendere rapidamente la micro-topologia OSM, verificare way e nodi associati all'arco GSFVG, diagnosticare segmentazioni, rami e differenze di rappresentazione e ridurre il tempo necessario a interpretare casi topologicamente complessi. Non viene invece utilizzato per trasferire automaticamente a GSFVG una classificazione bidirezionale o monodirezionale, né per stabilire senza controllo indipendente il verso legale di percorrenza.

**Implicazioni per P2.** Il dataset P2 già predisposto rimane invariato:

```text
INFERENTIAL_SAMPLE      120
STRATEGIC_SUPPLEMENT      7
---------------------------
totale review            127
```

Il benchmark Overpass non modifica il disegno campionario né le decisioni metodologiche precedentemente consolidate. La revisione P2 è stata successivamente completata utilizzando il layer e il navigatore già predisposti:

```text
P2_OSM_AS_GSFVG_DISPLAY
```

con la convenzione già validata secondo cui la freccia DISPLAY rappresenta fisicamente la direzione proposta da OSM tradotta nel riferimento della geometria originale GSFVG.

Overpass resta richiamabile soltanto come supporto diagnostico nei casi in cui la struttura topologica non risulti interpretabile direttamente in QGIS. La distinzione tra **120 casi inferenziali** e **7 casi strategici motorway/trunk** resta valida ai fini dell'interpretazione del campione P2, ma la chiusura della revisione non determina automaticamente il trattamento di P3/P4.

Dopo la chiusura P1/P2, l'obiettivo operativo viene spostato sulla validazione globale del grafo. La revisione esaustiva di P3/P4 è sospesa e sarà riaperta selettivamente soltanto quando reachability, connettività, continuità delle arterie principali o plausibilità dei percorsi indichino un impatto funzionale concreto.


#### 4.1.1.2 Benchmark di Overture Maps come rete stradale alternativa

Nell'ambito della selezione del grafo stradale operativo è stata valutata anche **Overture Maps Transportation** come possibile alternativa al Grafo Stradale della Regione Friuli Venezia Giulia (GSFVG) e alla rete derivata da OpenStreetMap (OSM). L'obiettivo del test non era verificare la sola completezza cartografica del dataset, ma determinarne l'effettiva idoneità alla costruzione di un grafo automobilistico diretto utilizzabile per il calcolo dei cammini minimi e per la successiva assegnazione all-or-nothing delle matrici OD.[^audit-overture-20260811]

È stato acquisito un estratto della rete Transportation comprendente integralmente il Friuli Venezia Giulia e il relativo intorno transfrontaliero. La rete automobilistica derivata da Overture comprende **246.223 feature**, per una lunghezza complessiva di circa **37.574,7 km**. La topologia è definita nativamente dallo schema Overture attraverso i `connector` associati ai segmenti stradali; tale struttura costituisce un vantaggio rispetto alla ricostruzione della connettività mediante semplice snapping o noding geometrico, in quanto consente di distinguere connessioni stradali effettive da mere intersezioni geometriche, come nel caso di ponti e sovrappassi.

La rete risultante mostra una buona connettività complessiva: la componente principale comprende il **96,954% dei 453.683 routing edge dotati di costo**. Anche un primo test di routing ha fornito risultati complessivamente positivi: tutti gli **11 itinerari campione** considerati hanno prodotto un cammino valido e non sono stati rilevati utilizzi contromano delle infrastrutture. Un itinerario è risultato anomalo in termini di deviazione rispetto al percorso atteso ed è stato pertanto mantenuto come caso diagnostico, senza considerarlo sufficiente per invalidare il comportamento generale della rete.

La principale criticità riguarda tuttavia la **direzionalità**. Dopo l'applicazione delle regole di accesso e dei relativi default previsti dallo schema Overture, **3.721 feature automobilistiche** rimangono non risolte dal punto di vista direzionale, pari all'**1,511% delle feature** e al **3,179% della lunghezza della rete**. Overture non elimina quindi il problema dell'interpretazione dei versi di percorrenza, ma lo rappresenta mediante uno schema più articolato basato su `access_restrictions`, condizioni di applicabilità e orientamento forward/backward rispetto alla geometria. L'assenza di una restrizione esplicita non può essere interpretata automaticamente come un valore nullo o come bidirezionalità certa, poiché parte del comportamento dipende da default applicativi associati alla classe stradale e alle regole locali.

Analogamente, l'informazione relativa alla velocità non è sufficientemente completa da consentire l'utilizzo diretto del dataset senza ulteriori assunzioni. Una velocità esplicita utilizzabile è disponibile per circa il **17,294% della rete automobilistica**; per la parte restante sarebbe comunque necessario ricorrere a una procedura di imputazione per classe stradale, analoga a quella già prevista nel modello operativo della tesi.

Per valutare direttamente l'affidabilità di Overture sui casi che avevano richiesto una revisione manuale del GSFVG è stato inoltre utilizzato come test set l'insieme delle feature regionali già validate. Su **247 casi direzionali confrontabili**, Overture risulta concorde con la decisione finale GSFVG in **186 casi**, discordante in **52** e non risolto in **9**, corrispondenti rispettivamente al **75,304%**, **21,053%** e **3,644%** del campione. Il risultato è migliore nei casi infrastrutturali più strutturati: le **2 rampe** confrontabili risultano entrambe concordi, mentre per gli **svincoli** sono stati osservati 73 casi concordi, 9 discordanti e 1 non risolto. Il livello di concordanza complessivo non è tuttavia sufficiente per utilizzare Overture come fonte automatica di correzione delle direzioni GSFVG.

Un ulteriore elemento rilevante è costituito dalla **provenance** del dato. Sul sottoinsieme automobilistico analizzato, il **99,739% delle feature** riporta OSM tra le fonti, mentre il contributo associato a TomTom è pari solamente allo **0,261%**. Nel campione disponibile la provenienza è dichiarata a livello di feature e non consente di attribuire separatamente a ciascuna fonte proprietà specifiche quali geometria, classificazione, direzione o velocità. Overture non può pertanto essere considerato, nel caso FVG, un benchmark sostanzialmente indipendente da OSM.

Sono state inoltre censite **9.578 `prohibited_transitions`** sulla rete automobilistica. Tali restrizioni di svolta non sono state applicate nel test preliminare di routing, poiché la loro integrazione avrebbe richiesto lo sviluppo di un ulteriore livello di preprocessing specifico del modello Overture. La loro presenza conferma la ricchezza dello schema, ma anche il costo aggiuntivo necessario per trasformare il dataset in un network routing-ready completo.

Alla luce di questi risultati, **Overture Maps non viene adottato come sostituto del GSFVG né come grafo operativo principale**. La topologia nativa basata sui connector, la buona continuità territoriale e la copertura transfrontaliera ne rendono comunque utile l'impiego come **terza fonte di controllo**, soprattutto per la verifica di connettività, svincoli, rampe e continuità oltre i confini regionali.

La configurazione metodologica adottata rimane pertanto:

- **GSFVG** come riferimento istituzionale e cartografico regionale;
- **OSM** come principale fonte operativa complementare per l'aggiornamento e la verifica degli attributi necessari al routing;
- **Overture Maps** come benchmark aggiuntivo mirato, da utilizzare per controlli topologici e per la verifica di casi critici, senza trasferirne automaticamente direzioni, accessi o altri attributi al grafo operativo.

Il confronto conferma quindi che, allo stato attuale, **non emerge una rete gratuita e aperta chiaramente superiore alla combinazione GSFVG + audit/integrazione OSM** per la costruzione del grafo automobilistico regionale necessario alla tesi. La validazione funzionale del grafo effettivamente adottato dovrà verificare in primo luogo l'esistenza e la plausibilità dei cammini minimi tra le zone OD prima di procedere all'assegnazione all-or-nothing.

#### 4.1.1.3 Precedente metodologico: costruzione del modello di rete stradale nella tesi di Tufaro

Un precedente particolarmente rilevante per la costruzione del grafo stradale regionale è costituito dalla tesi di dottorato di Teresa Tufaro, *Valutazione del Rischio Sismico di un Modello di Rete Stradale per la Regione Friuli Venezia Giulia (NE)*, sviluppata a partire dal database regionale EAGLE FVG e finalizzata alla costruzione di un modello di rete stradale (MRS) per l'analisi della vulnerabilità e del rischio sismico.[^tufaro2022]

L'interesse di tale lavoro per il presente studio deriva dal fatto che viene utilizzata una base stradale GSFVG sostanzialmente riconducibile alla stessa famiglia di dati regionali adottata nel presente progetto e che l'intera procedura di trasformazione del dato geografico in grafo è documentata anche mediante il codice di calcolo riportato in appendice.

##### Ricostruzione della rete a partire dal GSFVG

La procedura di Tufaro può essere schematizzata nelle seguenti fasi:

1. **conversione della geometria stradale** mediante il programma `conv.f`;
2. **identificazione delle connessioni topologiche** mediante `nodes.f`;
3. **semplificazione e riduzione del grafo** mediante `nodesR.f`;
4. **associazione dei comuni alla rete** mediante `com.f`;
5. **generazione dei percorsi tra comuni** mediante `per.f`.

Il programma `conv.f` trasforma la rete geografica in un insieme di **76.350 segmenti stradali**. Per ciascun segmento vengono trasferiti nel file intermedio `str.csv` esclusivamente:

- longitudine e latitudine dell'estremo iniziale;
- longitudine e latitudine dell'estremo finale;
- lunghezza del segmento.

Il dato è quindi ridotto, in questa fase, alla sola geometria necessaria alla ricostruzione della connettività.

Il numero di 76.350 segmenti assume particolare interesse perché coincide con le **76.350 parti lineari** individuate indipendentemente nell'audit del `GSFVG_IRDAT.shp` utilizzato nel presente lavoro. Tale corrispondenza costituisce un forte indizio della sostanziale equivalenza geometrica delle due basi di partenza, senza tuttavia dimostrare che si tratti della medesima versione del dataset o di file perfettamente identici.

Il programma `nodes.f` considera inizialmente separatamente i due estremi di ciascun segmento, producendo quindi **152.700 estremi potenziali**. Gli estremi appartenenti a segmenti differenti vengono riconosciuti come coincidenti quando la loro distanza è inferiore a **0,1 m**. La soglia utilizzata da Tufaro è coerente con i risultati dell'audit topologico svolto nel presente progetto, nel quale la struttura della componente principale del GSFVG è risultata stabile utilizzando tolleranze comprese tra 0,10 e 1 m.

Successivamente, `nodesR.f` procede alla semplificazione della struttura eliminando coincidenze e ridondanze, alcuni piccoli circuiti e alcuni elementi esterni alla regione, contraendo inoltre sequenze di segmenti prive di incroci intermedi. Il risultato dichiarato è un grafo ridotto a **29.695 nodi**.

Questa procedura costituisce un precedente utile per distinguere due problemi che nel presente lavoro devono rimanere separati:

- la **ricostruzione della topologia geometrica**, cioè stabilire quali segmenti siano effettivamente connessi;
- la **definizione della percorribilità operativa**, cioè stabilire in quale direzione ciascun arco possa essere legalmente utilizzato nel routing.

##### Trattamento della direzionalità: risultato dell'analisi del codice

L'analisi dei programmi riportati in appendice evidenzia una differenza metodologica fondamentale rispetto alle esigenze del presente lavoro.

Nel passaggio effettuato da `conv.f`, gli attributi direzionali originari del GSFVG **non vengono trasferiti nel file** **`str.csv`**: ogni segmento viene rappresentato esclusivamente attraverso i due estremi e la lunghezza.

Il successivo programma `nodes.f` costruisce inoltre esplicitamente le relazioni di adiacenza in **entrambi i versi** tra gli estremi di ogni segmento. Per una coppia di nodi collegati (i) e (l), il codice registra infatti sia la connessione $i\rightarrow l$ sia la connessione $l\rightarrow i$. La medesima logica è adottata per gli estremi riconosciuti come coincidenti.

Da ciò si deduce che il grafo computazionale utilizzato da Tufaro è, dal punto di vista della connettività, **non orientato o equivalentemente rappresentato mediante archi reciproci**. Questo non significa che il GSFVG originario non contenesse informazioni sulla direzionalità, ma che tali informazioni non vengono conservate nella struttura sulla quale vengono successivamente ricercati i percorsi.

La scelta risulta coerente con l'obiettivo del lavoro di Tufaro, rivolto soprattutto alla connettività della rete e alla disponibilità di percorsi alternativi in seguito all'interruzione di ponti e viadotti. Per un modello di assegnazione del traffico, e ancor più per il successivo FRLM, la stessa semplificazione non è invece direttamente trasferibile: rendere indiscriminatamente reciproci gli archi consentirebbe potenzialmente l'utilizzo contromano di carreggiate separate, rampe e svincoli.

Il precedente di Tufaro **non costituisce pertanto una validazione dell'ipotesi secondo cui tutti gli archi con** **`TRIM_USAGE=0`** **possano essere considerati automaticamente bidirezionali**. Esso dimostra piuttosto che, nel precedente modello MRS, il problema della direzionalità è stato escluso dalla rappresentazione computazionale mediante una scelta topologica compatibile con il diverso obiettivo dello studio.

##### Associazione dei comuni alla rete

Nel modello MRS ciascun comune viene associato al **nodo della rete più prossimo**. Il programma `com.f` utilizza quindi un solo nodo stradale per rappresentare l'accesso alla rete del comune di origine e un solo nodo per quello di destinazione.

Tale soluzione costituisce una baseline semplice e riproducibile, ma introduce una concentrazione artificiale dei flussi su un unico accesso. Nel presente lavoro viene pertanto mantenuta una formulazione più generale, nella quale ogni zona (o) è collegata a un insieme ristretto di nodi stradali ammissibili:

$$
\Gamma_o^{\mathrm L}\subseteq\mathcal V_{\mathrm L},
$$

eventualmente dotati di pesi $\lambda_{ou}^{\mathrm L}$. Il metodo di Tufaro potrà essere utilizzato come caso limite con $|\Gamma_o^{\mathrm L}|=1$ e come benchmark diagnostico, ma non viene assunto come configurazione definitiva.

Va inoltre considerata la differenza temporale nella geografia amministrativa: Tufaro lavora con **220 comuni**, mentre il modello corrente utilizza la configurazione comunale aggiornata. I conteggi delle relazioni OD non sono quindi direttamente confrontabili senza tenere conto delle modifiche amministrative intervenute.

##### Generazione dei percorsi

Anche il procedimento utilizzato per individuare gli itinerari differisce dal routing previsto nel presente progetto.

Il programma `per.f` applica una ricerca iterativa orientata geometricamente verso la destinazione, con procedure di ritorno all'indietro quando il percorso incontra un vicolo cieco. Dopo l'individuazione del primo itinerario, una quota pari al **20% dei segmenti utilizzati viene eliminata casualmente** per favorire la ricerca di percorsi alternativi. Per ciascun collegamento vengono effettuati fino a **10.000 tentativi** e vengono conservati fino a dieci itinerari, purché la loro lunghezza non superi tre volte la distanza in linea d'aria tra origine e destinazione. Con tali condizioni, 49 degli 8.020 collegamenti non producono una soluzione accettabile.

Questa procedura è funzionale alla ricerca di ridondanza della rete, ma non equivale a un algoritmo standard di minimo costo. Nel presente progetto il routing light interno è ora congelato sul **grafo diretto turn-aware B5** con `TIME_B5` come impedenza canonica: per ogni access-pair esiste un singolo path time-optimal materializzato nella Fase 5.7. Eventuali scenari alternativi o multipath, se mai introdotti, dovranno essere versioni separate e non modificare il sistema interno frozen.

La tesi di Tufaro presenta inoltre un'incongruenza interna nei conteggi finali dei percorsi: in alcune sintesi del modello vengono indicati **6.397 percorsi**, in una successiva sezione dei risultati **76.397**, mentre la descrizione di `per.f` parla di **quasi 80.000 percorsi**. Tale discordanza consiglia di non utilizzare il numero complessivo di percorsi come elemento di validazione quantitativa del presente grafo. Restano invece robustamente confrontabili le caratteristiche della rete di partenza e della procedura topologica.

##### Implicazioni per il trattamento del GSFVG nel presente lavoro

Il confronto con Tufaro permette di consolidare alcune considerazioni metodologiche.

In primo luogo, il GSFVG costituisce un precedente effettivamente utilizzato per costruire una rete stradale computazionale su scala regionale. La coincidenza del numero di segmenti e la procedura di riconoscimento dei nodi confermano la plausibilità dell'utilizzo del GSFVG come **backbone geometrico e topologico** del modello corrente.

In secondo luogo, il precedente conferma che **topologia e direzionalità devono essere trattate come due livelli distinti**. La connettività geometrica può essere ricostruita indipendentemente dalla direzione di marcia; il grafo destinato all'assegnazione dei flussi deve tuttavia applicare successivamente le regole di percorrenza forward e backward.

In terzo luogo, il precedente non risolve direttamente l'attuale ambiguità dei **64.477 archi con** **`TRIM_USAGE=0`**. Nel dataset utilizzato nel presente progetto tale popolazione comprende:

```
30.418   DBPRIOR_ST=1     DBPRIOR_TI=1
    20   DBPRIOR_ST=1     DBPRIOR_TI=3
    46   DBPRIOR_ST=1     DBPRIOR_TI=NULL
33.993   DBPRIOR_ST=NULL  DBPRIOR_TI=NULL

```

Poiché la definizione ufficiale di `TRIM_USAGE=0` può rappresentare il caso bidirezionale ma non permette, isolatamente, di escludere tutti gli altri casi previsti dalla semantica della banca dati, il comportamento adottato da Tufaro non può essere utilizzato come giustificazione sufficiente per convertire automaticamente l'intera popolazione in archi bidirezionali.

Il contributo del precedente è tuttavia importante nel definire una strategia proporzionata. Non appare necessario ricostruire manualmente decine di migliaia di archi per ottenere la topologia della rete; è invece opportuno:

1. mantenere il GSFVG come backbone geometrico;
2. conservare separatamente gli attributi originali e le decisioni operative derivate;
3. utilizzare `TRIM_USAGE`, `DBPRIOR_ST` e `DBPRIOR_TI` come primo livello semantico;
4. isolare preventivamente gli elementi speciali o non riconducibili alla normale viabilità automobilistica;
5. sottoporre la popolazione ancora ambigua a verifiche indipendenti e proporzionate, anziché a revisione manuale esaustiva;
6. utilizzare OSM e, ove utile, Overture come fonti di controllo, senza sostituire automaticamente il riferimento regionale;
7. sottoporre il grafo risultante a una successiva **validazione funzionale dei cammini**, verificando raggiungibilità, assenza di percorrenze contromano, corretto comportamento di rampe e svincoli e plausibilità degli itinerari.

In questo senso, la tesi di Tufaro non fornisce un grafo direttamente riutilizzabile per l'assegnazione del traffico, ma costituisce un importante **precedente metodologico per la trasformazione topologica del GSFVG**. Il presente lavoro ne conserva gli elementi utili — utilizzo della base regionale, ricostruzione della connettività, tolleranza geometrica e semplificazione topologica — introducendo tuttavia una rappresentazione diretta della percorribilità necessaria per gli obiettivi di stima della domanda, assegnazione dei flussi e localizzazione delle infrastrutture di ricarica.

**Riferimento:** Tufaro, T., *Valutazione del Rischio Sismico di un Modello di Rete Stradale per la Regione Friuli Venezia Giulia (NE)*, Tesi di Dottorato, XXXV ciclo, A.A. 2021/2022.

#### 4.1.1.4 Consolidamento direzionale e costruzione della rete GSFVG operativa

La fase di revisione e consolidamento della direzionalità della rete regionale GSFVG è stata completata mediante una procedura separata dalla successiva definizione delle restrizioni di accesso, delle impedenze e dei costi di percorrenza. L'obiettivo è stato produrre una copia operativa persistente e versionata nella quale la direzione di percorrenza di ciascun arco fosse esplicitamente definita e direttamente utilizzabile nelle successive elaborazioni di routing, senza modificare né sovrascrivere l'informazione originaria.

##### Audit delle sorgenti direzionali

La rete sorgente `GSFVG_IRDAT_FULL` contiene **76.349 feature** e utilizza il campo ufficiale `TRIM_USAGE` per rappresentare la direzionalità:

| `TRIM_USAGE` | Significato operativo | Numero archi |
|---|---|---:|
| `0` | bidirezionale | 64.477 |
| `1` | monodirezionale nel verso della geometria | 8.704 |
| `2` | monodirezionale in verso opposto alla geometria | 2.552 |
| `NULL` | direzione originariamente non risolta | 616 |

Il valore di 76.349 si riferisce alle feature del layer sorgente; il precedente audit topologico ha invece analizzato **76.350 parti lineari**, per effetto della decomposizione geometrica utilizzata in quella specifica verifica.

I **616** valori originariamente `NULL` sono stati integralmente risolti mediante revisione manuale, suddivisa in 54 archi SC, 363 archi AS mainline e 199 archi AS residui. L'audit conclusivo ha verificato una copertura di 616/616 `ID1`, senza missing, extra o sovrapposizioni. Le decisioni consolidate risultano pari a **557 `FWD_ONLY`**, **1 `BWD_ONLY`** e **58 `BIDIRECTIONAL`**.

Per `TRIM_USAGE=0`, il gruppo **P1** è stato revisionato integralmente su 136 archi, con 45 `FWD_ONLY`, 58 `BWD_ONLY` e 33 `BIDIRECTIONAL`. Per il gruppo **P2**, costituito da una popolazione di 1.067 archi, sono stati revisionati 127 casi — 120 appartenenti al campione inferenziale e 7 al supplemento strategico — con esito finale **65 `FWD_ONLY`**, **46 `BWD_ONLY`** e **16 `BIDIRECTIONAL`**.

Per P1 e P2 è stata verificata la coerenza tra verdetto manuale e direzione finale, senza casi `UNRESOLVED`. Le verifiche utilizzano, a seconda del caso, Google Maps, Google Street View e controlli topologici OSM. Gli override manuali consolidati sono quindi **879 `ID1` distinti**:

$$
616+136+127=879.
$$

I **940 casi P2 non revisionati**, così come i gruppi **P3** e **P4** non sottoposti a revisione esaustiva, mantengono **nella baseline GSFVG qui documentata** la semantica conservativa della fonte regionale, cioè `TRIM_USAGE=0 → BIDIRECTIONAL`. Tale regola descrive il prodotto storico GSFVG e non implica che questi casi debbano essere ulteriormente revisionati dopo la decisione di switch.

##### Regole di consolidamento

La copia operativa applica una precedenza esplicita e riproducibile:

1. `TRIM_USAGE=1 → FWD_ONLY`;
2. `TRIM_USAGE=2 → BWD_ONLY`;
3. `TRIM_USAGE=0 → BIDIRECTIONAL` come default;
4. i 616 archi originariamente `NULL` ricevono la decisione manuale consolidata;
5. le revisioni P1 e P2 approvate costituiscono override espliciti esclusivamente su archi originariamente `TRIM_USAGE=0`.

Gli override vengono applicati utilizzando `ID1` come chiave logica. Qualsiasi duplicazione dello stesso `ID1`, riferimento a un arco inesistente o incompatibilità tra gruppo di review e valore originario di `TRIM_USAGE` determina un **hard failure** della procedura, evitando risoluzioni implicite o precedenze silenziose.

##### Dataset operativo GSFVG v01

Il risultato della procedura è il GeoPackage:

```text
C:\Tesi\Tesi_QGIS\02_package\grafo_operativo_gsfvg\GSFVG_operativo_direzionale_v01.gpkg
```

contenente il layer spaziale:

```text
GSFVG_operativo_direzionale_v01
```

e le tabelle non spaziali:

```text
direction_overrides_v01
build_metadata
```

Il dataset rimane in **EPSG:25833**, CRS nativo della sorgente GSFVG. In questa fase non viene effettuata alcuna riproiezione, così da isolare completamente il consolidamento attributivo/direzionale da eventuali trasformazioni geometriche.

Il layer principale conserva integralmente i **26 attributi GSFVG originari**, incluso `TRIM_USAGE`, e aggiunge i campi operativi:

```text
dir_final
dir_fwd_ok
dir_bwd_ok
dir_source
review_group
review_scope
review_outcome
review_status
verification_source
review_note
gsfvg_fid_original
```

`ID1` rimane la chiave logica principale. `gsfvg_fid_original` materializza invece il FID della feature sorgente utilizzata durante la revisione; il `fid` tecnico generato dal GeoPackage non viene utilizzato come identificativo stabile.

La direzione finale viene tradotta automaticamente nei due flag elementari di percorrenza:

| `dir_final` | `dir_fwd_ok` | `dir_bwd_ok` |
|---|---:|---:|
| `FWD_ONLY` | 1 | 0 |
| `BWD_ONLY` | 0 | 1 |
| `BIDIRECTIONAL` | 1 | 1 |

La combinazione `(0,0)` non è ammessa. I flag sono derivati esclusivamente da `dir_final` e non vengono importati dai CSV di revisione. Essi costituiscono quindi l'interfaccia diretta tra il dataset GIS consolidato e la futura costruzione del grafo computazionale.

`dir_source` distingue tre provenienze:

- `GSFVG_EXPLICIT`, per gli archi con `TRIM_USAGE=1/2`;
- `GSFVG_DEFAULT`, per gli archi che mantengono la classificazione bidirezionale originaria;
- `MANUAL_REVIEW`, per gli archi interessati dagli 879 override consolidati.

Nei casi manuali vengono inoltre conservati gruppo di review, eventuale scope, outcome originale, stato, fonte di verifica e nota. I placeholder `<NOTE>` privi di contenuto informativo sono normalizzati a campo vuoto.

##### Audit trail e riproducibilità

La tabella `direction_overrides_v01` contiene un record per ciascuno degli **879 override manuali** e costituisce l'audit trail normalizzato della revisione. Per ogni `ID1` vengono conservati almeno il FID GSFVG originale, il valore originario di `TRIM_USAGE`, il gruppo e lo scope della review, la direzione finale, il verdetto originale, lo stato della revisione, la fonte di verifica, la nota e il CSV sorgente.

La tabella `build_metadata` registra gli input utilizzati nella build, i rispettivi hash SHA-256, il numero di record, le righe fisiche ed effettive dei CSV, i timestamp e le informazioni di build. Per il dataset GSFVG, essendo la sorgente uno Shapefile multi-file, viene calcolato un manifest SHA-256 dei componenti del dataset anziché il solo hash del file `.shp`.

Il GeoPackage finale della build v01 presenta SHA-256:

```text
24a699f71f5ad115afe2f9fa96c0f2b51334e022f18e5d3f47d3bc873cd0036a
```

##### Audit finale della copia operativa

La build è stata sottoposta a un audit completo source-output. Sono stati verificati **76.349 `ID1` univoci**, **76.349 FID sorgente materializzati**, **76.349 geometrie presenti** e nessuna geometria mancante. La firma geometrica complessiva coincide con quella del GSFVG sorgente e non è stata rilevata alcuna modifica geometrica.

Tutti i 26 attributi originali sono stati inoltre verificati feature per feature. Una diagnostica intermedia aveva evidenziato un arrotondamento dei campi `BEGIN`, `END` e `ORIG_LENGT` dovuto alla precisione dichiarata dallo Shapefile; la build è stata corretta serializzando tali campi come `Double` senza imporre la precisione nominale dello Shapefile. L'audit finale ha ottenuto **zero mismatch attributivi** e identità della firma complessiva degli attributi tra sorgente e output.

La distribuzione finale per provenienza della direzione è:

| Provenienza | Archi |
|---|---:|
| `GSFVG_EXPLICIT` | 11.256 |
| `GSFVG_DEFAULT` | 64.214 |
| `MANUAL_REVIEW` | 879 |
| **Totale** | **76.349** |

La distribuzione finale delle direzioni operative è:

| Direzione finale | Archi |
|---|---:|
| `FWD_ONLY` | 9.371 |
| `BWD_ONLY` | 2.657 |
| `BIDIRECTIONAL` | 64.321 |
| **Totale** | **76.349** |

L'audit finale ha inoltre verificato:

- zero `UNRESOLVED`;
- zero `dir_final` invalidi;
- zero incoerenze tra `dir_final` e i flag di percorrenza;
- zero errori nella tabella degli override;
- zero errori nei metadata.

##### Consolidamento dell'ambiente QGIS

Completata la build, è stata effettuata anche una procedura di housekeeping del progetto QGIS. Il layer temporaneo `P2_OSM_AS_GSFVG_DISPLAY`, precedentemente residente in memoria, è stato materializzato nel GeoPackage persistente:

```text
C:\Tesi\Tesi_QGIS\02_package\revisione_direzioni_gsfvg\GSFVG_P2_review_display_127.gpkg
```

La persistenza è stata verificata confrontando tutti i **127 `ID1`**, gli attributi e le geometrie con il memory layer originario, ottenendo zero mismatch e firme source-output identiche.

Il layer persistente è stato collocato unicamente in:

```text
04_REVISIONE_DIREZIONI/02_P2_CHIUSO
```

Il precedente nodo errato in `01_TERRITORIO` è stato eliminato e il gruppo `02_P2_IN_CORSO` è stato rinominato `02_P2_CHIUSO`.

Il layer operativo `GSFVG_operativo_direzionale_v01` è collocato in `05_GRAFO_OPERATIVO`. Il progetto QGIS è stato salvato senza modifiche pendenti e con backup preventivi dello stato su disco e dello stato live.

##### Stato metodologico al termine del gate

Con questa procedura il **consolidamento direzionale della rete GSFVG era stato chiuso**. Nella baseline storica, `GSFVG_operativo_direzionale_v01` costituiva il confine tra il dataset regionale e le procedure di review, da un lato, e il successivo grafo computazionale, dall'altro.

Quella versione storica conteneva esclusivamente la semantica direzionale consolidata: **non vi erano ancora applicate restrizioni di accesso, velocità, tempi di percorrenza, impedenze o costi generalizzati**. Le corrispondenti regole sviluppate sulla pipeline OSM preliminare rimasero allora disponibili come base metodologica; il successivo `SWITCH_OSM` ha reso superato il previsto trasferimento sistematico al backbone GSFVG.

Nella fase storica precedente allo `SWITCH_OSM` si prevedeva che il grafo computazionale GSFVG utilizzasse direttamente `dir_fwd_ok` e `dir_bwd_ok` per generare gli archi orientati, senza reinterpretare `TRIM_USAGE`. Quel passaggio non è più un'attività corrente: la pipeline GSFVG è stata superseduta dal backbone OSM frozen della Fase 5.6.

### 4.1.2 Rete OSM light preliminare e PBF di riferimento

È stato congelato l'estratto Geofabrik `nord-est_2026-08-03.osm.pbf`. La data nel nome del file identifica la fotografia della rete impiegata e deve essere conservata nei metadati di ogni prodotto derivato. I layer descritti in questa sezione appartengono alla **prima elaborazione OSM**, nata inizialmente come fonte complementare al GSFVG e utile a consolidare regole su accessi, direzioni e velocità. Dopo il Gate shadow lo **stesso PBF congelato** diventa la sorgente della nuova pipeline operativa OSM della Fase 5.6; la costruzione routing-ready B1--B5 è documentata nella §4.4.7. I prodotti preliminari qui descritti restano quindi provenance e base metodologica, non il prodotto finale del nuovo backbone.

L'area di lavoro di questa **elaborazione OSM preliminare storica** comprendeva il FVG e una fascia esterna di 20 km verso il territorio italiano confinante. Poiché l'estratto non comprendeva Austria e Slovenia, il disegno dell'epoca prevedeva gateway di confine sulla rete regionale. Questa formulazione resta provenance e non descrive la baseline LIGHT v0 corrente, che è FVG-only e rinvia l'external demand a una estensione downstream separata.

Nel **layer OSM preliminare** destinato alla mobilità light erano comprese le seguenti classi `highway`:

```text
motorway, motorway_link,
trunk, trunk_link,
primary, primary_link,
secondary, secondary_link,
tertiary, tertiary_link,
unclassified, residential, living_street
```

Nella pipeline preliminare la classe `service` era conservata in un layer ausiliario separato, destinato al controllo e alla costruzione di collegamenti verso aree di servizio, parcheggi, distributori e siti candidati; tale scelta appartiene alla provenance precedente e non definisce da sola la semantica del backbone B1--B5 oggi frozen. Percorsi pedonali o ciclabili, sentieri, strade in costruzione o proposte e oggetti non interpretabili come archi carrabili sono esclusi dal grafo principale.

I prodotti correnti sono `osm_rete_light_principale_fvg_20km`, `osm_viabilita_service_fvg_20km`, `osm_rete_light_principale_fvg_20km_singleparts` e `osm_rete_light_fvg_20km_tag_estratti`. Quest'ultimo contiene 22 campi e conserva `other_tags` insieme alle colonne estratte, evitando la perdita dell'informazione sorgente.

Le classi `unclassified`, `residential` e `living_street` non sono eliminate a priori, perché possono essere necessarie per assicurare l'accessibilità di alcuni comuni. Sono trattate principalmente come connettori alla rete gerarchicamente superiore e ricevono costi o penalità coerenti con il loro ruolo, da sottoporre ad analisi di sensibilità, per scoraggiarne l'uso improprio nei percorsi regionali. Tale protezione è completata dalla scelta di pochi accessi zonali significativi in $\Gamma_o^{\mathrm L}$.

Il passaggio a geometrie singole ha prodotto 112.466 `LineString` in EPSG:32632. Questo risultato non costituisce il backbone topologico finale: i layer OSM saranno utilizzati per il confronto con GSFVG e per il trasferimento validato degli attributi, evitando di introdurre connessioni automatiche in corrispondenza di ponti, gallerie o attraversamenti a quote differenti.

### 4.1.3 Censimento completo di `other_tags` e perimetro informativo

Sul layer `osm_rete_light_fvg_20km_tag_estratti` è stato eseguito un censimento completo di `other_tags` per tutte le 112.466 geometrie.[^censimento-other-tags] L'operazione ha trasformato il campo da contenitore opaco a inventario controllabile e ha prodotto i seguenti risultati:

| Indicatore | Valore |
|---|---:|
| geometrie con `other_tags` valorizzato | 86.244 |
| geometrie con `other_tags` nullo o vuoto | 26.222 |
| errori di parsing | 0 |
| chiavi residue distinte | 446 |
| occorrenze di coppie chiave--valore sulle geometrie | 312.530 |
| combinazioni distinte chiave--valore nel file completo | 4.597 |

La quantità 312.530 non indica combinazioni distinte: rappresenta il numero complessivo di occorrenze di coppie chiave--valore osservate sulle geometrie. Le combinazioni distinte sono invece 4.597 e il file completo ne conserva i conteggi. Questa distinzione terminologica deve essere mantenuta nei resoconti e negli output.

Il censimento ha individuato le seguenti chiavi precedentemente non estratte ma immediatamente rilevanti per accesso e direzione:

| Chiave | Geometrie |
|---|---:|
| `vehicle` | 80 |
| `vehicle:conditional` | 2 |
| `motor_vehicle:conditional` | 20 |
| `access:conditional` | 37 |
| `access:backward` | 18 |
| `motor_vehicle:backward` | 7 |
| `oneway:conditional` | 27 |

I conteggi descrivono la presenza delle chiavi sulle geometrie e non sono necessariamente disgiunti. Le sette chiavi sono state selezionate, estratte e conservate insieme a `oneway`, `junction`, `access`, `motor_vehicle` e `motorcar` per consolidare accessibilità e direzionalità operative. Nel censimento non compare `motorcar:conditional`; nella versione corrente non viene pertanto creato un campo strutturalmente vuoto.

Per le fasi successive sono inoltre rilevanti: `maxspeed`, `maxspeed:forward`, `maxspeed:backward`, `maxspeed:conditional`, `maxspeed:type`, `zone:maxspeed`, `surface` e `smoothness` per velocità e costo; `layer`, `bridge` e `tunnel` per la costruzione topologica; `hgv`, `maxweight`, `maxheight`, `maxlength` e `maxaxleload` soprattutto per il futuro grafo heavy. In particolare, due linee che si incrociano in pianta non devono essere connesse automaticamente quando rappresentano livelli differenti: in OSM le vie connesse alla stessa quota condividono un nodo, mentre attraversamenti a quote diverse non lo condividono e sono qualificati anche mediante `layer=*`, `bridge=*` o attributi equivalenti.[^osm-nodes-crossings]

Le 191 chiavi restituite dal filtro automatico di richiamo non costituiscono l'elenco finale dei campi operativi. Il filtro massimizza il richiamo e include falsi positivi quali `destination`, `destination:forward`, `destination:ref`, `cycleway:right:oneway`, `internet_access` e `access:name`. In particolare, `destination=*` identifica destinazioni indicate seguendo una strada e non equivale a una restrizione `access=destination`.[^osm-destination] Ogni chiave deve quindi essere selezionata in base alla propria semantica documentata, non soltanto alla presenza di una sottostringa.

Il censimento riguarda esclusivamente gli attributi associati alle geometrie lineari del layer e conservati in `other_tags`. Non costituisce un inventario completo di tutte le regole di routing presenti nel PBF: le restrizioni di svolta sono normalmente modellate mediante relazioni `type=restriction` con membri `from`, `via` e `to`, mentre cancelli, dissuasori e altre barriere possono essere oggetti puntuali con proprie regole di accesso.[^osm-turn-restrictions][^osm-barriers] Relazioni e nodi richiedono pertanto estrazioni e controlli dedicati prima di considerare completo il grafo operativo.

### 4.1.4 Direzionalità e stato dell'informazione

Il tag OSM originale e la sua interpretazione operativa sono mantenuti distinti. Prima di definire i campi operativi si riepiloga il censimento della direzionalità statica.

Il censimento di `tag_oneway` sulle 112.466 geometrie ha restituito:

| Valore | Geometrie |
|---|---:|
| `yes` | 28.024 |
| `no` | 3.885 |
| `NULL` | 80.460 |
| `alternating` | 95 |
| `reversible` | 1 |
| `-1` | 1 |

Il controllo congiunto con `tag_junction` ha individuato 3.568 casi `NULL + roundabout`, 12 `yes + roundabout`, 22 `NULL + circular` e 5 `yes + circular`. Questi conteggi costituiscono il punto di partenza per la normalizzazione e restano associati alla fotografia OSM del 3 agosto 2026.

Sulla base di tali evidenze, si adottano i campi:

```text
direction_code:
  1  = percorribile nel verso della geometria
 -1  = percorribile nel verso opposto alla geometria
  0  = percorribile in entrambi i versi
NULL = non rappresentabile automaticamente

direction_status:
standard
implicit_roundabout
implicit_motorway
alternating
reversible
conditional
to_review
```

La codifica preliminare segue queste regole: `oneway=yes` produce `direction_code=1`, `oneway=-1` produce `-1` e `oneway=no` produce `0`; `junction=roundabout` e `highway=motorway` implicano normalmente il senso unico quando manca un'esplicita deroga.[^osm-oneway] I valori `alternating` e `reversible` rimangono distinti e non sono ridotti automaticamente a un arco diretto ordinario. Per `junction=circular`, invece, il senso unico è frequente ma non implicito in modo sicuro: in assenza di `oneway=yes/no` il caso entra inizialmente in `to_review`, poiché esistono anche circolazioni bidirezionali; l'esito del successivo audit mirato è registrato nella Sezione 4.1.6.[^osm-circular]

Il censimento ha inoltre individuato 27 geometrie con `oneway:conditional`.[^censimento-other-tags] Di queste, 26 riportano l'alternanza stagionale e oraria `yes @ (Jul-Aug 09:00-13:00); -1 @ (Jul-Aug 14:00-17:00)`; il caso residuo deve essere conservato e verificato separatamente. Questi archi non sono inclusi nei 95 casi `oneway=alternating` né nel singolo `oneway=reversible` e costituiscono una famiglia distinta di direzionalità dinamica. La sintassi condizionale esprime un'eccezione valida soltanto quando la condizione è soddisfatta e deve essere valutata insieme alla regola ordinaria o implicita.[^osm-conditional]

I valori nulli di `oneway` non sono sinonimo di dato errato e non richiedono un controllo manuale esaustivo. Dopo l'applicazione delle regole implicite documentate, la percorribilità residua è assunta provvisoriamente bidirezionale soltanto per le classi e i casi compatibili con tale default; origine della regola, eccezioni e casi dubbi restano registrati in `direction_status`. La normalizzazione delle restrizioni di accesso segue il principio gerarchico della sezione successiva.

### 4.1.5 Gerarchia dei tag di accesso per le automobili

Per il routing light, `access`, `vehicle`, `motor_vehicle` e `motorcar` non rappresentano restrizioni indipendenti da sommare. Appartengono alla gerarchia dei tag OSM di accesso, nella quale una chiave riferita a una modalità più specifica può sostituire il valore ereditato dalla chiave genitrice.[^osm-access-hierarchy]

Per le automobili l'ordine di prevalenza è:

```text
tag_motorcar
    ↓ prevale su
tag_motor_vehicle
    ↓ prevale su
tag_vehicle
    ↓ prevale su
tag_access
```

`motorcar=*` esprime la regola specifica per le automobili; `motor_vehicle=*` riguarda la classe più ampia dei veicoli a motore, comprendente automobili, motocicli, mezzi pesanti e autobus; `vehicle=*` riguarda i veicoli in generale; `access=*` stabilisce una regola generale per tutte le modalità di trasporto.[^osm-motor-vehicle] Pertanto, se più tag statici sono valorizzati sul medesimo arco, la regola effettiva per un'automobile è il primo valore disponibile procedendo dal livello più specifico al più generale.

Il censimento ha confermato che `vehicle=*` non è soltanto una possibilità teorica: compare su 80 geometrie ed è stato incluso nell'estrazione operativa. La precedente catena ridotta `motorcar` → `motor_vehicle` → `access` è quindi sostituita dalla gerarchia completa sopra riportata.

Per esempio, la combinazione `access=private`, `motor_vehicle=destination` e `motorcar=yes` rende l'arco accessibile alle automobili secondo `motorcar=yes`: il tag specifico corregge i valori ereditati dai livelli superiori. Ne consegue che le frequenze calcolate separatamente sui diversi campi possono sovrapporsi e non devono essere sommate per stimare il numero di archi aperti o limitati.

La gerarchia modale statica non è tuttavia sufficiente. Sono presenti `access:conditional` su 37 geometrie, `vehicle:conditional` su 2 e `motor_vehicle:conditional` su 20, oltre a `access:backward` su 18 e `motor_vehicle:backward` su 7. Le frequenze possono sovrapporsi e restano non additive. Secondo la semantica OSM, a parità di modalità una restrizione direzionale prevale su quella non direzionale e una restrizione condizionale prevale sulla regola non condizionale della stessa modalità e direzione; `forward` e `backward` sono definiti rispetto al verso della way OSM.[^osm-conditional] Non essendo stato rilevato `motorcar:conditional`, non viene creato un campo vuoto dedicato nella versione corrente.

La classificazione operativa dell'accesso deve essere costruita arco per arco risolvendo prima la specificità modale e poi le eventuali regole direzionali e condizionali applicabili. Quando nessun tag pertinente è presente, il valore non è dedotto mediante una somma né interpretato automaticamente come dato mancante: si applica il valore predefinito implicato dalla tipologia `highway` e dal profilo di routing pertinente, mantenendo tracciata la natura implicita della decisione.[^osm-access-hierarchy] I tag di accesso descrivono primariamente l'autorizzazione legale; l'idoneità fisica o strategica dell'arco resta affidata ad attributi e costi distinti.

### 4.1.6 Normalizzazione metodologica di accessibilità e direzionalità della rete light

La costruzione del grafo stradale operativo è stata preceduta da un censimento completo degli attributi residui contenuti nel campo `other_tags` del layer neutrale `osm_rete_light_fvg_20km_tag_estratti`. Il controllo ha interessato tutte le 112.466 geometrie della rete: 86.244 presentavano `other_tags` valorizzato, 26.222 un valore nullo o vuoto e nessuna geometria ha generato errori di parsing. Sono state individuate 446 chiavi distinte, per complessive 312.530 occorrenze di tag; tali occorrenze non devono essere confuse con le 4.597 combinazioni chiave--valore distinte censite.[^censimento-other-tags]

Il censimento ha mostrato che la sola estrazione iniziale dei campi `access`, `motor_vehicle` e `motorcar` non era sufficiente. Sono state quindi integrate anche la chiave `vehicle`, le restrizioni riferite al verso opposto e le principali regole condizionali.[^normalizzazione-accesso-direzione] L'accessibilità automobilistica di base è stata ricostruita applicando la gerarchia di specificità:

```text
motorcar → motor_vehicle → vehicle → access → default implicito
```

Il primo valore disponibile nella gerarchia determina la regola effettiva per l'automobile. La classificazione è stata effettuata separatamente nel verso della geometria e nel verso opposto; in quest'ultimo sono applicati, quando presenti, `access:backward` e `motor_vehicle:backward`.[^osm-access-hierarchy] Le classi risultanti e la loro traduzione operativa sono:

| Classe di accesso | Codice per verso | Interpretazione |
|---|---:|---|
| `open` | 1 | accesso ordinario |
| `local_restricted` | 2 | accesso locale o traffico di destinazione |
| `permission_restricted` | 0 | non disponibile per l'assegnazione automobilistica ordinaria |
| `authorized_only` | 0 | non disponibile per l'assegnazione automobilistica ordinaria |
| `prohibited` | 0 | non disponibile per l'assegnazione automobilistica ordinaria |

Le classi originarie sono conservate nei campi `access_fwd_class` e `access_bwd_class`: la convergenza delle ultime tre classi nel codice 0 non elimina quindi la causa specifica dell'esclusione.[^normalizzazione-accesso-direzione]

La direzionalità statica è stata ricostruita a partire da `oneway` e `junction`, distinguendo sensi unici espliciti, senso opposto alla geometria, rotatorie implicitamente monodirezionali e geometrie bidirezionali o trattate staticamente come tali. Prima della costruzione topologica, il termine «arco» indica qui il record lineare candidato a diventare arco del grafo. Dopo l'audit mirato la partizione consolidata è:[^audit-direzionalita-finale]

| Percorribilità statica | Geometrie |
|---|---:|
| soltanto nel verso della geometria (`direction_code=1`) | 31.614 |
| soltanto nel verso opposto | 1 |
| bidirezionale o trattata staticamente come tale (`direction_code=0`) | 80.851 |
| **Totale** | **112.466** |

La distribuzione finale del campo `direction_status` è:

| `direction_status` | Geometrie |
|---|---:|
| `default_bidirectional` | 76.870 |
| `explicit_oneway` | 28.024 |
| `explicit_bidirectional` | 3.885 |
| `implicit_roundabout` | 3.568 |
| `alternating` | 95 |
| `implicit_circular` | 22 |
| `explicit_reverse` | 1 |
| `reversible` | 1 |
| **Totale** | **112.466** |

L'audit mirato ha riguardato 118 geometrie: 95 `alternating`, 1 `reversible` e 22 `junction=circular` prive di `oneway` esplicito.[^audit-direzionalita-finale] L'inclusione di `alternating` e `reversible` rimane una semplificazione statica intenzionale: le geometrie sono considerate percorribili nei due versi, ma non necessariamente nello stesso momento, e conservano uno stato speciale per una successiva penalizzazione legata all'attesa. Il caso `reversible` di Via dei Bagni Nuova è stato verificato manualmente come ponte a senso alternato regolato da semaforo. I 22 casi `junction=circular` sono stati invece riclassificati come monodirezionali nel verso della geometria, portando `direction_code=1` da 31.592 a 31.614 e `direction_code=0` da 80.873 a 80.851.

Le 59 occorrenze condizionali di accesso sono costituite da 37 `access:conditional`, 2 `vehicle:conditional` e 20 `motor_vehicle:conditional`. L'audit ha verificato che appartengono a 59 geometrie distinte: nel dataset corrente nessuna geometria contiene contemporaneamente due delle tre famiglie condizionali, benché ciascuna possa coesistere con una regola di accesso di base. Cinquantotto valori sono sintatticamente leggibili e uno è sospetto. Nel primo grafo statico resta valida la regola di base e la presenza della condizione è registrata mediante `access_conditional_flag`.[^normalizzazione-accesso-direzione][^osm-conditional]

Le 27 occorrenze `oneway:conditional` comprendono 26 valori sintatticamente leggibili e uno sospetto. Anche queste condizioni sono conservate ma non applicate al grafo statico: la loro attivazione richiede data e ora di riferimento e sarà trattata mediante scenari successivi. In questo modo una regola stagionale o oraria non viene trasformata arbitrariamente in una proprietà permanente dell'arco.[^osm-conditional]

I campi definitivi che combinano accesso e direzione per ciascun verso sono:

```text
routing_fwd_status
routing_fwd_code
routing_bwd_status
routing_bwd_code
```

Nei campi `routing_*_code`, 0 indica un verso non disponibile, 1 un accesso ordinario e 2 un accesso locale o riservato al traffico di destinazione. Questa semantica è distinta da quella di `direction_code`, nel quale -1 indica il solo verso opposto alla geometria, 0 la bidirezionalità o il suo trattamento provvisorio e 1 il solo verso della geometria.

Quando accesso e direzione chiudono simultaneamente lo stesso verso, la direzione ha priorità nella costruzione dello status operativo e il risultato è `closed_direction`. La restrizione di accesso non viene persa perché resta disponibile in `access_fwd_class` o `access_bwd_class`. Sono stati inoltre introdotti `routing_fwd_closure_cause` e `routing_bwd_closure_cause`, con dominio `none`, `access`, `direction` e `both`, rendendo l'audit indipendente dall'ordine delle condizioni nel `CASE`.[^audit-direzionalita-finale]

I conteggi operativi finali del layer validato sono:[^audit-direzionalita-finale]

| Verso | Accesso ordinario | Accesso locale | Chiuso per accesso | Chiuso per direzione | Totale |
|---|---:|---:|---:|---:|---:|
| verso della geometria | 110.325 | 500 | 1.640 | 1 | 112.466 |
| verso opposto | 78.961 | 424 | 1.467 | 31.614 | 112.466 |

I controlli finali non hanno evidenziato valori nulli né incoerenze tra status, codici operativi e cause di chiusura.[^audit-direzionalita-finale] Il layer conclusivo della normalizzazione semantica OSM è `osm_rete_light_fvg_20km_routing_statico_validato` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg`; esso costituisce la fonte complementare degli attributi da confrontare e trasferire sulla copia operativa GSFVG.

Il termine «validato» indica che direzione e accessibilità sono state normalizzate semanticamente rispetto ai tag disponibili e sottoposte a controlli automatici di completezza e coerenza su tutte le 112.466 geometrie. Non indica una verifica manuale di ogni record, la costruzione già avvenuta del grafo nodi--archi, il calcolo dei cammini o l'applicazione completa delle `restriction relation`. La validazione visuale successiva sarà concentrata sul backbone strategico e sul sottografo effettivamente utilizzato dopo il primo all-or-nothing.

La produzione di questo layer ha completato la componente semantica OSM del punto 4; la successiva selezione e l'audit topologico di GSFVG ne hanno consolidato il backbone principale. Restano come limiti espliciti: rete riferita alla sola mobilità light; rappresentazione statica priva di data e ora; regole condizionali conservate ma non applicate; restrizioni di svolta e barriere puntuali OSM non ancora integrate; copertura estera Austria--Slovenia incompleta; grafo computazionale diretto e trasferimento degli attributi alla copia operativa GSFVG ancora da completare. Google Maps, Street View e altri servizi esterni potranno essere usati soltanto come benchmark per controlli mirati degli archi ad alto impatto. Il trattamento strutturale del codice 2 è consolidato nella Sezione 4.1.8; restano invece da definire eventuali funzioni di attesa per `alternating` e `reversible`.

### 4.1.7 Normalizzazione delle velocità OSM e separazione dalla velocità di modello

Il punto 5 definisce le velocità operative e l'impedenza base della rete light ed è chiuso a livello metodologico sui layer OSM complementari; l'applicazione definitiva richiede il trasferimento validato degli attributi sulla copia operativa GSFVG. Il primo all-or-nothing minimizzerà un costo temporale statico, più rappresentativo della scelta automobilistica rispetto alla sola distanza: un percorso più lungo su viabilità veloce può richiedere meno tempo di uno più corto su strade locali. Distanza e tempo restano attributi distinti; la distanza è comunque necessaria per autonomia, consumo energetico e successiva formulazione FRLM. Lunghezze ellissoidali e tempi per verso sono stati calcolati e validati; nella formulazione base non sono applicate penalità numeriche ulteriori e i cammini non sono ancora stati calcolati.[^normalizzazione-velocita-osm]

In OSM `maxspeed=*` rappresenta normalmente la velocità massima legale, non una velocità media di percorrenza. I valori sono espressi implicitamente in km/h quando non è indicata un'altra unità; eventuali limiti differenti nei due versi sono rappresentati mediante `maxspeed:forward=*` e `maxspeed:backward=*`, riferiti rispettivamente al verso della way OSM e a quello opposto.[^osm-maxspeed] Valori contestuali come `IT:urban`, `maxspeed:type=*` e `zone:maxspeed=*` descrivono limiti impliciti o la tipologia del limite e richiedono una traduzione documentata nel corrispondente valore numerico.[^osm-maxspeed-type][^osm-default-speed-limits]

Sulle 112.466 geometrie, `speed_fwd_obs_kmh` è valorizzato per 27.484 geometrie e `speed_bwd_obs_kmh` per 27.490. I valori ricavati variano tra 5 e 130 km/h; 93 geometrie presentano valori differenti nei due versi e 4 contengono `maxspeed` condizionale o variabile. Il controllo della normalizzazione ha restituito zero anomalie.[^velocita-modello-base]

La normalizzazione già eseguita ha prodotto i campi:

```text
speed_fwd_obs_kmh
speed_bwd_obs_kmh
```

Il suffisso `obs` significa «ricavato dai dati OSM» e non «misurato sul campo». I valori specifici `maxspeed:forward` e `maxspeed:backward` prevalgono sul `maxspeed` generale per il rispettivo verso. Sono inoltre tradotti i valori contestuali riconosciuti, per esempio `maxspeed=IT:urban` in 50 km/h e `maxspeed:type=IT:zone30` in 30 km/h. Valori multipli o ambigui quali `30;50` e `70,50,30` non sono trasformati arbitrariamente in media, minimo o massimo, ma restano irrisolti e tracciati per il controllo.

Il risultato di questa sola fase descrittiva è il layer:

```text
osm_rete_light_fvg_20km_velocita_osm_normalizzate
```

Il controllo comunicato non ha evidenziato anomalie strutturali; ciò è compatibile con la presenza di valori ambigui esplicitamente conservati come irrisolti. Questa normalizzazione riguarda i limiti OSM e non costituisce ancora il costo di percorrenza.[^normalizzazione-velocita-osm]

Nel successivo passaggio del punto 5 è stata costruita una velocità operativa di modello distinta dai limiti OSM:

```text
speed_class_default_kmh
speed_fwd_model_kmh
speed_bwd_model_kmh
speed_fwd_model_source
speed_bwd_model_source
```

I campi `speed_fwd_model_source` e `speed_bwd_model_source` conservano la provenienza del valore applicato. Come benchmark iniziale sono adottati i valori `speeds.highway` del profilo automobilistico ufficiale di OSRM. Per evitare la dipendenza dal ramo mobile `master`, la sorgente riproducibile è fissata al tag `v26.7.3`, verificato come coerente con la tabella consultata il 5 agosto 2026 su `master`.[^osrm-car-profile]

| Classe OSM `highway` | `speed_class_default_kmh` |
|---|---:|
| `motorway` | 90 |
| `motorway_link` | 45 |
| `trunk` | 85 |
| `trunk_link` | 40 |
| `primary` | 65 |
| `primary_link` | 30 |
| `secondary` | 55 |
| `secondary_link` | 25 |
| `tertiary` | 40 |
| `tertiary_link` | 20 |
| `unclassified` | 25 |
| `residential` | 25 |
| `living_street` | 10 |

Il profilo OSRM assegna inoltre 15 km/h a `service`, ma tale classe resta nel layer ausiliario e non entra automaticamente nella rete light principale. Il suo eventuale impiego come connettore sarà quindi parametrizzato e validato separatamente.

La documentazione OSRM distingue percorso più corto e più rapido, raccomanda che la velocità rappresenti la migliore stima della velocità effettivamente praticata e mostra che, ponendo il rate uguale alla velocità, il routing minimizza il tempo anziché la distanza.[^osrm-profile-docs] I valori della tabella sono pertanto velocità convenzionali di routing: non sono limiti normativi né misure empiriche specifiche del Friuli Venezia Giulia.

La regola operativa adottata e applicata usa il limite OSM come tetto. Per un verso $d\in\{\mathrm{fwd},\mathrm{bwd}\}$:

$$
v_{a,d}^{\mathrm{model}}
=
\begin{cases}
\min\!\left(v_{a,d}^{\mathrm{obs}},v_{h(a)}^{\mathrm{class}}\right), & \text{se }v_{a,d}^{\mathrm{obs}}\text{ è disponibile},\\
v_{h(a)}^{\mathrm{class}}, & \text{altrimenti}.
\end{cases}
$$

Un limite OSM inferiore riduce quindi la velocità operativa, mentre un limite superiore non trasforma automaticamente la velocità legale in velocità media. L'adozione della sola tabella `speeds.highway` come benchmark, combinata con il limite OSM, è un adattamento metodologico del profilo e non una replica integrale di `car.lua`: il profilo ufficiale considera anche superficie, `tracktype`, `smoothness`, limiti impliciti, penalità e altri handler.[^osrm-car-profile] Queste ulteriori componenti non sono state applicate implicitamente.

Il layer risultante è `osm_rete_light_fvg_20km_velocita_modello_base`. Tutte le 112.466 geometrie possiedono `speed_class_default_kmh`, `speed_fwd_model_kmh`, `speed_bwd_model_kmh` e le due sorgenti, senza valori nulli; le velocità di modello sono comprese tra 5 e 90 km/h e 69 geometrie differiscono tra forward e backward. I controlli complessivi hanno restituito zero anomalie.[^velocita-modello-base]

| Sorgente forward | Geometrie |
|---|---:|
| `class_default_missing` | 84.979 |
| `class_operational_cap` | 17.091 |
| `osm_limit_binding` | 10.211 |
| `osm_equals_class` | 182 |
| `class_default_compound` | 3 |
| **Totale** | **112.466** |

| Sorgente backward | Geometrie |
|---|---:|
| `class_default_missing` | 84.973 |
| `class_operational_cap` | 17.088 |
| `osm_limit_binding` | 10.220 |
| `osm_equals_class` | 182 |
| `class_default_compound` | 3 |
| **Totale** | **112.466** |

Per ogni verso percorribile il tempo base è calcolato da lunghezza e velocità di modello:

$$
t_{a,d}^{\mathrm{base}}=\frac{\ell_a}{v_{a,d}^{\mathrm{model}}},
$$

con $\ell_a$ in km, $v_{a,d}^{\mathrm{model}}$ in km/h e $t_{a,d}^{\mathrm{base}}$ in ore. I versi con `routing_*_code=0` sono esclusi dal routing indipendentemente dall'eventuale valore di velocità memorizzato. Le eventuali componenti di attesa per `alternating`, `reversible` o altre condizioni restano separate dal tempo fisico base e appartengono a estensioni successive. Gli archi con codice 2 non ricevono una penalità numerica generica: sono gestiti strutturalmente secondo la Sezione 4.1.8.

La costruzione delle velocità di modello, il calcolo delle lunghezze ellissoidali e dei campi `time_fwd_base_s` e `time_bwd_base_s`, nonché la validazione dei tempi per verso e l'adozione del tempo statico come costo ordinario, chiudono la definizione metodologica del punto 5 sui layer OSM complementari. Il matching e il trasferimento sulla copia operativa GSFVG devono essere documentati e validati prima del routing definitivo. Il confronto con ISTAT/TomTom e con itinerari esterni resta un controllo di robustezza successivo. La velocità di modello non determina la percorribilità: un verso con `routing_fwd_code=0` o `routing_bwd_code=0` resta escluso indipendentemente dal valore di velocità memorizzato.

### 4.1.8 Trattamento degli archi ad accesso locale o `destination`

È consolidata la decisione di non rappresentare i versi con `routing_*_code=2` mediante una penalità numerica generica. Questi archi non sono semplicemente meno convenienti: sono percorribili per esigenze di accesso locale o per raggiungere una destinazione. Una penalità fissa sarebbe inoltre sensibile alla segmentazione OSM e potrebbe essere applicata più volte alla medesima strada soltanto perché suddivisa in più geometrie.

Nella prima implementazione si adotta quindi la seguente regola:

- `code=0`: verso escluso dal grafo;
- `code=1`: verso appartenente alla rete ordinaria, con costo pari al tempo statico di percorrenza;
- `code=2`: verso escluso dalla rete di attraversamento, ma conservato come possibile collegamento iniziale o finale tra una zona e la rete ordinaria.

Gli archi locali saranno gestiti nella costruzione degli accessi zonali $\Gamma_o^{\mathrm L}$, evitando che vengano utilizzati come scorciatoie nei tratti intermedi dei percorsi. Per la rete core, il costo di routing coincide inizialmente con il tempo statico di percorrenza, senza ulteriori penalità non giustificate. Questa scelta è una regola di ammissibilità topologica e di ruolo nel percorso, non una modifica della velocità dell'arco.[^decisione-accesso-locale]

## 4.2 Connessione tra comuni e rete fisica

La presente sezione documenta la **regola canonica degli accessi costruita per la baseline GSFVG**. I passaggi diagnostici che hanno motivato la scelta — in particolare i casi di Sacile e Codroipo — sono documentati nella Sezione 4.4.

> **Stato dopo lo switch OSM.** `Gamma_L_comuni_fvg_v01` resta un prodotto valido, auditato e fondamentale per la provenance del progetto, ma **non è più la sorgente operativa degli accessi per le elaborazioni downstream**. La nuova $\Gamma_o^{L,OSM}$ riutilizza i principi di cardinalità, anchoring e diversificazione, ma li ridefinisce su una rappresentazione strutturale specifica di OSM basata su `structural nodes` e `topological segments`; si veda la §4.4.7.

### Motivazione

**Idea in termini semplici.** La domanda di mobilità è nota a livello di comune, ma le automobili si muovono su una rete formata da nodi e archi stradali. Occorre quindi costruire, per ogni comune, una piccola “porta di ingresso e uscita” verso la rete. L'insieme di queste porte è indicato con $\Gamma_o^{\mathrm L}$. La scelta è importante: un nodo molto vicino al centroide può essere geometricamente conveniente ma, a causa dei sensi di marcia, può essere un pessimo punto attraverso cui rappresentare tutti gli spostamenti del comune.

La matrice di domanda della tesi è definita su zone comunali, mentre il modello di rete stradale GSFVG è rappresentato mediante un grafo diretto:

$$
G=(\mathcal V,\mathcal A).
$$

Un comune non coincide pertanto con un nodo della rete. Prima del calcolo dei shortest path e delle successive assegnazioni di traffico è necessario definire il modo con cui ogni zona comunale viene collegata al grafo.

Per ciascun comune $o$ viene introdotto un insieme di nodi di accesso alla rete light:

$$
\Gamma_o^{\mathrm L}\subseteq\mathcal V.
$$

I nodi appartenenti a $\Gamma_o^{\mathrm L}$ non rappresentano punti fisici di generazione puntuale di tutti gli spostamenti comunali. Essi costituiscono invece una rappresentazione zonale dei principali punti mediante i quali la domanda associata al comune viene immessa nel grafo o estratta da esso.

La costruzione di tali accessi è stata preceduta da una fase diagnostica nella quale ciascun comune era rappresentato dal solo nodo GSFVG più vicino al relativo centroide di popolazione:

$$
\gamma_o^{(1)}
=
\arg\min_{v\in\mathcal V}
d(\mathbf p_o^{\mathrm{pop}},v).
$$

Tale configurazione, equivalente a:

$$
|\Gamma_o^{\mathrm L}|=1,
$$

si è dimostrata utile come benchmark, ma non sufficientemente robusta come specificazione definitiva.

In particolare, l'applicazione delle direzioni stradali validate aveva apparentemente prodotto una perdita di 424 relazioni OD rispetto alla baseline Gate 4. La successiva analisi ha mostrato che tale effetto era interamente attribuibile ai nodi nearest scelti per Sacile e Codroipo e non a errori delle direzioni stradali.

Questo risultato ha evidenziato una proprietà generale del problema: in una rete diretta il nodo geometricamente più vicino a un centroide può trovarsi su una micro-topologia locale che permette l'ingresso ma non l'uscita, o viceversa, e può quindi rappresentare in maniera inadeguata l'accessibilità complessiva di un'intera zona comunale.

La definizione definitiva di $\Gamma_o^{\mathrm L}$ è stata pertanto sviluppata mediante una specifica analisi di sensibilità.

### Giant SCC e ammissibilità direzionale

Nel seguito, **v01** indica il dataset GIS nel quale è stata consolidata la semantica direzionale (`GSFVG_operativo_direzionale_v01`), mentre **v02** indica la versione computazionale successivamente utilizzata per l'analisi degli accessi. La distinzione è utile perché il primo è un prodotto GIS versionato, mentre il secondo è la rappresentazione a nodi e archi su cui vengono eseguite le analisi di grafo.

Il grafo operativo GSFVG v02 contiene:

$$
61.947
$$

nodi computazionali e:

$$
140.213
$$

relazioni dirette nel `DiGraph` utilizzato per le analisi di connettività.

La rete presenta 625 Strongly Connected Components (SCC). La componente fortemente connessa principale, nel seguito indicata come `giant SCC`, contiene:

$$
61.109
$$

nodi.

Una Strongly Connected Component è un insieme massimale di nodi tale che, per ogni coppia $u,v$ appartenente alla componente, esiste sia un percorso diretto:

$$
u\leadsto v
$$

sia:

$$
v\leadsto u.
$$

L'appartenenza alla giant SCC viene quindi utilizzata come criterio di ammissibilità degli accessi comunali, poiché impedisce di associare una zona a nodi localmente isolati o inseriti in componenti direzionali marginali della rete.

Tale criterio deve però essere interpretato correttamente.

Una volta imposto che almeno un accesso di ogni comune appartenga alla medesima giant SCC, la raggiungibilità fra tali accessi è garantita per definizione matematica. La reachability OD non viene pertanto riutilizzata come prova indipendente della qualità della rete dopo l'introduzione di questo filtro. Il gate di reachability del grafo rimane quello eseguito precedentemente mediante accessi diagnostici.

### Inventario iniziale dei candidati

Per ciascuno dei 215 comuni FVG sono stati individuati i 20 nodi computazionali GSFVG più vicini al centroide di popolazione.

Sono stati quindi analizzati:

$$
215\cdot20=4.300
$$

candidati.

Per ciascun candidato sono stati registrati almeno:

- distanza dal centroide;
- grado entrante e uscente;
- WCC e SCC di appartenenza;
- appartenenza alla giant SCC;
- feature GSFVG incidenti;
- classi e denominazioni delle strade incidenti;
- stato direzionale degli archi locali.

Il primo risultato ha mostrato che il criterio nearest-node costituisce una buona approssimazione nella maggioranza dei casi:

$$
210/215
$$

comuni, pari al 97,67%, presentano già il nodo più vicino all'interno della giant SCC.

I cinque casi residui sono:

- Campolongo Tapogliano;
- Porpetto;
- Claut;
- Codroipo;
- Sacile.

Per tutti i 215 comuni è comunque disponibile almeno un candidato appartenente alla giant SCC entro i primi 10 nodi per distanza.

La distanza aggiuntiva necessaria per raggiungere il primo nodo della giant SCC nei cinque casi problematici è risultata:

| Comune | incremento rispetto al nearest |
|---|---:|
| Sacile | 14,3 m |
| Codroipo | 36,4 m |
| Claut | 47,7 m |
| Porpetto | 56,7 m |
| Campolongo Tapogliano | 131,9 m |

Il risultato conferma che il problema non consiste in una generale scarsa connessione fra centroidi e rete, ma in un numero limitato di configurazioni locali sensibili alla direzionalità.

### Sensibilità alla numerosità degli accessi

Sono state successivamente confrontate configurazioni con:

$$
|\Gamma_o^{\mathrm L}|=
1,\;2,\;3,\;5.
$$

La semplice selezione dei $k$ nodi ammissibili più vicini si è tuttavia dimostrata insufficiente.

Nella configurazione con tre accessi nearest (`G3`):

- 109 comuni presentano almeno una coppia di accessi distante meno di 50 m;
- 162 comuni presentano almeno una coppia distante meno di 100 m;
- 204 comuni su 215 presentano almeno due accessi che condividono una medesima feature `ID1` incidente.

La numerosità dei nodi non può quindi essere interpretata direttamente come numerosità degli accessi indipendenti.

Tre vertici distinti del grafo possono infatti appartenere alla stessa strada o alla stessa micro-topologia e non fornire una reale diversificazione dell'associazione zona--rete.

Per questo motivo alla prossimità geometrica è stato affiancato un criterio di **diversità topologica**.

### Appartenenza territoriale

È stato verificato che i candidati selezionati appartengano territorialmente al comune rappresentato.

È stata ammessa una tolleranza di 2 m dal confine amministrativo per assorbire eventuali effetti numerici delle geometrie.

Dei 4.300 candidati analizzati:

$$
4.263
$$

risultano interni o entro 2 m dal proprio comune, mentre 37 sono esterni.

Tuttavia:

- nessuno dei 215 nearest-node è esterno al comune;
- nessuno dei primi nodi giant-SCC è esterno al comune;
- tutti i 215 comuni possiedono candidati giant-SCC interni nei primi 20 nodi.

L'appartenenza al territorio comunale può quindi essere utilizzata come requisito di ammissibilità senza generare zone prive di accesso.

### Diversità topologica degli accessi

Per ogni nodo $v$ si definisce:

$$
I(v)
$$

come l'insieme degli identificativi `ID1` delle feature GSFVG incidenti al nodo.

Due accessi $u$ e $v$ sono considerati topologicamente insufficientemente distinti se:

$$
I(u)\cap I(v)\neq\varnothing.
$$

La selezione degli accessi richiede pertanto:

$$
I(\gamma_{or})
\cap
I(\gamma_{os})
=
\varnothing
\qquad
\forall r\neq s.
$$

Questo vincolo impedisce, per esempio, di rappresentare come due accessi indipendenti due estremi o due nodi appartenenti allo stesso segmento stradale GSFVG.

Sono state inoltre confrontate soglie di separazione spaziale minima pari a:

$$
0,\;50,\;100,\;150,\;200\text{ m}.
$$

La configurazione con separazione minima di 100 m è risultata la più robusta.

Con i vincoli:

$$
v\in\text{giant SCC},
$$

$$
v\in o,
$$

$$
I(\gamma_{or})\cap I(\gamma_{os})=\varnothing,
$$

e:

$$
d(\gamma_{or},\gamma_{os})\ge100\text{ m},
$$

è possibile individuare tre accessi per tutti i:

$$
215/215
$$

comuni.

Una soglia di 150 m rende invece impossibile una tripletta per un comune; con 200 m le eccezioni diventano due. La soglia di 100 m viene pertanto adottata come compromesso fra diversificazione e prossimità.

### Selezione anchored

Per preservare l'interpretabilità della rappresentazione zonale è stato infine adottato un criterio `anchored`.

Si definisce innanzitutto l'insieme dei nodi ammissibili:

$$
E_o
=
\left\{
v\in\mathcal V:
v\in o,
\;
v\in\text{giant SCC}
\right\}.
$$

L'accesso primario del comune viene fissato come:

$$
\gamma_{o1}
=
\arg\min_{v\in E_o}
d(\mathbf p_o^{\mathrm{pop}},v).
$$

Il nodo $\gamma_{o1}$ costituisce quindi sempre il nodo ammissibile più vicino al centroide di popolazione.

Gli accessi secondari:

$$
\gamma_{o2},\gamma_{o3}
$$

vengono individuati mediante ricerca combinatoria tra i primi 20 candidati, imponendo:

$$
d(\gamma_{or},\gamma_{os})
\ge100\text{ m}
$$

e:

$$
I(\gamma_{or})
\cap
I(\gamma_{os})
=
\varnothing
\qquad
\forall r\neq s.
$$

Fra le combinazioni ammissibili viene preferita, in ordine lessicografico, quella che:

1. minimizza la distanza dal centroide dell'accesso più lontano;
2. minimizza la somma delle distanze dei tre accessi;
3. massimizza la separazione minima fra gli accessi.

La configurazione anchored è risultata fattibile per:

$$
215/215
$$

comuni.

La diagnostica spaziale risultante è:

| indicatore | valore |
|---|---:|
| mediana dell'extra-distanza del nodo più lontano | 126,7 m |
| 95° percentile dell'extra-distanza | 394,6 m |
| massimo dell'extra-distanza | 711,5 m |
| mediana della separazione minima fra accessi | 149,3 m |
| separazione minima osservata | 100,5 m |
| massimo rank originario utilizzato nei top-20 | 19 |

La cardinalità strutturale proposta viene pertanto fissata a:

$$
\boxed{
|\Gamma_o^{\mathrm L}|=3
}
$$

con un accesso primario ancorato al nodo ammissibile più prossimo e due accessi secondari spazialmente e topologicamente distinti.

### Pesi degli accessi

Poiché i tre accessi non devono necessariamente rappresentare quote uguali della domanda comunale, a ciascun nodo:

$$
u\in\Gamma_o^{\mathrm L}
$$

viene associato un peso:

$$
\lambda_{ou}^{\mathrm L}\in[0,1]
$$

con:

$$
\sum_{u\in\Gamma_o^{\mathrm L}}
\lambda_{ou}^{\mathrm L}
=
1.
$$

Il peso rappresenta la quota della domanda della zona $o$ immessa o estratta dal grafo attraverso l'accesso $u$.

È stata valutata una funzione di decadimento esponenziale basata sulla distanza relativa rispetto all'accesso primario:

$$
\Delta d_{ou}
=
d_{ou}-d_{o1},
$$

$$
\lambda_{ou}^{\mathrm L}
=
\frac{
\exp\left(-\Delta d_{ou}/\tau\right)
}{
\sum_{v\in\Gamma_o^{\mathrm L}}
\exp\left(-\Delta d_{ov}/\tau\right)
}.
$$

L'utilizzo della distanza relativa, anziché assoluta, evita che differenze dovute esclusivamente alla posizione del centroide rispetto alla rete modifichino artificialmente la concentrazione dei pesi.

Sono stati confrontati:

$$
\tau=
50,\;100,\;200,\;300,\;500\text{ m},
$$

oltre alla configurazione a pesi uniformi.

Valori molto bassi di $\tau$ concentrano eccessivamente il traffico sull'accesso primario e tendono quindi a ricondurre il modello verso il caso $|\Gamma_o^{\mathrm L}|=1$.

Con $\tau=100$ m, ad esempio, 163 comuni su 215 assegnano più del 50% del peso al nodo primario e 66 comuni assegnano meno del 10% al terzo accesso.

Al contrario, $\tau=500$ m produce una distribuzione molto vicina ai pesi uniformi e riduce sensibilmente il ruolo della prossimità.

La configurazione:

$$
\tau=300\text{ m}
$$

costituisce allo stato corrente il compromesso preferito.

Per essa si ottengono:

| indicatore | valore |
|---|---:|
| peso primario mediano | 0,417 |
| 95° percentile del peso primario | 0,617 |
| massimo peso primario | 0,793 |
| peso terziario mediano | 0,268 |
| 5° percentile del peso terziario | 0,166 |
| comuni con peso primario > 0,50 | 33/215 |
| comuni con peso primario > 0,70 | 3/215 |
| comuni con peso primario > 0,90 | 0/215 |
| comuni con peso terziario < 0,10 | 2/215 |

L'indice:

$$
N_o^{\mathrm{eff}}
=
\frac{1}{
\sum_{u\in\Gamma_o^{\mathrm L}}
(\lambda_{ou}^{\mathrm L})^2
}
$$

può essere interpretato come numero effettivo di accessi utilizzati.

Per $\tau=300$ m la mediana regionale è:

$$
N_o^{\mathrm{eff}}=2,894,
$$

indicando che la formulazione conserva effettivamente il carattere multi-accesso senza assegnare quote identiche ai tre nodi.

### Materializzazione canonica e chiusura del gate degli accessi comunali light

L'analisi di sensibilità descritta nei paragrafi precedenti ha consentito di trasformare la formulazione inizialmente parametrica degli accessi comunali in una regola operativa unica e riproducibile.

Per ciascun comune interno FVG $o\in\mathcal M$, l'insieme degli accessi light viene definitivamente fissato a:

$$
\boxed{
|\Gamma_o^{\mathrm L}|=3
}
$$

con:

$$
\Gamma_o^{\mathrm L}
=
\{
\gamma_{o1},
\gamma_{o2},
\gamma_{o3}
\}.
$$

I tre nodi non rappresentano tre stazioni di ricarica e non sono candidati infrastrutturali in senso stretto. Essi costituiscono i punti attraverso i quali la zona comunale viene collegata matematicamente alla rete stradale fisica GSFVG per la costruzione dei percorsi e la successiva assegnazione della domanda.

In altri termini:

- il comune rimane una zona OD;
- il centroide di popolazione rimane il riferimento territoriale della zona;
- i nodi $\Gamma_o^{\mathrm L}$ appartengono al grafo stradale;
- gli accessi non modificano la geometria del GSFVG;
- gli accessi non coincidono automaticamente con futuri siti di ricarica.

#### Regola definitiva di ammissibilità

Per ogni comune viene inizialmente considerato un insieme esplorativo costituito dai 20 nodi computazionali GSFVG più vicini al centroide di popolazione.

Un nodo $u$ è considerato ammissibile come accesso light del comune $o$ se soddisfa contemporaneamente:

$$
u\in\mathcal V_{\mathrm L},
$$

$$
u\in o,
$$

e:

$$
u\in\mathrm{SCC}_{\mathrm{giant}}.
$$

L'appartenenza territoriale è verificata rispetto al poligono comunale, ammettendo una tolleranza geometrica di 2 m per evitare che minime discrepanze numeriche lungo i confini amministrativi producano classificazioni artificialmente esterne.

L'appartenenza alla giant SCC garantisce invece che il nodo faccia parte della principale componente fortemente connessa del grafo diretto.

Tale requisito deve essere interpretato esclusivamente come criterio di ammissibilità degli accessi.

Infatti, per definizione di Strongly Connected Component, se due nodi appartengono alla stessa SCC allora esiste un percorso diretto dal primo al secondo e anche nel verso opposto. Di conseguenza, una volta imposto che tutti gli accessi comunali appartengano alla medesima giant SCC, la raggiungibilità reciproca fra i comuni non costituisce più una verifica indipendente della qualità del grafo.

Il gate di reachability utilizzato per validare la rete rimane pertanto quello svolto precedentemente; la reachability successiva alla costruzione di $\Gamma_o^{\mathrm L}$ è una conseguenza della regola di accesso e non deve essere utilizzata come prova circolare della correttezza della rete.

#### Accesso primario

La selezione viene effettuata mediante una regola `anchored`.

L'accesso primario:

$$
\gamma_{o1}
$$

è sempre il nodo ammissibile più vicino al centroide di popolazione:

$$
\gamma_{o1}
=
\arg\min_{u\in E_o}
d(\mathbf p_o^{\mathrm{pop}},u),
$$

dove:

$$
E_o
=
\left\{
u\in\mathcal V_{\mathrm L}:
u\in o,\;
u\in\mathrm{SCC}_{\mathrm{giant}}
\right\}.
$$

La scelta anchored conserva quindi un riferimento principale immediatamente interpretabile e impedisce che la ricerca della migliore tripletta elimini il nodo ammissibile geometricamente più vicino soltanto per migliorare la disposizione degli accessi secondari.

Nel dataset definitivo tale proprietà è stata verificata per:

$$
215/215
$$

comuni.

#### Accessi secondari e diversità topologica

Gli accessi:

$$
\gamma_{o2},
\gamma_{o3}
$$

vengono selezionati fra i candidati rimanenti mediante ricerca combinatoria.

La scelta non è effettuata semplicemente prendendo il secondo e il terzo nodo più vicini, perché le analisi preliminari hanno mostrato che più nodi geometricamente distinti possono appartenere alla stessa micro-topologia stradale e risultare quindi ridondanti.

Per ogni nodo $u$ si definisce:

$$
I(u)
$$

come l'insieme degli identificativi `ID1` delle feature GSFVG incidenti sul nodo.

Per ogni coppia di accessi dello stesso comune viene imposto:

$$
I(\gamma_{or})
\cap
I(\gamma_{os})
=
\varnothing
\qquad
\forall r\neq s.
$$

Due accessi non possono quindi condividere alcuna feature GSFVG incidente.

Questo vincolo evita, per esempio, che due estremi della medesima feature stradale vengano interpretati come due accessi zonali indipendenti.

Viene inoltre imposto un requisito di separazione geometrica:

$$
d(\gamma_{or},\gamma_{os})
\geq
100\text{ m}
\qquad
\forall r\neq s.
$$

La soglia di 100 m deriva dall'analisi di sensibilità effettuata sulle alternative 50, 100, 150 e 200 m.

La soglia di 100 m consente una tripletta ammissibile per tutti i 215 comuni, mentre soglie superiori iniziano a produrre eccezioni. Essa viene pertanto adottata come compromesso empirico fra prossimità al centroide e diversificazione spaziale degli accessi.

Fra le coppie di accessi effettivamente materializzate la minima separazione osservata è:

$$
100{,}487\text{ m},
$$

quindi nessuna coppia viola il vincolo.

Sono inoltre risultate:

$$
0
$$

coppie con `ID1` incidente condiviso.

Fra tutte le combinazioni ammissibili dei due accessi secondari viene scelta, mantenendo fisso $\gamma_{o1}$, quella che in ordine lessicografico:

1. minimizza la distanza dal centroide dell'accesso più lontano;
2. minimizza la somma delle distanze dei tre accessi;
3. massimizza la separazione minima fra gli accessi;
4. utilizza infine l'identificativo del nodo come criterio deterministico di tie-break.

La regola garantisce quindi riproducibilità della selezione.

#### Pesi di accesso definitivi

I tre nodi non rappresentano quote necessariamente uguali della domanda comunale.

A ciascun accesso:

$$
u\in\Gamma_o^{\mathrm L}
$$

viene assegnato un peso:

$$
\lambda_{ou}^{\mathrm L}\in(0,1)
$$

con:

$$
\sum_{u\in\Gamma_o^{\mathrm L}}
\lambda_{ou}^{\mathrm L}
=
1.
$$

Il parametro:

$$
\lambda_{ou}^{\mathrm L}
$$

rappresenta la quota della domanda light associata alla zona $o$ che viene immessa o estratta dalla rete attraverso il nodo $u$.

I pesi sono determinati mediante decadimento esponenziale della distanza relativa rispetto all'accesso primario.

Si definisce:

$$
\Delta d_{ou}
=
d_{ou}-d_{o1},
$$

dove:

- $d_{ou}$ è la distanza euclidea tra il centroide di popolazione del comune $o$ e l'accesso $u$;
- $d_{o1}$ è la distanza del primary access.

La funzione adottata è:

$$
\lambda_{ou}^{\mathrm L}
=
\frac{
\exp\left(-\Delta d_{ou}/\tau\right)
}{
\displaystyle
\sum_{v\in\Gamma_o^{\mathrm L}}
\exp\left(-\Delta d_{ov}/\tau\right)
}.
$$

L'analisi di sensibilità ha confrontato:

$$
\tau
=
50,\;100,\;200,\;300,\;500\text{ m},
$$

oltre alla configurazione con pesi uniformi.

La configurazione definitivamente adottata per la versione corrente è:

$$
\boxed{
\tau=300\text{ m}
}
$$

e viene identificata operativamente come:

`EXP_REL_300`.

Valori inferiori concentravano eccessivamente la domanda sull'accesso primario, riducendo di fatto il vantaggio della rappresentazione multi-accesso; valori superiori tendevano invece ad avvicinare la distribuzione ai pesi uniformi.

Con $\tau=300$ m la distribuzione finale dei pesi presenta:

| accesso | mediana $\lambda$ | p05 | p95 |
|---|---:|---:|---:|
| primary, ordine 1 | 0,417 | 0,357 | 0,617 |
| secondary, ordine 2 | 0,312 | 0,207 | 0,361 |
| secondary, ordine 3 | 0,268 | 0,166 | 0,315 |

Per misurare quanto la distribuzione mantenga effettivamente attivi più accessi è stato inoltre utilizzato il numero effettivo:

$$
N_o^{\mathrm{eff}}
=
\frac{
1
}{
\displaystyle
\sum_{u\in\Gamma_o^{\mathrm L}}
\left(\lambda_{ou}^{\mathrm L}\right)^2
}.
$$

Si ottengono:

$$
\operatorname{median}
\left(
N_o^{\mathrm{eff}}
\right)
=
2{,}894,
$$

$$
p05
\left(
N_o^{\mathrm{eff}}
\right)
=
2{,}203,
$$

e:

$$
\min
\left(
N_o^{\mathrm{eff}}
\right)
=
1{,}535.
$$

La configurazione mantiene quindi, nella maggioranza dei comuni, un comportamento effettivamente multi-accesso pur attribuendo maggiore importanza ai nodi più prossimi al riferimento territoriale.

#### Dataset canonico degli accessi comunali light

La regola è stata infine materializzata in un dataset persistente e versionato.

Il prodotto canonico è:

`C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_light\Gamma_L_comuni_fvg_v01.gpkg`

con layer:

`Gamma_L_comuni_fvg_v01`.

È disponibile anche il gemello tabellare:

`C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_light\Gamma_L_comuni_fvg_v01.csv`

e il manifest di provenienza:

`C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_light\Gamma_L_comuni_fvg_v01_manifest.json`.

Il dataset contiene:

$$
215
$$

comuni,

$$
3
$$

accessi per comune e quindi:

$$
\boxed{
645
}
$$

record complessivi.

Il GeoPackage contiene inoltre la tabella:

`Gamma_L_metadata_v01`

con 18 record di metadata e provenance.

Il dataset deriva dalla baseline stradale:

`GSFVG_operativo_direzionale_v02.gpkg`

con SHA-256:

`5afb71c500c2ff2cd779cee1601fb42354eadcf754ddc83ac2fc5fc481d7c02e`.

La provenienza degli input intermedi utilizzati per la materializzazione è registrata tramite checksum:

- inventario candidati: `33812fa0bd8a8c329babfa9a18ac6619b8746aebc1ecf00e26331529b3ce2d60`;
- selezione anchored: `d227e62fc10b7b65ae7a1e89f75d9af62b70d9d19ed2a543705f7718e592af78`;
- sensitivity dei pesi: `a89f3b81061ae85e0d806a9ab24a3e5432d17f83c22791f02cd706fdff2b1a84`.

Gli output definitivi hanno checksum:

- GeoPackage `Gamma_L_comuni_fvg_v01.gpkg`:

`425dab4a127aeca2169911763f48238275d1dffe553d857a44c0fda773406179`;

- CSV `Gamma_L_comuni_fvg_v01.csv`:

`1a6d4d8075f84a9c98a39b4bde072ae15575a9a8dc5c02b39420fdb6cdd16386`.

Il dataset canonico contiene 42 campi e conserva, oltre alle chiavi comunali e nodali, le informazioni necessarie alla tracciabilità della selezione:

- `PRO_COM` e `COMUNE`;
- `access_order` e `access_role`;
- `node_id`;
- coordinate del nodo;
- distanza dal centroide;
- distanza relativa dal primary;
- `lambda_L`;
- parametro `tau_m`;
- rank del candidato nell'inventario originario;
- in-degree e out-degree del nodo;
- identificativi delle feature incidenti;
- classi e denominazioni stradali incidenti;
- WCC e SCC;
- appartenenza alla giant SCC;
- appartenenza al comune;
- regola di selezione;
- regola di ponderazione;
- versione e SHA-256 del grafo sorgente.

Il GeoPackage degli accessi rimane un prodotto distinto dal GeoPackage stradale. Questa separazione è deliberata: il primo descrive la relazione fra zone OD e nodi della rete, mentre il secondo descrive la rete stradale fisica.

#### Audit finale del dataset

La materializzazione è stata sottoposta a un hard audit indipendente dal semplice completamento dello script.

Sono stati verificati:

- 215 comuni distinti;
- 645 record;
- esattamente 3 accessi per ciascun comune;
- ordini di accesso $1,2,3$ completi per tutti i comuni;
- nessun `node_id` duplicato all'interno dello stesso comune;
- primary access coincidente con il nearest-node ammissibile per 215/215 comuni;
- 645/645 accessi appartenenti alla giant SCC;
- 645/645 accessi appartenenti al proprio comune;
- 0 coppie con separazione inferiore a 100 m;
- separazione minima osservata pari a 100,487 m;
- 0 coppie con `ID1` incidente condiviso;
- tutti i pesi strettamente positivi;
- somma dei pesi pari a 1 per ciascuno dei 215 comuni;
- massimo errore numerico sulla somma dei pesi pari a $3{,}331\times10^{-16}$;
- massimo errore sulla ricostruzione di $\Delta d$ pari a $5{,}684\times10^{-14}$ m;
- massimo errore rispetto alla formula teorica di $\lambda$ pari a $1{,}110\times10^{-16}$;
- rilettura completa del GeoPackage con 645 record e 215 comuni;
- presenza e rilettura dei 18 record della tabella metadata;
- `PRAGMA integrity_check = ok`.

La distribuzione finale delle distanze dal centroide è:

| indicatore | valore |
|---|---:|
| mediana distanza primary | 130,0 m |
| mediana distanza di tutti gli accessi | 204,3 m |
| p95 distanza di tutti gli accessi | 658,4 m |
| massima distanza osservata | 1.075,1 m |

I valori più elevati non vengono interpretati come errori automatici: nei comuni caratterizzati da insediamenti dispersi, morfologia montana o centroide relativamente distante dalla rete rappresentata, anche il nodo ammissibile più vicino può essere fisicamente distante dal punto rappresentativo. Per questo motivo non è stato imposto un raggio assoluto rigido dal centroide, che avrebbe prodotto esclusioni arbitrarie fra comuni territorialmente differenti.

#### Decisione del gate

A seguito della sensitivity e dell'audit del prodotto persistente, il gate relativo agli accessi comunali light viene considerato definitivamente chiuso per la versione corrente del modello.

Sono congelate le seguenti decisioni:

$$
\boxed{
|\Gamma_o^{\mathrm L}|=3
}
$$

per tutti i 215 comuni;

$$
\boxed{
d(\gamma_{or},\gamma_{os})\geq100\text{ m}
}
$$

per ogni coppia di accessi;

$$
\boxed{
I(\gamma_{or})\cap I(\gamma_{os})=\varnothing
}
$$

per ogni coppia distinta;

e:

$$
\boxed{
\tau=300\text{ m}
}
$$

per la ponderazione `EXP_REL_300`.

Il riferimento operativo canonico da utilizzare nelle elaborazioni successive è quindi:

`Gamma_L_comuni_fvg_v01`.

Le precedenti Run 1--4 e i relativi prodotti contenuti in `03_output_temporanei` rimangono parte della documentazione diagnostica e della provenance, ma non devono più essere utilizzati come sorgente operativa quando è disponibile il dataset canonico.

#### Conseguenze per il seguito del modello

La chiusura di $\Gamma_o^{\mathrm L}$ completa il collegamento fra il livello zonale comunale e il livello fisico della rete stradale.

Da questo punto in avanti gli input canonici del ramo light sono almeno:

1. il grafo operativo direzionale:

`GSFVG_operativo_direzionale_v02.gpkg`;

2. gli accessi comunali:

`Gamma_L_comuni_fvg_v01.gpkg`.

Il prossimo gate è il primo audit sistematico dei shortest path sulla rete.

La prima implementazione utilizzerà come impedenza esclusivamente la lunghezza geometrica degli archi, mantenendo separato il problema topologico e metrico da quello successivo delle velocità e dei tempi di percorrenza.

Per i 215 comuni interni devono essere considerate:

$$
215\cdot214
=
46.010
$$

relazioni OD ordinate con origine e destinazione differenti.

L'audit dovrà produrre almeno:

- percorsi minimi fra gli accessi comunali;
- distanze di rete;
- identificazione degli accessi di origine e destinazione coinvolti;
- sequenza ordinata degli archi e degli `ID1` percorsi;
- insieme degli archi appartenenti ad almeno uno shortest path, indicato con $\mathcal A^{\mathrm{SP}}$;
- numero di relazioni OD che utilizzano ciascun arco, indicato con $N_a^{\mathrm{OD}}$;
- confronto fra i versi $o\rightarrow d$ e $d\rightarrow o$;
- individuazione di percorsi anomali, deviazioni eccessive o altre configurazioni che possano richiedere una revisione mirata del grafo.

Per questa fase dovrà essere preservata l'identità degli archi paralleli. Il `DiGraph` collassato utilizzato nelle precedenti analisi di reachability era sufficiente per stabilire l'esistenza o meno di un percorso, ma non è sufficiente per attribuire senza ambiguità un shortest path a specifiche feature GSFVG quando più archi collegano la stessa coppia ordinata di nodi.

L'audit metrico dovrà quindi utilizzare un `MultiDiGraph`, oppure un'eventuale trasformazione equivalente che selezioni deterministicamente l'arco di costo minimo conservandone esplicitamente `ID1`, indice della parte e metadati di provenienza.

I pesi $\lambda_{ou}^{\mathrm L}$ descrivono la ripartizione della domanda fra gli accessi comunali e non devono essere confusi con il costo degli archi stradali. La modalità esatta con cui i tre accessi di origine e i tre accessi di destinazione vengono combinati nella costruzione dei percorsi e nella successiva assegnazione deve essere mantenuta esplicita e verificabile nel gate shortest-path.

Non vengono ancora introdotti:

- velocità operative;
- tempi di percorrenza;
- costi generalizzati;
- calibrazione rispetto ai tempi ISTAT/TomTom;
- flussi pendolari o non pendolari;
- assegnazione all-or-nothing;
- calibrazione rispetto ai conteggi ANAS.

Tali passaggi appartengono ai gate successivi.

**Stato finale del gate accessi comunali GSFVG:** CHIUSO — **baseline storica**.

**Dataset canonico storico:** `Gamma_L_comuni_fvg_v01`.

**Stato operativo corrente:** il backbone è OSM frozen. La nuova $\Gamma_o^{L,OSM}$ è stata materializzata e validata: E0, E1, E2 ed E3 sono `PASS`, `Gamma_OSM` ed `EXP_REL_300` sono `FROZEN`. La specificazione corrente è nella §4.4.7.

## 4.3 Ruoli distinti di QGIS e Python

**QGIS** è lo strumento principale per preparare e validare le zone, creare e verificare i punti rappresentativi, preparare il grafo stradale, associare i centroidi al grafo, controllare le postazioni ANAS, svolgere analisi spaziali, visualizzare e validare i risultati ed esportare layer e tabelle pulite.

**Python** è lo strumento principale per costruire le matrici, calcolare la matrice dei costi, gestire matrici sparse, calcolare shortest path e assegnazioni, calibrare i parametri della domanda, analizzare i residui, eseguire un'eventuale matrix estimation ed esportare i flussi di percorso per il FRLM.

Questa ripartizione è funzionale: non implica che le elaborazioni elencate siano già state eseguite.

## 4.4 Strategia di validazione del grafo

La strategia di validazione è stata applicata dapprima al GSFVG e ha successivamente condotto, attraverso SP2 e il Gate OSM shadow, alla decisione `SWITCH_OSM`. **Il backbone operativo corrente è quindi OSM**; GSFVG resta una baseline storica e un benchmark istituzionale. Il principio generale non cambia: non si richiede una certificazione manuale esaustiva dell'intera cartografia, ma una validazione progressiva e proporzionata all'impatto sul routing, con riaperture locali soltanto quando un'anomalia può alterare materialmente OD, assegnazione o FRLM. Le §§4.4.3--4.4.5 documentano il percorso di validazione che ha portato alla decisione; le §§4.4.6--4.4.7 descrivono lo switch e il nuovo stato operativo OSM.

### 4.4.1 Validazione minima prima del routing

Prima della prima assegnazione devono essere verificati almeno:

- autostrade, carreggiate separate e relative rampe;
- rotatorie, svincoli e grandi direttrici statali e regionali;
- connettività generale e componenti isolate;
- futuri gateway e collegamenti alle sezioni ANAS;
- nodi di accesso comunali;
- restrizioni di accesso o direzione palesemente incompatibili.

Lo scopo non è certificare ogni strada, ma impedire che errori macroscopici determinino già i cammini minimi e quindi il sottoinsieme di rete che sarà considerato rilevante.

### 4.4.2 Validazione guidata dall'assegnazione

Dopo l'all-or-nothing preliminare si estrae il sottografo effettivamente utilizzato:

$$
\mathcal A^{\mathrm{used}}
=
\bigcup_{(o,d):\,T_{od}^{\mathrm L}>0} q_{od}^{*},
$$

dove $q_{od}^{*}$ è il cammino minimo scelto per la relazione $(o,d)$. Per ogni arco di $\mathcal A^{\mathrm{used}}$ si calcolano almeno numero di percorsi OD incidenti, flusso totale assegnato, associazione a sezioni ANAS, appartenenza a gateway o accessi dei candidati e presenza di attributi direzionali dubbi.

I controlli GIS e i confronti esterni manuali con Google Maps e Street View sono quindi prioritizzati su:

- archi dubbi utilizzati nei corridoi principali;
- rampe e svincoli effettivamente percorsi;
- archi con elevato numero di OD o flusso assegnato;
- itinerari anomali e differenze rilevanti rispetto a `KM_TOT`, `TEP_TOT` e `TTP_TOT`;
- sezioni ANAS, gateway e accessibilità dei candidati alla ricarica.

Google Maps e Street View costituiscono esclusivamente riferimenti esterni per ispezioni manuali mirate e non sostituiscono né diventano sorgente del grafo. Ogni correzione deve essere motivata e registrata nella copia operativa derivata da GSFVG; il file originale `GSFVG_IRDAT.shp` e l'estratto OSM congelato restano immutati. Dopo le correzioni si ripetono routing e assegnazione; il ciclo termina quando gli archi critici e i principali indicatori si stabilizzano entro criteri di tolleranza da documentare.


### 4.4.3 Validazione funzionale del grafo mediante connettività e cammini minimi OD

La validazione del grafo stradale non viene interpretata come necessità di certificare esaustivamente ogni arco della rete regionale. L'obiettivo operativo è garantire che la porzione di rete in grado di influenzare i percorsi Origine-Destinazione utilizzati dal modello presenti connettività, direzionalità e costi sufficientemente affidabili.

Per i 215 comuni del Friuli Venezia Giulia esistono, considerando le relazioni direzionali e omettendo gli spostamenti intracomunali,

$$
215(215-1)=46.010
$$

possibili coppie OD ordinate. Tali relazioni non vengono ridotte a 23.005 coppie non ordinate, poiché la rete operativa è diretta: sensi unici, rampe, restrizioni e costi eventualmente differenti per verso possono produrre cammini differenti o perfino condizioni di raggiungibilità differenti tra $(o\rightarrow d)$ e $(d\rightarrow o)$.

La validazione funzionale viene articolata in due procedure complementari. Nel passaggio operativo corrente viene inoltre introdotta, come specificato nella Sezione 4.4.4, una **baseline conservativa** $G^{\mathrm{base}}$ nella quale gli archi P3/P4 non ancora verificati mantengono temporaneamente `TRIM_USAGE=0`.

#### A. Audit conservativo di connettività

Si distinguono quindi:

$$
G^{\mathrm{full}},
$$

che rappresenta il grafo operativo completo prima di eventuali esclusioni diagnostiche;

$$
G^{\mathrm{base}},
$$

che costituisce lo scenario di lavoro corrente e conserva P3/P4 non verificati come bidirezionali; e, quando utile per isolare criticità,

$$
G^{\mathrm{safe}},
$$

ottenuto da $G^{\mathrm{base}}$ escludendo temporaneamente specifici archi la cui percorribilità o direzionalità richieda un test più severo.

Le esclusioni in $G^{\mathrm{safe}}$ sono esclusivamente diagnostiche e non implicano la cancellazione degli archi dal dataset sorgente. La baseline $G^{\mathrm{base}}$ resta invece lo scenario primario su cui misurare il comportamento funzionale della rete.

Sul grafo $G^{\mathrm{safe}}$ vengono analizzati:

- componenti debolmente connesse;
- componenti fortemente connesse;
- raggiungibilità direzionale tra gli accessi associati ai comuni;
- numero e quota delle coppie OD che rimangono percorribili;
- eventuali aree o corridoi che risultano separati dalla rete principale.

Per ogni arco problematico escluso $a$ può essere valutato il beneficio derivante dal suo reinserimento:

$$
\Delta R_a
=
R(G^{\mathrm{safe}}+a)-R(G^{\mathrm{safe}}),
$$

dove $R(G)$ rappresenta il numero di relazioni OD direzionali raggiungibili sul grafo $G$.

Quando sono disponibili flussi OD preliminari, la stessa misura può essere ponderata rispetto alla domanda:

$$
\Delta F_a
=
\sum_{(o,d)}
T_{od}^{\mathrm L}
\left[
I_{od}(G^{\mathrm{safe}}+a)
-
I_{od}(G^{\mathrm{safe}})
\right],
$$

dove $I_{od}(G)=1$ se la relazione $(o\rightarrow d)$ è raggiungibile e $0$ altrimenti.

Questa analisi consente di individuare gli archi problematici con maggiore rilevanza topologica e di assegnare priorità alla loro verifica. Un arco la cui correzione ristabilisce centinaia o migliaia di relazioni OD assume quindi priorità superiore rispetto a un elemento locale che non modifica la raggiungibilità delle relazioni considerate dal modello.

#### B. Audit dei cammini minimi e confronto con riferimenti esterni

In parallelo viene calcolato, per ogni coppia OD direzionale considerata, il cammino minimo sulla rete operativa:

$$
q_{od}^{*}
=
\operatorname*{arg\,min}_{q\in\mathcal Q_{od}}
c_q,
$$

registrando almeno:

- raggiungibilità;
- tempo di percorrenza;
- distanza;
- successione degli archi;
- eventuale differenza tra percorso di andata e di ritorno.

L'unione degli archi utilizzati dai cammini minimi definisce un sottografo funzionale:

$$
\mathcal A^{\mathrm{SP}}
=
\bigcup_{o\neq d} q_{od}^{*}.
$$

Il rapporto

$$
\frac{|\mathcal A^{\mathrm{SP}}|}{|\mathcal A|}
$$

misura direttamente quale quota della rete fisica risulta effettivamente necessaria per i cammini intercomunali considerati.

Per ciascun arco $a$ viene inoltre calcolato il numero di relazioni OD i cui cammini minimi lo attraversano:

$$
N_a^{\mathrm{OD}}
=
\sum_{o\neq d}
\mathbf 1
\left(
a\in q_{od}^{*}
\right).
$$

Quando sono disponibili i flussi OD, viene calcolata anche la corrispondente misura ponderata:

$$
F_a^{\mathrm{OD}}
=
\sum_{o\neq d}
T_{od}^{\mathrm L}
\mathbf 1
\left(
a\in q_{od}^{*}
\right).
$$

Tali indicatori permettono di distinguere una **backbone funzionale** della rete dagli archi con rilevanza trascurabile per l'assegnazione regionale.

I costi ottenuti dal grafo vengono quindi confrontati, dove disponibili e semanticamente compatibili, con i riferimenti esterni presenti nei dati ISTAT/TomTom, in particolare con i campi relativi a tempi e distanze già associati alle relazioni OD.

Il confronto non è utilizzato come vincolo rigido né come prova automatica di errore del grafo. Viene impiegato come strumento diagnostico per individuare:

- OD prive di percorso;
- deviazioni macroscopiche rispetto alle distanze di riferimento;
- tempi di percorrenza anormalmente elevati;
- forti asimmetrie tra andata e ritorno non spiegate dalla struttura della rete;
- percorsi che evitano arterie evidentemente più plausibili;
- itinerari che presentano deviazioni concentrate in prossimità di rampe, svincoli o collegamenti direzionali dubbi.

Per ciascuna relazione anomala viene conservato l'insieme degli archi appartenenti al relativo cammino minimo. Si definisce quindi, per ogni arco:

$$
H_a
=
\sum_{(o,d)\in\mathcal O^{\mathrm{anom}}}
w_{od}
\mathbf 1
\left(
a\in q_{od}^{*}
\right),
$$

dove $\mathcal O^{\mathrm{anom}}$ rappresenta l'insieme delle OD classificate come anomale e $w_{od}$ può incorporare l'entità dell'anomalia o il volume di traffico della relazione.

Un arco attraversato da numerosi cammini anomali viene pertanto identificato automaticamente come possibile **hotspot topologico o direzionale**.

#### C. Prioritizzazione degli archi da verificare

Gli archi ancora dubbi vengono ordinati secondo una priorità funzionale costruita combinando almeno:

1. contributo alla raggiungibilità della rete;
2. numero di cammini minimi OD che utilizzano l'arco;
3. eventuale flusso OD complessivamente associato;
4. frequenza dell'arco nei percorsi classificati come anomali;
5. appartenenza ad arterie strategiche, svincoli, rampe, gateway o collegamenti con sezioni ANAS.

In forma generale:

$$
P_a
=
f
\left(
\Delta R_a,
\Delta F_a,
N_a^{\mathrm{OD}},
F_a^{\mathrm{OD}},
H_a,
S_a
\right),
$$

dove $S_a$ rappresenta un indicatore di rilevanza strategica.

La funzione $P_a$ non deve necessariamente essere trasformata fin dall'inizio in un indice numerico unico. Nella prima applicazione può essere sufficiente una classificazione gerarchica fondata sui singoli indicatori.

Gli archi con priorità maggiore vengono sottoposti a verifica mirata attraverso:

- attributi ufficiali GSFVG;
- confronto geometrico e direzionale con OSM;
- Overture Maps come ulteriore controllo;
- Overpass esclusivamente come supporto diagnostico;
- verifica cartografica manuale quando necessaria.

L'esito della verifica viene incorporato esclusivamente nella copia operativa versionata del grafo.

#### D. Logica iterativa e criterio di arresto

La procedura segue un ciclo:

$$
\text{grafo}
\rightarrow
\text{reachability}
\rightarrow
\text{shortest path}
\rightarrow
\text{anomalie}
\rightarrow
\text{archi prioritari}
\rightarrow
\text{validazione}
\rightarrow
\text{nuovo grafo}.
$$

Dopo ogni gruppo significativo di correzioni vengono ricalcolati gli indicatori precedenti.

La validazione funzionale può essere considerata sufficientemente stabile quando:

- non rimangono relazioni OD interne rilevanti irraggiungibili senza una spiegazione topologica plausibile;
- non emergono rotture direzionali sistematiche;
- le anomalie macroscopiche di tempo e distanza sono state ricondotte a cause note o corrette;
- gli archi ad alta frequenza nei cammini OD non presentano criticità direzionali irrisolte;
- ulteriori correzioni locali producono variazioni trascurabili sui cammini e sugli indicatori aggregati.

Il criterio metodologico finale non è quindi la completa eliminazione di ogni incertezza locale, ma la dimostrazione che le incertezze residue non alterano materialmente raggiungibilità, cammini minimi e successiva assegnazione dei flussi.

#### Roadmap operativa della validazione funzionale

1. **Costruire e versionare la baseline conservativa $G^{\mathrm{base}}$**, applicando le decisioni consolidate per `TRIM_USAGE=1/2`, per i 616 NULL, per P1 e per i 127 casi P2 revisionati; gli archi P3/P4 non ancora verificati mantengono temporaneamente `TRIM_USAGE=0`. Una variante $G^{\mathrm{safe}}$ con esclusioni mirate può essere costruita soltanto come diagnostica aggiuntiva.
2. **Eseguire l'audit di connettività**, individuando componenti deboli e forti e quantificando la raggiungibilità direzionale tra i comuni.
3. **Quando viene utilizzato $G^{\mathrm{safe}}$, attribuire agli archi esclusi un contributo di riconnessione**, identificando quelli il cui reinserimento recupera il maggior numero di relazioni OD o la maggiore quantità di domanda.
4. **Definire gli accessi comunali necessari al test** e calcolare i cammini minimi direzionali tra i 215 comuni, mantenendo separate andata e ritorno.
5. **Estrarre il sottografo dei cammini minimi** $\mathcal A^{\mathrm{SP}}$ e calcolare per ogni arco il numero di OD e, quando disponibile, il flusso totale che lo utilizza.
6. **Confrontare tempi e distanze di rete con i riferimenti ISTAT/TomTom**, dopo averne verificato la compatibilità semantica, classificando le OD anomale.
7. **Retro-proiettare le anomalie sugli archi**, individuando gli elementi ricorrenti nei percorsi irraggiungibili, eccessivamente lunghi o temporalmente implausibili.
8. **Costruire una shortlist prioritaria di archi da verificare**, combinando rilevanza per la connettività, uso nei cammini minimi, presenza nelle anomalie e importanza strategica.
9. **Validare e correggere soltanto gli archi prioritari**, utilizzando GSFVG, OSM, Overture e controllo manuale secondo la gerarchia metodologica già adottata.
10. **Rigenerare il grafo e ripetere l'audit** fino alla stabilizzazione di raggiungibilità, cammini e indicatori di anomalia.
11. **Congelare e versionare il grafo operativo validato**, conservando gli archi esclusi o incerti nel dataset sorgente e documentando le modifiche applicate alla sola copia operativa.
12. **Utilizzare il grafo così validato per la matrice dei costi e per l'assegnazione all-or-nothing**, mantenendo la possibilità di riaprire localmente la validazione solo qualora le successive analisi ANAS evidenzino nuove anomalie di percorso.


### 4.4.4 Passaggio dalla revisione direzionale all'audit funzionale del grafo

A seguito della chiusura delle revisioni manuali **P1** e **P2**, il trattamento esaustivo delle ulteriori discordanti direzionali **P3** e **P4** viene temporaneamente sospeso.

La decisione è motivata dalla necessità di verificare preliminarmente se le incertezze direzionali residue abbiano un impatto effettivamente rilevante sul comportamento del grafo regionale utilizzato per le successive fasi di routing e ricostruzione OD.

L'obiettivo non è ottenere una ricostruzione esaustiva della disciplina locale della circolazione, ma costruire un grafo sufficientemente robusto per:

- connettere i comuni FVG alla rete;
- calcolare shortest path tra origini e destinazioni;
- supportare l'assegnazione all-or-nothing;
- associare successivamente i flussi agli archi;
- alimentare la ricostruzione della matrice OD;
- fornire i percorsi necessari al successivo modello FRLM.

#### Scenario operativo di base

Lo stato direzionale che alimenterà il primo scenario funzionale $G^{\mathrm{base}}$ è ora materializzato nel layer persistente `GSFVG_operativo_direzionale_v01`. In esso:

1. i valori GSFVG `TRIM_USAGE = 1` e `TRIM_USAGE = 2` rimangono autorevoli;
2. i **616** casi originariamente `TRIM_USAGE = NULL` utilizzano le decisioni manuali consolidate;
3. i **136 casi P1** utilizzano le decisioni manuali definitive;
4. i **127 casi P2 revisionati manualmente** utilizzano le decisioni definitive, pari a 65 `FWD_ONLY`, 46 `BWD_ONLY` e 16 `BIDIRECTIONAL`;
5. i **940 casi P2 fuori campione** e gli archi **P3/P4 non verificati** mantengono la bidirezionalità originaria `TRIM_USAGE = 0`.

Questa configurazione è **conservativa rispetto alla connettività**: evita di introdurre sensi unici non verificati che potrebbero interrompere artificialmente il grafo. Il layer direzionale non è però ancora il grafo computazionale finale: accessi, velocità, tempi e costi devono essere applicati in passaggi successivi e separatamente auditabili.

Gli attributi sorgente GSFVG non vengono sovrascritti; le decisioni sono applicate esclusivamente alla copia operativa versionata.

#### Audit funzionale della rete

La validazione viene spostata dal livello della singola discordanza direzionale al comportamento complessivo del grafo.

L'audit deve verificare almeno:

- numero e dimensione delle componenti connesse;
- componente principale;
- componenti fortemente connesse nel grafo direzionale;
- reachability tra i comuni FVG;
- comuni o nodi non raggiungibili;
- dead end sospetti introdotti dalle direzioni;
- continuità delle principali arterie regionali;
- presenza di interruzioni anomale su autostrade, strade statali e principali collegamenti regionali;
- shortest path tra coppie OD rappresentative;
- eventuali deviazioni macroscopicamente implausibili;
- confronto qualitativo e, dove possibile, quantitativo di distanze e tempi con riferimenti esterni semanticamente compatibili.

Particolare attenzione viene riservata ai collegamenti strategici, includendo almeno:

- Udine–Trieste;
- Udine–Pordenone;
- Trieste–Pordenone;
- Tolmezzo–Udine;
- ulteriori relazioni necessarie a coprire le principali direttrici regionali.

Questi itinerari campione affiancano, senza sostituirlo, l'audit sistematico sulle **46.010 OD direzionali** definito nella Sezione 4.4.3.

#### Utilizzo delle anomalie per la revisione selettiva

Le anomalie individuate dall'audit funzionale vengono retro-proiettate sugli archi responsabili.

La revisione P3/P4 non viene quindi affrontata mediante controllo manuale esaustivo dei **1.346 archi residui**, ma mediante una procedura selettiva orientata all'impatto sul routing.

Sono prioritariamente riesaminati gli archi che:

- interrompono la reachability tra comuni;
- modificano le componenti fortemente connesse;
- provocano dead end anomali;
- interrompono direttrici principali;
- generano deviazioni rilevanti nei shortest path;
- appartengono a un numero elevato di cammini minimi OD.

Gli archi P3/P4 che non producono effetti apprezzabili sul comportamento regionale del grafo possono mantenere il trattamento conservativo, salvo ulteriori evidenze provenienti dalle successive fasi di assegnazione o dal confronto con ANAS.

#### Scenario di sensibilità

Dopo la costruzione della baseline può essere predisposto, esclusivamente come **stress test**, uno scenario alternativo $G^{\mathrm{stress}}_{\mathrm{OSM}}$ nel quale le indicazioni one-way OSM dei casi P3/P4 vengono applicate in modo più esteso.

Il confronto tra $G^{\mathrm{base}}$ e $G^{\mathrm{stress}}_{\mathrm{OSM}}$ consente di misurare:

- variazioni della connettività;
- variazioni della reachability;
- variazioni dei shortest path;
- OD interessate;
- archi responsabili delle differenze.

Lo scenario alternativo **non costituisce una validazione automatica delle direzioni OSM** e non viene utilizzato come sostituto della baseline. La sua funzione è esclusivamente diagnostica: identificare quali discordanti residue possano avere conseguenze materiali sul routing regionale.

#### Criterio decisionale

Il trattamento definitivo di P3 e P4 viene deciso soltanto dopo l'audit funzionale.

Il principio metodologico adottato è:

> la revisione manuale viene concentrata sugli errori direzionali che producono conseguenze funzionali sul grafo regionale, evitando una ricostruzione esaustiva della micro-topologia urbana quando questa non influenza i percorsi utilizzati dal modello.

Il passaggio consente di allineare lo sforzo di validazione allo scopo effettivo del grafo: supportare in modo affidabile shortest path, assegnazione OD e successiva localizzazione delle infrastrutture di ricarica.

#### Esito del primo ciclo di validazione funzionale: connettività direzionale e reachability dei P3/P4

A seguito del consolidamento della copia direzionale `GSFVG_operativo_direzionale_v01`, è stato eseguito il primo ciclo operativo della strategia di validazione funzionale definita nella presente sezione.

L'obiettivo specifico di questo ciclo non era dimostrare che ogni arco del GSFVG fosse correttamente rappresentato in ogni dettaglio locale, né verificare esaustivamente tutti i residui P3/P4. L'obiettivo era più circoscritto e direttamente legato all'utilizzo successivo della rete: verificare se le incertezze direzionali residue potessero alterare materialmente la possibilità di costruire percorsi tra le zone comunali del Friuli Venezia Giulia.

Il ciclo ha quindi risposto alla domanda:

> i residui P3/P4 ancora non verificati contengono archi la cui direzionalità può interrompere o alterare in maniera materialmente rilevante la raggiungibilità tra i comuni FVG?

La risposta è stata ottenuta mediante una sequenza iterativa di stress test, attribuzione delle anomalie agli archi, revisione manuale e nuova verifica del comportamento del grafo.

##### Richiamo terminologico: grafo, nodi, archi e direzionalità

La rete stradale viene rappresentata mediante un grafo diretto:

$$
G=(\mathcal V,\mathcal A),
$$

dove:

- $\mathcal V$ è l'insieme dei nodi;
- $\mathcal A$ è l'insieme degli archi diretti.

Nel contesto della rete stradale, un nodo rappresenta un punto di connessione topologica tra segmenti della rete, mentre un arco rappresenta una possibilità di movimento da un nodo verso un altro.

La distinzione tra geometria stradale e arco diretto è fondamentale.

Una feature GSFVG geometricamente rappresentata da una linea con estremi $u$ e $v$ può generare:

$$
(u,v)
$$

se è ammessa la percorrenza nel verso della geometria, e:

$$
(v,u)
$$

se è ammessa la percorrenza nel verso opposto.

Una feature bidirezionale genera quindi entrambi gli archi:

$$
(u,v),(v,u),
$$

mentre una feature monodirezionale ne genera soltanto uno.

Nel layer operativo tale informazione è materializzata dai campi:

```text
dir_fwd_ok
dir_bwd_ok
```

che vengono tradotti direttamente nella costruzione del grafo senza reinterpretare `TRIM_USAGE`.

##### Baseline e primo stress test P3/P4

Il riferimento del ciclo è il grafo corrispondente allo stato direzionale consolidato al Gate 4. Con gli accessi comunali diagnostici allora utilizzati, la baseline presenta:

```text
OD ordinate totali          46.010
OD raggiungibili            45.370
OD non raggiungibili           640
```

Per misurare il ruolo potenziale dei P3/P4, i **1.346 archi residui** sono stati temporaneamente esclusi dalla percorribilità. In tale scenario conservativo la reachability scende a:

```text
OD raggiungibili            43.893
OD non raggiungibili         2.117
OD perse rispetto Gate 4     1.477
```

La forte differenza non viene interpretata come prova che tutti i 1.346 archi siano direzionalmente errati. Lo stress test serve esclusivamente a identificare quali elementi abbiano effettiva capacità di riconnessione.

##### Attribuzione della perdita di reachability

Il reinserimento diagnostico dei singoli archi ha individuato **sei ID1 con contributo individuale positivo alla riconnessione**:

```text
ID1 2762     ΔR = 420
ID1 20894    ΔR = 209
ID1 20974    ΔR = 209
ID1 21669    ΔR = 209
ID1 51544    ΔR = 209
ID1 67316    ΔR = 209
```

L'analisi delle catene minime di riconnessione ha inoltre mostrato che, tra le 1.477 OD perse:

```text
1.047 OD richiedono 1 arco P3/P4
  425 OD richiedono 2 archi P3/P4
    5 OD richiedono 3 archi P3/P4
```

È stata identificata anche una catena funzionale specifica nell'area di **Vajont**, costituita dagli ID1 **25315** e **25414** di Via Colomber. Ciò dimostra che l'impatto funzionale non può essere attribuito esclusivamente agli archi con massimo $\Delta R$ individuale: alcuni collegamenti diventano rilevanti soltanto come combinazione.

##### Revisione manuale mirata

Sulla base dell'analisi precedente sono stati sottoposti a revisione manuale **8 casi P3/P4**:

| ID1 | Strada / località | Esito finale |
|---:|---|---|
| 25315 | Via Colomber | `BIDIRECTIONAL` |
| 25414 | Via Colomber | `BIDIRECTIONAL` |
| 2762 | SR 251 | `BIDIRECTIONAL` |
| 20894 | SP 16 | `BWD_ONLY` |
| 20974 | Via degli Elettricisti | `BWD_ONLY` |
| 21669 | Via Roma | `BIDIRECTIONAL` |
| 51544 | Via Pietà | `BWD_ONLY` |
| 67316 | Via Pietro Zorutti | `FWD_ONLY` |

I quattro override direzionali effettivi sono quindi:

```text
20894  → BWD_ONLY
20974  → BWD_ONLY
51544  → BWD_ONLY
67316  → FWD_ONLY
```

Gli altri quattro archi sono stati mantenuti bidirezionali. Nel complesso, i verdetti comprendono tre casi `MODELLING_DIFFERENCE_GSFVG_OK`, un caso `GSFVG_OK_BIDIRECTIONAL` e quattro casi `OSM_ONEWAY_CONFIRMED`.

Questa selezione realizza esattamente la logica prevista dalla metodologia: la revisione manuale viene concentrata sugli archi dimostrati rilevanti per la connettività, anziché sull'intera popolazione P3/P4.

##### Verifica del residuo P3/P4

Dopo l'applicazione dei quattro override effettivi, i casi P3/P4 ancora non revisionati scendono da 1.346 a **1.338**.

Uno stress test ulteriore sul residuo ha prodotto:

```text
OD aggiuntive perse per i 1.338 P3/P4 residui = 0
```

Rispetto all'obiettivo specifico del ciclo, il risultato è decisivo: **nessuno dei 1.338 residui P3/P4 ancora non revisionati risulta necessario per recuperare ulteriore reachability tra gli accessi comunali utilizzati**.

Il gate P3/P4 sulla sola raggiungibilità può quindi essere considerato **convergente**. Ciò non equivale ad affermare che ogni residuo sia localmente corretto, ma dimostra che una revisione manuale esaustiva dei 1.338 casi non è giustificata dalla connettività intercomunale.

##### Diagnosi delle 424 OD residue: ruolo degli accessi comunali

Dopo l'applicazione degli override P3/P4, il grafo `G_iter1` presentava:

```text
OD raggiungibili            44.946
differenza rispetto Gate 4     424
```

Poiché il residuo P3/P4 non produceva ulteriori perdite, le 424 OD sono state investigate separando il problema della **rete stradale** da quello dell'**associazione delle zone comunali ai nodi del grafo**.

Per questo primo ciclo era stata adottata, a fini diagnostici, una configurazione minimale:

$$
|\Gamma_o^{\mathrm L}|=1,
$$

ossia un solo nodo di accesso per comune, scelto come nodo GSFVG più vicino al punto comunale.

L'analisi ha identificato due accessi critici.

**Sacile.** Il nodo originariamente associato era:

```text
node 49570
distanza dal punto comunale = 96,1 m
in-degree = 3
out-degree = 0
fuori dalla giant SCC
```

Il nodo più vicino appartenente alla giant strongly connected component è:

```text
node 48739
distanza = 110,5 m
incremento = +14,4 m
```

Sostituendo esclusivamente l'accesso di Sacile si recuperano **211 OD**, portando la reachability a **45.157/46.010**.

**Codroipo.** Il nodo originariamente associato era:

```text
node 37780
distanza dal punto comunale = 61,6 m
in-degree = 1
out-degree = 3
fuori dalla giant SCC
```

Il nodo più vicino appartenente alla giant SCC è:

```text
node 39946
distanza = 98,1 m
incremento ≈ +36,4 m
```

Sostituendo esclusivamente l'accesso di Codroipo si recuperano **212 OD**, portando la reachability a **45.158/46.010**.

Sostituendo **entrambi** gli accessi, la reachability torna esattamente a:

```text
45.370 / 46.010
```

cioè al valore del Gate 4.

Il recupero congiunto è pari a 424 OD: oltre alle 211 OD attribuibili a Sacile e alle 212 attribuibili a Codroipo separatamente, viene recuperata anche la relazione direzionale la cui fattibilità richiede contemporaneamente entrambi gli accessi corretti.

##### Interpretazione metodologica

Il primo ciclo consente di separare due classi di problema che, osservando soltanto una matrice di reachability, sarebbero indistinguibili:

1. **problemi interni alla rete**, dovuti alla direzionalità degli archi;
2. **problemi di accesso zonale**, dovuti alla scelta del nodo con cui il comune viene connesso al grafo.

Nel caso analizzato, la perdita iniziale di 1.477 OD nello stress test P3/P4 ha effettivamente permesso di individuare un piccolo gruppo di archi funzionalmente rilevanti e quattro correzioni direzionali. Dopo tali correzioni, però, le 424 OD mancanti rispetto al Gate 4 non erano imputabili ad altri P3/P4: erano prodotte interamente dalla configurazione provvisoria degli accessi di Sacile e Codroipo.

Ne deriva una regola importante per le fasi successive:

> la mancata raggiungibilità di una zona non deve essere attribuita automaticamente a un difetto della rete stradale; prima di riaprire la revisione degli archi occorre verificare che il relativo insieme di accesso $\Gamma_o^{\mathrm L}$ sia correttamente collegato alla componente direzionale principale del grafo.

La configurazione `|Γ|=1` rimane quindi un benchmark diagnostico utile, ma **non può essere congelata semplicemente scegliendo il nodo geometricamente più vicino**. La definizione finale degli accessi comunali deve tenere conto almeno dell'appartenenza alla componente funzionale principale e, nella successiva analisi di sensibilità, della possibilità di utilizzare più accessi per comune.

##### Chiusura del gate reachability P3/P4

Al termine del ciclo:

```text
P3/P4 iniziali                         1.346
P3/P4 revisionati manualmente              8
override direzionali effettivi             4
P3/P4 residui                           1.338
OD aggiuntive perse per residuo             0
reachability con accessi corretti   45.370 / 46.010
```

Il **gate P3/P4 relativo alla reachability viene quindi chiuso**.

Gli archi residui non vengono dichiarati universalmente corretti: restano disponibili per eventuali verifiche successive qualora emergano anomalie nei shortest path, nelle distanze, nei tempi di percorrenza, nel confronto con ISTAT/TomTom o nell'associazione con le sezioni ANAS. Viene invece respinta, perché non supportata dall'evidenza funzionale, l'ipotesi di una revisione manuale esaustiva dei 1.338 P3/P4 residui.

Il consolidamento degli accessi comunali è stato successivamente completato nella Sezione 4.2. Il livello successivo della validazione funzionale è quindi l'**audit metrico dei shortest path**, documentato nella Sezione 4.4.5, che separa inizialmente la sola distanza geometrica dalle successive verifiche su velocità e tempi.


### 4.4.4.1 Gate preliminare OSM shadow — disegno sperimentale (CHIUSO)

> **Stato:** questa sezione conserva il disegno sperimentale approvato prima del test. Il Gate è stato successivamente eseguito e chiuso con decisione `SWITCH_OSM`; risultati e decisione sono riportati nella §4.4.6. Le stop rule qui definite vanno lette insieme all'evidenza effettivamente osservata: due `GSFVG_DATA_MISSING` indipendenti ad alto costo di correzione, entrambi coperti nativamente da OSM, sono risultati sufficienti per interrompere l'audit GSFVG estensivo anche in assenza di `REPRESENTATION_ERROR` confermati.

Durante SP2 sono state individuate **1.818 relazioni OD** per le quali:

$$
KM\_TOT < D_{\min}^{GSFVG},
$$

e una shortlist operativa di **125 OD**. Tali valori sono indicatori diagnostici: non equivalgono a 1.818 o 125 errori dimostrati. Una sola causa locale può generare molte OD anomale e, in altri casi, l'anomalia può dipendere da dati mancanti, ambiguità o differenze di rappresentazione. La review manuale SP2 resta quindi il meccanismo principale per stabilire la causa effettiva.

È approvato un **Gate preliminare OSM shadow**. Durante il test:

$$
\boxed{GSFVG=\text{baseline operativa corrente}}
$$

ma ciò non implica una presunzione forte di permanenza. Il lavoro già svolto su GSFVG è un **sunk cost**: non costituisce argomento per mantenerlo, pur fornendo evidenza empirica sul `manual correction burden` già osservato.

La domanda decisionale è prospettica:

$$
\boxed{\text{da oggi, quale soluzione permette di arrivare prima a un network sufficientemente affidabile?}}
$$

e confronta principalmente:

$$
T_{\mathrm{finish}}^{GSFVG}
\quad\text{vs}\quad
T_{\mathrm{switch+finish}}^{OSM}.
$$

#### Cosa il Gate non autorizza

Il Gate non autorizza:

- costruzione di un backbone OSM regionale completo;
- costruzione di $\Gamma^{L,OSM}$ per tutti i 215 comuni;
- audit sistematico dell'intera rete OSM;
- seconda pipeline completa di shortest path;
- sostituzione automatica di GSFVG.

La sequenza approvata è:

$$
\boxed{
SP2\ manuale
\rightarrow
causa\ verificata
\rightarrow
test\ OSM\ locale
\rightarrow
decisione
}
$$

con l'obiettivo di ottenere il massimo valore informativo possibile con il minimo lavoro aggiuntivo.

#### Time-box e unità sperimentale

Il budget massimo indicativo è:

$$
\boxed{\text{circa una giornata effettiva di lavoro}}
$$

come tetto, non come tempo da consumare obbligatoriamente. Se una decisione affidabile può essere presa prima, il gate deve chiudersi anticipatamente.

L'unità sperimentale non è la singola OD ma il **sito fisico causale indipendente**. Devono essere distinti:

$$
N_{OD},\qquad N_{cause},\qquad N_{siti}.
$$

Il primo gate shadow utilizza indicativamente **3–5 siti indipendenti**, includendo preferibilmente almeno un `DATA_ERROR`, almeno due sospetti `REPRESENTATION_ERROR`, almeno un controllo negativo con GSFVG già verificato come corretto e un eventuale quinto sito solo se aggiunge informazione indipendente.

#### Test locale e isolamento della causa

Quando possibile, il controfattuale OSM viene eseguito fra punti immediatamente **a monte e a valle della zona causale**. In questo modo si isolano:

$$
\boxed{\text{topologia}+\text{regole di percorrenza}}
$$

senza confonderle con snapping comunale, costruzione di $\Gamma_o^L$, centroidi o scelta degli accessi.

Un confronto regionale completo, se mai autorizzato, dovrà invece utilizzare accessi propri per ciascun network:

$$
(G^{GSFVG},\Gamma^{GSFVG})
\quad\text{vs}\quad
(G^{OSM},\Gamma^{OSM}).
$$

Il routing OSM locale deve usare, per quanto necessario e compatibile con il time-box, la semantica utile al routing: `oneway`, roundabout, `access`, `motor_vehicle`/`motorcar`, rampe, carreggiate separate, ponti, tunnel, livelli e turn restrictions. Le turn restrictions richiedono cautela perché non sono automaticamente rappresentate da un normale grafo nodo-arco.

Se ottenere un controfattuale affidabile richiede già una pipeline OSM regionale complessa, ciò costituisce di per sé evidenza negativa sul costo dello switch e può determinare lo stop anticipato del gate.

#### Ground truth indipendente e tassonomia causale

Per ogni sito deve essere stabilito preliminarmente, mediante evidenza indipendente, quale movimento sia realmente consentito. OSM non può essere benchmark di se stesso. Le evidenze possono includere Google Maps, indicazioni di routing, Street View, segnaletica, geometria/configurazione fisica del sito e altre fonti indipendenti appropriate.

Il confronto diretto OSM-vs-GSFVG **non è sempre possibile**. GSFVG può avere dati mancanti o valori ambigui; questi casi non devono essere forzati nelle categorie di errore.

La tassonomia minima è:

- `DATA_ERROR`: informazione GSFVG disponibile e interpretabile, ma errata; il fenomeno è rappresentabile correggendo un attributo;
- `REPRESENTATION_ERROR`: la topologia/geometria GSFVG non possiede sufficienti gradi di libertà per rappresentare correttamente i movimenti reali tramite una semplice modifica attributiva;
- `GSFVG_DATA_MISSING`: l'informazione necessaria al routing non è disponibile;
- `GSFVG_UNRESOLVED`: l'informazione esiste ma è ambigua o non consente una decisione affidabile;
- `OTHER`: causa differente.

Un caso:

$$
\text{direzione GSFVG errata}
\rightarrow
\text{override}
\rightarrow
\text{routing corretto}
$$

è un `DATA_ERROR` e non costituisce, da solo, ragione sufficiente per cambiare backbone.

Un caso nel quale una singola geometria GSFVG aggrega movimenti reali con regole differenti e nessuna scelta fra `BIDIRECTIONAL`, `FWD_ONLY` e `BWD_ONLY` riesce a rappresentarli correttamente è invece un candidato `REPRESENTATION_ERROR`.

`GSFVG_DATA_MISSING` e `GSFVG_UNRESOLVED` contribuiscono al costo di review GSFVG ma **non costituiscono automaticamente evidenza di un vantaggio strutturale OSM**. In tali casi OSM viene confrontato direttamente con la realtà verificata.

#### Benchmark ISTAT/TomTom

Il Gate shadow non usa MAE/RMSE rispetto a `KM_TOT` come criterio globale di scelta. `KM_TOT` è la lunghezza dell'itinerario TomTom scelto minimizzando il **tempo**, mentre SP2 GSFVG minimizza la **lunghezza geometrica**.

Pertanto:

$$
KM\_TOT < D_{\min}^{GSFVG}
$$

resta un indicatore diagnostico unilaterale, ma non rende automaticamente migliore il network con distanza mediamente più vicina a `KM_TOT`.

#### Manual correction burden

Si distinguono:

$$
B_{GSFVG}^{past},
\qquad
B_{GSFVG}^{future},
\qquad
B_{OSM}^{switch}.
$$

$B_{GSFVG}^{past}$ è il lavoro già sostenuto: utile come evidenza empirica, ma non come costo decisionale futuro.

$B_{GSFVG}^{future}$ è il costo residuo atteso per completare audit, diagnosi, correzioni e regressioni GSFVG.

$B_{OSM}^{switch}$ è il costo atteso per costruire, validare e integrare OSM fino al livello richiesto downstream.

La decisione riguarda quindi soprattutto:

$$
\boxed{
B_{GSFVG}^{future}
\quad\text{vs}\quad
B_{OSM}^{switch}
}
$$

#### Stop rule simmetriche

**STOP OSM shadow.** Il ramo OSM viene arrestato rapidamente se le anomalie GSFVG sono prevalentemente `DATA_ERROR` locali e correggibili, OSM richiede patch comparabili, introduce anomalie proprie, gli esiti restano ambigui, non emerge vantaggio strutturale ricorrente o una pipeline affidabile eccede il time-box.

$$
\boxed{
STOP\ OSM
\rightarrow
GSFVG
\rightarrow
chiusura\ audit
\rightarrow
roadmap
}
$$

**STOP audit GSFVG estensivo.** Se emergono rapidamente in più siti indipendenti `REPRESENTATION_ERROR` reali, ricorrenza di problemi strutturalmente coerenti, OSM nativamente corretto e un plausibile vantaggio sul costo residuo, non si continua a verificare decine di casi solo per accumulare evidenza ridondante.

$$
\boxed{
evidenza\ strutturale\ sufficiente
\rightarrow
STOP\ audit\ GSFVG\ estensivo
\rightarrow
switch\ feasibility
}
$$

La `switch feasibility` deve stimare rapidamente:

$$
G^{OSM}_{operativo}
\rightarrow
\Gamma^{OSM}
\rightarrow
QA
\rightarrow
ripartenza\ downstream.
$$

Solo dopo tale verifica può essere assunta una decisione definitiva di sostituzione del backbone.

#### Principio decisionale finale

Il Gate non cerca il network teoricamente più perfetto. Il criterio è:

$$
\boxed{\text{quale soluzione, da oggi, può essere resa sufficientemente affidabile prima?}}
$$

GSFVG resta la baseline operativa durante l'esperimento, ma non viene privilegiato per il lavoro già svolto. Il valore marginale di ulteriore audit viene considerato in entrambe le direzioni: si interrompe OSM quando non produce rapidamente evidenza utile e si interrompe il patching GSFVG estensivo quando l'evidenza strutturale è già sufficiente a rendere razionale una valutazione immediata dello switch.

### 4.4.5 Fase 5.4 — Audit metrico degli shortest path sulla rete GSFVG

#### Obiettivo della fase

Conclusa la costruzione della baseline stradale operativa `GSFVG_operativo_direzionale_v02` e chiuso il gate degli accessi comunali light $\Gamma_o^{\mathrm L}$, è stata avviata la prima fase sistematica di calcolo dei percorsi Origine-Destinazione sulla rete stradale.

**Idea in termini semplici.** Fino a questo punto è stato verificato che la rete sia connessa in modo coerente e che ogni comune disponga di accessi affidabili al grafo. Il passo successivo consiste nel chiedere alla rete non soltanto *se* un percorso esiste, ma *quale sia il percorso più corto*, quanto misuri e quali archi utilizzi. In questa fase si usa volutamente la sola distanza geometrica, così da non confondere eventuali problemi topologici o metrici con le ipotesi successive sulle velocità.

L'obiettivo non è ancora costruire la matrice definitiva dei costi né assegnare i flussi di traffico. L'audit verifica separatamente:

1. la riproducibilità topologica del grafo;
2. la corretta associazione degli accessi comunali al grafo;
3. la raggiungibilità reciproca degli accessi;
4. la correttezza metrica dei cammini minimi;
5. la sensibilità della distanza OD rispetto ai tre accessi comunali;
6. l'asimmetria indotta dalla direzionalità della rete;
7. quali archi della rete vengono effettivamente utilizzati dai cammini minimi.

Il costo di ciascun arco $a$ è esclusivamente geometrico:

$$
c_a=l_a,
$$

dove $l_a$ è la lunghezza dell'arco in metri, calcolata nel CRS metrico nativo EPSG:25833.

Non vengono ancora introdotti velocità, tempi di percorrenza, congestione, pedaggi o costi generalizzati. I pesi $\lambda$ degli accessi **non modificano il costo degli archi né il calcolo dei singoli shortest path**; vengono utilizzati soltanto più avanti come diagnostica per confrontare possibili regole di aggregazione delle nove combinazioni access-to-access.

---

#### SP0 — Graph and Access Regression

Prima di calcolare gli shortest path è stata eseguita una run di regressione denominata **SP0 — graph and access regression**.

Una *regressione*, in questo contesto, non è una regressione statistica: è un controllo di riproducibilità. Lo scopo è ricostruire da zero il grafo e gli accessi a partire dai prodotti canonici e verificare che i numeri ottenuti coincidano esattamente con quelli già consolidati. In caso contrario, non sarebbe metodologicamente corretto procedere con i shortest path.

##### Ricostruzione canonica dei nodi

Le 76.349 feature fisiche del GSFVG operativo sono state esplose nelle rispettive parti lineari, ottenendo:

- **76.349** feature fisiche;
- **76.350** parti lineari;
- **152.700** endpoint.

I nodi computazionali non sono stati ricavati dalle intersezioni geometriche interne e non è stato effettuato alcuno snapping generalizzato. Sono stati utilizzati esclusivamente gli estremi delle parti lineari.

Gli endpoint sono stati raggruppati mediante una ricerca spaziale `dwithin` con tolleranza:

$$
\varepsilon=0{,}10\ \text{m},
$$

e clustering **Union-Find**. Union-Find è una struttura algoritmica che consente di raggruppare in modo efficiente tutti gli endpoint appartenenti allo stesso cluster di prossimità.

Il `node_id` è stato assegnato in modo deterministico seguendo l'ordine originario degli endpoint: il primo cluster incontrato riceve `node_id = 0`, il successivo `node_id = 1` e così via.

La coordinata rappresentativa di ciascun nodo è la media aritmetica delle coordinate degli endpoint appartenenti al cluster.

La procedura ha riprodotto esattamente:

$$
61.947\ \text{nodi computazionali}.
$$

##### Regressione topologica

Il grafo diretto collassato a `DiGraph` ha riprodotto integralmente i valori consolidati della baseline:

- nodi: **61.947**;
- archi diretti: **140.213**;
- Weakly Connected Components (WCC): **10**;
- nodi della giant WCC: **61.914**;
- Strongly Connected Components (SCC): **625**;
- nodi della giant SCC: **61.109**.

Una **WCC — Weakly Connected Component** è una componente nella quale i nodi risultano collegati se si ignora temporaneamente il verso degli archi.

Una **SCC — Strongly Connected Component** è invece una componente di un grafo diretto nella quale, per ogni coppia di nodi $u,v$, esiste sia un percorso $u\rightarrow v$ sia un percorso $v\rightarrow u$.

La **giant SCC** è la SCC di dimensione maggiore ed è quindi la parte principale della rete nella quale tutti i nodi sono reciprocamente raggiungibili rispettando la direzionalità.

##### Regressione degli accessi $\Gamma_o^{\mathrm L}$

Il dataset canonico contiene:

$$
215\ \text{comuni}\times3\ \text{accessi}=645\ \text{accessi}.
$$

I 645 `node_id` degli accessi sono tutti distinti.

I controlli hanno verificato:

- **645/645** accessi correttamente mappati sul grafo;
- **0** `node_id` mancanti;
- **645/645** accessi appartenenti alla giant SCC;
- errore massimo sulla somma dei pesi $\lambda$:

$$
3{,}33\times10^{-16};
$$

- errore massimo nella regressione delle coordinate:

$$
9{,}33\times10^{-10}\ \text{m}.
$$

Tali differenze sono esclusivamente di ordine floating-point e sono prive di significato geometrico.

**Esito SP0:** superato senza anomalie. Non emerge alcuna evidenza per riaprire né la topologia GSFVG né il gate degli accessi comunali.

---

#### SP1 — Access-to-Access Distance Shortest Paths

Superata SP0, è stata eseguita la run **SP1 — access-to-access distance shortest paths**.

Per ogni coppia ordinata di comuni $o\neq d$ esistono tre accessi in origine e tre accessi in destinazione. Ogni relazione comunale genera quindi:

$$
3\times3=9
$$

relazioni access-to-access.

Con 215 comuni, il numero delle OD comunali ordinate è:

$$
215\times214=46.010.
$$

Il numero complessivo di relazioni access-to-access è pertanto:

$$
46.010\times9=414.090.
$$

##### Distinzione tra OD comunale e relazione access-to-access

Questa distinzione è centrale per il seguito della tesi.

La quantità:

$$
D_G(u,v)
$$

rappresenta la distanza di shortest path sulla rete fra **due specifici nodi di accesso** $u$ e $v$.

La relazione comunale:

$$
o\rightarrow d
$$

è invece un oggetto zonale. Poiché ogni comune dispone di tre accessi, una singola OD comunale dispone inizialmente di **nove valori** $D_G(u,v)$.

SP1 calcola tutti questi percorsi, ma **non decide ancora come i nove risultati debbano essere combinati** per rappresentare la OD comunale. Separare questi due livelli evita di trasformare prematuramente un comune multi-accesso in un unico nodo artificiale.

---

#### MultiDiGraph fisico e DiGraph ausiliario

Per il calcolo sono stati mantenuti due livelli di rappresentazione del grafo.

##### MultiDiGraph fisico

Il `MultiDiGraph` conserva separatamente tutte le **directed edge instances** fisiche.

Una directed edge instance è una specifica istanza orientata di una parte stradale. Due archi possono quindi avere gli stessi nodi iniziale e finale senza essere la stessa feature fisica.

Ogni edge conserva almeno:

- `ID1`;
- `gsfvg_fid_original`;
- `part_idx`;
- `dir_final`;
- `dir_source`;
- `length_m`;
- `orientation`.

Sono state generate:

$$
140.668
$$

directed edge instances fisiche.

##### DiGraph ausiliario

Un normale `DiGraph` può conservare un solo collegamento per ciascuna coppia ordinata di nodi $u\rightarrow v$.

Poiché nel GSFVG esistono archi paralleli, il `DiGraph` ausiliario contiene:

$$
140.213
$$

archi.

La differenza:

$$
140.668-140.213=455
$$

corrisponde a directed edge instances appartenenti a coppie parallele.

Sono state identificate:

- **455** coppie ordinate con parallelismo;
- molteplicità massima pari a **2**;
- **32** casi di parità esatta nel costo minimo.

Per ciascuna coppia $u\rightarrow v$, il grafo ausiliario conserva l'edge fisico di lunghezza minima.

In caso di parità viene applicato un **tie-break deterministico**, cioè una regola convenzionale che sceglie sempre lo stesso arco tra alternative di costo identico. Il tie-break non afferma che un arco sia fisicamente “più corretto” dell'altro: serve esclusivamente a rendere il calcolo riproducibile.

Il `MultiDiGraph` resta quindi la rappresentazione fisica autorevole, mentre il `DiGraph` è una struttura computazionale ausiliaria ottimizzata per l'algoritmo shortest-path.

---

#### Algoritmo shortest path

È stato utilizzato l'algoritmo di **Dijkstra**.

Dijkstra calcola, a partire da un nodo sorgente, la distanza minima verso tutti gli altri nodi raggiungibili quando i costi degli archi sono non negativi. Poiché in questa fase il costo è una lunghezza, tale condizione è sempre rispettata.

Non è stato necessario eseguire 414.090 calcoli indipendenti. Poiché i 645 accessi sono nodi distinti, sono stati eseguiti:

$$
645
$$

**single-source Dijkstra**.

Un single-source Dijkstra costruisce in un'unica esecuzione l'albero dei cammini minimi dalla sorgente verso tutti i nodi raggiungibili.

Per ciascuna delle 645 sorgenti sono poi stati estratti i 642 accessi appartenenti agli altri 214 comuni:

$$
214\times3=642.
$$

Si ottiene quindi:

$$
645\times642=414.090
$$

relazioni access-to-access pur eseguendo soltanto 645 alberi di shortest path.

La correttezza dell'implementazione deterministica è stata verificata confrontando tre sorgenti con l'implementazione NetworkX, ottenendo:

$$
\max|\Delta|=0\ \text{m}.
$$

---

#### Risultati della raggiungibilità e coerenza metrica

Sono state calcolate tutte le:

$$
414.090
$$

relazioni previste.

Risultato:

- reachable: **414.090/414.090**;
- unreachable: **0**.

L'assenza di coppie non raggiungibili è coerente con la regola di selezione degli accessi nella giant SCC. Come già chiarito nel gate $\Gamma_o^{\mathrm L}$, tale risultato non costituisce una nuova validazione indipendente della reachability del grafo: è soprattutto un controllo di coerenza della ricostruzione e del mapping degli accessi.

Le distanze access-to-access osservate sono comprese fra:

$$
1.511{,}406\ \text{m}
$$

e:

$$
176.796{,}408\ \text{m}.
$$

L'errore massimo fra la distanza restituita da Dijkstra e la somma delle lunghezze degli archi ricostruiti nel path è:

$$
4{,}07\times10^{-10}\ \text{m},
$$

quindi numericamente trascurabile.

---

#### Utilizzo degli archi: $\mathcal A^{\mathrm{SP}}_{\mathrm{ACCESS}}$ e $N_a^{\mathrm{ACCESS}}$

Dai 414.090 shortest path sono state ricostruite complessivamente:

$$
141.595.042
$$

occorrenze di directed edge.

Sono state utilizzate almeno una volta:

$$
61.852
$$

directed edge instances fisiche sulle 140.668 disponibili.

Si definisce provvisoriamente:

$$
\mathcal A^{\mathrm{SP}}_{\mathrm{ACCESS}}
$$

come l'insieme degli archi fisici che compaiono in almeno uno shortest path access-to-access.

Questo insieme **non coincide ancora** con l'insieme definitivo $\mathcal A^{\mathrm{SP}}$ delle OD comunali, perché ogni OD comunale genera ancora nove path distinti.

Analogamente si definisce:

$$
N_a^{\mathrm{ACCESS}}
$$

come il numero di shortest path access-to-access che utilizzano l'arco $a$.

Il controllo globale restituisce:

$$
\sum_a N_a^{\mathrm{ACCESS}}=141.595.042,
$$

esattamente pari al numero complessivo di attraversamenti di edge ricostruiti.

La notazione $N_a^{\mathrm{OD}}$ viene riservata alla fase successiva, quando sarà definita la regola con cui le nove combinazioni vengono trasformate nella rappresentazione comunale della domanda.

---

#### Audit dell'asimmetria

Poiché il grafo è diretto, non è necessariamente vero che:

$$
D_G(u,v)=D_G(v,u).
$$

L'asimmetria può derivare legittimamente da sensi unici, rampe, svincoli, carreggiate separate e differenze fra i percorsi di andata e ritorno.

Sono state considerate le **207.045 coppie non ordinate di accessi**.

Per ciascuna coppia è stato calcolato:

$$
\rho_{uv}
=
\frac{
\max\left[D_G(u,v),D_G(v,u)\right]
}{
\min\left[D_G(u,v),D_G(v,u)\right]
}.
$$

Interpretazione:

- $\rho=1$: perfetta simmetria;
- $\rho=1{,}05$: una direzione è circa il 5% più lunga dell'altra;
- valori molto superiori a 1 indicano asimmetrie crescenti.

I risultati sono:

$$
\operatorname{median}(\rho)=1{,}004581,
$$

$$
p95(\rho)=1{,}051639,
$$

$$
\max(\rho)=4{,}060971.
$$

La differenza assoluta massima osservata fra andata e ritorno è:

$$
27{,}154\ \text{km}.
$$

La mediana molto vicina a 1 e il 95° percentile pari a circa 1,052 indicano che la rete è sostanzialmente simmetrica nella grande maggioranza dei collegamenti. La coda superiore della distribuzione deve tuttavia essere sottoposta ad audit mirato prima della chiusura del gate metrico.

---

#### Sensibilità delle OD comunali ai tre accessi

Per ciascuna delle 46.010 OD comunali sono disponibili nove distanze access-to-access.

Si definisce lo **spread delle nove alternative**:

$$
S_{od}
=
D^{\max}_{od}-D^{\min}_{od}.
$$

Lo spread misura quanto cambia la distanza della stessa OD quando si modifica la combinazione degli accessi di origine e destinazione.

I risultati sono:

$$
\operatorname{median}(S_{od})=0{,}924\ \text{km},
$$

$$
p95(S_{od})=3{,}648\ \text{km}.
$$

Per la maggioranza delle OD la scelta fra i diversi accessi modifica quindi la distanza di rete in misura relativamente contenuta. Ciò suggerisce che la rappresentazione multi-accesso cattura una variabilità locale reale senza alterare, nella maggior parte dei casi, la struttura generale dei percorsi regionali.

---

#### Confronto preliminare fra MIN-PATH e PRODUCT-LAMBDA

SP1 non chiude ancora la modalità di aggregazione delle nove alternative. Sono però state confrontate due rappresentazioni candidate.

##### MIN-PATH

La regola **MIN-PATH** selezionerebbe, per ogni OD comunale:

$$
D^{\mathrm{MIN}}_{od}
=
\min_{
u\in\Gamma_o^{\mathrm L},
\;v\in\Gamma_d^{\mathrm L}
}
D_G(u,v).
$$

L'intero flusso della OD verrebbe quindi associato a una sola delle nove coppie di accessi.

Per tutte le 46.010 OD è stata osservata una sola coppia strettamente minima:

- OD con più di una combinazione esattamente minima: **0**.

MIN-PATH sarebbe quindi completamente deterministico.

Il suo limite è concettuale: dopo aver costruito tre accessi e i relativi pesi, l'intera OD verrebbe nuovamente concentrata su una sola coppia di nodi, senza utilizzare direttamente $\lambda$ nella distribuzione della domanda.

##### PRODUCT-LAMBDA

La formulazione candidata **PRODUCT-LAMBDA** conserva invece tutte le nove combinazioni assegnando a ciascuna:

$$
\omega_{oduv}
=
\lambda_{ou}^{\mathrm L}
\lambda_{dv}^{\mathrm L}.
$$

Poiché:

$$
\sum_u\lambda_{ou}^{\mathrm L}=1
\qquad\text{e}\qquad
\sum_v\lambda_{dv}^{\mathrm L}=1,
$$

si ottiene:

$$
\sum_u\sum_v\omega_{oduv}=1.
$$

La distanza media associata alla OD diventerebbe:

$$
D^{\mathrm{PL}}_{od}
=
\sum_u\sum_v
\omega_{oduv}D_G(u,v).
$$

In SP1 questa quantità è utilizzata **soltanto come diagnostica** e non rappresenta ancora una decisione metodologica congelata.

La differenza:

$$
D^{\mathrm{PL}}_{od}-D^{\mathrm{MIN}}_{od}
$$

presenta:

$$
\operatorname{median}=0{,}464\ \text{km},
$$

$$
p95=1{,}907\ \text{km}.
$$

Il costo metrico aggiuntivo associato al mantenimento della rappresentazione multi-accesso appare quindi relativamente contenuto per la grande maggioranza delle OD.

Il risultato costituisce una prima evidenza favorevole alla conservazione effettiva della struttura multi-accesso nelle successive fasi di assegnazione, ma **non è ancora sufficiente per congelare PRODUCT-LAMBDA**.

---

#### Stato del gate SP0–SP1 dopo lo switch di backbone

SP0 e SP1 restano **computazionalmente superati sulla baseline GSFVG** e mantengono pieno valore come audit trail:

- la baseline topologica era stata riprodotta;
- tutti gli accessi GSFVG erano correttamente associati;
- tutte le 414.090 relazioni access-to-access erano state calcolate;
- gli shortest path risultavano metricamente e numericamente coerenti;
- il `MultiDiGraph` preservava l'identità fisica degli archi paralleli.

La successiva rappresentazione comunale MIN-PATH vs PRODUCT-LAMBDA **non è stata congelata**. SP2 ha infatti portato al Gate OSM shadow e quindi alla decisione di cambiare backbone prima di completare tale scelta.

Di conseguenza:

- SP0/SP1 GSFVG restano benchmark storico e test di metodologia;
- non vengono proseguite campagne SP2 estensive sul GSFVG;
- $\mathcal A^{\mathrm{SP}}$ e $N_a^{\mathrm{OD}}$ definitivi dovranno essere ricostruiti sul backbone OSM dopo il freeze di $\Gamma_{OSM}$;
- la modalità di combinazione dei tre accessi verrà riesaminata sul nuovo network, senza trasferire automaticamente una decisione non ancora congelata.


### 4.4.6 Chiusura del Gate OSM shadow e decisione `SWITCH_OSM`

#### Esiti causali del Gate

Il Gate OSM shadow è stato chiuso dopo **tre siti causali fisicamente indipendenti** e **un controllo negativo valido**. L'obiettivo non era stabilire quale sorgente fosse cartograficamente “migliore” in assoluto, ma confrontare il costo residuo atteso delle due strategie.

##### SHADOW-01 — Cavazzo Carnico → Amaro

Il caso era già stato identificato come errore direzionale GSFVG. L'arco causale `ID1 14897` risultava `BWD_ONLY`, mentre la realtà richiedeva `FWD_ONLY`.

Il controfattuale GSFVG aveva mostrato che il semplice override direzionale riduceva la distanza minima da circa:

$$
12{,}752\ \text{km}
$$

a:

$$
5{,}252\ \text{km},
$$

senza regressioni di reachability.

OSM conteneva nativamente la rampa corrispondente come `highway=primary_link`, `oneway=yes`, nel verso corretto.

Il sito è quindi classificato:

```text
cause_class               = DATA_ERROR
local_verdict             = OSM_DATA_ADVANTAGE_ONLY
structural_advantage_OSM  = NO
```

Il caso dimostra un vantaggio informativo OSM, ma non un limite strutturale del modello GSFVG: la geometria regionale esisteva ed era sufficiente una correzione attributiva.

##### SHADOW-02 — Cercivento ↔ Paluzza

La review manuale ha verificato che il collegamento automobilistico reale utilizza:

```text
Via di Sot
→ Via dal Fiume
→ raccordo verso SS52bis / SS52
→ Paluzza
```

Il routing contemporaneo restituisce circa 2,8--2,9 km, mentre GSFVG produceva circa:

$$
5{,}994\ \text{km}
$$

nei due versi.

La catena GSFVG di Via di Sot risultava presente e continua, ma il successivo collegamento lungo Via dal Fiume era assente sia dal grafo operativo sia dalla sorgente regionale originale.

Classificazione:

```text
manual_verdict                       = GRAPH_MISSING_LINK
cause_class                          = GSFVG_DATA_MISSING
simple_attribute_override_sufficient = NO
GSFVG_correction_burden              = HIGH
```

OSM conteneva invece nativamente la direttrice completa, con continuità automobilistica e raccordi coerenti:

```text
OSM_native_correct        = YES
OSM_patch_required        = NONE
OSM_coverage_advantage    = YES
structural_advantage_OSM  = NO
```

Il caso non è classificato `REPRESENTATION_ERROR`: GSFVG potrebbe teoricamente rappresentarlo, ma richiederebbe introduzione di nuova geometria e topologia.

##### SHADOW-03 — Ovaro ↔ Raveo

La review ha individuato un secondo deficit di copertura indipendente. Il collegamento reale utilizza:

```text
SR355
→ raccordo verso Via Muina / SP35
→ Via Muina
→ Raveo
```

Google Maps restituisce circa 6,2 km nel verso Ovaro → Raveo e 6,1 km nel verso opposto, rispetto a un benchmark ISTAT di 6,7 km. GSFVG produceva invece circa:

$$
12{,}2977\ \text{km}
$$

in entrambi i versi.

La verifica ha stabilito che Via Muina manca anche da `GSFVG_IRDAT_FULL` originale.

Classificazione:

```text
manual_verdict                       = GRAPH_MISSING_LINK
cause_class                          = GSFVG_DATA_MISSING
simple_attribute_override_sufficient = NO
GSFVG_correction_burden              = HIGH

OSM_native_correct        = YES
OSM_patch_required        = NONE
OSM_coverage_advantage    = YES
structural_advantage_OSM  = NO
```

La ripetizione dello stesso tipo di deficit su due siti fisici indipendenti determina:

```text
REPEATED_GSFVG_COVERAGE_DEFICIT = YES
```

#### Controllo negativo OSM

Un primo candidato, `ID1 2762`, è stato escluso perché non disponeva di ground truth indipendente sufficientemente specifica.

Il controllo valido è stato eseguito su:

```text
ID1 15304
Via Leonardo Andervolti
```

già classificato nella review GSFVG come `MODELLING_DIFFERENCE_GSFVG_OK → BIDIRECTIONAL`.

La micro-topologia OSM rappresenta il primo verso direttamente su una way monodirezionale e il verso opposto mediante un breve percorso complementare sulle strade locali adiacenti. Entrambi i movimenti reali restano disponibili senza patch.

Esito:

```text
GSFVG_native_correct        = YES
OSM_native_correct          = YES
new_error_introduced_by_OSM = NO
local_verdict               = BOTH_CORRECT
confidence                  = HIGH
```

Il controllo mostra che la maggiore granularità OSM non ha introdotto una regressione proprio in un sito nel quale la rappresentazione aggregata GSFVG era già stata giudicata corretta.

#### Bilancio finale

```text
siti causali testati                = 3
controlli negativi validi           = 1

DATA_ERROR                          = 1
GSFVG_DATA_MISSING                  = 2
REPRESENTATION_ERROR confermati     = 0

OSM native correct sui causali      = 3/3
OSM coverage advantage              = 2
controlli negativi peggiorati       = 0

REPEATED_GSFVG_COVERAGE_DEFICIT     = YES
```

Il caso SP2 Chions → Sesto al Reghena è stato classificato separatamente come `BOUNDARY_EFFECT / OTHER / REGIONAL_NETWORK_DOMAIN_LIMITATION` e non è stato utilizzato come evidenza discriminante del Gate, perché riguarda il dominio spaziale della rete e non un difetto interno specifico della sorgente.

L'assenza di `REPRESENTATION_ERROR` confermati **non è sufficiente per mantenere GSFVG**. Il Gate era stato definito per stimare il costo residuo prospettico. Due siti indipendenti mostrano collegamenti automobilistici reali assenti già dalla sorgente GSFVG originale, presenti e routable nativamente in OSM. Correggerli sul backbone regionale avrebbe richiesto nuova geometria, snapping, topologia, direzione, identificativi persistenti e nuove regressioni.

Questa evidenza è stata giudicata sufficiente per:

$$
\boxed{
\text{STOP audit GSFVG estensivo}
}
$$

anche senza `REPRESENTATION_ERROR` confermati. La regola effettivamente applicata è quindi più generale della formulazione preliminare: **deficit di copertura ripetuti, indipendenti e ad alto correction burden possono essere evidenza sufficiente di un maggiore costo residuo GSFVG** quando OSM li rappresenta nativamente e senza patch equivalenti.

#### Decisione di backbone

A seguito del Gate viene assunta la decisione:

$$
\boxed{
\text{SWITCH\_OSM}
}
$$

OSM diventa il **backbone operativo light** da costruire e validare per le successive fasi della tesi.

GSFVG non viene eliminato. Rimane:

- benchmark storico;
- fonte istituzionale di confronto;
- supporto diagnostico;
- riferimento per i casi già revisionati.

Viene invece interrotto il ciclo estensivo:

$$
\text{anomalia}
\rightarrow
\text{review GSFVG}
\rightarrow
\text{patch}
\rightarrow
\text{regressione}
\rightarrow
\text{nuova anomalia}.
$$

Non vengono avviate ulteriori campagne P1/P2/P3/P4 o SP2 estensive sul GSFVG, salvo una specifica necessità locale emersa nelle fasi downstream.

Il nuovo criterio operativo è:

$$
\boxed{
\text{costruire un backbone OSM sufficientemente affidabile, validarlo e procedere}
}
$$

e non eliminare ogni possibile incertezza locale della rete.

### 4.4.7 Fase 5.6 — Backbone OSM operativo e freeze definitivo di $\Gamma_{OSM}$

#### Costruzione del backbone routing OSM

La rete operativa è stata costruita sul PBF Geofabrik Nord-Est Italia congelato:

```text
nord-est_2026-08-03.osm.pbf
```

per garantire riproducibilità temporale.

La pipeline B1--B5 tratta OSM come **rete di routing**, non come semplice insieme di LineString. La percorribilità deve quindi rispettare, per quanto rilevante al dominio light, struttura delle way, direzioni, `oneway`, reverse-oneway, roundabout, accessibilità veicolare, rampe/link, bridge, tunnel, layer, connessioni topologiche reali e restrizioni di svolta.

La distinzione è importante: con turn restrictions, la fattibilità di una mossa può dipendere non soltanto dal nodo corrente, ma anche dall'arco con cui il veicolo è arrivato. Per questo il dominio B5 viene trattato come dominio **turn-aware**.

Ai fini del Gate degli accessi sono consolidati almeno:

$$
|V_{\mathrm{B3,\ giant\ SCC}}|=889.440
$$

e:

$$
|V_{\mathrm{B5,\ mutual}}|=889.423.
$$

La riduzione:

$$
889.440\rightarrow889.423
$$

è molto contenuta, ma permette di mantenere la semantica richiesta dal routing turn-aware.

Il backbone B1--B5 resta separato dalla costruzione degli accessi: **$\Gamma_{OSM}$ non modifica il network di routing per rendersi fattibile**.

#### Backbone OSM canonico

La sorgente congelata è:

```text
C:\Tesi\Tesi_QGIS\00_originali\rete_stradale\osm\nord-est_2026-08-03.osm.pbf
```

con SHA-256:

```text
e3b8be938c6acc58be516d988f0162c768e092a5674398d9c0f43ea9d9663813
```

La pipeline canonica presenta le seguenti cardinalità:

| blocco | indicatore | valore |
|---|---|---:|
| B2 | physical nodes | 912.562 |
| B2 | physical segments | 944.219 |
| B2 | directed edges | 1.698.857 |
| B2 | CORE edges | 1.691.766 |
| B2 | LOCAL edges | 7.091 |
| B3 | giant SCC nodes | 889.440 |
| B4 | relations | 6.876 |
| B4 | compiled sequences | 6.833 |
| B4 | CORE sequences | 6.821 |
| B5 | base states | 897.857 |
| B5 | total states | 904.607 |
| B5 | transitions | 1.699.994 |

`G_OSM_operativo_v01.gpkg` è la **rappresentazione GIS dei 944.219 segmenti fisici**. Non coincide, da solo, con il routing canonico completo. La semantica di routing è costituita congiuntamente da:

$$
\boxed{
\text{B2 directed graph}
+
\text{B4 compiled turn restrictions}
+
\text{B5 selective state-expanded turn-aware graph}
}
$$


#### Perché raw node e `way_id` non sono equivalenti a nodo e `ID1` GSFVG

È stata testata e scartata la trasposizione diretta:

```text
raw OSM node ↔ nodo GSFVG
OSM way_id   ↔ ID1 GSFVG
```

Una way OSM contiene spesso molti vertici intermedi utilizzati soltanto per descrivere la forma geometrica. Considerarli tutti accessi funzionali trasformerebbe la maggiore granularità geometrica in una falsa diversificazione.

Anche `way_id` è representation-dependent: una way può attraversare più discontinuità topologiche e, viceversa, uno stesso asse stradale può essere diviso in più way per ragioni estranee alla rappresentazione zonale.

Per questo $\Gamma_{OSM}$ usa uno skeleton strutturale derivato dal backbone, ma distinto dalla rappresentazione completa usata per il routing.

#### Structural nodes

Uno `structural node` è un nodo OSM del dominio turn-aware che rappresenta una reale discontinuità topologica e non un semplice shape point.

Sono stati individuati:

$$
131.871
$$

structural nodes.

Rispetto ai:

$$
889.423
$$

nodi del dominio B5 mutual, vengono esclusi dalla sola rappresentazione $\Gamma$:

$$
757.552
$$

nodi shape-like.

Questi nodi **non vengono rimossi dal backbone**: restano necessari alla geometria e al routing, ma non sono candidati indipendenti per l'accesso zonale.

#### Topological segments

Un `topological segment` è una catena di segmenti OSM consecutivi delimitata da structural nodes.

La rete strutturale contiene:

$$
167.033
$$

topological segments.

Per un accesso $u$, sia $I(u)$ l'insieme dei topological segments incidenti. Due accessi dello stesso comune sono considerati topologicamente indipendenti se:

$$
I(u)\cap I(v)=\varnothing.
$$

La regola sostituisce il precedente uso di `ID1` GSFVG e impedisce che due punti distinti ma appartenenti alla stessa struttura locale vengano interpretati come accessi indipendenti.

#### Sensitivity della cardinalità

La cardinalità $K=3$ della precedente $\Gamma_{GSFVG}$ non è stata trasferita automaticamente. È stata sottoposta a nuova sensitivity sullo skeleton OSM.

| $K$ | comuni con topological segment condiviso | comuni con separazione minima <100 m |
|---:|---:|---:|
| 1 | 0/215 | — |
| 2 | 125/215 | 119/215 |
| 3 | 186/215 | 168/215 |
| 5 | 214/215 | 207/215 |

La selezione dei soli $K$ structural nodes più vicini è quindi fortemente ridondante. Come già osservato sul GSFVG, la molteplicità degli accessi deve derivare da una selezione esplicitamente diversificata.

#### Regola anchored strutturale di $\Gamma_o^{L,OSM}$

La sensitivity conferma:

$$
\boxed{
|\Gamma_o^{L,OSM}|=3
}
$$

per ciascuno dei 215 comuni.

La regola strutturale congelata è:

1. primary = structural node ammissibile più vicino al centroide di popolazione;
2. selezione di due secondary;
3. nessun topological segment incidente condiviso;
4. separazione euclidea pairwise almeno 100 m;
5. ottimizzazione lessicografica deterministica.

Mantenendo fisso il primary, l'ordine lessicografico:

1. minimizza la massima distanza dal centroide;
2. minimizza la somma delle tre distanze;
3. massimizza la minima separazione pairwise;
4. applica il `node_id` come tie-break deterministico.

#### Candidate search adattiva

Con $K=3$ e separazione minima 100 m, la profondità fissa della candidate pool produce:

```text
top20 → 209/215 comuni fattibili
top25 → 214/215 comuni fattibili
top30 → 215/215 comuni fattibili
```

I soli comuni che richiedono una profondità superiore a 20 sono:

| Comune | rank minimo richiesto |
|---|---:|
| Majano | 21 |
| Rigolato | 21 |
| Pontebba | 23 |
| Preone | 25 |
| San Lorenzo Isontino | 25 |
| Amaro | 29 |

Sulla popolazione completa:

$$
\operatorname{median}(rank)=6,
\qquad
p95(rank)=16,
\qquad
\max(rank)=29.
$$

Il fallimento di `top20` è quindi **candidate-pool truncation**, non non-fattibilità della regola.

Non viene congelato `top30`. La regola definitiva è:

$$
\boxed{
\text{candidate search adattiva}
}
$$

ordinata per distanza e progressivamente estesa fino alla prima profondità sufficiente a determinare la tripletta anchored ottima. Il massimo `rank=29` è un risultato empirico del PBF congelato, non un parametro metodologico.

#### Evoluzione E0

La progettazione strutturale è passata attraverso quattro versioni diagnostiche:

- `E0_v01 — REJECTED`: raw node + `way_id`, troppo dipendente dalla rappresentazione OSM;
- `E0_v02 — SUPERSEDED`: prima nozione di segmento topologico, poi corretta;
- `E0_v03 — PASS`: $K=3$, distanza minima 100 m, fattibilità 215/215 con ricerca adattiva;
- `E0_v04 — PASS`: final cardinality audit e coerenza fra candidati, domini B3/B5, structural nodes e topological segments.

Il final audit E0 restituisce:

```text
candidate rows              = 10.750
comuni                      = 215

B3 giant SCC nodes          = 889.440
B5 mutual nodes             = 889.423

structural nodes            = 131.871
candidate non-structural    = 0

topological segments        = 167.033
candidate senza incidence   = 0
```

Il Gate strutturale è quindi chiuso **senza modificare B1--B5**.

#### `Gamma_OSM` — esito finale

La costruzione di `Gamma_OSM` è stata articolata in quattro Gate:

```text
E0 = structural/cardinality design
E1 = materializzazione
E2 = validazione full access-to-access
E3 = freeze/promozione canonica
```

L'esito finale è:

```text
E0 = PASS
E1 = PASS
E2 = PASS
E3 = PASS

Gamma_OSM   = FROZEN
EXP_REL_300 = FROZEN
```

La regola congelata rimane:

$$
\boxed{
|\Gamma_o^{L,OSM}|=3
}
$$

per ogni comune FVG, con:

- primary = nearest eligible structural node al centroide di popolazione;
- due secondary topologicamente indipendenti;
- separazione pairwise almeno 100 m;
- nessun topological segment incidente condiviso;
- candidate search **ADAPTIVE**;
- ottimizzazione lessicografica: minima massima distanza dal centroide, minima somma delle distanze, massima separazione minima, tie-break deterministico sul `node_id`.

Il parametro `top30` **non appartiene alla metodologia**. Il massimo candidate rank osservato, pari a 29, è soltanto una proprietà empirica dello snapshot OSM congelato del 3 agosto 2026.

La rappresentazione strutturale frozen è:

```text
B5 mutual nodes              = 889.423
structural nodes             = 131.871
shape-like nodes esclusi     = 757.552
topological segments         = 167.033
```

La trasposizione `raw OSM node ↔ nodo GSFVG` e `OSM way_id ↔ ID1 GSFVG` resta definitivamente scartata perché representation-dependent. La rappresentazione Gamma è derivata dal backbone B5 ma **non modifica il backbone di routing**.

#### E1 — materializzazione: PASS

E1 ha materializzato:

```text
Comuni                            = 215
Accessi                           = 645
Accessi per comune                = 3

Selection failures                = 0
Determinism failures              = 0
Duplicate structural node/comune  = 0
Duplicate access_order            = 0
Primary non-nearest               = 0
Pair separation violations        = 0
Topological independence failures = 0
```

Candidate rank:

```text
median = 6
p95    = 16
max    = 29
```

Distanza primary--centroide:

```text
median = 93.52 m
p95    = 446.10 m
max    = 776.24 m
```

Extra-distance dell'accesso più lontano rispetto al primary:

```text
median = 114.00 m
p95    = 319.97 m
max    = 665.43 m
```

Separazione minima della tripletta:

```text
min    = 100.17 m
median = 130.67 m
p95    = 368.44 m
max    = 675.81 m
```

#### `EXP_REL_300`: FROZEN

La funzione di ponderazione conserva:

$$
\tau=300\ \mathrm{m}.
$$

L'audit finale restituisce:

```text
lambda > 0                  = 645/645
max errore somma/comune     = 2.220e-16

median lambda primary       = 0.4034
median lambda secondary 2   = 0.3202
median lambda secondary 3   = 0.2790

effective access count:
median                       = 2.92
min                          = 1.74
```

Pertanto:

```text
EXP_REL_300 = FROZEN
```

#### E2 — full Dijkstra turn-aware: PASS

E2 utilizza il **B5 selective state-expanded turn-aware graph**.

La source semantics è il `base-state` dell'accesso. La destination semantics è il minimo fra tutti gli state B5 associati al nodo fisico destinazione.

```text
Dijkstra                       = 645
matrice                        = 645 × 645
celle                          = 416.025
finite                         = 416.025

self                           = 645
intra-comunali ordinate        = 1.290
inter-comunali ordinate        = 414.090
off-diagonal ordinate          = 415.380

unreachable off-diagonal       = 0
unreachable intermunicipal     = 0
unreachable intramunicipal     = 0
triplette 6/6 reach FAIL       = 0/215

zero-time off-diagonal         = 0
negative-time                  = 0
physical lower-bound violation = 0
```

Asymmetry ratio:

```text
median = 1.0067
p95    = 1.0405
p99    = 1.0829
max    = 2.6920
```

Le code elevate sono localizzate e non costituiscono un bias sistemico.

Coerenza esterna delle triplette:

```text
external-out median ratio:
median = 1.0114
p95    = 1.0487
max    = 1.1405

external-in median ratio:
median = 1.0110
p95    = 1.0498
max    = 1.1512
```

Esito E2:

```text
SYSTEMIC issues = 0
BLOCKING issues = 0
E2              = PASS
```

#### E3 — freeze e promozione canonica: PASS

E3 ha promosso i prodotti auditati a riferimenti persistenti e versionati. Non introduce una nuova regola di selezione: congela la materializzazione E1, i pesi `EXP_REL_300` e la matrice E2 come artefatti canonici.

Package `Gamma_OSM`:

```text
C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_osm_light\
```

Artefatti principali:

| artefatto | SHA-256 |
|---|---|
| `Gamma_OSM_L_comuni_fvg_v01.gpkg` | `b899a2e0e29d7ef366c42f4b68ef43a25b4ba1150052b6275770dd9ae23e1d3f` |
| `Gamma_OSM_L_comuni_fvg_v01.csv` | `a2ec905f84a2df2f566d670a7522ed583e4d9235664f77e3920f96e3fca1ebb5` |
| `Gamma_OSM_E2_time_matrix_645x645_v01.npy` | `b787665e7c4064cf6ec68e1905d8db5654a69aad1538919f1d0a92adafa006db` |
| `Gamma_OSM_E2_intermunicipal_pairs_v01.csv` | `07d371be745d37115186cd9cba74727f03de1b2d82a59b1f07764943d9e5ac85` |
| `Gamma_OSM_FINAL_manifest_v01.json` | `31ee2d78cd06c75cdf07014a8cecac4af2c58adeb26ec7285362ab4c02b11416` |

#### Final Functional Gate del backbone OSM

Il freeze di Gamma non è stato considerato sufficiente a congelare automaticamente il backbone. Il backbone B1--B5 è stato quindi sottoposto a tre controlli finali indipendenti dal semplice completamento della pipeline:

```text
F1 = B5 SHADOW REGRESSION
F2 = OSM BUILD FIDELITY / DIRECTIONAL AUDIT
F3 = REGIONAL TIME-OPTIMAL METRIC AUDIT
```

##### F1 — B5 shadow regression: PASS

Sono stati riutilizzati come regression tests, senza riaprire la review manuale, i quattro siti con ground truth già congelata:

- SHADOW-01 — Cavazzo Carnico → Amaro;
- SHADOW-02 — Cercivento ↔ Paluzza;
- SHADOW-03 — Ovaro ↔ Raveo;
- SHADOW-04B — Via Andervolti, controllo negativo.

```text
Sites               = 4
Technical failures  = 0
Real B5 regressions = 0
F1                  = PASS
```

Tutte le ground truth consolidate durante il Gate preliminare OSM shadow sono riprodotte dal backbone B5 reale.

##### F2 — build fidelity / directional audit: PASS

F2 ha verificato la coerenza:

$$
\text{frozen OSM PBF}
\rightarrow
\text{B2 directed translation}
\rightarrow
\text{B5 state-expanded routing behaviour}.
$$

```text
Way semantic samples      = 6
Turn restriction samples  = 4
Total samples             = 10
Pipeline discordances     = 0
```

Le classi way testate comprendono `oneway=yes`, `oneway=-1`, bidirezionalità esplicita, roundabout, motorway/link-ramp e access-restricted/local. Le quattro famiglie di turn restriction campionate sono `NO_SEQUENCE / VIA_NODE`, `NO_SEQUENCE / VIA_WAY`, `ONLY_SEQUENCE / VIA_NODE` e `ONLY_SEQUENCE / VIA_WAY`.

Tutti i campioni hanno prodotto comportamento B5 coerente con la semantica dello snapshot OSM.

##### F3 — regional time-optimal metric audit: PASS_NO_SYSTEMIC_SIGNAL

Il confronto metrico finale è stato costruito coerentemente con la semantica ISTAT/TomTom. Per ciascuna access-pair il percorso è selezionato minimizzando il tempo B5:

$$
p^*=\arg\min_p TIME_{B5}(p),
$$

quindi la distanza di confronto è ottenuta sommando le lunghezze fisiche degli edge appartenenti al path time-optimal:

$$
D_{OSM,TIME}=\sum_{a\in p^*}\ell_a.
$$

Per questo gate diagnostico l'aggregazione comunale utilizza `PRODUCT-LAMBDA` con i pesi `EXP_REL_300` frozen.

```text
OD intercomunali ISTAT/TomTom = 10.951
access-pair analizzate         = 98.559

N_finite                       = 10.951
N_unreachable                  = 0
max regressione vs E2          = 0.000e+00 s
```

Distribuzione di:

$$
\frac{D_{OSM,TIME}^{PRODUCT-LAMBDA}}{KM_{TOT}}
$$

```text
p01 = 0.684329
p05 = 0.799860
p25 = 0.962764
p50 = 1.009302
p75 = 1.059736
p95 = 1.206755
p99 = 1.459747

Signed gap median      = +0.2598 km
MAE                    = 2.3490 km
Median absolute error  = 1.3055 km
p95 absolute error     = 8.0440 km
Pendolari-weighted MAE = 1.8222 km
```

Spread fra le nove access-pair:

```text
Access-pair distance spread:
median = 0.8390 km
p95    = 3.4069 km

Access-pair time spread:
median = 84.58 s
p95    = 273.12 s
```

Sono state identificate 201 OD estreme. La classificazione complessiva è:

```text
EXPECTED_MODEL_DIFFERENCE = 10.750
ACCESS_EFFECT             = 23
LOCAL_ROUTING_ANOMALY     = 178
BOUNDARY_OR_DOMAIN        = 0
POSSIBLE_SYSTEMIC         = 0
UNRESOLVED                = 0
```

Non emergono segnali sistemici:

```text
SYSTEMIC_CONNECTIVITY_ERROR = NO
SYSTEMIC_DISTANCE_INFLATION = NO
SYSTEMIC_ROUTING_BIAS       = NO
```

I 178 casi `LOCAL_ROUTING_ANOMALY` non giustificano una nuova campagna manuale. Il Gate è finalizzato a individuare bias sistemici materiali per assignment e FRLM, non a eliminare ogni possibile anomalia locale. Il rapporto mediano:

$$
\frac{D_{OSM,TIME}^{PRODUCT-LAMBDA}}{KM_{TOT}}=1.009302
$$

è coerente con l'assenza di inflazione metrica aggregata sistematica.

##### Freeze rule finale

Poiché:

```text
F1 = PASS
F2 = PASS
F3 = PASS_NO_SYSTEMIC_SIGNAL
Gamma_OSM = FROZEN
SYSTEMIC issues = 0
BLOCKING issues = 0
```

si dichiara:

$$
\boxed{
G_{OSM,operativo}=FROZEN
}
$$

ed è applicata la decisione:

```text
NETWORK AUDIT = STOP
```

La fase di costruzione e validazione generale della rete stradale light è quindi **formalmente chiusa**. Una fase downstream può riaprirla esclusivamente dimostrando un errore `BLOCKING` materiale per OD assignment o FRLM; in tal caso deve essere riaperto soltanto il componente causale necessario.

#### Package canonico del backbone frozen

Package:

```text
C:\Tesi\Tesi_QGIS\02_package\grafo_operativo_osm
```

| artefatto | contenuto | SHA-256 |
|---|---|---|
| `G_OSM_operativo_v01.gpkg` | 944.219 segmenti fisici GIS | `f1d87245d1bc28f3ecab16e126514f8a3ab718b73ce7bd244db2f12628697ef3` |
| `G_OSM_FINAL_manifest_v01.json` | manifest finale | `c4ea80c9c660f6a513b20400d0c3edb2f9e6ec10c3364f0a4b0beed91af55da3` |
| `G_OSM_FINAL_GATE_REPORT_v01.txt` | report del Final Functional Gate | checksum registrato nel package finale |

Il GeoPackage fisico non sostituisce B2, B4 e B5: il routing canonico frozen resta la composizione **B2 directed graph + B4 compiled turn restrictions + B5 selective state-expanded turn-aware graph**.

#### Stato finale della Fase 5.6

```text
FASE 5.6                         = COMPLETATA
G_OSM_operativo                 = FROZEN
Gamma_OSM                       = FROZEN
EXP_REL_300                     = FROZEN
NETWORK AUDIT                   = STOP

SYSTEMIC issues aperte          = 0
BLOCKING issues aperte          = 0
```

Il backbone operativo light della tesi è OSM. GSFVG resta baseline storica, benchmark istituzionale, supporto diagnostico e fonte di confronto per casi già verificati.

Alla chiusura della Fase 5.6 il blocco successivo era la materializzazione di impedenza e percorsi OD. Tale blocco è stato completato e congelato nella **Fase 5.7**, documentata nella §4.4.8. In seguito la Fase 5.8A ha congelato gli input territoriali Gravity v0 e ha superseduto la precedente priorità gateway: il `NEXT` corrente è **Fase 5.8B — Gravity Model v0 FVG-only**, senza riapertura del routing interno.


### 4.4.8 Fase 5.7 — Sistema canonico dei percorsi OD interni OSM

#### Chiusura del gate

La Fase 5.7 trasforma il backbone e gli accessi già congelati nella Fase 5.6 in un **sistema persistente di percorsi OD interni FVG**, utilizzabile direttamente dalle elaborazioni downstream senza dover ricalcolare il routing a ogni passaggio.

Lo stato ratificato è:

```text
FASE 5.7                         = CLOSED / FROZEN
OD_PATH_SYSTEM_OSM              = FROZEN
CANONICAL_ROUTE_IMPEDANCE       = TIME_B5
DISTANCE                        = PATH_ATTRIBUTE
PRODUCT_LAMBDA_PATH_WEIGHTS     = FROZEN
ACCESS_WEIGHT_MODEL             = EXP_REL_300
NETWORK AUDIT                   = STOP
SYSTEMIC issues                 = 0
BLOCKING issues                 = 0
```

Non sono richiesti ulteriori test della Fase 5.7.

#### Contratto di routing frozen

Per ogni coppia ordinata di accessi, il percorso canonico è:

$$
\boxed{
p^*=\arg\min_p TIME_{B5}(p)
}
$$

La distanza non determina il percorso. Viene calcolata **dopo** la scelta del path e conservata come attributo:

$$
\boxed{
DISTANCE=PATH\_ATTRIBUTE
}
$$

Il routing canonico rimane quello già congelato nella Fase 5.6:

$$
\boxed{
\text{B2 directed graph}
+
\text{B4 compiled turn restrictions}
+
\text{B5 selective state-expanded turn-aware graph}
}
$$

La specificazione frozen non include congestione, costo generalizzato, pedaggi, penalità di comfort o di classe stradale, turn penalty artificiali, scelta stocastica del percorso, $k$-shortest paths o diversion paths. Questi elementi non devono essere retro-iniettati nel package 5.7.

#### Dominio OD interno materializzato

Il dominio della Fase 5.7 comprende tutti i 215 comuni FVG e tutte le relazioni comunali ordinate con origine diversa dalla destinazione:

$$
215\times214=46.010.
$$

Ogni comune possiede tre accessi `Gamma_OSM`. Per ogni OD comunale vengono quindi considerate:

$$
3\times3=9
$$

coppie ordinate di accessi, per un totale di:

$$
46.010\times9=414.090
$$

access-pair path.

L'esito della materializzazione è:

```text
municipal OD                    = 46.010 / 46.010
access-pair paths               = 414.090 / 414.090
finite                          = 414.090
unreachable                     = 0
coverage                        = 100%
```

Il dominio non è limitato alle sole 10.951 OD ISTAT/TomTom con valore positivo utilizzate nel gate metrico F3. La copertura completa $215\times214$ è necessaria perché una futura seed gravitazionale può generare domanda anche su celle oggi nulle nei dati osservati.

#### Pesi `PRODUCT-LAMBDA`

Per l'accesso di origine $a$ e l'accesso di destinazione $b$:

$$
w_{ab}
=
\lambda_{o,a}\lambda_{d,b},
$$

dove i $\lambda$ provengono dal modello frozen `EXP_REL_300`.

Per ogni OD comunale:

$$
\sum_{a,b} w_{ab}=1,
$$

con massimo errore numerico osservato:

```text
5.551e-16
```

Il significato metodologico è preciso: i nove access-pair **non sono nove percorsi alternativi fra cui l'utente sceglie**. La struttura è:

$$
\boxed{
\text{flusso OD comunale}
\rightarrow
\text{PRODUCT-LAMBDA sui 9 access-pair}
\rightarrow
\text{un solo path TIME\_B5-optimal per ciascun access-pair}
}
$$

`PRODUCT-LAMBDA` descrive quindi la rappresentazione multi-accesso della zona, non un modello di route choice multipath.

#### Ricostruzione dei path e validazione

La ricostruzione dei path è completa:

```text
Path reconstruction                         = 100%

path reconstruction failures                = 0
sequence discontinuity                      = 0
source mismatch                             = 0
destination mismatch                        = 0
forbidden transition violation              = 0
transition-count mismatch                   = 0
time consistency failure                    = 0
distance consistency failure                = 0
```

Le sequenze congelate contengono:

```text
stored B5 transition slots = 598.707.601
```

La regressione completa contro il dominio F3 restituisce:

```text
98.559 / 98.559 = PASS

max time error       = 9.095e-13 s
max distance error   = 9.022e-10 m
max pair-weight error= 9.714e-17
```

Si tratta di residui numerici irrilevanti e non costituiscono nuove issue.

#### Architettura di storage frozen

Il package canonico è:

```text
C:\Tesi\Tesi_QGIS\02_package\od_paths_osm_light\
```

L'architettura evita un GeoPackage massivo path-edge e separa:

$$
\boxed{
\text{RELATIONAL PATH METADATA}
+
\text{CSR-LIKE PATH SEQUENCES}
+
\text{B5 FROZEN BACKBONE}
}
$$

Gli artefatti canonici esplicitati sono:

| artefatto | cardinalità / tipo | SHA-256 |
|---|---:|---|
| `OSM_OD_access_paths_v01.csv` | 414.090 record | `3c0a8786a05719db4ca2a4258250bde8937b8dd017d93fea0c8f4a8a101c8dd3` |
| `OSM_OD_municipal_summary_v01.csv` | 46.010 record | `dfa2db5e3b18c9c1c7f05e2c7b7445848904971f35c916390bfc4d1b34571441` |
| `OSM_OD_path_offsets_v01.npy` | 414.091 elementi | `478efd3a3f6eba6964db9f0a785dfd9405d5ae61af30e4f84538b0699a7a3a08` |
| `OSM_OD_transition_slots_v01.npy` | 598.707.601 elementi, `int32` | `2a6b06d21b6d3eea7132a0154bbb4c74d305a4ea582d780e07b5abeed24d2c1d` |
| `OSM_OD_sequence_contract_v01.json` | contratto sequenze | `59ea1121d56c7cc1515e5aa5d40087a66fcf51ce6a1b1b763797baed70844364` |
| `OSM_OD_F1_regression_v01.csv` | regressione finale | `11310670ed7ccf2cb99d2a56b59f656e64940cd68354d712944b4a77e379b2bc` |
| `OSM_OD_PATHS_FINAL_REPORT_v01.txt` | final gate report | `370aff831af437cc8f09d895f16d78f9fd8ed3856bbb2a726863c1b078787075` |
| `OSM_OD_PATHS_manifest_v01.json` | manifest finale | `9c3279c604685dbb8ebf52d18910658efc5fb7fe0ab1069d36d313476abb8fff` |

Il package finale registra complessivamente **9 file** per circa **2,371 GB**; il manifest costituisce l'inventario autorevole dell'intero contenuto.

```text
Atomic promotion = PASS
SHA verification = PASS
```

La geometria resta ricostruibile senza duplicarla nel package:

```text
transition slot
→ B5
→ edge_id
→ B2 directed edge
→ physical OSM segment
```

#### Confine fra Fase 5.7 e gateway esterni

`OD_PATH_SYSTEM_OSM` frozen riguarda esclusivamente:

$$
215\times214
$$

OD comunali **interne FVG**.

Le zone esterne costituiscono una estensione downstream separata e versionata. Pertanto non si deve:

- modificare il package 5.7;
- aggiungere zone esterne dentro i suoi artefatti;
- rigenerare i 414.090 path interni;
- alterare `Gamma_OSM`;
- riaprire il routing interno.

Al momento della chiusura della Fase 5.7 il gate previsto era la rappresentazione di gateway e domanda esterna come nuovo strato, mantenendo il sistema interno frozen come dipendenza immutabile. Questa priorità è **storica** ed è stata successivamente superseduta dalla decisione di regia post-5.8A: la prima baseline LIGHT v0 viene chiusa sul solo dominio FVG e il `NEXT` corrente diventa la Fase 5.8B — Gravity Model v0 FVG-only. Gateway ed external demand rimangono una estensione downstream separata.

#### Stato finale della Fase 5.7

```text
FASE 5.7                         = CLOSED / FROZEN
OD_PATH_SYSTEM_OSM              = FROZEN
CANONICAL_ROUTE_IMPEDANCE       = TIME_B5
DISTANCE                        = PATH_ATTRIBUTE
PRODUCT_LAMBDA_PATH_WEIGHTS     = FROZEN
ACCESS_WEIGHT_MODEL             = EXP_REL_300
SYSTEMIC                        = 0
BLOCKING                        = 0
STOP FASE 5.7
NEXT_AT_5_7_CLOSE              = GATEWAY / EXTERNAL DEMAND  [STORICO / SUPERSEDED]
```

---

# 5. Corridoi e direzioni di marcia

## 5.1 Insieme dei corridoi

Si definisce:

$$
\mathcal R=\{r_1,r_2,\ldots,r_{|\mathcal R|}\}
$$

come l'insieme degli assi stradali considerati rilevanti ai fini della ricarica.

Un corridoio può corrispondere a:

- un'autostrada;
- una strada statale;
- un asse TEN-T;
- una sequenza coerente di archi della rete;
- una direttrice regionale funzionalmente continua.

## 5.2 Posizione longitudinale

Per ogni candidato $c$ appartenente al corridoio $r$, si definisce:

$$
\xi_{rc}\geq 0
$$

come la posizione longitudinale del candidato lungo il corridoio, misurata da un'origine convenzionale.

La distanza lungo il corridoio tra due candidati $c$ e $c'$ è:

$$
d_r(c,c')=|\xi_{rc}-\xi_{rc'}|.
$$

Questa distanza non deve essere sostituita dalla distanza euclidea.

## 5.3 Direzione

Si definisce una funzione:

$$
\operatorname{dir}:\mathcal C_s\rightarrow\mathcal D
$$

con:

$$
\mathcal D=\{+,-,\pm\}
$$

dove:

- $+$: candidato accessibile nella direzione positiva del corridoio;
- $-$: candidato accessibile nella direzione negativa;
- $\pm$: candidato effettivamente accessibile da entrambe le direzioni.

Due nodi posti su carreggiate opposte non devono essere automaticamente aggregati. Possono essere considerati un unico candidato bidirezionale soltanto quando esiste un accesso realistico e compatibile con la regolazione stradale.

---

# 6. Nodi candidati alla ricarica

## 6.1 Definizione

Per ogni segmento $s$ si definisce:

$$
\mathcal C_s
$$

come l'insieme dei nodi candidati alla localizzazione della capacità di ricarica.

I candidati possono essere:

1. **reali**, se derivano da un oggetto fisico noto, come area di servizio, parcheggio, distributore o piazzola;
2. **virtuali**, se vengono introdotti dal modello per rappresentare una finestra territoriale di localizzazione;
3. **ibridi**, se un nodo virtuale rappresenta un gruppo di possibili siti fisici equivalenti ai fini della pianificazione strategica.

Un candidato virtuale non equivale a una raccomandazione catastale. Esso indica che la capacità dovrebbe essere collocata in una porzione limitata del corridoio e all'interno di un determinato comune.

## 6.2 Funzioni di associazione

Per ogni candidato $c\in\mathcal C_s$ si definiscono:

$$
\mu(c)\in\mathcal M
$$

comune nel quale ricade il candidato;

$$
\rho(c)\in\mathcal R
$$

corridoio al quale è associato;

$$
\operatorname{type}(c)\in\{\text{reale},\text{virtuale},\text{ibrido}\}
$$

tipologia del candidato.

Il candidato può quindi essere identificato dal quadruplo concettuale:

$$
\bigl(\mu(c),\rho(c),\operatorname{dir}(c),\operatorname{type}(c)\bigr).
$$

La connessione elettrica non è una funzione rigida candidato--cabina. Si definisce invece la relazione molti-a-molti:

$$
\mathcal G_s\subseteq\mathcal C_s\times\mathcal B,
$$

dove $(c,b)\in\mathcal G_s$ indica che la cabina $b$ è tecnicamente ammissibile per il candidato $c$. Ogni candidato può avere più alternative e ogni cabina può servire più candidati; la connessione effettiva è una decisione del modello basata su distanza, costo, capacità e vincoli tecnici.

## 6.3 Finestra di localizzazione

A ogni candidato si può associare una tolleranza geografica o stradale:

$$
\varepsilon_c^{\mathrm{loc}}\geq 0.
$$

Essa rappresenta la distanza massima entro la quale il policy maker può individuare il sito esecutivo senza modificare sostanzialmente il significato strategico della soluzione.

Questa tolleranza:

- non deve essere interpretata come automaticamente valida per qualsiasi strada;
- deve rispettare l'accessibilità reale dal corridoio;
- deve distinguere le direzioni di marcia quando necessario;
- deve essere verificata rispetto alle norme applicabili.

Nel caso delle infrastrutture conteggiate ai fini AFIR lungo la rete TEN-T, il riferimento dei 3 km riguarda la distanza di guida dall'uscita più vicina per la qualificazione del gruppo di ricarica come infrastruttura “lungo” la rete TEN-T. Non costituisce una regola universale secondo cui ogni punto entro 3 km sia tecnicamente equivalente.

---

# 7. Generazione dei candidati

## 7.1 Criteri di costruzione

La normativa definisce requisiti minimi di copertura che devono essere rispettati, ma non determina a priori una griglia uniforme di candidati con passo fisso. L'insieme dei siti sarà costruito e raffinato caso per caso considerando:

- topologia e direzioni della rete;
- infrastrutture di ricarica esistenti;
- distribuzione della domanda e corridoi serviti;
- vincoli tecnici, territoriali ed elettrici;
- trattabilità computazionale.

Il modello potrà attivare una densità di stazioni superiore al minimo normativo quando necessaria per esigenze locali o per migliorare la soluzione globale. La soglia normativa resta un vincolo di copertura, non il passo di generazione dei candidati.

## 7.2 Procedura proposta

Per ogni segmento $s$ e corridoio $r$:

1. si selezionano gli archi fisici appartenenti al corridoio;
2. si ordinano gli archi secondo la posizione longitudinale;
3. si inseriscono come candidati gli oggetti reali già noti;
4. si identificano i cambi di comune e gli accessi principali;
5. si creano nodi concettuali comune-corridoio;
6. si aggiungono nodi virtuali soltanto dove domanda, copertura, accessibilità o vincoli tecnici rendono utile ampliare le alternative;
7. ogni nuovo nodo viene agganciato alla rete spezzando l'arco fisico interessato;
8. si assegnano comune, corridoio, direzione, tipo e alternative ammissibili di connessione elettrica;
9. si eliminano o fondono candidati soltanto se realmente equivalenti per accessibilità e funzione.

## 7.3 Pseudocodice

```text
per ogni segmento s:
    costruisci il grafo stradale operativo G_s^R

    per ogni corridoio r:
        estrai la sequenza ordinata degli archi del corridoio
        inserisci candidati reali già noti
        inserisci candidati ai principali accessi e passaggi comunali

        valuta lacune di copertura e necessità locali
        genera candidati virtuali soltanto dove giustificato
        spezza gli archi fisici nei punti selezionati

        assegna a ogni candidato:
            comune
            corridoio
            direzione
            tipo
            cabine elettriche compatibili
            posizione longitudinale
```

---

# 8. Riferimenti normativi come parametri del modello

## 8.1 Principio generale

La norma non deve essere utilizzata soltanto per decidere quanto fitta debba essere la griglia dei candidati. Deve essere rappresentata anche attraverso vincoli separati, perché il modello potrebbe altrimenti scegliere candidati troppo distanti tra loro.

Si definiscono quindi:

- $\overline D_{sr}^{\mathrm{norm}}$: distanza massima ammessa tra gruppi di ricarica;
- $\underline P_{sr}^{\mathrm{norm}}$: potenza aggregata minima richiesta;
- $\underline n_{sr}^{\mathrm{norm}}$: numero minimo di punti di una data potenza;
- $\mathcal R_s^{\mathrm{norm}}\subseteq\mathcal R$: corridoi soggetti al requisito;
- $Y_{sr}^{\mathrm{target}}$: anno-obiettivo della disposizione.

## 8.2 Quadro AFIR provvisorio da verificare

Il riferimento europeo preliminare è il Regolamento (UE) 2023/1804.[^afir2023] Valori, scadenze, soglie e appartenenza dei corridoi alle reti interessate non sono assunti come definitivamente verificati nel presente aggiornamento.

Per i veicoli leggeri:

- rete centrale TEN-T: distanza massima 60 km in ciascuna direzione;
- entro il 31 dicembre 2025: almeno 400 kW aggregati e almeno un punto da 150 kW;
- entro il 31 dicembre 2027: almeno 600 kW aggregati e almeno due punti da 150 kW;
- rete globale TEN-T: distanza massima 60 km; potenze e scadenze differenziate, con 300 kW nella prima fase e 600 kW nella fase successiva.

Per i veicoli pesanti:

- entro il 31 dicembre 2030, rete centrale TEN-T: distanza massima 60 km, almeno 3.600 kW aggregati e almeno due punti da 350 kW;
- entro il 31 dicembre 2030, rete globale TEN-T: distanza massima 100 km, almeno 1.500 kW aggregati e almeno un punto da 350 kW;
- sono inoltre previsti requisiti specifici per aree di parcheggio sicure e protette e nodi urbani.

Questi valori non devono essere applicati indiscriminatamente a tutta la rete regionale. Devono essere associati soltanto ai corridoi, agli anni-obiettivo, alle direzioni e alle eventuali deroghe effettivamente pertinenti.

Prima della versione finale della tesi saranno ricontrollati testo vigente, scadenze, soglie applicabili e classificazione dei corridoi regionali. I requisiti saranno applicati esclusivamente alle categorie di infrastruttura, alle tratte e agli orizzonti temporali pertinenti.

## 8.3 Vincolo di copertura normativa

Per ogni corridoio normato $r$, segmento $s$ e direzione, si costruisce un insieme di finestre longitudinali:

$$
\mathcal W_{sr}.
$$

Ogni finestra $w\in\mathcal W_{sr}$ rappresenta un intervallo del corridoio nel quale deve essere presente almeno un candidato attivato affinché non si crei una distanza superiore alla soglia.

Si definisce:

$$
\mathcal C_{sr}(w)
=
\{c\in\mathcal C_s:\rho(c)=r,\ c\text{ ricade nella finestra }w\}.
$$

Un vincolo di copertura può essere scritto come:

$$
\sum_{c\in\mathcal C_{sr}(w)}x_c^s\geq 1
\qquad
\forall w\in\mathcal W_{sr}.
$$

La costruzione delle finestre deve includere correttamente:

- estremi del corridoio;
- infrastrutture esistenti;
- direzioni di marcia;
- eventuali deroghe;
- nodi esterni immediatamente adiacenti al confine regionale.

---

# 9. Matrici OD e percorsi

## 9.1 Matrice OD

Per ciascun segmento $s$, il flusso OD è:

$$
T_{od}^s\geq 0
\qquad
\forall o,d\in\mathcal Z,
\quad o\neq d.
$$

La versione base considera soltanto relazioni interzonali. Gli spostamenti intrazonali $T_{oo}$ sono esclusi perché non generano percorsi intercomunali utili alla localizzazione HPC sulla rete regionale; potranno essere rappresentati separatamente in futuro se rilevanti per la ricarica locale.

La mobilità leggera e quella pesante hanno matrici distinte:

$$
\mathbf T^{\mathrm L}
\neq
\mathbf T^{\mathrm H}.
$$

La matrice può derivare da fonti e metodi diversi. Per esempio:

### Mobilità leggera

$$
T_{od}^{\mathrm L}
=
C_{od}^{\mathrm{pend}}
+
N_{od}^{\mathrm{nonpend}}
+
E_{od}^{\mathrm{est}}.
$$

### Mobilità pesante

Costruzione e calibrazione della matrice heavy-duty sono rinviate a una fase successiva. La formulazione heavy precedente, inclusi eventuali termini di correzione basati sui conteggi, è provvisoria e non viene adottata nella fase corrente. Dovrà essere riesaminata separatamente rispettando il principio che i conteggi sono osservati sui link e non identificano direttamente le celle OD.

### 9.1.1 Architettura corrente della matrice light

Per $o,d\in\mathcal Z$, $C_{od}^{\mathrm{pend}}$ è la componente pendolare e $N_{od}^{\mathrm{nonpend}}$ la componente leggera non pendolare. **La baseline LIGHT v0 corrente è FVG-only su 215 comuni**: la componente esterna $E_{od}^{\mathrm{est}}$ non è implementata nella v0 e resta una estensione downstream separata. Pertanto, nella baseline v0, $T_{od}^{\mathrm L}$ contiene soltanto le componenti interne FVG ed è espressa in veicoli/giorno medio annuo. La formulazione generale con $E_{od}^{\mathrm{est}}$ resta valida come architettura futura, ma non descrive lo stato implementativo della v0.

#### Conversione minima dei pendolari in veicoli/giorno medio annuo — decisione consolidata

Il campo **Pendolari**, indicato nel seguito con $P_{od}^{\mathrm{ISTAT}}$, misura le persone occupate residenti nell'origine $o$ che raggiungono il luogo abituale di lavoro nella destinazione $d$ almeno tre giorni alla settimana e rientrano giornalmente alla residenza.[^istat2021] Il dato descrive quindi persone e relazioni abituali casa--lavoro, non veicoli né spostamenti osservati in un giorno specifico.

Per convertire le persone in veicoli si adotta la quota modale regionale:

$$
s_{\mathrm{driver}}=0{,}711,
$$

corrispondente alla quota di occupati del Friuli Venezia Giulia che utilizza l'auto privata come **conducente** per raggiungere il luogo di lavoro.[^istat-driver-fvg-2019] Non si introduce un coefficiente separato di occupazione del veicolo: la quota non identifica gli occupanti complessivi di un'auto, ma direttamente i conducenti; nella conversione minima ciascun conducente genera pertanto un veicolo.

Il ritorno non è trattato come un coefficiente comportamentale da stimare. Poiché la definizione ISTAT richiede il rientro giornaliero alla residenza, per ogni relazione osservata $o\to d$ si costruiscono due flussi direzionali simmetrici: l'andata $o\to d$ e il ritorno $d\to o$. Questa scelta conserva esplicitamente la direzione dei flussi e non raddoppia il valore di una singola direzione.

Per riportare la relazione abituale al **giorno medio annuo** si assume:

$$
n_{\mathrm{pend}}=220\ \text{giorni/anno},
\qquad
g_{\mathrm{ann}}=\frac{220}{365}=0{,}60274.
$$

Il valore di 220 giorni è un'ipotesi operativa di pianificazione, coerente con intervalli e applicazioni utilizzati in piani degli spostamenti casa--lavoro ISTAT,[^istat-pscl-roma-2022][^istat-pscl-piemonte-2022] ma non è una misura osservata specificamente per il Friuli Venezia Giulia.

La conversione definitiva per ciascuna direzione è:

$$
C_{od}^{\mathrm{and}}
=
P_{od}^{\mathrm{ISTAT}}\,s_{\mathrm{driver}}\,g_{\mathrm{ann}},
\qquad
C_{do}^{\mathrm{rit}}=C_{od}^{\mathrm{and}}.
$$

Il coefficiente risultante è quindi:

$$
\alpha_{\mathrm{pend}}
=
0{,}711\cdot\frac{220}{365}
=
0{,}42855.
$$

Di conseguenza, 100 pendolari ISTAT producono **42,855 veicoli/giorno medio annuo in andata** e **42,855 veicoli/giorno medio annuo in ritorno**. Il coefficiente $0{,}42855$ non è un dato osservato: deriva dalla combinazione della quota modale osservata $s_{\mathrm{driver}}$ e dell'ipotesi di annualizzazione $n_{\mathrm{pend}}=220$.

La sensibilità alla sola ipotesi sul numero di giorni di pendolarismo è:

| Giorni di pendolarismo annui $n_{\mathrm{pend}}$ | $\alpha_{\mathrm{pend}}=0{,}711\,n_{\mathrm{pend}}/365$ |
|---:|---:|
| 200 | 0,38959 |
| 220 | 0,42855 |
| 230 | 0,44803 |

La variante a 220 giorni costituisce la specificazione base; le varianti a 200 e 230 giorni sono scenari di sensibilità e non modificano la decisione consolidata.

### 9.1.2 Fase 5.8A — input territoriali RAW della Gravity v0

La **Fase 5.8A è CLOSED / FROZEN**. Il suo obiettivo era predisporre e congelare i tre vettori territoriali RAW necessari alla successiva Gravity v0, senza normalizzarli e senza costruire il modello gravitazionale.

La baseline è riferita ai **215 comuni FVG** e usa i seguenti input canonici:

| Componente | Definizione frozen | Riferimento | Copertura | Somma di controllo |
|---|---|---|---:|---:|
| produzione territoriale RAW | `PARCO_AUTO_v0 = ACI_AUTOVETTURE_2024` | ACI — *Autoritratto 2024, Parco veicolare* | 215/215 | 828.909 autovetture |
| attrazione turistica RAW | `TURISMO_v0 = PRESENZE_TOTALI_2024` | ISTAT | 215/215 | 10.143.980 presenze |
| attrazione commerciale RAW | `GDO_v0 = STRUCTURED_RETAIL_AREA_2023` | Osservatorio regionale del commercio FVG | 215/215 | 1.480.796,13 m² |

#### Parco auto

Per `PARCO_AUTO_v0` vengono utilizzate le autovetture ACI 2024. Le 20 autovetture associate a `COMUNE = NON DEFINITO` sono escluse e **non imputate**. La somma comunale frozen è 828.909 autovetture.

#### Turismo

`TURISMO_v0` utilizza le **presenze turistiche totali 2024**, non i soli posti letto. La copertura finale 215/215 è composta da:

- 178 comuni `OBSERVED`;
- 35 comuni `IMPUTED_BEDS_RESIDUAL`;
- 2 comuni `ZERO_CAPACITY`.

L'imputazione è applicata esclusivamente ai 35 comuni oscurati e preserva il totale regionale. La regola è:

$$
TURISMO\_RAW_j
=
14.944\,\frac{BEDS_j}{944}.
$$

Il totale regionale frozen è 10.143.980 presenze.

#### GDO

`GDO_v0 = STRUCTURED_RETAIL_AREA_2023` misura la **superficie comunale di vendita della distribuzione commerciale strutturata medio-grande (>400 m²)** secondo le categorie dell'Osservatorio regionale del commercio FVG. La formula frozen è:

$$
GDO\_RAW_j
=
MEDIA\_SUP\_MQ_j
+
GRANDE\_SINGOLA\_MQ_j
+
CENTRO\_COMMERCIALE\_MQ_j
+
COMPLESSO\_COMMERCIALE\_MQ_j.
$$

Il riferimento temporale è il **31/12/2023**. La copertura è 215/215 comuni, la somma è 1.480.796,13 m² e il `DOUBLE COUNTING CHECK = PASS`. I tre comuni con valore regionale nullo — Drenchia, Grimacco e San Floriano del Collio-Števerjan — sono documentati mediante evidenza esterna; la provenance distingue `REGIONAL_TABLE` ed `EXTERNAL_EVIDENCE`. Non viene introdotta alcuna imputazione positiva.

#### Master input table canonica

L'artefatto canonico è:

`Gravity_v0_territorial_inputs_raw.xlsx`

con:

```text
STATUS = CANONICAL
FROZEN = YES
SHA256 = 3dab79eabaf4e7f55a0e8244f02a1ebbae1f80355c0c3882a3b03ec90ff2724e
SIZE   = 64.156 bytes
SHEET  = MASTER_RAW
```

La tabella contiene almeno `PRO_COM`, `COMUNE`, `PARCO_AUTO_RAW`, `TURISMO_RAW`, `GDO_RAW`, le rispettive informazioni di fonte/provenance e `BEDS_2024`. L'hard QA finale registra 215 righe, 215 `PRO_COM` distinti, 0 duplicati, 0 missing, 0 valori negativi e 0 cross-source mismatch; tutte le somme di controllo coincidono con i totali frozen sopra riportati.

**Confine del gate:** la 5.8A congela gli input RAW e la loro provenance. Non normalizza gli indicatori, non sceglie $\beta$ o $Q$, non costruisce $N_{ij}^{0}$, non esegue assignment e non utilizza ANAS. Queste attività appartengono alla Fase 5.8B o ai gate downstream successivi.

### 9.1.3 Seed non pendolare gravitazionale

La componente non pendolare è stata studiata nella Fase 5.8E mediante una **specifica parametrica FVG-only sperimentale**, fondata sulla struttura:

$$
W_{ij}(\beta)=P_iA_j\exp(-\beta c_{ij}),
$$

$$
S_{ij}(\beta)=
\frac{W_{ij}(\beta)}
{\displaystyle\sum_{r\neq s}W_{rs}(\beta)},
$$

$$
N_{ij}(Q,\beta)=Q\,S_{ij}(\beta),
\qquad
N_{ii}=0.
$$

Nella specifica testata:

- $P_i$ è il proxy di parco auto normalizzato L1;
- $A_j=0{,}5\,tourism\_regional\_share_j+0{,}5\,GDO\_regional\_share_j$;
- $c_{ij}$ è l'impedenza comunale `TIME_B5 PRODUCT-LAMBDA`, espressa in minuti;
- $Q$ è la domanda sintetica LIGHT intercomunale non pendolare complessiva;
- $\beta$ è il parametro di deterrenza esponenziale, in $1/\mathrm{min}$.

Questa formulazione **non è frozen come modello finale**. Dopo la Fase 5.8E deve essere trattata come:

```text
INCUMBENT REFERENCE / FVG-ONLY EXPERIMENTAL MODEL
```

La base teorica della stima parametrica da conteggi resta preservata, ma $Q$, $\beta$ e la specificazione finale della Gravity devono essere riaperti soltanto dopo l'estensione del dominio di domanda external/transborder.

Nel modello gravitazionale doppiamente vincolato:

$$
T_{od}=A_o^{\mathrm{bal}}O_o B_d^{\mathrm{bal}}D_d f(c_{od};\beta),
$$

$O_o$ e $D_d$ sono rispettivamente i margini di origine e destinazione. I fattori di bilanciamento $A_o^{\mathrm{bal}}$ e $B_d^{\mathrm{bal}}$ sono calcolati iterativamente, una volta noti margini, $\beta$ e costi; non sono parametri indipendenti da stimare liberamente. La distribuzione gravitazionale dei viaggi resta distinta dalla successiva stima o revisione della matrice mediante conteggi.

### 9.1.4 Seed light complessiva

Per la **baseline LIGHT v0 FVG-only** la seed da costruire in 5.8B è:

$$
T_{od}^{0,\mathrm L,v0}
=
C_{od}^{\mathrm{pend}}+N_{od}^{0},
\qquad o,d\in\mathcal M_{FVG},\ o\neq d.
$$

La componente esterna è esplicitamente:

```text
EXTERNAL_OD = NOT_IMPLEMENTED / DOWNSTREAM EXTENSION
```

L'architettura generale futura rimane:

$$
T_{od}^{0,\mathrm L,full}
=
C_{od}^{\mathrm{pend}}+N_{od}^{0}+E_{od}^{0},
$$

ma $E_{od}^{0}$ non appartiene alla prima baseline v0 e non deve richiedere modifiche al package interno frozen della Fase 5.7.

### 9.1.5 Assegnazione alla rete e confronto con ANAS

Per ogni coppia $(o,d)$ e link $a\in\mathcal A_{\mathrm L}$, la matrice di assegnazione OD--link è definita da:

$$
p_{od}^{a,\mathrm L}
=
\sum_{q\in\mathcal Q_{od}^{\mathrm L}}
\pi_{q\mid od}^{\mathrm L}\,\delta_{aq},
$$

dove $\pi_{q\mid od}^{\mathrm L}$ è la quota del flusso light $o\to d$ assegnata al percorso $q$ e $\delta_{aq}$ vale 1 se il link $a$ appartiene a $q$, 0 altrimenti. Il flusso stimato è:

$$
\widehat y_a^{\mathrm L}
=
\sum_{o\in\mathcal Z}\sum_{d\in\mathcal Z}
p_{od}^{a,\mathrm L}T_{od}^{\mathrm L}.
$$

Per il dominio light interno FVG il contratto di percorso non è più una proposta futura: è congelato dalla Fase 5.7. Per ciascuna coppia ordinata di accessi viene utilizzato un unico percorso $p^*=\arg\min TIME_{B5}$; la distanza è `PATH_ATTRIBUTE`. Il flusso di ciascuna OD comunale viene distribuito sui nove access-pair mediante `PRODUCT-LAMBDA`, con pesi $\lambda_{o,a}\lambda_{d,b}$ derivati da `EXP_REL_300`, e ciascun access-pair utilizza il proprio singolo path `TIME_B5`-ottimo. Non vi sono scelta stocastica, $k$-shortest o diversion paths nel sistema frozen. Tutte le 46.010 OD comunali interne sono già coperte, comprese quelle oggi nulle nei dati ISTAT; ciò consente alla futura seed gravitazionale di generare domanda senza richiedere il ricalcolo del routing interno.

Soltanto per una futura seed gravitazionale molto densa potrà eventualmente essere introdotta una soglia **sulla domanda da assegnare**, non sul dominio dei path già materializzato. Qualsiasi soglia dovrà quantificare il flusso escluso, documentare la regola di taglio e verificare mediante sensibilità che corridoi e flussi sui link non cambino in misura sostanziale; il package dei 414.090 path interni resta comunque invariato.

La notazione distingue $y_a^{\mathrm{obs}}$, conteggio osservato sul link, da $\widehat y_a$, flusso stimato. Gli apici $\mathrm{cal}$ e $\mathrm{val}$ distinguono osservazioni di calibrazione e validazione; fonte e classe possono essere aggiunte, per esempio $y_a^{\mathrm{ANAS,L,obs}}$.

Il confronto sui link è:

$$
y_a^{\mathrm{ANAS,L,obs}}
\approx
\widehat y_a^{\mathrm L}
=
\sum_o\sum_d p_{od}^{a,\mathrm L}
\left(C_{od}^{\mathrm{pend}}+N_{od}^{\mathrm{nonpend}}+E_{od}^{\mathrm{est}}\right).
$$

Se si usa una componente residua, la sottrazione del pendolarismo avviene **sui link dopo l'assegnazione**:

$$
r_a
=
y_a^{\mathrm{ANAS,L,obs}}
-
\sum_o\sum_d p_{od}^{a,\mathrm L}C_{od}^{\mathrm{pend}}.
$$

La serie ANAS 2017--2025 viene conservata come patrimonio informativo, ma il target canonico della Gravity v0 è ora **unicamente il 2024**. I valori 2017--2023 possono supportare la valutazione di stabilità e l'eventuale inferenza di un 2024 mancante; il 2025 è riservato a `TEMPORAL_VALIDATION_2025` e non sostituisce direttamente il target. La futura selezione primaria dipenderà dapprima dalla classe qualitativa A/B e, in un gate separato, dalla contaminazione esterna e dalla rappresentatività spaziale. La matrice non deve riprodurre perfettamente ogni conteggio: gli scarti possono derivare da misura, copertura, disallineamento temporale, associazione al grafo, scelta dei percorsi o traffico esterno--esterno non rappresentato. Residui positivi sistematici sui corridoi di confine possono segnalare tale transito mancante e non devono essere assorbiti forzatamente dalle OD modellate.

La compatibilità ISTAT 2021--ANAS non è un vincolo bloccante: ISTAT fornisce la struttura spaziale delle relazioni abituali, mentre ANAS misura traffico medio sui link. Le anomalie pandemiche interessano soprattutto i conteggi 2020--2021 e, da verificare, parte del 2022. Non si richiede coincidenza temporale o quantitativa perfetta, ma coerenza complessiva verificata mediante assegnazione, calibrazione e residui.

Non va eseguita una sottrazione diretta nelle celle OD. I conteggi non identificano autonomamente la matrice: calibrano o aggiornano una seed strutturata. L'impostazione resta una **procedura ibrida di stima OD basata su seed gravitazionale e calibrazione mediante conteggi assegnati alla rete**.

#### 9.1.5.1 Decision record ANAS Gravity v0 — target 2024 e quality eligibility

La selezione dei conteggi ANAS utilizzabili dalla Gravity v0 viene governata da due decisioni metodologiche **ratificate e frozen**. Il freeze riguarda le regole di costruzione del target e di ammissibilità qualitativa; **non** congela ancora la classificazione concreta delle singole 20 sezioni logiche, perché la precedente readiness era stata costruita nel quadro 2024/25 e deve essere ricalcolata specificamente sul target 2024.

##### ANAS_V0_DECISION_01 — anno canonico di calibrazione

```text
ANAS_CALIBRATION_REFERENCE_YEAR = 2024
STATUS = RATIFIED / FROZEN
```

La scelta del 2024 deriva da quattro considerazioni convergenti:

1. **copertura diretta recente:** nel 2024 sono osservate direttamente 10/20 sezioni logiche, contro 8/20 nel 2025. Il massimo storico disponibile è 13/20 nel 2022, ma il 2024 offre un compromesso migliore fra recenza e copertura;
2. **coerenza temporale con la Gravity v0:** gli input territoriali frozen sono centrati sostanzialmente sul 2024; usare lo stesso anno per il target ANAS evita di calibrare una struttura territoriale 2024 contro un target 2025 senza una necessità metodologica specifica;
3. **target annuale unico:** la baseline canonica non deve essere una pseudo-baseline 2024/25 nella quale sezioni diverse rappresentano anni differenti;
4. **preservazione del 2025:** il dato 2025 non viene scartato, ma cambia ruolo e viene riservato a `TEMPORAL_VALIDATION_2025`, cioè validazione temporale e sensitivity fuori dall'anno di calibrazione.

La gerarchia conseguente è:

```text
2024 OBSERVED valido
    = preferred evidence

2024 MISSING + sufficient history
    = inference candidate

2024 MISSING + history insufficiente
    = D / UNUSABLE

2025
    = TEMPORAL_VALIDATION_2025
    = NOT primary calibration target
    = NON sostituisce direttamente un 2024 mancante
```

##### ANAS_V0_DECISION_02 — quality eligibility

```text
A — HIGH
B — MEDIUM
    -> CANDIDATE_PRIMARY_CALIBRATION_SET

C — LOW
    -> SENSITIVITY / DIAGNOSTIC ONLY

D — UNUSABLE
    -> EXCLUDED FOR INSUFFICIENT INFORMATION

STATUS = RATIFIED / FROZEN
```

La classificazione misura esclusivamente **qualità e completabilità del dato**: copertura temporale, stabilità della serie e affidabilità dell'eventuale inferenza. Non misura ancora l'idoneità spaziale o causale della sezione a rappresentare una Gravity FVG-only.

È quindi congelata la separazione:

```text
DATA QUALITY FILTER
!=
EXTERNAL CONTAMINATION FILTER
```

Una sezione di classe `A` può avere una serie eccellente ma risultare successivamente poco adatta alla calibrazione primaria se fortemente esposta a traffico di confine, transito esterno, autostrada o altre componenti non rappresentate dalla baseline FVG-only. Viceversa una sezione `B` interna può essere informativa. `BORDER EXPOSURE`, `THROUGH TRAFFIC`, `MOTORWAY`, `EXTERNAL TRAFFIC` e `SPATIAL REPRESENTATIVENESS` rimangono metadata/filtri successivi e **non** devono essere incorporati nella classe qualitativa A/B/C/D.

L'audit preliminare indica che l'informazione storica è parzialmente, ma non universalmente, sufficiente: l'ordine di grandezza atteso è circa 12 sezioni solide e al massimo circa 15 utilizzabili. Questi numeri sono diagnostici e **non** costituiscono una membership frozen. Le sezioni prive di evidenza sufficiente non devono essere ricostruite artificialmente solo per aumentare la numerosità del campione.

Il backtest già svolto costituisce evidenza a favore della fattibilità dell'inferenza sulle serie appropriate: il benchmark `LOCF` ha ottenuto un errore percentuale assoluto medio di circa **5,46%** e mediano di circa **3,83%**. `LOCF` rimane tuttavia il **benchmark da battere**, non l'algoritmo finale congelato.

##### Elementi esplicitamente non congelati

Restano aperti e devono essere decisi nei gate successivi:

- membership finale delle 20 sezioni nelle classi `A/B/C/D`;
- valori 2024 inferiti per le sezioni mancanti;
- algoritmo finale di inferenza;
- filtri per esposizione esterna e traffico di attraversamento;
- pesi nel fit;
- scelta fra Huber, WLS, L1 o altre loss;
- parametri `n`, $\beta$ e `Q` della calibrazione Gravity.

##### Fase 5.8C — ricalcolo della readiness sul target 2024

La 5.8C deve produrre una proposta completa sulle 20 sezioni logiche, registrando per ciascuna almeno:

```text
SECTION_ID
ROAD
2024_STATUS
2024_OBSERVED_VALUE
INFERENCE_REQUIRED
INFERENCE_FEASIBLE
PROPOSED_INFERENCE_METHOD
INFERRED_2024_VALUE
CONFIDENCE_CLASS
EVIDENCE_YEARS
STABILITY_CLASS
NOTES
```

La gerarchia esecutiva è:

1. usare `OBSERVED_2024` quando valido;
2. se il 2024 manca, inferire solo quando la storia fornisce supporto sufficiente;
3. se l'inferenza non è difendibile, classificare la sezione `D / UNUSABLE`;
4. mantenere il 2025 separato come `TEMPORAL_VALIDATION_2025`.

L'output richiesto è `ANAS_2024_TARGET_PROPOSAL`, con conteggi di osservati, inferiti A/B e totali A/B/C/D, più la tabella completa delle 20 sezioni.

La 5.8C **non** deve ancora filtrare per esposizione esterna, assegnare fit weights, scegliere una loss di calibrazione o stimare parametri della Gravity. Il suo scopo è costruire un target 2024 documentato e qualitativamente classificato, lasciando ai gate successivi l'idoneità causale/spaziale delle sezioni e il fit del modello.

### 9.1.6 Possibile matrix estimation successiva

Come fase metodologica eventuale, non ancora adottata definitivamente, si potrà valutare:

$$
\min_{T}
\left[
\lambda D(T,T^0)
+
\sum_{a\in\mathcal A_{\mathrm{obs}}}
w_a
\left(y_a^{\mathrm{obs}}-\sum_o\sum_d p_{od}^{a,\mathrm L}T_{od}\right)^2
\right].
$$

Qui $T^0$ è la seed, $T$ la matrice aggiornata, $D(T,T^0)$ una penalizzazione dell'allontanamento dalla seed, $\mathcal A_{\mathrm{obs}}$ l'insieme dei link osservati, $y_a^{\mathrm{obs}}$ il conteggio, $w_a$ il peso di qualità e $\lambda$ il compromesso tra fedeltà alla seed e adattamento. Funzione di distanza, pesi, regolarizzazione e vincoli restano questioni aperte; non si deve forzare un adattamento perfetto.

### 9.1.7 Catena di trasformazione verso il FRLM

$$
T_{od}^{\mathrm L}
\longrightarrow
\text{percorsi OD}
\longrightarrow
\text{flussi sugli archi}
\longrightarrow
\text{flussi di percorso per il FRLM}.
$$

La matrice OD, i flussi sui link e i flussi di percorso sono oggetti distinti. I conteggi agiscono nel confronto sui link; il FRLM riceve flussi associati a percorsi stradali.

### 9.1.8 Fase 5.8E — calibrazione FVG-only, test di specificazione e stop pre-freeze

#### Stato formale

```text
FASE 5.8E =
CLOSED / STOP / NOT_READY FOR PARAMETER FREEZE

Q =
NOT FROZEN

beta =
NOT FROZEN

GLOBAL EXPONENTIAL =
INCUMBENT REFERENCE ONLY

ORIGIN-CONSTRAINED =
TESTED / REJECTED AS BASELINE

COMBINED DETERRENCE =
TESTED / REJECTED AS BASELINE

GENERAL_TIME_B5_FAILURE =
NO

ASSIGNMENT REPRESENTATION =
SECTION-SPECIFIC LIMITATION CONFIRMED

920022 =
NOT CLEAN FOR GRAVITY GENERALIZATION VALIDATION

920040 =
REMAINS USEFUL DIAGNOSTIC SIGNAL

ANAS 2025 =
TEMPORAL HOLDOUT / NOT USED

NEXT =
EXTERNAL / TRANSBORDER DEMAND COMPONENT
```

Il backbone e il sistema dei path restano invariati:

```text
G_OSM_operativo       = REMAINS FROZEN
Gamma_OSM             = REMAINS FROZEN
EXP_REL_300           = REMAINS FROZEN
OD_PATH_SYSTEM_OSM    = REMAINS FROZEN
TIME_B5               = REMAINS FROZEN
PRODUCT_LAMBDA        = REMAINS FROZEN
```

#### Baseline Gravity studiata in 5.8E

Il riferimento sperimentale FVG-only è:

$$
W_{ij}(\beta)=P_iA_j\exp(-\beta c_{ij}),
$$

$$
S_{ij}(\beta)
=
\frac{W_{ij}(\beta)}
{\displaystyle\sum_{r\neq s}W_{rs}(\beta)},
$$

$$
N_{ij}(Q,\beta)=Q\,S_{ij}(\beta),
$$

$$
T_{ij}^{LIGHT}=C_{ij}^{ISTAT}+N_{ij},
\qquad
N_{ii}=0.
$$

Con:

```text
P_i =
L1-normalized parco auto proxy

A_j =
0.5 * tourism regional share
+
0.5 * GDO regional share

c_ij =
TIME_B5 PRODUCT-LAMBDA municipal impedance [min]

Q =
total synthetic intermunicipal non-pendular LIGHT demand [veh/day]

beta =
exponential deterrence parameter [1/min]
```

La specifica costituisce `INCUMBENT REFERENCE / FVG-ONLY EXPERIMENTAL MODEL`, non un modello finale frozen.

#### Calibration method

La calibrazione da conteggi ANAS resta concettualmente link-based:

$$
\widehat Y_a(Q,\beta)
=
\sum_{ij}p_{ij}^{a}
\left[
C_{ij}^{ISTAT}+N_{ij}(Q,\beta)
\right].
$$

L'objective baseline utilizzata è:

$$
J(Q,\beta)
=
\sum_a
\left[
Y_a-\widehat Y_a(Q,\beta)
\right]^2.
$$

Contratto:

```text
LOSS =
UNWEIGHTED SSE / NLLS

CUSTOM WEIGHTS =
NO

ROBUST LOSS =
NO

RELATIVE SQUARED ERROR =
DIAGNOSTIC ONLY
```

La log-regression diretta sui conteggi ANAS non è applicabile: un link count aggrega numerose OD,

$$
Y_a=\sum_{ij}p_{ij}^{a}T_{ij},
$$

e il logaritmo non trasforma questo problema in una regressione lineare OD-level.

`HYMAN = NOT APPLICABLE NOW`, perché non è disponibile una observed trip-length distribution rappresentativa della componente non pendolare.

#### Profiling di $Q$

Per una deterrence shape fissata, il modello resta lineare rispetto alla scala $Q$. Nel caso exponential:

$$
\widehat Y_a=C_a+Q\,G_a(\beta),
$$

con:

$$
C_a=\sum_{ij}p_{ij}^{a}C_{ij}^{ISTAT},
$$

$$
G_a(\beta)=\sum_{ij}p_{ij}^{a}S_{ij}(\beta).
$$

A $\beta$ fissato, $Q$ può quindi essere determinato analiticamente tramite least squares. Il problema si riduce a:

$$
\beta
\rightarrow
G_a(\beta)
\rightarrow
Q^*(\beta)
\rightarrow
J_{\mathrm{profile}}(\beta).
$$

Il principio di profiling resta metodologicamente valido anche se la specifica Gravity verrà successivamente rivista.

#### E3-A — exponential FVG-only candidate

Risultato storico/diagnostico:

```text
beta =
0.045953794473 1/min

Q =
1552629.518131 veh/day

J =
1056095.651965

c_half =
ln(2)/beta
≈ 15.083568 min
```

Il profilo $J(\beta)$ presentava:

- unico minimo interno;
- nessun minimo multiplo rilevante;
- nessun boundary optimum;
- buona profile stability locale.

Pertanto:

```text
NUMERICAL IDENTIFIABILITY =
SUPPORTED

EMPIRICAL / SPATIAL ADEQUACY =
NOT ESTABLISHED
```

Il buon profilo numerico non viene interpretato come validazione empirica del modello.

#### E4 — identifiability vs generalization

Per il candidato exponential:

```text
IDENTIFIABILITY =
PASS

PROFILE_STABILITY =
PASS

LOO_STABILITY =
WEAK

LOSS_SENSITIVITY =
ACCEPTABLE / MODERATE

Q_PLAUSIBILITY =
HIGH_BUT_DEFENSIBLE
```

Il problema principale non è l'identificabilità numerica di $\beta$, ma la mancata generalizzazione uniforme della stessa coppia $Q,\beta$ fra corridoi differenti. La SSE sulle PRIMARY risultava inoltre fortemente concentrata su `920032`.

#### E4-B — spatial generalization failure

Sulle sezioni model-exposed non usate nella calibrazione sono emersi mismatch rilevanti.

`920022 — RA13`:

```text
OBSERVED =
23143 veh/day

MODELLED =
169730.252 veh/day

RELATIVE ERROR ≈
-633%
```

`920040 — SS54`:

```text
OBSERVED =
14659 veh/day

MODELLED =
42439.478 veh/day

RELATIVE ERROR ≈
-190%
```

L'attribuzione OD ha mostrato che gli eccessi non derivano da una singola OD anomala: la domanda assegnata è distribuita su numerose OD.

```text
FEW_OD_ERROR =
NOT SUPPORTED

SINGLE_PATH_ERROR =
NOT SUPPORTED AS GENERAL EXPLANATION
```

Il segnale resta compatibile con:

```text
DEMAND_MODEL_MISMATCH
MODEL_SCALE_MISMATCH
ASSIGNMENT_REPRESENTATION
DOMAIN_MISMATCH
```

e impedisce il freeze di $Q$ e $\beta$.

#### Test origin-constrained — formulazione

È stata testata:

$$
N_{ij}
=
Q\,P_i
\frac{
A_j\exp(-\beta c_{ij})
}{
\displaystyle\sum_{s\neq i}A_s\exp(-\beta c_{is})
}.
$$

Per costruzione:

$$
\sum_jN_{ij}=QP_i,
\qquad
\sum_{ij}N_{ij}=Q.
$$

Interpretazione:

- $Q$: traffico non pendolare totale prodotto nel dominio;
- $P_i$: quota di $Q$ prodotta dall'origine $i$;
- $A_j\exp(-\beta c_{ij})/\sum_sA_s\exp(-\beta c_{is})$: distribuzione delle destinazioni del traffico che parte da $i$.

La normalizzazione globale $\sum_jA_j=1$ è utile per interpretazione e confrontabilità, ma **non è matematicamente necessaria** nella destination choice origin-constrained: una costante moltiplicativa comune agli $A_j$ si semplifica.

È invece strutturalmente necessaria la normalizzazione per origine:

$$
\frac{
A_j\exp(-\beta c_{ij})
}{
\sum_sA_s\exp(-\beta c_{is})
},
$$

affinché le destination shares di ciascuna origine sommino a 1.

La variante era stata proposta perché, nella normalizzazione globale originale:

$$
N_{i+}
=
Q
\frac{
P_i\sum_jA_j\exp(-\beta c_{ij})
}{
Z(\beta)
},
$$

e quindi la produzione effettiva dell'origine non coincide necessariamente con $QP_i$: è influenzata anche dall'accessibilità dell'origine alle destinazioni.

#### Test origin-constrained — risultato empirico

R1:

```text
IMPLEMENTATION =
PASS

MATHEMATICAL QA =
PASS

SPATIAL GENERALIZATION =
WORSE OVERALL
```

Candidato diagnostico:

```text
beta ≈
0.032095691453 1/min

Q ≈
990072.131283 veh/day

NEW 3-PRIMARY SSE ≈
9.443 million

OLD SSE ≈
1.056 million
```

Sezioni diagnostiche:

```text
920022:
OLD error ≈ -633%
NEW error ≈ -692%

920040:
OLD error ≈ -190%
NEW error ≈ -83%
```

`920040` migliora, ma il pattern regionale peggiora nel complesso.

Conclusione:

```text
ORIGIN-CONSTRAINED =
MATHEMATICALLY VALID
BUT
EMPIRICALLY REJECTED AS BASELINE v0
```

La produzione non controllata del modello globale non è quindi una spiegazione sufficiente del failure.

#### Combined / Tanner deterrence

È stata inoltre testata:

$$
f(c)=c^n\exp(-\beta c),
\qquad
n\geq0,\ \beta\geq0.
$$

La famiglia contiene l'exponential come caso speciale $n=0$. Per $n>0,\beta>0$:

$$
c_{\mathrm{peak}}=\frac{n}{\beta}.
$$

Anche qui $Q$ è profilabile; la ricerca numerica diventa:

$$
(n,\beta)
\rightarrow
Q^*(n,\beta)
\rightarrow
J_{\mathrm{profile}}(n,\beta).
$$

Candidato diagnostico:

```text
n ≈
1.255285016510

beta ≈
0.089101695826 1/min

Q ≈
1846499.386614 veh/day

c_peak ≈
14.088 min

PRIMARY SSE ≈
0
```

La quasi perfetta interpolazione delle tre PRIMARY non costituisce evidenza di superiorità: tre osservazioni PRIMARY e tre parametri effettivi $Q,n,\beta$ rendono il risultato `SATURATION-AFFECTED`; $n$ e $\beta$ risultano inoltre fortemente correlati.

```text
IDENTIFIABILITY =
WEAK / SATURATION-AFFECTED
```

Generalizzazione:

```text
EXPOSED SENSITIVITY =
6

IMPROVED =
0

WORSENED =
6
```

Esempi:

```text
920022:
EXPONENTIAL error ≈ -633%
COMBINED error ≈ -681%

920040:
EXPONENTIAL error ≈ -190%
COMBINED error ≈ -250%
```

Pertanto:

```text
COMBINED DETERRENCE =
TESTED / REJECTED AS BASELINE CANDIDATE
```

La forma esponenziale della deterrenza non appare la causa primaria del failure.

#### Risultato cumulativo R1 + R2

```text
A. PRODUZIONE NON VINCOLATA
   test = ORIGIN-CONSTRAINED
   risultato = NON RISOLVE IL FAILURE

B. DETERRENZA ESPONENZIALE TROPPO RIGIDA
   test = COMBINED / TANNER
   risultato = NON RISOLVE IL FAILURE
               e peggiora la generalizzazione
```

Non è quindi supportato attribuire il problema principalmente a:

- global normalization di $P_i$;
- exponential deterrence shape.

Restano invece rilevanti:

```text
DEMAND PROXY STRUCTURE
DESTINATION ATTRACTIVENESS STRUCTURE
LOW-DIMENSIONAL GRAVITY LIMITATION
ASSIGNMENT REPRESENTATION
MISSING EXTERNAL DEMAND DOMAIN
MISSING INTRAZONAL DOMAIN
```

#### Final assignment sanity check

Su `920022 — RA13` e `920040 — SS54` è stato svolto un controllo manuale limitato dei top contributor.

Per i top contributor controllati, tutte le:

$$
3\times3=9
$$

combinazioni `PRODUCT-LAMBDA` attraversavano la sezione analizzata.

Questi nove path **non sono nove route-choice alternatives**. Sono:

```text
3 origin accesses
×
3 destination accesses
=
9 access-pair combinations
```

e ciascuna access pair usa un solo deterministic `TIME_B5` shortest path.

##### 920022 — assignment representation signal

Per `MONFALCONE → TRIESTE`, il frozen assignment porta 9/9 path su `920022`, ma un percorso principale/plausibile non richiede RA13; RA13 è una alternativa competitiva.

Per `TRIESTE → GRADO`, il frozen assignment porta ancora 9/9 path su `920022`, mentre esiste un percorso plausibile che evita RA13.

Pertanto:

$$
p_{ij}^{920022}=1
$$

significa:

> 100% dei frozen deterministic `TIME_B5` access-pair paths utilizza RA13,

non:

> 100% del traffico reale OD utilizza RA13.

Classificazione:

```text
920022_ASSIGNMENT =
SECTION-SPECIFIC ASSIGNMENT REPRESENTATION LIMITATION

920022 =
NOT A CLEAN INDEPENDENT GRAVITY VALIDATION SECTION
```

##### 920040 — assignment sanity

Per `UDINE → REMANZACCO` il passaggio su `SS54 / 920040` è risultato geograficamente e operativamente plausibile.

```text
920040 =
NO OBVIOUS ASSIGNMENT ANOMALY ON CHECKED TOP OD
```

La forte sovrastima su `920040` rimane quindi un segnale diagnostico importante.

#### Interpretazione dell'assignment

Non viene concluso:

```text
G_OSM_operativo is wrong
TIME_B5 is wrong
```

Il risultato è:

```text
GENERAL_TIME_B5_FAILURE =
NO

ISOLATED_PATH_ERROR =
NO

SECTION-SPECIFIC ASSIGNMENT REPRESENTATION LIMITATION =
YES
```

Il limite riguarda la differenza fra deterministic shortest-path assignment e real route-choice distribution.

Non esiste evidenza sufficiente per riaprire `G_OSM_operativo` o `OD_PATH_SYSTEM_OSM`; entrambi restano frozen.

Prima del FRLM non si deve interpretare automaticamente $p_{ij}^{a}=1$ come `100% real-world route share`. I frozen paths sono deterministic model paths, non empirical stochastic route-choice shares.

Non viene autorizzato ora un nuovo route-choice model. Il tema sarà rivalutato soltanto se resterà materialmente blocking dopo l'ampliamento del dominio di domanda.

#### Missing domain — external / transborder

Il modello 5.8E rappresenta soltanto:

```text
FVG INTERNAL INTERMUNICIPAL OD
```

e non include:

- external → internal;
- internal → external;
- external → external / through traffic;
- Austria;
- Slovenia;
- resto d'Italia;
- traffico transfrontaliero;
- intrazonal municipal traffic.

I conteggi ANAS osservano invece tutto il traffico che attraversa la sezione.

Esiste quindi un mismatch fra:

```text
MODEL DEMAND DOMAIN
```

e:

```text
ANAS MEASUREMENT DOMAIN
```

La precedente selezione di poche sezioni a bassa external exposure è stata sufficiente per una baseline esplorativa FVG-only, ma limita numerosità dei counts, rappresentatività spaziale, capacità diagnostica e robustezza dell'identificazione.

#### Nuova priorità metodologica

Il prossimo passo **non** è moltiplicare ulteriormente le varianti Gravity.

$$
\boxed{
\text{NEXT = EXTERNAL / TRANSBORDER DEMAND COMPONENT}
}
$$

Scope preliminare:

```text
A. INTERNAL → EXTERNAL
B. EXTERNAL → INTERNAL
C. EXTERNAL → EXTERNAL / THROUGH

REST OF ITALY
AUSTRIA
SLOVENIA
```

Architettura:

```text
EXTERNAL ZONES
+
GATEWAY / CONNECTION TO FROZEN INTERNAL NETWORK
```

L'estensione deve essere:

```text
SEPARATE / VERSIONED / ADDITIVE
```

e non deve rigenerare in place il sistema interno frozen.

Lo scopo è anche ridurre il mismatch fra model demand domain e ANAS measurement domain. Dopo l'estensione potranno essere rivalutati eligibility delle sezioni ANAS, calibration design, numero di segnali indipendenti, $Q$, $\beta$, generalizzazione regionale e residual assignment limitation.

#### ANAS 2025

```text
ANAS 2025 =
FINAL TEMPORAL HOLDOUT
```

Non è stato utilizzato:

- nella calibrazione exponential;
- nel test origin-constrained;
- nel test combined;
- nei sanity check di assignment.

Non deve essere usato per scegliere la nuova specifica, costruire la domanda external, selezionare parametri o scegliere sezioni in funzione del loro fit 2025.

#### Principio metodologico da preservare

Il fallimento di una specifica Gravity nel riprodurre alcuni conteggi di link non deve essere attribuito automaticamente alla sola domanda OD.

Il confronto:

$$
\text{OD demand}
\rightarrow
\text{deterministic path assignment}
\rightarrow
\text{link counts}
$$

combina almeno tre livelli distinti:

1. **DEMAND GENERATION / DISTRIBUTION**;
2. **ROUTE / ASSIGNMENT REPRESENTATION**;
3. **OBSERVATION DOMAIN OF TRAFFIC COUNTS**.

La Fase 5.8E ha mostrato empiricamente che questi livelli possono interagire. Per questo il passo successivo è migliorare prima la coerenza tra il dominio di domanda modellato e il dominio osservato dai conteggi introducendo la componente external/transborder, non aumentare ulteriormente la flessibilità della Gravity.

## 9.2 Insieme dei percorsi

Per ogni relazione $(o,d)$ e segmento $s$, si definisce:

$$
\mathcal Q_{od}^s
$$

come l'insieme dei percorsi ammessi tra la zona $o$ e la zona $d$.

L'insieme complessivo è:

$$
\mathcal Q^s
=
\bigcup_{o,d\in\mathcal Z}\mathcal Q_{od}^s.
$$

Ogni percorso $q\in\mathcal Q_{od}^s$ è una successione ordinata di archi fisici:

$$
q=(a_1,a_2,\ldots,a_{n_q}).
$$

Si definiscono le funzioni:

$$
\operatorname{org}(q)=o,
\qquad
\operatorname{dst}(q)=d.
$$

## 9.3 Quote di assegnazione

A ciascun percorso si associa una quota:

$$
\pi_{q\mid od}^s\in[0,1]
$$

con:

$$
\sum_{q\in\mathcal Q_{od}^s}\pi_{q\mid od}^s=1
\qquad
\forall o,d,s.
$$

Il flusso sul percorso è:

$$
f_q^s
=
\pi_{q\mid od}^s T_{od}^s,
$$

per $q\in\mathcal Q_{od}^s$. Nel dominio light interno FVG, dopo la Fase 5.7, la notazione va letta nel contratto frozen: il flusso comunale è ripartito sui nove access-pair mediante `PRODUCT-LAMBDA` e ciascun access-pair possiede **un solo** path `TIME_B5`-ottimo. `PRODUCT-LAMBDA` non è route choice multipath. Eventuali modelli stocastici o di deviazione appartengono a estensioni future separate e non modificano `OD_PATH_SYSTEM_OSM_v01`.

## 9.4 Indicatori di attraversamento

Si definisce:

$$
\delta_{aq}^s=
\begin{cases}
1 & \text{se il percorso }q\text{ utilizza l'arco }a,\\
0 & \text{altrimenti},
\end{cases}
$$

ed analogamente:

$$
\delta_{cq}^s=
\begin{cases}
1 & \text{se il percorso }q\text{ attraversa o può accedere al candidato }c,\\
0 & \text{altrimenti}.
\end{cases}
$$

Il flusso assegnato all'arco fisico è:

$$
F_a^s
=
\sum_{q\in\mathcal Q^s}f_q^s\delta_{aq}^s.
$$

Il flusso che attraversa il candidato è:

$$
\Phi_c^s
=
\sum_{q\in\mathcal Q^s}f_q^s\delta_{cq}^s.
$$

È fondamentale distinguere:

$$
\Phi_c^s
\neq
\text{domanda di ricarica al candidato }c.
$$

Non tutti i veicoli che attraversano un nodo devono ricaricare.

---

# 10. Strategie di ricarica

## 10.1 Definizione

Per ogni percorso $q\in\mathcal Q^s$, si definisce:

$$
\mathcal K_q^s
$$

come l'insieme delle strategie di ricarica ammissibili.

Una strategia $k\in\mathcal K_q^s$ specifica:

- in quali candidati il veicolo si ferma;
- l'ordine delle fermate;
- l'energia caricata in ciascuna fermata;
- la fattibilità rispetto ad autonomia e stato di carica;
- eventuali vincoli operativi e normativi.

## 10.2 Indicatore di utilizzo del candidato

Si definisce:

$$
a_{cqk}^s=
\begin{cases}
1 & \text{se la strategia }k\text{ del percorso }q\text{ usa il candidato }c,\\
0 & \text{altrimenti}.
\end{cases}
$$

Necessariamente:

$$
a_{cqk}^s\leq\delta_{cq}^s.
$$

## 10.3 Energia caricata per veicolo

Si definisce:

$$
e_{cqk}^s\geq 0
$$

come l'energia caricata da un singolo veicolo del flusso $q$, nel candidato $c$, adottando la strategia $k$.

L'unità di misura è:

$$
e_{cqk}^s=\text{kWh/veicolo}.
$$

Se il candidato non viene utilizzato:

$$
a_{cqk}^s=0
\quad\Rightarrow\quad
e_{cqk}^s=0.
$$

Il parametro $e_{cqk}^s$ deve dipendere dalla sequenza precedente delle fermate. Non è sufficiente assumere che ogni veicolo effettui una ricarica completa o che il numero di ricariche sia uguale al rapporto tra distanza e autonomia.

## 10.4 Tempo di occupazione

Si definisce:

$$
h_{cqk}^s\geq 0
$$

come il tempo durante il quale un singolo veicolo occupa un punto di ricarica nel candidato $c$, lungo il percorso $q$, adottando la strategia $k$.

In prima approssimazione:

$$
h_{cqk}^s
=
\frac{e_{cqk}^s}{p_s^{\mathrm{eff}}}
+h_s^{\mathrm{overhead}},
$$

dove:

- $p_s^{\mathrm{eff}}$ è la potenza media effettiva di ricarica;
- $h_s^{\mathrm{overhead}}$ include ingresso, collegamento, scollegamento e uscita.

---

# 11. Variabili decisionali del modello separato

Il modello viene risolto inizialmente per un singolo segmento fissato $s$. Le variabili mantengono comunque l'apice $s$ per rendere esplicita la separazione.

| Variabile | Dominio | Significato |
|---|---|---|
| $x_c^s$ | $\{0,1\}$ | 1 se il candidato $c$ viene attivato per il segmento $s$ |
| $y_c^s$ | $\mathbb Z_{\geq 0}$ | numero di punti di ricarica installati nel candidato $c$ |
| $\chi_{cb}^s$ | $\{0,1\}$ | 1 se il candidato $c$ attivato è connesso alla cabina ammissibile $b$ |
| $y_{cb}^s$ | $\mathbb Z_{\geq 0}$ | punti del candidato $c$ alimentati dalla cabina $b$ |
| $z_{qk}^s$ | $[0,1]$ | quota del flusso del percorso $q$ assegnata alla strategia $k$ |
| $u_q^s$ | $[0,1]$ | quota complessiva del flusso del percorso $q$ servita |
| $P_b^{\mathrm{exp},s}$ | $\mathbb R_{\geq 0}$ | potenza aggiuntiva richiesta alla cabina $b$ nel modello del segmento $s$ |

## 11.1 Perché $z_{qk}^s$ è continua

Nel file `Modello_FRLM_Tesi.md` la variabile associata alla strategia era binaria. Nel modello capacitato è generalmente più utile consentire la suddivisione del flusso:

$$
z_{qk}^s\in[0,1].
$$

In questo modo parti diverse dello stesso flusso possono utilizzare strategie diverse. Ciò evita che tutta la domanda di una relazione OD sia obbligata a utilizzare la stessa sequenza di stazioni.

---

# 12. Parametri infrastrutturali ed economici

| Parametro | Significato | Unità |
|---|---|---|
| $C_c^{\mathrm{fix},s}$ | costo fisso di attivazione del candidato | € |
| $C_c^{\mathrm{unit},s}$ | costo per punto di ricarica | €/punto |
| $C_b^{\mathrm{grid}}$ | costo di espansione della cabina | €/kW |
| $p_s^{\mathrm{unit}}$ | potenza nominale del singolo punto | kW |
| $\overline y_c^s$ | numero massimo di punti installabili | punti |
| $P_b^{\mathrm{avail}}$ | capacità residua disponibile presso la cabina | kW |
| $\gamma_c^s$ | fattore di contemporaneità della potenza installata | adimensionale |
| $H_c^{\mathrm{op}}$ | ore giornaliere equivalenti di esercizio | h/giorno |
| $\eta_c^{\mathrm{chg},s}$ | rendimento complessivo di ricarica | adimensionale |
| $\eta_s^{\mathrm{cov}}$ | quota minima di flusso da servire | adimensionale |

---

# 13. Formulazione base del modello per un segmento

## 13.1 Funzione obiettivo

Per un segmento fissato $s$, la formulazione base minimizza il costo infrastrutturale e di rete:

$$
\min TC^s
=
\sum_{c\in\mathcal C_s}
\left(
C_c^{\mathrm{fix},s}x_c^s
+
C_c^{\mathrm{unit},s}y_c^s
\right)
+
\sum_{b\in\mathcal B}
C_b^{\mathrm{grid}}P_b^{\mathrm{exp},s}.
$$

La quota minima servita viene imposta mediante un vincolo esplicito, evitando che il risultato dipenda da una penalità arbitraria per il traffico non servito.

Una variante coerente con il file originario può aggiungere un termine di penalità, ma tale parametro deve essere sottoposto ad analisi di sensibilità.

## 13.2 Assegnazione delle strategie

$$
\sum_{k\in\mathcal K_q^s}z_{qk}^s
=
u_q^s
\qquad
\forall q\in\mathcal Q^s.
$$

Se si impone la copertura completa:

$$
u_q^s=1.
$$

## 13.3 Collegamento tra strategie e candidati attivati

$$
z_{qk}^s
\leq
x_c^s
\qquad
\forall q\in\mathcal Q^s,
\forall k\in\mathcal K_q^s,
\forall c\in\mathcal C_s:
a_{cqk}^s=1.
$$

Una strategia può essere utilizzata soltanto se tutti i candidati richiesti dalla strategia sono stati attivati.

## 13.4 Collegamento tra apertura e numero di punti

$$
y_c^s
\leq
\overline y_c^s x_c^s
\qquad
\forall c\in\mathcal C_s,
$$

$$
y_c^s
\geq
x_c^s
\qquad
\forall c\in\mathcal C_s.
$$

## 13.5 Domanda energetica giornaliera al candidato

L'energia giornaliera assegnata al candidato è:

$$
E_c^s
=
\sum_{q\in\mathcal Q^s}
\sum_{k\in\mathcal K_q^s}
f_q^s e_{cqk}^s z_{qk}^s.
$$

L'unità è:

$$
E_c^s=\text{kWh/giorno}.
$$

## 13.6 Capacità energetica giornaliera

$$
\sum_{q\in\mathcal Q^s}
\sum_{k\in\mathcal K_q^s}
f_q^s e_{cqk}^s z_{qk}^s
\leq
H_c^{\mathrm{op}}
\eta_c^{\mathrm{chg},s}
p_s^{\mathrm{unit}}y_c^s
\qquad
\forall c\in\mathcal C_s.
$$

Il lato sinistro è espresso in kWh/giorno. Il lato destro è la capacità energetica giornaliera dei punti installati.

## 13.7 Vincolo minimo di copertura

$$
\sum_{q\in\mathcal Q^s}f_q^s u_q^s
\geq
\eta_s^{\mathrm{cov}}
\sum_{q\in\mathcal Q^s}f_q^s.
$$

Il parametro $\eta_s^{\mathrm{cov}}$ permette di costruire una curva costo-copertura senza introdurre immediatamente una penalità monetaria per ogni veicolo non servito.

## 13.8 Vincolo di rete elettrica

La relazione $\mathcal G_s$ ammette più alternative di connessione per candidato. Nella formulazione base ogni stazione attivata sceglie una sola cabina:

$$
\sum_{b:(c,b)\in\mathcal G_s}\chi_{cb}^s=x_c^s
\qquad\forall c\in\mathcal C_s.
$$

I punti installati vengono attribuiti alla connessione selezionata:

$$
\sum_{b:(c,b)\in\mathcal G_s}y_{cb}^s=y_c^s,
$$

$$
0\leq y_{cb}^s\leq\overline y_c^s\chi_{cb}^s
\qquad\forall(c,b)\in\mathcal G_s.
$$

Il vincolo di capacità della cabina è:

$$
\sum_{c:(c,b)\in\mathcal G_s}
\gamma_c^s p_s^{\mathrm{unit}}y_{cb}^s
\leq
P_b^{\mathrm{avail}}+P_b^{\mathrm{exp},s}
\qquad\forall b\in\mathcal B.
$$

Una cabina può servire più candidati. Distanza, costo, capacità disponibile e vincoli tecnici determinano la connessione scelta; alimentazioni multiple della stessa stazione restano un'estensione futura.

## 13.9 Vincoli normativi

Per i corridoi soggetti a requisiti normativi si aggiungono i vincoli descritti nella Sezione 8.3:

$$
\sum_{c\in\mathcal C_{sr}(w)}x_c^s\geq1
\qquad
\forall r\in\mathcal R_s^{\mathrm{norm}},
\forall w\in\mathcal W_{sr}.
$$

Se è richiesta una potenza aggregata minima nel candidato attivato:

$$
p_s^{\mathrm{unit}}y_c^s
\geq
\underline P_{sr}^{\mathrm{norm}}x_c^s
$$

per i candidati ai quali il requisito si applica.

Nella formulazione base light tutte le colonnine HPC sono della stessa tipologia e potenza nominale; tecnologie multiple richiederebbero un'estensione con un indice dedicato.

---

# 14. Estensione temporale per il numero di punti di ricarica

Questa sezione descrive un **raffinamento futuro**, non la formulazione base. Inizialmente tutte le colonnine HPC hanno uguale tipologia e potenza nominale e il dimensionamento sceglie il numero $y_c^s$ in funzione del traffico assegnato e della domanda potenziale di ricarica. Profili orari, code e livelli di servizio non sono modellati in dettaglio nella prima implementazione. La sola energia giornaliera potrebbe tuttavia sottostimare i punti necessari quando gli arrivi sono concentrati; in tal caso si potrà introdurre la seguente estensione.

Si definisce:

$$
\theta_{q\tau}^s\in[0,1]
$$

come la quota del flusso giornaliero del percorso $q$ che si presenta nell'intervallo $\tau$, con:

$$
\sum_{\tau\in\mathcal T}\theta_{q\tau}^s=1.
$$

Il numero di ore-punto richieste nell'intervallo $\tau$ è:

$$
O_{c\tau}^s
=
\sum_{q\in\mathcal Q^s}
\sum_{k\in\mathcal K_q^s}
f_q^s
\theta_{q\tau}^s
h_{cqk}^s
z_{qk}^s.
$$

Se la durata dell'intervallo è $\Delta_\tau$, il vincolo di occupazione è:

$$
O_{c\tau}^s
\leq
\Delta_\tau y_c^s
\qquad
\forall c,\tau.
$$

Per introdurre un margine di servizio:

$$
O_{c\tau}^s
\leq
\alpha_s\Delta_\tau y_c^s,
\qquad 0<\alpha_s<1.
$$

Un valore inferiore a 1 evita di progettare una stazione costantemente satura.

---

# 15. Output del modello a livello comunale

Il modello decide su candidati $c$, ma i risultati possono essere aggregati per comune.

## 15.1 Numero di punti per comune

$$
Y_m^s
=
\sum_{c\in\mathcal C_s:\mu(c)=m}y_c^s.
$$

## 15.2 Potenza installata per comune

$$
P_m^s
=
\sum_{c\in\mathcal C_s:\mu(c)=m}
p_s^{\mathrm{unit}}y_c^s.
$$

## 15.3 Energia giornaliera servita per comune

$$
E_m^s
=
\sum_{c\in\mathcal C_s:\mu(c)=m}E_c^s.
$$

## 15.4 Flusso attraversante il comune-corridoio

Per il corridoio $r$:

$$
\Phi_{mr}^s
=
\sum_{c\in\mathcal C_s:\mu(c)=m,\rho(c)=r}\Phi_c^s.
$$

Questa somma deve essere interpretata con cautela, perché lo stesso flusso può attraversare più candidati appartenenti allo stesso comune. Per una rendicontazione senza doppio conteggio è preferibile calcolare direttamente il flusso dei percorsi che attraversano almeno un candidato della coppia comune-corridoio.

## 15.5 Forma raccomandata del risultato

Il risultato finale dovrebbe riportare almeno:

| Campo | Esempio concettuale |
|---|---|
| Segmento | leggero o pesante |
| Comune | comune raccomandato |
| Corridoio | asse stradale rilevante |
| Direzione | una o entrambe |
| Tipo di candidato | reale, virtuale o ibrido |
| Punti di ricarica | $Y_m^s$ o dettaglio per candidato |
| Potenza aggregata | $P_m^s$ |
| Energia giornaliera | $E_m^s$ |
| Cabina selezionata | $b$ tale che $\chi_{cb}^s=1$ |
| Tolleranza di localizzazione | $\varepsilon_c^{\mathrm{loc}}$ |

---

# 16. Specificità dei due segmenti

## 16.1 Mobilità leggera

La fase corrente termina con costruzione, assegnazione e validazione della matrice OD veicolare light. La conversione in domanda EV/HPC appartiene alla successiva fase FRLM e non è un prerequisito della ricostruzione della mobilità. In tale fase il modello potrà utilizzare:

- matrice pendolare ISTAT;
- componente non pendolare sintetica;
- zone esterne;
- conteggi stradali leggeri per calibrazione;
- quota di veicoli elettrici;
- probabilità che un viaggio richieda ricarica rapida;
- distribuzione dello stato di carica iniziale;
- disponibilità di ricarica domestica o a destinazione.

Penetrazione EV, autonomia, SoC iniziale, disponibilità di ricarica domestica e probabilità di ricarica lungo il percorso saranno parametri e scenari del FRLM. Il flusso stradale leggero non è automaticamente domanda HPC; una forma schematica successiva è:

$$
f_q^{\mathrm{L,HPC}}
=
f_q^{\mathrm L}
\cdot s_q^{\mathrm{BEV}}
\cdot p_q^{\mathrm{need}}
$$

dove:

- $s_q^{\mathrm{BEV}}$ è la quota di BEV;
- $p_q^{\mathrm{need}}$ è la probabilità di necessità di ricarica rapida lungo il percorso.

## 16.2 Mobilità pesante

Il modello pesante può utilizzare:

- matrice merci prior;
- conversione tonnellate-veicoli;
- quota di viaggi a vuoto;
- traffico internazionale;
- porti, interporti e poli produttivi;
- conteggi ANAS pesanti;
- autonomie e consumi dei mezzi pesanti;
- vincoli sui tempi di guida e sulle soste;
- potenze di ricarica dedicate ai veicoli pesanti.

Il percorso pesante può differire da quello leggero per la stessa coppia OD. La letteratura recente offre inoltre esempi di formulazioni MILP spaziali e capacitate per la localizzazione di infrastrutture dedicate al trasporto pesante.[^depadova2024]

## 16.3 Nessuna condivisione prematura dei parametri

Non devono essere condivisi automaticamente:

- autonomia;
- stato di carica iniziale;
- consumo specifico;
- potenza del punto di ricarica;
- tempo di occupazione;
- distribuzione temporale degli arrivi;
- costo per punto;
- candidati accessibili;
- percorsi.

---

# 17. Validazione indipendente

Prima di valutare un modello combinato, ciascun segmento deve superare controlli indipendenti.

## 17.1 Validazione della rete e dei percorsi

La rete è validata secondo la procedura in due momenti della Sezione 4.4: controllo topologico minimo prima del routing e controllo mirato del sottografo $\mathcal A^{\mathrm{used}}$ dopo l'assegnazione. Gli esiti devono documentare almeno connettività, direzioni, restrizioni, svincoli, accessi zonali, gateway, associazione delle sezioni ANAS, scostamenti dai riferimenti ISTAT/TomTom e stabilità dei percorsi dopo le correzioni.

## 17.2 Validazione della domanda

Per ogni segmento:

- confronto tra flussi assegnati e conteggi usati per la calibrazione;
- confronto con conteggi lasciati fuori dalla calibrazione;
- analisi degli errori per strada, provincia e classe di flusso;
- verifica dei flussi esterni;
- verifica della distanza media degli spostamenti;
- analisi di stabilità rispetto ai parametri della matrice OD.

## 17.3 Validazione della localizzazione

- percentuale di flusso servito;
- rispetto dell'autonomia;
- rispetto dei vincoli normativi;
- utilizzo medio e di picco delle stazioni;
- numero di stazioni sottoutilizzate;
- sensibilità a domanda, autonomia e stato di carica;
- stabilità della scelta comunale e del corridoio;
- carico sulle cabine primarie.

## 17.4 Criterio per passare al modello combinato

Il modello combinato deve essere sviluppato soltanto quando:

1. la domanda leggera è ritenuta sufficientemente plausibile;
2. la domanda pesante è ritenuta sufficientemente plausibile;
3. le soluzioni separate sono stabili in scenari ragionevoli;
4. è disponibile una mappatura affidabile dei candidati alle cabine;
5. la condivisione infrastrutturale può produrre un vantaggio misurabile.

---

# 18. Possibile modello combinato futuro

Nella fase corrente $\mathcal C_{\mathrm L}$ e $\mathcal C_{\mathrm H}$ restano separati. L'eventuale riconoscimento di candidati che rappresentano lo stesso sito fisico sarà definito soltanto nello sviluppo combinato, dopo la validazione autonoma del light e della successiva componente heavy.

Si definisce:

$$
\mathcal C=\mathcal C_{\mathrm L}\cup\mathcal C_{\mathrm H}.
$$

Si introduce una variabile condivisa:

$$
x_c\in\{0,1\}
$$

che indica l'apertura fisica complessiva del sito, e variabili separate:

$$
y_c^{\mathrm L},
\qquad
y_c^{\mathrm H}.
$$

Il collegamento è:

$$
y_c^s\leq\overline y_c^s x_c
\qquad
\forall c,s.
$$

La funzione obiettivo può distinguere:

- costo fisso condiviso;
- costo dei punti leggeri;
- costo dei punti pesanti;
- costo della rete elettrica.

Una forma schematica è:

$$
\min
\sum_{c\in\mathcal C}C_c^{\mathrm{site}}x_c
+
\sum_{s\in\mathcal S}
\sum_{c\in\mathcal C_s}C_c^{\mathrm{unit},s}y_c^s
+
\sum_{b\in\mathcal B}C_b^{\mathrm{grid}}P_b^{\mathrm{exp}}.
$$

Il vincolo elettrico condiviso è:

$$
\sum_{s\in\mathcal S}
\sum_{c:(c,b)\in\mathcal G_s}
\gamma_c^s p_s^{\mathrm{unit}}y_{cb}^s
\leq
P_b^{\mathrm{avail}}+P_b^{\mathrm{exp}}
\qquad
\forall b\in\mathcal B.
$$

Il modello combinato integra le risorse, non fonde le matrici OD.

---

# 19. Corrispondenza con `Modello_FRLM_Tesi.md`

| Notazione nel file originario | Notazione proposta | Motivazione |
|---|---|---|
| $N$ come insieme di nodi, prevalentemente comuni | $\mathcal M$ per i comuni e $\mathcal C_s$ per i candidati | separazione tra zona territoriale e candidato fisico/virtuale |
| $i\in N$ | $c\in\mathcal C_s$ | l'indice della decisione infrastrutturale è il candidato |
| $Q$ | $\mathcal Q^s$ | separazione per segmento |
| $K_q$ | $\mathcal K_q^s$ | strategie specifiche per segmento |
| $f_q$ | $f_q^s$ | flusso di percorso distinto per leggeri e pesanti |
| $x_i$ | $x_c^s$ | apertura del candidato per segmento |
| $y_i$ | $y_c^s$ | numero di punti nel candidato |
| $v_q$ binaria | $u_q^s\in[0,1]$ | possibilità di copertura parziale del flusso |
| $w_{kq}$ binaria | $z_{qk}^s\in[0,1]$ | possibilità di suddividere il flusso tra strategie |
| $P_{i,0}^{sub}$ | $P_b^{\mathrm{avail}}$ e relazione $\mathcal G_s$ | connessione candidato--cabina molti-a-molti scelta dal modello |
| capacità provvisoria in veicoli | capacità energetica e temporale | coerenza tra kWh, kW e numero di punti |

Il file originario deve quindi essere letto come prima bozza del modello di ottimizzazione. Il presente documento chiarisce il significato degli oggetti sui quali il modello opera.

---


# 20. Decisioni metodologiche consolidate

> **Come usare questa sezione.** È l'indice decisionale sintetico del progetto. Se serve sapere rapidamente *che cosa è stato deciso*, partire da qui; per motivazioni, formule, audit e numeri completi tornare poi alla sezione metodologica indicata dal tema. Questa sezione non sostituisce le spiegazioni estese e non deve diventare un secondo capitolo metodologico parallelo.

Le decisioni sono registrate in ordine cronologico. **Quando una decisione successiva modifica lo stato operativo senza cancellare la validità storica di quella precedente, prevale la decisione con numero maggiore**. In particolare, le decisioni 145--158 documentano lo switch e il design iniziale di `Gamma_OSM`; le decisioni 159--176 registrano il freeze definitivo della Fase 5.6; le decisioni 177--186 chiudono la Fase 5.7; le decisioni **187--194** registrano la chiusura della Fase 5.8A; le decisioni **195--196** congelano anno di riferimento e quality eligibility ANAS; le decisioni **197--202** chiudono e congelano le Fasi 5.8R2 e 5.8D e fissano il contratto pre-calibrazione della Gravity v0. Le formulazioni precedenti restano provenance storica quando esplicitamente supersedute.

1. La fase corrente del ramo domanda è dedicata alla **calibrazione numerica della Gravity LIGHT v0**. Le Fasi 5.8A, 5.8R2 e 5.8D sono chiuse/frozen; il `NEXT` è la stima di $Q$ e $\beta$ sui tre segnali ANAS PRIMARY 2024. Costruzione e calibrazione heavy-duty restano rinviate e la formulazione heavy resta provvisoria.
2. La matrice light è espressa in veicoli/giorno medio annuo, coerentemente con il TGMA ANAS.
3. La versione base include soltanto relazioni interzonali $o\neq d$; $T_{oo}$ è escluso e potrà essere modellato separatamente in futuro.
4. Il campo **Pendolari** rappresenta occupati che raggiungono il luogo abituale di lavoro almeno tre giorni alla settimana e rientrano giornalmente alla residenza. La conversione minima adotta $s_{\mathrm{driver}}=0{,}711$, non applica un coefficiente di occupazione, costruisce andata e ritorno come flussi direzionali simmetrici e usa l'ipotesi operativa $n_{\mathrm{pend}}=220$, ottenendo $\alpha_{\mathrm{pend}}=0{,}42855$ per ciascuna direzione.
5. L'architettura generale della matrice light distingue componente pendolare, non pendolare ed esterna. **La baseline LIGHT v0 corrente è però FVG-only**: usa le sole componenti interne $C_{ij}^{ISTAT}+N_{ij}^{0}$; `EXTERNAL_OD` è `NOT_IMPLEMENTED / DOWNSTREAM EXTENSION`.
6. Il perimetro territoriale, infrastrutturale e decisionale coincide con il confine amministrativo FVG; soltanto i tratti regionali alimentano copertura e domanda energetica.
7. Le 2.895 relazioni disponibili rappresentano esclusivamente FVG--esterno; le relazioni interne sono separate, quelle esterno--FVG mancanti e il transito rinviato.
8. **KM_TOT**, **TEP_TOT** e **TTP_TOT** sono riferimenti ISTAT/TomTom usati come benchmark esterni; il routing operativo corrente è quello OSM frozen `B2 + B4 + B5` con impedenza `TIME_B5`. I riferimenti esterni non determinano il percorso canonico.
9. La diversa temporalità ISTAT 2021--ANAS non è bloccante: le fonti hanno ruoli diversi e richiedono coerenza complessiva, non coincidenza perfetta.
10. Le serie ANAS 2017--2025 sono la principale osservazione disponibile sui link, ma il target canonico Gravity v0 è `2024`: i valori storici precedenti supportano qualità/stabilità e inferenza, mentre il 2025 è il **final temporal holdout**. La Fase 5.8D ha congelato il set operativo pre-calibrazione: tre sezioni PRIMARY (`920034`, `920032`, `920039`) e 13 sezioni A/B/C di sensitivity. Nessun conteggio deve essere riprodotto forzatamente e il 2025 non può essere usato per tuning dei parametri.
11. I conteggi non identificano direttamente le celle OD; l'eventuale sottrazione pendolare avviene sui link dopo l'assegnazione.
12. Il collegamento OD--link è formalizzato da $p_{od}^{a,\mathrm L}=\sum_q\pi_{q\mid od}^{\mathrm L}\delta_{aq}$. Sul dominio interno FVG il contratto è frozen: `PRODUCT-LAMBDA` distribuisce ciascuna OD sui nove access-pair e ogni access-pair utilizza un solo percorso canonico $p^*=\arg\min TIME_{B5}$; la distanza è `PATH_ATTRIBUTE`.
13. Si distingue $y_a^{\mathrm{obs}}$ da $\widehat y_a$; gli apici $\mathrm{cal}$, $\mathrm{val}$, fonte e classe precisano il ruolo dell'osservazione.
14. Il comune è una zona simbolica e non un sito fisico. Nell'implementazione operativa OSM corrente ciascuno dei 215 comuni è collegato a tre accessi frozen $\Gamma_o^{L,OSM}$ con pesi `EXP_REL_300`; i centroidi restano riferimenti territoriali/provenance e non stazioni di ricarica.
15. I centroidi geometrici esterni sono riferimenti GIS provvisori; gateway e zonizzazione esterna saranno definiti soltanto dopo caricamento e validazione del grafo.
16. Le linee di desiderio non sono percorsi stradali e non costituiscono la base dell'assegnazione.
17. La normativa impone coperture minime, ma non viene adottata una griglia uniforme di candidati; i siti sono costruiti caso per caso e possono essere più densi del minimo.
18. Candidati light e heavy restano separati; eventuali siti fisici coincidenti saranno riconosciuti soltanto nel futuro modello combinato.
19. La connessione elettrica è una relazione molti-a-molti candidato--cabina; nella base ogni stazione sceglie una sola cabina ammissibile.
20. La conversione dei flussi light in domanda EV/HPC è successiva a costruzione, assegnazione e validazione della matrice veicolare.
21. Nella base tutte le colonnine HPC hanno uguale tipologia e potenza; il modello sceglie il numero di unità senza code, profili orari o livelli di servizio dettagliati.
22. Valori e requisiti AFIR sono provvisori e saranno verificati prima della redazione finale, applicandoli soltanto a tratte, categorie e orizzonti pertinenti.
23. Nel modello gravitazionale doppiamente vincolato i fattori di bilanciamento sarebbero calcolati iterativamente e non stimati come parametri liberi; **questa non è però la specifica v0 corrente**, che usa $P_i$ e $A_j$ come proxy relativi senza hard margins e non applica Furness in questa fase.
24. QGIS è dedicato soprattutto a preparazione e validazione spaziale; Python a matrici, costi, assegnazione, calibrazione e output FRLM.
25. Matrice OD, flussi sui link e flussi di percorso sono oggetti distinti; il FRLM riceve path flows.
26. Nodi zonali, nodi del grafo, candidati e stazioni fisiche finali sono entità diverse.
27. Il flusso attraversante un candidato non coincide con la domanda di ricarica.
28. L'eventuale modello combinato condivide risorse e rete elettrica, non fonde le matrici OD.
29. L'output strategico può essere aggregato a livello comune--corridoio senza coordinate esecutive definitive.
30. Il presente notebook è il riferimento unico per lo stato corrente; le bozze incompatibili hanno valore storico.
31. **Decisione storica della fase precedente allo `SWITCH_OSM`:** il dataset ufficiale GSFVG era stato scelto come backbone topologico principale della rete light e il file originale `GSFVG_IRDAT.shp` restava immutato; l'estratto Geofabrik `nord-est_2026-08-03.osm.pbf` era allora fonte complementare congelata. Questa decisione è successivamente superseduta dalle decisioni 147 e 159.
32. La validazione del grafo avviene in due momenti: controllo minimo del backbone prima del routing e verifica ad alta intensità del sottografo effettivamente usato dopo l'all-or-nothing.
33. I valori nulli di `oneway` non sono controllati uno per uno: si applicano regole OSM tracciabili e si sottopongono a verifica manuale i casi critici per backbone, flussi, ANAS, gateway e candidati.
34. Nei layer OSM complementari, `unclassified`, `residential` e `living_street` restano disponibili come possibili connettori e `service` resta in un layer ausiliario; il loro ruolo nel backbone GSFVG dipende dal matching, dalla classificazione regionale e dalla validazione locale.
35. Per le matrici osservate si instradano tutte le OD con flusso positivo; un'eventuale soglia sulla seed gravitazionale densa è ammissibile solo quantificando il flusso escluso e verificandone la sensibilità.
36. Per le automobili la restrizione statica effettiva segue la gerarchia `motorcar` → `motor_vehicle` → `vehicle` → `access`: il valore più specifico prevale su quello generale e i conteggi separati non sono additivi.
37. Il censimento completo di `other_tags` comprende 112.466 geometrie, 446 chiavi residue, 312.530 occorrenze di coppie chiave--valore e 4.597 combinazioni distinte; occorrenze e combinazioni distinte non sono sinonimi.
38. Le restrizioni direzionali e condizionali sono sovrapposizioni della regola statica, non categorie da sommare: `oneway:conditional` resta distinto da `oneway=alternating` e `oneway=reversible`.
39. Le 191 chiavi selezionate automaticamente costituiscono un filtro di richiamo da sottoporre a selezione semantica, non l'elenco dei campi operativi.
40. Il censimento del layer lineare non copre automaticamente relazioni di svolta e barriere puntuali; tali elementi devono essere estratti dal PBF e integrati con procedure dedicate.
41. L'accesso automobilistico è normalizzato separatamente nei campi `access_fwd_class` e `access_bwd_class`: `open` confluisce nel codice 1, `local_restricted` nel codice 2 e `permission_restricted`, `authorized_only` e `prohibited` nel codice 0.
42. Il primo grafo è statico: le restrizioni `*:conditional` sono conservate e validate sintatticamente, ma non applicate finché non viene definito uno scenario con data e ora di riferimento. Le 59 occorrenze condizionali di accesso appartengono a 59 geometrie distinte e sono segnalate da `access_conditional_flag`.
43. I campi `routing_fwd_code` e `routing_bwd_code` sintetizzano accesso e direzione: 0 significa verso non disponibile, 1 accesso ordinario e 2 accesso locale o riservato al traffico di destinazione. Questa semantica non coincide con `direction_code=0`, che indica bidirezionalità o trattamento provvisorio come tale.
44. Dopo l'audit delle 118 geometrie speciali, la direzionalità statica ripartisce le 112.466 geometrie in 31.614 percorribili soltanto nel verso della geometria, 1 soltanto nel verso opposto e 80.851 bidirezionali o trattate staticamente come tali; queste ultime comprendono 3.885 `explicit_bidirectional`, 76.870 `default_bidirectional`, 95 `alternating` e 1 `reversible`.
45. `alternating` e `reversible` sono trattati intenzionalmente come percorribili nei due versi nel modello statico, pur conservando uno stato speciale per una futura penalizzazione legata all'attesa. Il caso di Via dei Bagni Nuova è stato verificato manualmente come ponte a senso alternato regolato da semaforo.
46. I 22 `junction=circular` privi di `oneway` sono stati riclassificati come monodirezionali nel verso della geometria; i conteggi operativi finali sono 110.325 ordinari, 500 locali, 1.640 chiusi per accesso e 1 chiuso per direzione nel verso della geometria, e 78.961 ordinari, 424 locali, 1.467 chiusi per accesso e 31.614 chiusi per direzione nel verso opposto.
47. In caso di chiusura simultanea, `closed_direction` ha priorità nello status finale; le classi di accesso restano conservate. I campi `routing_fwd_closure_cause` e `routing_bwd_closure_cause` distinguono `none`, `access`, `direction` e `both`.
48. Il layer OSM consolidato è `osm_rete_light_fvg_20km_routing_statico_validato` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg` e costituisce la fonte complementare per trasferire e validare impedenze, velocità e tempi di percorrenza sulla copia operativa GSFVG.
49. I controlli computazionali finali non hanno rilevato valori nulli né incoerenze tra status, codici operativi e cause di chiusura; la distribuzione forward e backward post-audit è registrata nel presente notebook.
50. `speed_fwd_obs_kmh` e `speed_bwd_obs_kmh` rappresentano limiti o valori di velocità ricavati dai tag OSM per verso; `obs` indica provenienza dal dato OSM e non una misurazione sul campo né una velocità media di percorrenza.
51. La normalizzazione dei dati OSM di velocità è completata nel layer `osm_rete_light_fvg_20km_velocita_osm_normalizzate`; i valori ambigui non sono aggregati arbitrariamente e restano irrisolti e tracciati.
52. Le velocità operative iniziali per classe sono fissate come benchmark dalla tabella `speeds.highway` del profilo automobilistico ufficiale OSRM v26.7.3 e applicate nel layer `osm_rete_light_fvg_20km_velocita_modello_base`; quando un valore OSM direzionale è ricavabile si adotta la regola `speed_model = min(speed_obs, speed_class)`, altrimenti il default di classe.
53. Il primo all-or-nothing minimizzerà il costo temporale statico; la distanza resta separata per autonomia, consumo energetico e FRLM. Lunghezze ellissoidali e tempi statici forward e backward sono stati calcolati e validati sui layer OSM complementari e dovranno essere trasferiti e controllati sulla copia operativa GSFVG. Nella rete ordinaria il costo coincide con il tempo statico: i versi con `routing_*_code=0` sono esclusi, quelli con codice 1 sono utilizzabili nel routing ordinario e quelli con codice 2 sono conservati esclusivamente per i collegamenti iniziali o finali delle zone.
54. I campi `speed_*_obs_kmh` sono limiti o valori ricavati dai tag OSM e non misure empiriche; i campi `speed_*_model_kmh` sono stime operative di routing da validare e calibrare rispetto a ISTAT/TomTom e a itinerari esterni.
55. La velocità non determina la percorribilità: l'inclusione di un verso nel grafo dipende esclusivamente da `routing_fwd_code` e `routing_bwd_code`; i campi di velocità possono restare valorizzati anche quando il corrispondente verso è chiuso.
56. Il punto 4 della roadmap è completato il 6 agosto 2026 con la selezione di GSFVG come backbone topologico, supportata dall'audit di connettività, e con il mantenimento di OSM come fonte complementare. Il punto 5 è chiuso a livello metodologico con classi stradali e velocità di modello, direzionalità e accessi, lunghezze ellissoidali, tempi forward e backward e costo temporale statico validati sui layer OSM; resta il trasferimento tecnico e validato sulla copia operativa GSFVG. Prima dell'avvio operativo del punto 6 deve essere chiuso il trattamento di `TRIM_USAGE=0`; successivamente si procederà alla costruzione degli insiemi di accesso comunali $\Gamma_o^{\mathrm L}$ e dei relativi pesi.
57. I versi con `routing_*_code=2` non ricevono una penalità numerica generica: sono esclusi dalla rete core di attraversamento e possono essere utilizzati soltanto come collegamenti iniziali o finali degli accessi zonali $\Gamma_o^{\mathrm L}$. Questa regola evita scorciatoie improprie e dipendenza del costo dalla segmentazione OSM.

58. Nel grafo GSFVG `TRIM_USAGE` è l'attributo primario per la percorribilità direzionale; `DIR`, `ENTEXT` e `GEOMETRY_R` descrivono la struttura LRS e non sono utilizzati isolatamente per dedurre il senso legale di marcia.
59. I 616 archi originariamente privi di `TRIM_USAGE` sono stati revisionati manualmente sull'intera popolazione. L'esito validato è 557 `FWD_ONLY`, 1 `BWD_ONLY`, 58 `BIDIRECTIONAL` e 0 `UNRESOLVED`; i tre blocchi SC, AS mainline e AS residui coprono esattamente 616/616 archi, senza sovrapposizioni, mancanti o estranei.
60. Il tentativo di attribuzione automatica collettiva dei 363 AS mainline è stato scartato dopo che il controllo campionario indipendente ha individuato almeno un errore. Il campionamento ha quindi svolto una funzione di falsificazione e ha determinato il passaggio alla revisione manuale completa dell'intera popolazione.
61. Gli attributi GSFVG originali restano immutati; le decisioni ricostruite, la percorribilità forward e backward, la fonte, la confidenza e le note di validazione sono conservate in output e campi derivati separati della copia operativa.
62. Gli archi con `TRIM_USAGE` già valorizzato non sono assunti automaticamente come pronti per il routing. In particolare, `TRIM_USAGE=0` richiede controllo semantico con `DBPRIOR_ST` e `DBPRIOR_TI`; i 33.993 casi con entrambi gli attributi nulli e i 20 elementi `DBPRIOR_TI=3` restano oggetto di verifica dedicata.
63. Per gli 11.256 archi con `TRIM_USAGE=1/2` la direzione operativa è congelata sulla codifica ufficiale GSFVG: `TRIM_USAGE=1` → `FWD_ONLY`, `TRIM_USAGE=2` → `BWD_ONLY`. L'audit indipendente OSM ↔ GSFVG è concluso con 7.456 `AGREEMENT_HIGH`, 3.009 `CONFLICT_HIGH`, 52 `AMBIGUOUS_MATCH`, 128 `AMBIGUOUS_DIRECTION` e 611 `INSUFFICIENT_MATCH`; una discordanza OSM non modifica automaticamente GSFVG.
64. Il matching OSM ↔ GSFVG è stato verificato manualmente in uno specifico regime geometrico robusto mediante 24 casi bilanciati, con esito 24/24 `MATCH_VALID`. In tale regime ricadono 1.459 dei 3.009 conflitti HIGH e 1.143 dei 2.190 `OPPOSITE_ONEWAY`; ciò dimostra che una quota consistente delle discordanti è reale rispetto alle due fonti, senza determinare quale rappresenti correttamente la situazione corrente.
65. Non viene eseguita un'ulteriore revisione manuale massiva dei 3.009 conflitti. Nel futuro layer operativo le direzioni `TRIM_USAGE=1/2` restano GSFVG-primary e OSM-QA, con campi derivati almeno per `direction_source`, `direction_conflict_osm`, `direction_conflict_type` e `osm_match_confidence`. I 616 `TRIM_USAGE=NULL` restano un caso separato già ricostruito manualmente.
66. Overture Maps Transportation è stato sottoposto a benchmark come possibile rete alternativa. Pur mostrando topologia nativa a connector, buona continuità e routing preliminare positivo, presenta 3.721 feature con direzione non risolta, velocità esplicita disponibile soltanto sul 17,294% della rete automobilistica, 9.578 `prohibited_transitions` non ancora applicate e una provenance dominata da OSM (99,739% delle feature analizzate). Non viene pertanto adottato come grafo operativo principale.
67. **Nella fase storica precedente allo `SWITCH_OSM`**, la configurazione metodologica della rete light era **GSFVG + OSM**, con Overture come terza fonte di controllo mirata. Sul test set di 247 casi GSFVG già validati, Overture risulta concorde in 186 casi, discordante in 52 e non risolto in 9; tali esiti non autorizzano trasferimenti automatici di direzione, accesso o altri attributi.
68. Prima dell'assegnazione all-or-nothing deve essere eseguita una validazione funzionale del grafo adottato sulle relazioni OD direzionali: per i 215 comuni interni, escludendo gli intracomunali, il dominio completo comprende 46.010 coppie ordinate. La validazione verifica raggiungibilità, plausibilità dei cammini minimi, asimmetrie di verso e anomalie di tempo/distanza; le deviazioni non costituiscono automaticamente errori ma casi diagnostici.
69. La tesi di Tufaro costituisce un precedente metodologico diretto per la trasformazione topologica del GSFVG: il lavoro parte da 76.350 segmenti, riconosce coincidenze degli estremi con soglia di 0,1 m e riduce il grafo a 29.695 nodi. La coincidenza del numero di segmenti con l'audit corrente rafforza la plausibilità del GSFVG come backbone, senza dimostrare l'identità delle versioni del dataset.[^tufaro2022]
70. Il grafo computazionale di Tufaro non conserva la direzionalità originaria: `conv.f` trasferisce estremi e lunghezza, mentre `nodes.f` costruisce adiacenze reciproche. Tale scelta è coerente con un'analisi di connettività e ridondanza, ma **non è trasferibile al routing diretto della presente tesi** e non costituisce evidenza sufficiente per trattare automaticamente tutti i `TRIM_USAGE=0` come bidirezionali.
71. L'associazione comunale di Tufaro al nodo stradale più prossimo è adottabile soltanto come baseline con $|\Gamma_o^{\mathrm L}|=1$; la configurazione corrente mantiene insiemi di accesso multipli e pesati per ridurre la concentrazione artificiale dei flussi.
72. La procedura `per.f` di Tufaro ricerca itinerari alternativi mediante euristica geometrica, backtracking e rimozione casuale del 20% dei segmenti del primo percorso; non equivale a uno shortest path standard. Il primo routing della presente tesi resta quindi un all-or-nothing su grafo diretto con costo temporale statico. I conteggi discordanti dei percorsi riportati nella tesi di Tufaro non vengono usati come benchmark quantitativo.

---


73. Per `TRIM_USAGE=0`, OSM è utilizzato come segnale indipendente di QA e non come fonte sostitutiva automatica. La shortlist one-way con match `HIGH/STRICT` comprende 3.349 archi; 212 casi sono riconducibili a rotatorie e 588 a carreggiate o rami separati, lasciando 2.549 discordanti residue stratificate in P1--P4.
74. Il gruppo P1 (`STRICT` + `DBPRIOR_ST=1`) è chiuso mediante revisione manuale esaustiva 136/136. Gli esiti finali sono 103 archi effettivamente monodirezionali e 33 da mantenere bidirezionali; nessun caso resta `UNRESOLVED`. Questo risultato conferma che il segnale OSM strict è fortemente informativo ma non sufficientemente affidabile per una correzione automatica dell'intera popolazione.
75. Per P2 (`STRICT` + `DBPRIOR_ST` non compilato) non viene eseguito un trasferimento automatico né una revisione esaustiva dell'intera popolazione di 1.067 archi. Il disegno è stato fissato in 127 casi, di cui 120 `INFERENTIAL_SAMPLE` e 7 `STRATEGIC_SUPPLEMENT`; la revisione manuale è chiusa con 65 `FWD_ONLY`, 46 `BWD_ONLY`, 16 `BIDIRECTIONAL` e 0 `UNRESOLVED`.
76. La distinzione 120 inferenziali + 7 strategici resta valida per interpretare P2, ma i risultati non vengono automaticamente trasferiti alla popolazione residua. I 940 archi P2 non revisionati mantengono `TRIM_USAGE=0`; la revisione esaustiva di P3/P4 resta sospesa e subordinata all'evidenza funzionale prodotta dal routing.
77. La correzione delle differenze locali di micro-topologia è subordinata all'impatto sulla percorribilità e sulla connettività del grafo regionale. Il modello non mira a ricostruire esaustivamente la disciplina urbana, ma a garantire cammini minimi intercomunali robusti per l'assegnazione dei flussi.
78. Overpass Turbo / Overpass API non costituisce una fonte indipendente da OSM: interroga lo stesso database già rappresentato nell'estratto Geofabrik. Può essere utilizzato per ispezionare way, nodi, tag e micro-topologia locale, ma non come fonte autonoma di verità direzionale.
79. Il benchmark del classificatore topologico Overpass sui 136 casi P1 ha prodotto 43 `AUTO_ONEWAY_CANDIDATE`, 77 `AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE` e 16 `MANUAL_REVIEW_REQUIRED`. A fronte di una copertura teorica dell'88,24%, l'accuratezza delle classificazioni automatiche è risultata del 55,00%, con 3 falsi one-way, 51 falsi bidirezionali e 54 errori complessivi; il classificatore automatico viene pertanto scartato.
80. La precisione di `AUTO_ONEWAY_CANDIDATE` (93,02%) non autorizza comunque un trasferimento automatico della direzione OSM, poiché nel benchmark compare un caso noto in cui la monodirezionalità è corretta ma il verso OSM è opposto a quello verificato indipendentemente. `AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE`, con precisione 33,77%, non è utilizzabile come regola decisionale.
81. Overpass può essere richiamato soltanto come supporto diagnostico nei casi topologicamente complessi. Il layer usato per la revisione P2 resta `P2_OSM_AS_GSFVG_DISPLAY`, nel quale la freccia DISPLAY rappresenta la direzione OSM tradotta nel riferimento della geometria GSFVG originale; il benchmark Overpass non autorizza alcuna estensione automatica a P3/P4.
82. La validazione funzionale pre-AON distingue $G^{\mathrm{full}}$, la baseline conservativa $G^{\mathrm{base}}$ e, se utile, una variante diagnostica $G^{\mathrm{safe}}$ con esclusioni mirate. Nella baseline P3/P4 non verificati restano bidirezionali; le eventuali esclusioni in $G^{\mathrm{safe}}$ non modificano il dataset sorgente.
83. La rilevanza topologica di un arco dubbio può essere misurata tramite il recupero di raggiungibilità $\Delta R_a$ e, quando esiste una domanda preliminare, tramite il recupero di flusso $\Delta F_a$; tali indicatori servono a prioritizzare la verifica degli archi che incidono materialmente sulla rete regionale.
84. L'insieme $\mathcal A^{\mathrm{SP}}$, definito come unione degli archi appartenenti ai cammini minimi delle OD considerate, costituisce il sottografo funzionale pre-assegnazione. Per ogni arco si calcolano almeno $N_a^{\mathrm{OD}}$ e, quando disponibile la domanda, $F_a^{\mathrm{OD}}$. Questo sottografo è distinto da $\mathcal A^{\mathrm{used}}$, che viene estratto successivamente dall'assegnazione delle sole OD con flusso positivo.
85. Il confronto dei costi di rete con `KM_TOT`, `TEP_TOT`, `TTP_TOT` e altri riferimenti ISTAT/TomTom è diagnostico e subordinato alla compatibilità semantica. Le anomalie OD vengono retro-proiettate sugli archi mediante un indicatore $H_a$, così da individuare hotspot topologici o direzionali ricorrenti.
86. La priorità di revisione degli archi combina raggiungibilità, uso nei cammini minimi, eventuale flusso, frequenza nei percorsi anomali e rilevanza strategica. Non è obbligatorio condensare tali grandezze in un unico indice numerico: una classificazione gerarchica è sufficiente nella prima applicazione.
87. La validazione funzionale è iterativa e termina quando le incertezze residue non alterano materialmente raggiungibilità, cammini minimi e indicatori aggregati. L'obiettivo non è eliminare ogni incertezza locale, ma dimostrare la robustezza funzionale del grafo rispetto agli usi della tesi.
88. Dopo la chiusura delle revisioni manuali P1 e P2, la revisione esaustiva dei 363 archi P3 e dei 983 archi P4 è sospesa. Il loro stato non verificato viene mantenuto nella baseline come `TRIM_USAGE=0`, privilegiando la conservazione della connettività rispetto all'introduzione di sensi unici non confermati.
89. Lo scenario operativo $G^{\mathrm{base}}$ applica le direzioni ufficiali `TRIM_USAGE=1/2`, le decisioni manuali sui 616 NULL, le decisioni P1 e le decisioni disponibili sui 127 casi P2 revisionati; P3/P4 non verificati restano bidirezionali.
90. La revisione P3/P4 viene riaperta selettivamente sugli archi che producono effetti materiali su reachability, componenti fortemente connesse, dead end, continuità delle arterie principali, shortest path o frequenza di utilizzo nelle OD.
91. È ammesso uno scenario di sensibilità $G^{\mathrm{stress}}_{\mathrm{OSM}}$ che applichi in modo più esteso le indicazioni one-way OSM di P3/P4 esclusivamente come stress test. Le differenze rispetto alla baseline servono a individuare archi e OD sensibili, non a validare automaticamente OSM.
92. Il criterio di chiusura della fase direzionale diventa funzionale: non è richiesta la revisione esaustiva della micro-topologia urbana se le incertezze residue non alterano materialmente i percorsi e gli indicatori utilizzati dal modello.
93. Il consolidamento direzionale è materializzato nel GeoPackage `GSFVG_operativo_direzionale_v01.gpkg`, mantenuto in EPSG:25833 e costruito senza modificare geometrie o attributi GSFVG originari.
94. Gli override manuali consolidati sono 879 `ID1` distinti: 616 casi originariamente `TRIM_USAGE=NULL`, 136 P1 e 127 P2. `ID1` è la chiave logica; duplicazioni, riferimenti inesistenti o incompatibilità di scope determinano hard failure della build.
95. `dir_final` è l'unica sorgente dei flag `dir_fwd_ok` e `dir_bwd_ok`; le combinazioni ammesse sono `(1,0)` per `FWD_ONLY`, `(0,1)` per `BWD_ONLY` e `(1,1)` per `BIDIRECTIONAL`. La combinazione `(0,0)` non è ammessa.
96. La build v01 conserva 26 attributi originali e aggiunge campi operativi e di audit trail; `gsfvg_fid_original` materializza il FID sorgente, mentre il `fid` GeoPackage non è considerato identificativo stabile.
97. L'audit finale della build direzionale ha verificato 76.349 geometrie e attributi source-output con zero mismatch, zero `UNRESOLVED`, zero `dir_final` invalidi e zero incoerenze dei flag. La distribuzione finale è 9.371 `FWD_ONLY`, 2.657 `BWD_ONLY` e 64.321 `BIDIRECTIONAL`.
98. `GSFVG_operativo_direzionale_v01` contiene esclusivamente la semantica direzionale consolidata. Restrizioni di accesso, velocità, tempi, impedenze/costi e l'eventuale riproiezione devono essere applicati successivamente come trasformazioni separate e auditabili prima della costruzione/validazione definitiva del grafo computazionale.
99. Il primo stress test funzionale sui 1.346 P3/P4 ha ridotto la reachability da 45.370 a 43.893 OD raggiungibili e ha identificato un insieme ristretto di archi con contributo alla riconnessione; la perdita nello scenario di esclusione è usata come diagnostica, non come prova di errore dei singoli archi.
100. Sono stati revisionati manualmente 8 P3/P4 funzionalmente prioritari; quattro hanno prodotto override effettivi: `20894`, `20974`, `51544 → BWD_ONLY` e `67316 → FWD_ONLY`.
101. Dopo tali revisioni, i 1.338 P3/P4 residui producono **zero OD aggiuntive perse** nello stress test residuo. Il gate P3/P4 relativo alla sola reachability è quindi considerato chiuso/convergente; i residui vengono riaperti soltanto in presenza di successive anomalie di percorso.
102. La differenza di 424 OD osservata in `G_iter1` rispetto al Gate 4 non era dovuta ai P3/P4, ma ai nodi di accesso diagnostici di Sacile e Codroipo, entrambi esterni alla giant SCC. Sostituendoli rispettivamente con i nodi 48739 e 39946, la reachability torna esattamente a 45.370/46.010.
103. La configurazione `|Γ_o^L|=1` resta una baseline diagnostica ma il nodo di accesso non deve essere scelto sulla sola distanza geometrica. La definizione finale di `Γ_o^L` deve controllare almeno l'appartenenza alla componente direzionale funzionale principale e deve essere confrontata con una configurazione multi-accesso.
104. Una OD non raggiungibile non deve essere attribuita automaticamente a un arco errato: prima di riaprire la revisione della rete occorre distinguere tra difetto interno al grafo e difetto dell'associazione zona–rete.
105. Chiuso il gate reachability P3/P4, la validazione funzionale prosegue su shortest path, distanze, tempi, asimmetrie direzionali e confronto con riferimenti esterni, mantenendo la possibilità di retro-proiettare nuove anomalie sugli archi responsabili.
106. La configurazione nearest-node $|\Gamma_o^{\mathrm L}|=1$ resta esclusivamente una baseline diagnostica: la selezione degli accessi comunali deve considerare anche l'ammissibilità direzionale e non soltanto la distanza geometrica.
107. Un nodo è ammissibile come accesso light se appartiene territorialmente al comune rappresentato e alla giant SCC del grafo computazionale. Il filtro giant-SCC impedisce di associare la zona a micro-componenti direzionali marginali; di conseguenza la reachability fra accessi filtrati non costituisce una validazione indipendente successiva del grafo.
108. Per ciascuno dei 215 comuni vengono considerati i 20 nodi computazionali più vicini al centroide di popolazione. Tutti i comuni dispongono di almeno un candidato giant-SCC interno entro i primi 10; 210/215 hanno già il nearest node nella giant SCC.
109. La cardinalità strutturale degli accessi light viene fissata a $|\Gamma_o^{\mathrm L}|=3$. La selezione è `anchored`: il primo accesso è il nodo ammissibile più vicino al centroide; i due secondari sono scelti fra i top-20 preservando prossimità e diversità.
110. Due accessi dello stesso comune devono essere distanti almeno 100 m e non possono condividere alcun `ID1` GSFVG incidente. La soglia di 100 m è il massimo compromesso verificato che consente una tripletta ammissibile per 215/215 comuni; 150 m produce già un'eccezione e 200 m due.
111. La selezione anchored è fattibile per 215/215 comuni. La mediana dell'extra-distanza dell'accesso più lontano è 126,7 m, il p95 394,6 m e il massimo 711,5 m; la separazione minima mediana è 149,3 m e il minimo osservato 100,5 m.
112. I pesi $\lambda_{ou}^{\mathrm L}$ sono definiti mediante decadimento esponenziale sulla distanza relativa dall'accesso primario. La configurazione preferita è $\tau=300$ m: peso primario mediano 0,417, peso terziario mediano 0,268 e numero effettivo mediano di accessi $N_o^{\mathrm{eff}}=2,894$.
113. Il gate degli accessi comunali light è definitivamente chiuso: la regola anchored $|\Gamma_o^{\mathrm L}|=3$ è stata materializzata e auditata nel dataset canonico `Gamma_L_comuni_fvg_v01`.
114. `Gamma_L_comuni_fvg_v01` contiene 645 record, pari a 3 accessi per ciascuno dei 215 comuni. Il primary coincide con il nearest-node ammissibile per 215/215 comuni; tutti i 645 accessi appartengono al proprio comune e alla giant SCC.
115. La separazione minima materializzata fra accessi dello stesso comune è 100,487 m e non esistono coppie con `ID1` incidente condiviso. La regola di tie-break deterministica utilizza, dopo i tre criteri geometrici/topologici, l'identificativo del nodo.
116. La ponderazione definitiva della versione corrente è `EXP_REL_300`, con $\tau=300$ m e pesi strettamente positivi normalizzati a 1 per ciascun comune. La mediana di $N_o^{\mathrm{eff}}$ è 2,894; il p05 è 2,203 e il minimo 1,535.
117. Il prodotto canonico è `C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_light\Gamma_L_comuni_fvg_v01.gpkg`; il CSV gemello è `Gamma_L_comuni_fvg_v01.csv` e il manifest è `Gamma_L_comuni_fvg_v01_manifest.json`. Il GeoPackage contiene inoltre `Gamma_L_metadata_v01` con 18 record.
118. `Gamma_L_comuni_fvg_v01.gpkg` ha SHA-256 `425dab4a127aeca2169911763f48238275d1dffe553d857a44c0fda773406179`; il CSV ha SHA-256 `1a6d4d8075f84a9c98a39b4bde072ae15575a9a8dc5c02b39420fdb6cdd16386`.
119. Il dataset degli accessi deriva da `GSFVG_operativo_direzionale_v02.gpkg`, SHA-256 `5afb71c500c2ff2cd779cee1601fb42354eadcf754ddc83ac2fc5fc481d7c02e`. I checksum degli input intermedi di inventario, selezione anchored e sensitivity pesi sono registrati nella provenance del gate.
120. L'hard audit del dataset ha verificato cardinalità, ordini 1--3, unicità dei nodi per comune, giant SCC, appartenenza territoriale, separazione, diversità `ID1`, formula di $\Delta d$, formula e normalizzazione di $\lambda$, rilettura GeoPackage/metadata e `PRAGMA integrity_check = ok`, senza errori.
121. `Gamma_L_comuni_fvg_v01` è un prodotto separato dal GeoPackage stradale: descrive la relazione zona--rete e non siti di ricarica. Le precedenti Run 1--4 restano documentazione diagnostica/provenance e non sono più sorgenti operative quando è disponibile il dataset canonico.
122. Il gate successivo agli accessi è l'audit dei shortest path metrici sulla baseline `GSFVG_operativo_direzionale_v02 + Gamma_L_comuni_fvg_v01`. La prima impedenza è la lunghezza geometrica; devono essere preservati gli archi paralleli mediante `MultiDiGraph` o trasformazione equivalente e l'identità `ID1`/parte deve restare ricostruibile.
123. SP0 è il controllo di regressione computazionale di grafo e accessi: la ricostruzione canonica con endpoint, tolleranza `dwithin` 0,10 m e clustering Union-Find riproduce esattamente 61.947 nodi.
124. La regressione topologica SP0 riproduce 140.213 archi del `DiGraph`, 10 WCC, giant WCC di 61.914 nodi, 625 SCC e giant SCC di 61.109 nodi. Tutti i 645 accessi canonici sono mappati, distinti e appartenenti alla giant SCC.
125. SP1 considera separatamente le 9 combinazioni fra 3 accessi di origine e 3 di destinazione per ciascuna delle 46.010 OD comunali ordinate, ottenendo 414.090 relazioni access-to-access. In questa fase non viene ancora definita l'aggregazione comunale.
126. La rappresentazione fisica autorevole di SP1 è un `MultiDiGraph` con 140.668 directed edge instances. Il `DiGraph` ausiliario contiene 140.213 coppie ordinate dopo la selezione deterministica dell'edge di lunghezza minima; esistono 455 coppie ordinate parallele, molteplicità massima 2 e 32 parità esatte di costo.
127. Gli shortest path metrici sono calcolati con 645 single-source Dijkstra, uno per ciascun accesso. Il confronto di tre sorgenti con NetworkX restituisce differenza massima pari a 0 m; la differenza massima fra costo Dijkstra e somma delle lunghezze del path ricostruito è $4{,}07\times10^{-10}$ m.
128. Tutte le 414.090 relazioni access-to-access risultano raggiungibili. Tale risultato è un controllo di coerenza di SP1, non una nuova prova indipendente di reachability, perché gli accessi sono stati selezionati nella giant SCC.
129. L'insieme provvisorio $\mathcal A^{\mathrm{SP}}_{\mathrm{ACCESS}}$ contiene 61.852 directed edge instances utilizzate almeno una volta; il numero totale di attraversamenti ricostruiti è 141.595.042. $N_a^{\mathrm{ACCESS}}$ è riservato ai path access-to-access, mentre $N_a^{\mathrm{OD}}$ verrà definito soltanto dopo la scelta della regola comunale.
130. L'asimmetria access-to-access presenta mediana $\rho=1{,}004581$, p95 $1{,}051639$, massimo $4{,}060971$ e differenza assoluta massima andata/ritorno 27,154 km. La coda superiore richiede audit mirato.
131. Lo spread fra le nove alternative di ciascuna OD ha mediana 0,924 km e p95 3,648 km; nella maggioranza delle OD la struttura multi-accesso modifica la distanza in misura contenuta.
132. MIN-PATH produce una sola combinazione strettamente minima per tutte le 46.010 OD e sarebbe quindi deterministico, ma concentra l'intera OD su una sola coppia di accessi.
133. PRODUCT-LAMBDA assegna alle nove combinazioni peso $\omega_{oduv}=\lambda_{ou}^{\mathrm L}\lambda_{dv}^{\mathrm L}$. La differenza diagnostica $D^{\mathrm{PL}}_{od}-D^{\mathrm{MIN}}_{od}$ ha mediana 0,464 km e p95 1,907 km. Il risultato è favorevole alla rappresentazione multi-accesso, ma non congela ancora PRODUCT-LAMBDA.
134. SP0 e SP1 sono computazionalmente superati. Resta aperto soltanto il gate metodologico sull'aggregazione comunale delle nove combinazioni; prima della decisione devono essere auditati asimmetrie estreme, spread, differenze PRODUCT-LAMBDA/MIN-PATH, archi ad alto $N_a^{\mathrm{ACCESS}}$, distribuzione degli `access_order` e sovrapposizione fra le anomalie.
135. Durante SP2, le 1.818 OD con $KM\_TOT < D_{\min}^{GSFVG}$ e la shortlist di 125 OD sono segnali diagnostici, non conteggi di errori dimostrati; la causa deve essere stabilita mediante review manuale.
136. È approvato un **Gate preliminare OSM shadow**. L'approvazione riguarda il disegno sperimentale; **non è approvata la migrazione del backbone a OSM**.
137. Durante il Gate shadow GSFVG resta la baseline operativa corrente, ma il lavoro già svolto non costituisce argomento per mantenerlo. La decisione futura confronta prospetticamente $T_{\mathrm{finish}}^{GSFVG}$ e $T_{\mathrm{switch+finish}}^{OSM}$.
138. Il Gate shadow è time-boxed a circa una giornata effettiva come tetto massimo e utilizza indicativamente 3–5 siti causali fisici indipendenti.
139. Il controfattuale OSM deve essere locale, preferibilmente fra punti immediatamente a monte e a valle della causa, per isolare topologia e regole di percorrenza senza introdurre accessi comunali o una pipeline regionale OSM.
140. La tassonomia causale minima è `DATA_ERROR`, `REPRESENTATION_ERROR`, `GSFVG_DATA_MISSING`, `GSFVG_UNRESOLVED`, `OTHER`. Missing e unresolved contribuiscono al costo di review ma non dimostrano automaticamente un vantaggio strutturale OSM.
141. OSM deve essere valutato rispetto a una ground truth indipendente e non può essere benchmark di se stesso. Il confronto diretto GSFVG-vs-OSM viene omesso quando GSFVG non offre un'informazione interpretabile.
142. Il Gate shadow è causale: `KM_TOT` non viene usato tramite MAE/RMSE come criterio globale di scelta perché TomTom è time-minimizing mentre SP2 corrente è distance-minimizing.
143. Le stop rule sono simmetriche: si arresta OSM se non emerge rapidamente vantaggio strutturale o se il costo di pipeline è eccessivo; si arresta l'audit GSFVG estensivo se più `REPRESENTATION_ERROR` indipendenti e un OSM nativamente corretto rendono razionale una `switch feasibility`.
144. Il criterio finale è prospettico: confrontare $B_{GSFVG}^{future}$ con $B_{OSM}^{switch}$ e scegliere la soluzione che, da oggi, può essere resa sufficientemente affidabile prima.
145. Il Gate preliminare OSM shadow è **CHIUSO** dopo 3 siti causali indipendenti e 1 controllo negativo valido: `DATA_ERROR=1`, `GSFVG_DATA_MISSING=2`, `REPRESENTATION_ERROR=0`, OSM native correct 3/3, coverage advantage 2, controlli negativi peggiorati 0.
146. Due `GSFVG_DATA_MISSING` indipendenti ad alto `correction burden`, entrambi presenti e routable nativamente in OSM, sono ritenuti evidenza sufficiente sul **costo residuo** anche in assenza di `REPRESENTATION_ERROR`. La precedente stop rule viene quindi interpretata in senso prospettico, non come requisito rigido di representation error.
147. È approvata la decisione $\text{SWITCH\_OSM}$: OSM diventa il backbone operativo light per le fasi successive. GSFVG resta benchmark storico, fonte istituzionale di confronto e supporto diagnostico.
148. L'audit GSFVG estensivo è arrestato. Non vengono avviate ulteriori campagne generiche P1/P2/P3/P4 o SP2 sul GSFVG; eventuali riaperture future devono essere locali, causali e motivate da un impatto materiale downstream.
149. Il backbone OSM operativo viene costruito sul PBF congelato `nord-est_2026-08-03.osm.pbf` mediante pipeline B1--B5 con semantica di routing, incluse le restrizioni di svolta rilevanti.
150. Ai fini del Gate $\Gamma_{OSM}$, il dominio B3 giant SCC contiene 889.440 nodi e il dominio B5 mutual/turn-aware 889.423 nodi. B1--B5 non vengono modificati automaticamente per rendere fattibile $\Gamma$.
151. La trasposizione `raw OSM node ↔ nodo GSFVG` e `way_id ↔ ID1` è rigettata come representation-dependent. $\Gamma_{OSM}$ usa una rappresentazione strutturale distinta dal backbone completo di routing.
152. La rappresentazione $\Gamma_{OSM}$ contiene 131.871 `structural nodes` e 167.033 `topological segments`; 757.552 nodi shape-like restano nel backbone di routing ma non sono candidati indipendenti di accesso.
153. La cardinalità strutturale è $ |\Gamma_o^{L,OSM}|=3 $ per 215/215 comuni, con primary anchored, due secondary, separazione pairwise minima 100 m e nessun topological segment incidente condiviso.
154. La selezione OSM utilizza una **candidate search adattiva**. `top20`, `top25` e `top30` sono risultati diagnostici, non parametri congelati; il massimo rank osservato è 29, mediana 6 e p95 16.
155. `E0_v01` è REJECTED, `E0_v02` SUPERSEDED, `E0_v03` PASS ed `E0_v04` PASS. Il final audit E0 conferma 10.750 candidate rows, 215 comuni, zero candidati non-structural e zero candidati senza incidence.
156. `Gamma structural design = CLOSED`, ma **$\Gamma_{OSM}$ non è ancora `FROZEN`**. E1 deve materializzare 645 accessi e auditare anche `EXP_REL_300`; E2 deve eseguire la validazione full 645-source sul B5 turn-aware.
157. Le anomalie E2 vengono classificate preliminarmente `GAMMA_LOCAL`, `ROUTING_LOCAL`, `SYSTEMIC`, `BLOCKING`; una anomalia Gamma non autorizza automaticamente modifiche al backbone B1--B5.
158. La condizione di freeze è $E1=PASS \land E2=PASS \Rightarrow \Gamma_{OSM}=FROZEN$. Dopo il freeze, la fase rete generale deve chiudersi e qualsiasi riapertura deve essere locale e causale.
159. La **Fase 5.6 è COMPLETATA**. Il backbone operativo light definitivo della tesi è OSM sullo snapshot `nord-est_2026-08-03.osm.pbf`; `G_OSM_operativo = FROZEN` e `NETWORK AUDIT = STOP`. GSFVG rimane baseline storica, benchmark istituzionale e supporto diagnostico.
160. La sorgente frozen del backbone è `C:\Tesi\Tesi_QGIS\00_originali\rete_stradale\osm\nord-est_2026-08-03.osm.pbf`, SHA-256 `e3b8be938c6acc58be516d988f0162c768e092a5674398d9c0f43ea9d9663813`.
161. Il routing canonico OSM **non coincide con il solo GeoPackage fisico**. È congelata la rappresentazione congiunta `B2 directed graph + B4 compiled turn restrictions + B5 selective state-expanded turn-aware routing`.
162. Le cardinalità canoniche B2 sono: 912.562 physical nodes, 944.219 physical segments, 1.698.857 directed edges, 1.691.766 CORE edges e 7.091 LOCAL edges; B3 contiene 889.440 nodi nella giant SCC.
163. B4 contiene 6.876 relations, 6.833 compiled sequences e 6.821 CORE sequences; B5 contiene 897.857 base states, 904.607 total states e 1.699.994 transitions.
164. Il Gate `Gamma_OSM` è definitivamente chiuso con `E0=PASS`, `E1=PASS`, `E2=PASS`, `E3=PASS`; pertanto `Gamma_OSM = FROZEN` ed `EXP_REL_300 = FROZEN`. Questa decisione supersede lo stato intermedio registrato alle decisioni 156 e 158.
165. È congelata la regola `|Gamma_o^{L,OSM}|=3`, con `K=3`, minimum pair separation 100 m, primary anchored, topological independence, candidate search `ADAPTIVE` e tie-break deterministico. `top30` non è un parametro metodologico; il massimo rank 29 è soltanto una proprietà empirica dello snapshot frozen.
166. È congelata la separazione fra `routing representation` e `Gamma structural representation`: 889.423 B5 mutual nodes, 131.871 structural nodes, 757.552 shape-like nodes esclusi dalla sola Gamma e 167.033 topological segments. Raw OSM node e `way_id` non sono equivalenti metodologici diretti di nodo e `ID1` GSFVG.
167. E1 ha materializzato 645 accessi, tre per ciascuno dei 215 comuni, con zero selection failure, determinism failure, duplicati di structural node/comune o `access_order`, primary non-nearest, pair separation violation e topological independence failure.
168. `EXP_REL_300` è congelato con `tau=300 m`; tutti i 645 pesi sono positivi, il massimo errore di somma per comune è `2.220e-16`, le mediane lambda sono 0,4034 / 0,3202 / 0,2790 e il numero effettivo di accessi ha mediana 2,92 e minimo 1,74.
169. E2 ha completato 645 Dijkstra turn-aware e la matrice 645×645: 416.025/416.025 celle finite, zero unreachable off-diagonal/intermunicipal/intramunicipal, zero zero-time off-diagonal, zero tempi negativi e zero physical lower-bound violation.
170. Le asimmetrie E2 sono localizzate: ratio mediano 1,0067, p95 1,0405, p99 1,0829 e massimo 2,6920; le mediane di coerenza esterna delle triplette sono 1,0114 in uscita e 1,0110 in ingresso. `SYSTEMIC issues = 0` e `BLOCKING issues = 0`.
171. Il Final Functional Gate F1 — B5 shadow regression è `PASS`: 4 siti verificati, 0 technical failures e 0 real B5 regressions rispetto alle ground truth congelate nel Gate shadow.
172. Il Final Functional Gate F2 — build fidelity/directional audit è `PASS`: 6 campioni di semantica way + 4 campioni di turn restriction, 10 campioni totali e 0 pipeline discordances fra PBF frozen, B2 e comportamento B5.
173. Il Final Functional Gate F3 — regional time-optimal metric audit è `PASS_NO_SYSTEMIC_SIGNAL`: 10.951 OD ISTAT/TomTom e 98.559 access-pair, 10.951 finite e 0 unreachable; il rapporto mediano `D_OSM_TIME_PRODUCT_LAMBDA / KM_TOT` è 1,009302, con `SYSTEMIC_CONNECTIVITY_ERROR=NO`, `SYSTEMIC_DISTANCE_INFLATION=NO`, `SYSTEMIC_ROUTING_BIAS=NO`.
174. I 178 casi classificati `LOCAL_ROUTING_ANOMALY` in F3 non autorizzano una nuova campagna manuale generale: il Gate cerca bias sistemici materiali per assignment/FRLM, non l'eliminazione di ogni anomalia locale.
175. I package canonici frozen sono `C:\Tesi\Tesi_QGIS\02_package\grafo_operativo_osm` e `C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_osm_light\`; checksum e artefatti canonici sono registrati nella §4.4.7 e nei manifest finali.
176. **Regola di riapertura:** non sono previste nuove campagne generiche di audit GSFVG, shadow OSM, directional review, Gamma sensitivity, B1--B5 structural build o regional metric audit. La fase rete può essere riaperta soltanto se una fase downstream dimostra un errore `BLOCKING` materiale per OD assignment o FRLM, e soltanto sul componente causale necessario.

177. La **Fase 5.7 è CLOSED / FROZEN**. `OD_PATH_SYSTEM_OSM = FROZEN`, `CANONICAL_ROUTE_IMPEDANCE = TIME_B5`, `DISTANCE = PATH_ATTRIBUTE`, `PRODUCT_LAMBDA_PATH_WEIGHTS = FROZEN`; `SYSTEMIC=0`, `BLOCKING=0` e `NETWORK AUDIT=STOP`.
178. Per ogni ordered access-pair il percorso canonico è $p^*=\arg\min TIME_{B5}$. La distanza viene calcolata ex-post sul path e non determina la scelta del percorso.
179. La Fase 5.7 non introduce congestion, generalized cost, toll, comfort penalty, road-class penalty, artificial turn penalty, stochastic route choice, $k$-shortest paths o diversion paths. Il routing interno resta `B2 + B4 + B5`.
180. Il dominio frozen comprende 215 comuni, 46.010 OD intercomunali ordinate, 3 accessi per comune, 9 access-pair per OD e 414.090 access-pair path. Tutti i 414.090 path sono finite e `unreachable=0`.
181. Il dominio path non è limitato alle 10.951 OD ISTAT positive: la copertura completa $215\times214$ è intenzionale perché la futura seed gravitazionale può attivare celle attualmente nulle.
182. `PRODUCT-LAMBDA` usa $w_{ab}=\lambda_{o,a}\lambda_{d,b}$, con $\lambda$ da `EXP_REL_300`; per ogni OD comunale la somma dei nove pesi vale 1 entro errore massimo `5.551e-16`. I nove access-pair rappresentano la distribuzione zonale multi-accesso, non nove alternative di route choice. Questa decisione supersede, per il backbone OSM corrente, lo stato provvisorio delle decisioni 133--134 riferite allo storico SP1 GSFVG.
183. La ricostruzione dei path è completa al 100%: zero failure di ricostruzione, continuità, estremi, transizioni vietate, conteggio transizioni, coerenza tempo e distanza. La full F3 regression è `98.559/98.559 PASS`.
184. L'architettura frozen è `relational path metadata + CSR-like path sequences + B5 frozen backbone`, nel package `C:\Tesi\Tesi_QGIS\02_package\od_paths_osm_light\`; non è richiesto un GeoPackage massivo path-edge.
185. Il package OD interno FVG della Fase 5.7 è **immutabile rispetto alle future zone esterne**. Gateway e domanda esterna devono essere aggiunti come estensione downstream separata e versionata, senza modificare o rigenerare i 414.090 path interni, `Gamma_OSM` o il routing B5.
186. Stato canonico post-5.7: `FASE 5.6=CLOSED/FROZEN`, `FASE 5.7=CLOSED/FROZEN`, `G_OSM_operativo=FROZEN`, `Gamma_OSM=FROZEN`, `EXP_REL_300=FROZEN`, `OD_PATH_SYSTEM_OSM=FROZEN`, `TIME_B5=CANONICAL_ROUTE_IMPEDANCE`, `DISTANCE=PATH_ATTRIBUTE`, `PRODUCT_LAMBDA_PATH_WEIGHTS=FROZEN`, `NETWORK AUDIT=STOP`.
187. La **baseline LIGHT v0 è FVG-only** sul dominio dei 215 comuni interni. `EXTERNAL_OD = NOT_IMPLEMENTED / DOWNSTREAM EXTENSION`; l'esterno sarà aggiunto in una fase separata senza modificare o rigenerare il sistema OD/path interno frozen della Fase 5.7.
188. La regola pendolare frozen della baseline è $C_{od}=Pendolari_{od}\cdot0{,}711\cdot220/365=Pendolari_{od}\cdot0{,}42855$ per ciascuna direzione, con costruzione coerente del ritorno.
189. `PARCO_AUTO_v0 = ACI_AUTOVETTURE_2024`, fonte ACI — *Autoritratto 2024, Parco veicolare*: copertura 215/215, totale 828.909 autovetture; le 20 autovetture `COMUNE = NON DEFINITO` sono escluse e non imputate.
190. `TURISMO_v0 = PRESENZE_TOTALI_2024`, fonte ISTAT: copertura 215/215, con 178 `OBSERVED`, 35 `IMPUTED_BEDS_RESIDUAL` e 2 `ZERO_CAPACITY`; totale regionale preservato 10.143.980 presenze. L'imputazione è limitata ai 35 comuni oscurati secondo $TURISMO\_RAW_j=14.944\,BEDS_j/944$.
191. `GDO_v0 = STRUCTURED_RETAIL_AREA_2023`: superficie comunale di vendita della distribuzione commerciale strutturata medio-grande (>400 m²), pari alla somma `MEDIA_SUP_MQ + GRANDE_SINGOLA_MQ + CENTRO_COMMERCIALE_MQ + COMPLESSO_COMMERCIALE_MQ`; riferimento 31/12/2023, copertura 215/215, totale 1.480.796,13 m², `DOUBLE COUNTING CHECK = PASS`. Drenchia, Grimacco e San Floriano del Collio-Števerjan sono documentati tramite `EXTERNAL_EVIDENCE`, senza imputazione positiva.
192. `Gravity_v0_territorial_inputs_raw.xlsx` è l'artefatto **CANONICAL / FROZEN** della Fase 5.8A, SHA-256 `3dab79eabaf4e7f55a0e8244f02a1ebbae1f80355c0c3882a3b03ec90ff2724e`, size 64.156 byte, sheet principale `MASTER_RAW`.
193. L'hard QA della master table è `PASS`: 215 righe, 215 `PRO_COM` distinti, 0 duplicati, 0 missing, 0 valori negativi, 0 cross-source mismatch; le somme di controllo sono 828.909 autovetture, 10.143.980 presenze e 1.480.796,13 m² di GDO.
194. La **Fase 5.8A è CLOSED / FROZEN**. Non sono state eseguite normalizzazione, scelta di $\beta$ o $Q$, costruzione di $N_{ij}^{0}$, assignment o calibrazione ANAS. Il `NEXT` canonico è **Fase 5.8B — Gravity Model v0 FVG-only**.
195. `ANAS_CALIBRATION_REFERENCE_YEAR = 2024` è **RATIFIED / FROZEN**. Il 2024 è il target annuale unico della calibrazione ANAS Gravity v0: 10/20 sezioni sono osservate direttamente nel 2024 contro 8/20 nel 2025; il 2024 è coerente con gli input territoriali della baseline e evita una baseline mista 2024/25. Il 2025 è preservato come `TEMPORAL_VALIDATION_2025` e non sostituisce direttamente un 2024 mancante. La membership concreta delle sezioni e i valori inferiti 2024 non sono ancora frozen.
196. La regola `QUALITY_ELIGIBILITY_RULE` è **RATIFIED / FROZEN**: `A/HIGH` e `B/MEDIUM` sono candidati al primary calibration set; `C/LOW` è riservata a sensitivity/diagnostic; `D/UNUSABLE` è esclusa per informazione insufficiente. La classe misura solo qualità, copertura, stabilità e affidabilità dell'eventuale inferenza ed è separata dai futuri filtri di `BORDER EXPOSURE`, `THROUGH TRAFFIC`, `MOTORWAY`, `EXTERNAL TRAFFIC` e rappresentatività spaziale. Il backtest `LOCF` (errore percentuale assoluto medio ~5,46%, mediano ~3,83%) resta evidenza/benchmark, non algoritmo finale congelato. La Fase 5.8C deve ricalcolare le classi sul target 2024 prima di qualunque fit.
197. La **Fase 5.8R2 è CLOSED / FROZEN**. Il metodo primario v0 è `PARAMETRIC_GRAVITY_MODEL_ESTIMATION_FROM_TRAFFIC_COUNTS`, con riferimento teorico operativo a *Modelling Transport*, §12.4.3. La struttura congelata è:
   $$
   W_{ij}=P_iA_j\exp(-\beta c_{ij}),\qquad
   S_{ij}=\frac{W_{ij}}{\sum_{rs}W_{rs}},\qquad
   N_{ij}=Q S_{ij},\qquad
   T_{ij}^{LIGHT}=C_{ij}^{ISTAT}+N_{ij}.
   $$
   $P_i$ è un `RELATIVE ORIGIN PROPENSITY PROXY` e $A_j$ un `RELATIVE ATTRACTION PROXY`; nessuno dei due costituisce un hard margin.
198. Per la v0 corrente: `HYMAN_NOW = NOT_APPLICABLE`, `FURNESS_NOW = NOT_APPROPRIATE`, `TRIPROPORTIONAL_NOW = NOT_APPROPRIATE`, `PARTIAL_MATRIX_NOW = NOT_APPROPRIATE`, `DIRECT_MATRIX_ESTIMATION_NOW = NOT_PRIMARY`. Un eventuale `MATRIX_UPDATING_LATER` è ammesso soltanto come `CONDITIONAL SECOND-STAGE UPGRADE` dopo la calibrazione/validazione parametrica primaria.
199. La deterrenza `EXPONENTIAL` selezionata nella Fase 5.8R2 è stata sottoposta al gate empirico 5.8E. **Post-5.8E non è frozen come specifica finale**: resta `INCUMBENT REFERENCE ONLY / FVG-ONLY EXPERIMENTAL MODEL`. La base teorica `PARAMETRIC_GRAVITY_MODEL_ESTIMATION_FROM_TRAFFIC_COUNTS` resta preservata. `Q` e `beta` non sono frozen.

200. La **Fase 5.8D resta CLOSED / FROZEN**. Sono state analizzate 16 sezioni A+B+C: `OSM_MATCH_OK=16`, `OSM_MATCH_UNCERTAIN=0`. Il PRIMARY resta `920034`, `920032`, `920039` (`PRIMARY_COUNT=3`); le restanti 13 sezioni formano il set `SENSITIVITY`. `MEASUREMENT_OPERATOR=BIDIRECTIONAL_SUM`. Queste selezioni descrivono il disegno sperimentale FVG-only 5.8E e non implicano che tutte le sezioni siano clean validation signals dopo l'audit di generalizzazione.

201. Il contratto calibrazione/validazione resta: `ANAS_2024 = CALIBRATION DATA`; `ANAS_2025 = FINAL TEMPORAL HOLDOUT`; `NO PARAMETER TUNING ON 2025`; `NO OD-SPECIFIC FUDGE FACTORS`; `NO MANUAL OD PATCHING TO MATCH ANAS`. ANAS 2025 non è stato usato nella 5.8E e non deve essere usato per scegliere specifica, domanda external, parametri o sezioni in funzione del loro fit 2025.

202. La **Fase 5.8E è CLOSED / STOP / NOT_READY FOR PARAMETER FREEZE**. La calibrazione numerica FVG-only è stata eseguita ma non autorizza il freeze di `Q` o `beta`. Il `NEXT` canonico non è più un ulteriore fit FVG-only: è **EXTERNAL / TRANSBORDER DEMAND COMPONENT**.
203. La specifica experimental/incumbent della 5.8E è $W_{ij}=P_iA_j\exp(-\beta c_{ij})$, $S_{ij}=W_{ij}/\sum_{r\neq s}W_{rs}$, $N_{ij}=Q S_{ij}$, $T_{ij}^{LIGHT}=C_{ij}^{ISTAT}+N_{ij}$, con $N_{ii}=0$, $P_i$ da parco auto normalizzato L1, $A_j=0{,}5\cdot tourism\_share_j+0{,}5\cdot GDO\_share_j$ e $c_{ij}=$ `TIME_B5 PRODUCT-LAMBDA` in minuti. Questa specifica **non è frozen come modello finale**.
204. La calibrazione 5.8E è link-based: $\widehat Y_a(Q,\beta)=\sum_{ij}p_{ij}^a[C_{ij}^{ISTAT}+N_{ij}(Q,\beta)]$, con objective baseline `UNWEIGHTED SSE / NLLS`, senza custom weights e senza robust loss; relative squared error è diagnostic only. La log-regression diretta sui conteggi ANAS non è applicabile perché ogni link count aggrega molte OD. `HYMAN_NOW = NOT_APPLICABLE`.
205. Per una deterrence shape fissata, $Q$ resta linearmente profilabile: $\widehat Y_a=C_a+QG_a(\theta)$. La sequenza `shape parameters → G_a → Q* analytically → J_profile` è una proprietà metodologica da preservare anche se la specifica Gravity viene rivista.
206. `E3-A EXPONENTIAL` resta `HISTORICAL / DIAGNOSTIC CANDIDATE`: $\beta=0{,}045953794473\ 1/min$, $Q=1.552.629{,}518131\ veh/day$, $J=1.056.095{,}651965$, $c_{half}\approx15{,}083568\ min$. Il profilo supporta `NUMERICAL IDENTIFIABILITY`, ma non stabilisce `EMPIRICAL / SPATIAL ADEQUACY`.
207. `E4`: `IDENTIFIABILITY=PASS`, `PROFILE_STABILITY=PASS`, `LOO_STABILITY=WEAK`, `LOSS_SENSITIVITY=ACCEPTABLE/MODERATE`, `Q_PLAUSIBILITY=HIGH_BUT_DEFENSIBLE`. Il failure principale è di generalizzazione spaziale, non di identificabilità numerica.
208. `E4-B`: sulle sezioni fuori calibrazione sono emersi mismatch importanti, fra cui `920022` (23.143 osservati vs 169.730,252 modellati, errore relativo circa -633%) e `920040` (14.659 vs 42.439,478, circa -190%). `FEW_OD_ERROR` e `SINGLE_PATH_ERROR` non sono supportati come spiegazione generale.
209. `ORIGIN-CONSTRAINED` è stato implementato e verificato matematicamente, ma **respinto come baseline v0** perché la generalizzazione complessiva peggiora: candidato $\beta\approx0{,}032095691453$, $Q\approx990.072{,}131283$, SSE PRIMARY circa 9,443 milioni contro circa 1,056 milioni dell'exponential. La normalizzazione globale di $A_j$ non è necessaria alle destination shares origin-constrained; la normalizzazione per origine è strutturalmente necessaria.
210. `COMBINED / TANNER`, $f(c)=c^n\exp(-\beta c)$, è stato testato e **respinto come baseline**. Candidato diagnostico $n\approx1{,}255285016510$, $\beta\approx0{,}089101695826\ 1/min$, $Q\approx1.846.499{,}386614$, $c_{peak}\approx14{,}088\ min$; PRIMARY SSE quasi nullo ma `IDENTIFIABILITY=WEAK/SATURATION-AFFECTED`, con 0/6 sensitivity migliorate e 6/6 peggiorate.
211. Il risultato cumulativo R1+R2 non supporta l'ipotesi che il failure dipenda principalmente dalla global normalization di $P_i$ o dalla sola exponential deterrence shape. Restano rilevanti `DEMAND PROXY STRUCTURE`, `DESTINATION ATTRACTIVENESS STRUCTURE`, `LOW-DIMENSIONAL GRAVITY LIMITATION`, `ASSIGNMENT REPRESENTATION`, `MISSING EXTERNAL DEMAND DOMAIN`, `MISSING INTRAZONAL DOMAIN`.
212. `PRODUCT-LAMBDA` mantiene il significato frozen di distribuzione sui 9 access-pair; **i 9 access-pair paths non sono 9 route-choice alternatives**. Ogni access-pair usa un solo deterministic `TIME_B5` shortest path.
213. `920022 — RA13` è classificata `SECTION-SPECIFIC ASSIGNMENT REPRESENTATION LIMITATION` e **non è una clean independent Gravity validation section** nella rappresentazione deterministic all-or-nothing corrente. `920040 — SS54` non mostra un'anomalia di assignment evidente sul top OD verificato e resta un segnale diagnostico utile.
214. `GENERAL_TIME_B5_FAILURE=NO` e non vi è evidenza sufficiente per riaprire `G_OSM_operativo` o `OD_PATH_SYSTEM_OSM`. `G_OSM_operativo`, `Gamma_OSM`, `EXP_REL_300`, `OD_PATH_SYSTEM_OSM`, `TIME_B5` e `PRODUCT_LAMBDA` restano frozen.
215. I deterministic path flows non devono essere interpretati come empirical stochastic route-choice shares. Un valore $p_{ij}^{a}=1$ significa che il 100% dei frozen deterministic access-pair paths attraversa la sezione, non che il 100% del traffico reale OD percorra quel link.
216. La baseline 5.8E copre soltanto `FVG INTERNAL INTERMUNICIPAL OD`; i conteggi ANAS osservano un dominio più ampio. Il mismatch fra `MODEL DEMAND DOMAIN` e `ANAS MEASUREMENT DOMAIN` è un limite metodologico materiale.
217. `EXTERNAL / TRANSBORDER DEMAND COMPONENT` è il **NEXT BLOCKING METHODOLOGICAL COMPONENT**. Deve coprire almeno `INTERNAL→EXTERNAL`, `EXTERNAL→INTERNAL`, `EXTERNAL→EXTERNAL/THROUGH`, con resto d'Italia, Austria e Slovenia, usando external zones + gateway/connection al network interno frozen.
218. L'estensione external/transborder deve essere `SEPARATE / VERSIONED / ADDITIVE` e non deve rigenerare in place il sistema interno frozen. Solo dopo tale estensione si riaprono calibration design, eligibility ANAS, identificazione $Q,\beta$, spatial generalization gate e infine ANAS 2025 holdout.
219. `ANAS 2025 = FINAL TEMPORAL HOLDOUT` senza eccezioni: non è stato usato nella 5.8E e non deve essere impiegato per specifica, costruzione external, tuning o selezione ex-post delle sezioni.
220. Principio post-5.8E: il fit link-based combina tre livelli — `DEMAND GENERATION/DISTRIBUTION`, `ROUTE/ASSIGNMENT REPRESENTATION`, `OBSERVATION DOMAIN OF TRAFFIC COUNTS`. Un mismatch sui counts non deve essere attribuito automaticamente alla sola domanda OD.

# 21. Ipotesi di lavoro e proposte metodologiche

Le voci seguenti non sono decisioni definitive:

- costruire nella 5.8B la componente non pendolare come seed gravitazionale FVG-only, definendo volume e deterrenza senza ancora coinvolgere ANAS;
- trasformare i vettori RAW frozen della 5.8A in indicatori normalizzati: la specificazione di lavoro della 5.8B usa `P_i = parco auto normalizzato` e `A_j = 0,5·turismo_normalizzato + 0,5·GDO_normalizzato`, da validare e congelare nel gate;
- definire gateway e aggregazioni esterne come **estensione downstream separata** dal sistema OD interno frozen e dalla baseline v0 FVG-only, scegliendo tra comuni, macrozone e porte di confine;
- usare pochi nodi di accesso per zona con pesi deterministici decrescenti con la distanza, parametrizzati tramite sensibilità;
- valutare un'eventuale matrix estimation regolarizzata dopo la prima assegnazione; eventuali scenari multipath o di deviazione, se mai introdotti, dovranno essere versioni separate e non alterare `OD_PATH_SYSTEM_OSM` frozen;
- introdurre in futuro la componente esterno--esterno, il modello heavy e il riconoscimento di siti condivisi;
- introdurre profili orari, code, livelli di servizio e tecnologie HPC multiple soltanto se necessari dopo la formulazione base.

---


# 22. Questioni ancora aperte

> **Come usare questa sezione.** Contiene soltanto problemi realmente non chiusi. Le Fasi 5.6, 5.7, 5.8A, 5.8R2 e 5.8D restano frozen; la Fase 5.8E è chiusa con `STOP / NOT_READY FOR PARAMETER FREEZE`. Rete OSM, `Gamma_OSM`, `EXP_REL_300`, `TIME_B5`, `OD_PATH_SYSTEM_OSM` e `PRODUCT_LAMBDA` non vengono riaperti. $Q$, $\beta$ e la specifica finale Gravity sono invece **non frozen**.

## A. External / transborder demand — NEXT blocking component

Definire una componente di domanda aggiuntiva che rappresenti separatamente:

- `INTERNAL → EXTERNAL`;
- `EXTERNAL → INTERNAL`;
- `EXTERNAL → EXTERNAL / THROUGH`;
- resto d'Italia;
- Austria;
- Slovenia;
- traffico transfrontaliero e di attraversamento.

L'obiettivo è ridurre il mismatch fra il dominio di domanda modellato e il dominio effettivamente misurato dai conteggi ANAS.

## B. Gateway / external zone architecture

Definire come collegare le external zones al backbone OSM frozen senza rigenerare il sistema interno.

Vincoli:

- estensione `SEPARATE / VERSIONED / ADDITIVE`;
- nessuna modifica in place dei 414.090 path interni;
- nessuna riapertura automatica di `G_OSM_operativo`, `Gamma_OSM`, `TIME_B5` o `PRODUCT_LAMBDA`;
- distinguere chiaramente `network extension` e `demand externalization`.

Le 2.895 relazioni di pendolarismo extra-regione già georeferenziate costituiscono un input disponibile da valutare nel disegno, non una soluzione completa del traffico esterno.

## C. Calibration redesign

Dopo l'estensione del dominio:

- riesaminare quali e quante sezioni ANAS siano utilizzabili come calibration e sensitivity;
- aumentare, se supportato, il numero di segnali spazialmente indipendenti;
- ridefinire il calibration core evitando di selezionare sezioni sulla base del fit 2025;
- mantenere `ANAS 2025 = FINAL TEMPORAL HOLDOUT`.

## D. Gravity final specification

```text
GLOBAL EXPONENTIAL =
INCUMBENT REFERENCE ONLY

ORIGIN-CONSTRAINED =
TESTED / REJECTED AS BASELINE

COMBINED / TANNER =
TESTED / REJECTED AS BASELINE

Q =
NOT FROZEN

beta =
NOT FROZEN
```

La specifica deve essere riaperta soltanto dopo l'ampliamento del measurement/demand domain, non moltiplicando ora nuove varianti FVG-only.

## E. Demand proxies

Dopo l'integrazione external verificare se $P_i=$ parco auto e $A_j=0{,}5\,tourism\_share_j+0{,}5\,GDO\_share_j$ siano sufficientemente affidabili per la baseline finale.

## F. Assignment representation

La deterministic `TIME_B5` all-or-nothing può sovrarappresentare alcuni corridoi quando esistono route alternatives competitive. `920022` ne costituisce un segnale section-specific.

Questo non è attualmente blocking e non autorizza un nuovo route-choice model né la riapertura del routing frozen. La questione va rivalutata dopo l'ampliamento del dominio di domanda e soltanto se resta materialmente rilevante.

## G. Intrazonal component

La mobilità intracomunale resta esclusa. Questo problema è distinto dalla external/transborder demand e deve essere affrontato separatamente.

## H. $Q$ / $\beta$ identification

La riapertura di $Q$ e $\beta$ avviene soltanto dopo:

1. external/transborder demand design;
2. gateway/external zone architecture;
3. materializzazione external;
4. integrazione internal + external;
5. reassessment ANAS.

## I. Matrix updating — eventuale secondo stadio

`DIRECT_MATRIX_ESTIMATION_NOW = NOT_PRIMARY`.

Un eventuale matrix updating regolarizzato resta un possibile upgrade successivo e non deve essere usato ora per correggere OD ad hoc o assorbire mismatch di dominio.

## J. Downstream light, FRLM e heavy

Dopo il freeze finale della domanda LIGHT restano da:

- final LIGHT OD;
- path flows;
- EV / charging demand;
- FRLM / Capacitated FRLM;
- successivamente heavy-duty, rete elettrica ed economia.

Non avviare questi blocchi prima della chiusura dell'estensione external e del nuovo gate Gravity.

# 23. Roadmap operativa proposta

> **Come usare questa sezione.** La Fase 5.8E ha chiuso l'esperimento di calibrazione FVG-only con `STOP / NOT_READY FOR PARAMETER FREEZE`. Il prossimo blocco non è una nuova variante Gravity: è l'ampliamento del dominio di domanda.

## Stato congelato da preservare

```text
G_OSM_operativo       = FROZEN
Gamma_OSM             = FROZEN
EXP_REL_300           = FROZEN
OD_PATH_SYSTEM_OSM    = FROZEN
TIME_B5               = FROZEN
PRODUCT_LAMBDA        = FROZEN

FASE 5.8A             = CLOSED / FROZEN
FASE 5.8R2            = CLOSED / FROZEN
FASE 5.8D             = CLOSED / FROZEN
FASE 5.8E             = CLOSED / STOP / NOT_READY FOR PARAMETER FREEZE

Q                     = NOT FROZEN
beta                  = NOT FROZEN
ANAS 2025             = FINAL TEMPORAL HOLDOUT
```

## NEXT canonico

$$
\boxed{
\text{EXTERNAL / TRANSBORDER DEMAND DESIGN}
}
$$

1. **EXTERNAL / TRANSBORDER DEMAND DESIGN**  
   Definire `INTERNAL→EXTERNAL`, `EXTERNAL→INTERNAL`, `EXTERNAL→EXTERNAL/THROUGH` e la granularità per resto d'Italia, Austria e Slovenia.

2. **GATEWAY / EXTERNAL ZONE ARCHITECTURE**  
   Collegare le external zones al backbone OSM frozen senza rigenerare il sistema interno.

3. **MATERIALIZATION OF EXTERNAL DEMAND COMPONENT**  
   Costruire una componente separata, versionata e additiva, con provenance e QA.

4. **INTERNAL + EXTERNAL LIGHT DEMAND INTEGRATION**  
   Integrare domanda interna ed external/transborder mantenendo distinti i contributi.

5. **REASSESS ANAS SECTION ELIGIBILITY**  
   Rivalutare quali sezioni siano realmente utilizzabili come calibration e sensitivity.

6. **REOPEN GRAVITY / SCALE CALIBRATION DESIGN WITH BROADER ANAS SET**  
   Riaprire $Q$, $\beta$, proxy e specifica Gravity soltanto a questo punto.

7. **SPATIAL GENERALIZATION GATE**  
   Verificare generalizzazione su corridoi non usati nel fit e distinguere demand mismatch, assignment representation e observation-domain effects.

8. **ANAS 2025 TEMPORAL HOLDOUT**  
   Validazione temporale finale senza tuning.

9. **FINAL LIGHT MATRIX**

10. **PATH FLOWS**

11. **EV / CHARGING DEMAND**

12. **FRLM / CAPACITATED FRLM**

## Stop operativo corrente

Non avviare ora:

```text
HEAVY
FRLM
CHARGING DEMAND
ANAS 2025 VALIDATION
NEW ROUTE-CHOICE MODEL
NEW GENERIC NETWORK AUDIT
```

La rete frozen può essere riaperta soltanto con evidenza `BLOCKING` specifica e causale.

## Principio guida

Prima di aumentare la flessibilità della Gravity, migliorare la coerenza:

$$
\text{MODEL DEMAND DOMAIN}
\leftrightarrow
\text{ANAS MEASUREMENT DOMAIN}.
$$

# 24. Registro aggiornamenti

> **Come usare questa sezione.** È un audit trail cronologico. Serve per ricostruire l'evoluzione del progetto e le ragioni storiche delle scelte, ma non deve essere usato come sorgente primaria dello stato corrente se una decisione è già stata consolidata altrove.

## 1 settembre 2026 — Freeze del build contract LaTeX

- Introdotto in §0.1.5 un **contratto permanente di authoring/export** per prevenire regressioni di compilazione nelle versioni future.
- Sintassi matematica canonica del notebook: `$...$` inline e `$$...$$` display; `\(...\)` e `\[...\]` non sono ammessi come delimitatori di authoring.
- `XeLaTeX` diventa il compiler canonico dell'output derivato.
- Introdotta la pipeline `build_tesi.py`: lint del notebook, esecuzione, rigenerazione `.md/.tex`, compilazione tramite `latexmk`/XeLaTeX, fail su errori bloccanti, report con hash e bundle Overleaf-ready.
- Il file `latexmkrc` incluso nel bundle Overleaf forza XeLaTeX e rende persistente il compiler contract.
- Regola `FAIL CLOSED`: nessun output futuro è considerato valido se il build report non restituisce `STATUS = PASS`.
- Nessuna modifica metodologica o scientifica introdotta.
## 1 settembre 2026 — Correzione di compatibilità dell'export LaTeX

- **Natura dell'intervento:** esclusivamente editoriale/tecnica; nessuna decisione metodologica, freeze, formula o risultato scientifico è stato modificato.
- **Problema individuato:** i delimitatori MathJax `\(...\)` e `\[...\]`, pur corretti nel rendering Jupyter, non venivano interpretati in modo affidabile dalla catena `nbconvert → Pandoc → LaTeX`, producendo comandi matematici come `\mathrm`, `\mathcal` e `\Gamma` fuori dalla modalità matematica.
- **Correzione:** gli inline math sono stati normalizzati a `$...$` e i display math a `$$...$$`, formati riconosciuti sia da Jupyter/MathJax sia dall'export Pandoc.
- **Correzioni sintattiche puntuali:** chiusi tre blocchi `\boxed{...}` storicamente privi della parentesi graffa finale e protetto l'underscore in `TIME\_B5` all'interno di un `\text{...}` matematico.
- **Compiler contract per l'output derivato:** per la compilazione del `.tex` generato utilizzare **XeLaTeX**, che gestisce correttamente i simboli Unicode presenti nel documento e nei blocchi verbatim. `pdfLaTeX` non è il compiler raccomandato per questo export.
- **Verifica:** il `.tex` rigenerato dal notebook è stato compilato con XeLaTeX senza errori bloccanti; restano soltanto warning tipografici/non bloccanti.
## 31 agosto 2026 — Chiusura Fase 5.8E e handoff external/transborder

- **E3-A exponential:** profile minimum interno chiaro; candidato $\beta=0{,}045953794473$, $Q=1.552.629{,}518131$, non frozen.
- **E4:** identifiability matematica `PASS`, profile stability `PASS`, generalizzazione spaziale insufficiente.
- **E4-B:** mismatch importanti fuori calibrazione e attribuzione diffusa su numerose OD; il failure non è spiegato da poche OD o da un singolo path.
- **R1 origin-constrained:** implementazione e mathematical QA `PASS`; performance regionale complessivamente peggiore; `TESTED / REJECTED AS BASELINE`.
- **R2 combined/Tanner:** nested/exponential QA superato, interpolazione quasi perfetta delle PRIMARY ma saturazione/weak identifiability; 0/6 sensitivity migliorate; `TESTED / REJECTED AS BASELINE`.
- **Assignment sanity:** nessun `GENERAL_TIME_B5_FAILURE`; `920022` mostra una `SECTION-SPECIFIC ASSIGNMENT REPRESENTATION LIMITATION`; sul top OD verificato `920040` non mostra un'anomalia equivalente.
- **Freeze network/path:** `G_OSM_operativo`, `Gamma_OSM`, `EXP_REL_300`, `OD_PATH_SYSTEM_OSM`, `TIME_B5`, `PRODUCT_LAMBDA` restano frozen.
- **Closure:** `FASE 5.8E = CLOSED / STOP / NOT_READY FOR PARAMETER FREEZE`; $Q$ e $\beta$ restano non frozen; exponential = `INCUMBENT REFERENCE ONLY`.
- **Measurement-domain finding:** FVG-only non coincide con il dominio misurato dai conteggi ANAS.
- **NEXT:** `EXTERNAL / TRANSBORDER DEMAND COMPONENT`, con external zones + gateway architecture separata, versionata e additiva.
- **ANAS 2025:** preservato come `FINAL TEMPORAL HOLDOUT`, non usato nella 5.8E e non utilizzabile per tuning o specifica.
## 28 agosto 2026 — Checkpoint pre-calibrazione Gravity v0: freeze 5.8R2 e 5.8D

- **Fase 5.8R2:** `CLOSED / FROZEN`.
- **Metodo primario v0:** `PARAMETRIC_GRAVITY_MODEL_ESTIMATION_FROM_TRAFFIC_COUNTS`; riferimento teorico operativo *Modelling Transport* §12.4.3.
- **Struttura Gravity frozen:** $W_{ij}=P_iA_j\exp(-\beta c_{ij})$; $S_{ij}=W_{ij}/\sum_{rs}W_{rs}$; $N_{ij}=Q S_{ij}$; $T_{ij}^{LIGHT}=C_{ij}^{ISTAT}+N_{ij}$.
- **Interpretazione dei proxy:** `P_i = RELATIVE ORIGIN PROPENSITY PROXY / NO HARD MARGIN`; `A_j = RELATIVE ATTRACTION PROXY / NO HARD MARGIN`.
- **Metodi non applicati ora:** `HYMAN_NOW = NOT_APPLICABLE`; `FURNESS_NOW = NOT_APPROPRIATE`; `TRIPROPORTIONAL_NOW = NOT_APPROPRIATE`; `PARTIAL_MATRIX_NOW = NOT_APPROPRIATE`; `DIRECT_MATRIX_ESTIMATION_NOW = NOT_PRIMARY`. `MATRIX_UPDATING_LATER = CONDITIONAL SECOND-STAGE UPGRADE`.
- **Deterrenza:** `GRAVITY_V0_DETERRENCE = EXPONENTIAL / FROZEN AS 5.8R2 TEST SPECIFICATION; POST-5.8E = INCUMBENT REFERENCE ONLY / NOT FINAL FREEZE`, $f(c)=\exp(-\beta c)$; `CALIBRATION_PARAMETERS = Q, beta`. La combined deterrence è rinviata a V1/futura estensione external-flow perché tre segnali PRIMARY indipendenti non supportano con sufficiente identificabilità una specifica a tre parametri nella v0.
- **Fase 5.8D:** `CLOSED / FROZEN`; 16 sezioni A+B+C analizzate, 16 `OSM_MATCH_OK`, 0 `OSM_MATCH_UNCERTAIN`.
- **PRIMARY ANAS:** `920034`, `920032`, `920039`; `PRIMARY_COUNT = 3`; tre segnali di corridoio distinti. Le restanti 13 sezioni A/B/C costituiscono `SENSITIVITY`.
- **Measurement operator:** `BIDIRECTIONAL_SUM` per tutte le 16 sezioni.
- **Perimetro:** `EXTERNAL_OD = NOT IMPLEMENTED`; `G_OSM_operativo` e `OD_PATH_SYSTEM_OSM` restano `UNCHANGED / FROZEN`.
- **Calibration/validation contract:** `ANAS_2024 = CALIBRATION DATA`; `ANAS_2025 = FINAL TEMPORAL HOLDOUT`; nessun parameter tuning sul 2025; nessun OD-specific fudge factor; nessuna patch manuale delle OD per riprodurre ANAS.
- **NEXT:** `GRAVITY v0 NUMERICAL CALIBRATION` con sequenza: objective/loss → stima $Q,\beta$ su PRIMARY 2024 → QA/identificabilità → holdout 2025 → freeze Gravity v0 → LIGHT OD finale → path flows → FRLM.
- **STOP del checkpoint:** nessuna calibrazione numerica è stata avviata in questo aggiornamento documentale.

## 26 agosto 2026 — Ratifica ANAS Gravity v0: target 2024 e quality eligibility

- **Decisione ANAS 01:** `ANAS_CALIBRATION_REFERENCE_YEAR = 2024`, `RATIFIED / FROZEN`.
- **Motivazione:** 10/20 sezioni osservate direttamente nel 2024 contro 8/20 nel 2025; il massimo storico 13/20 del 2022 è meno recente; il 2024 è coerente temporalmente con gli input territoriali della Gravity v0 e consente un target annuale unico.
- **2025:** preservato come `TEMPORAL_VALIDATION_2025`; non può sostituire direttamente un dato 2024 mancante.
- **Decisione ANAS 02:** `A/HIGH` e `B/MEDIUM -> CANDIDATE_PRIMARY_CALIBRATION_SET`; `C/LOW -> SENSITIVITY / DIAGNOSTIC ONLY`; `D/UNUSABLE -> EXCLUDED FOR INSUFFICIENT INFORMATION`.
- **Separazione frozen:** `DATA QUALITY FILTER != EXTERNAL CONTAMINATION FILTER`. Border exposure, through traffic, motorway/external contamination e rappresentatività spaziale restano valutazioni successive.
- **Backtest:** `LOCF` ha fornito errore percentuale assoluto medio ~5,46% e mediano ~3,83%; è benchmark corrente, non algoritmo finale congelato.
- **Non frozen:** membership A/B/C/D delle 20 sezioni, valori 2024 inferiti, algoritmo di inferenza, filtri external-exposure, fit weights, loss Huber/WLS/L1 e parametri del modello.
- **Handoff esecutivo:** `FASE 5.8C — ANAS 2024 TARGET READINESS`; output richiesto `ANAS_2024_TARGET_PROPOSAL`, con tabella completa delle 20 sezioni e conteggi A/B/C/D.
- **STOP 5.8C:** nessun filtro external exposure, nessun fit weight, nessuna calibrazione Gravity, nessuna stima di $n$, $\beta$ o $Q$.

## 26 agosto 2026 — Chiusura Fase 5.8A e freeze degli input territoriali Gravity v0

- **Fase 5.8A:** `CLOSED / FROZEN`.
- **Baseline LIGHT v0:** `FVG-ONLY`, dominio 215 comuni; `EXTERNAL_OD = NOT_IMPLEMENTED / DOWNSTREAM EXTENSION`.
- **Regola pendolare frozen:** `C_od = Pendolari_od × 0.711 × 220/365 = Pendolari_od × 0.42855` per ciascuna direzione, con ritorno costruito coerentemente.
- **PARCO_AUTO_v0:** `ACI_AUTOVETTURE_2024`, 215/215 comuni, totale 828.909; 20 autovetture `COMUNE = NON DEFINITO` escluse e non imputate.
- **TURISMO_v0:** `PRESENZE_TOTALI_2024`, 215/215; 178 `OBSERVED`, 35 `IMPUTED_BEDS_RESIDUAL`, 2 `ZERO_CAPACITY`; totale regionale preservato 10.143.980.
- **GDO_v0:** `STRUCTURED_RETAIL_AREA_2023`, riferimento 31/12/2023, 215/215, totale 1.480.796,13 m², `DOUBLE COUNTING CHECK = PASS`; provenance `REGIONAL_TABLE / EXTERNAL_EVIDENCE`, nessuna imputazione positiva.
- **Master table RAW:** `Gravity_v0_territorial_inputs_raw.xlsx = CANONICAL / FROZEN`, SHA-256 `3dab79eabaf4e7f55a0e8244f02a1ebbae1f80355c0c3882a3b03ec90ff2724e`, size 64.156 byte, sheet `MASTER_RAW`.
- **QA:** 215 righe, 215 `PRO_COM` distinti, 0 duplicati, 0 missing, 0 valori negativi, 0 cross-source mismatch; tutte le somme di controllo `PASS`.
- **Freeze precedenti preservati:** `G_OSM_operativo`, `Gamma_OSM`, `EXP_REL_300`, `OD_PATH_SYSTEM_OSM`, `TIME_B5`, `PRODUCT-LAMBDA`, 46.010 OD interne e 414.090 access-pair path restano immutati.
- **STOP 5.8A:** nessuna normalizzazione, nessuna scelta di $\beta$ o $Q$, nessuna costruzione di $N_{ij}^{0}$, nessun assignment e nessun uso di ANAS.
- **NEXT:** `FASE 5.8B — GRAVITY MODEL v0 FVG-ONLY`.

## 24 agosto 2026 — Chiusura Fase 5.7 e freeze del sistema OD path interno OSM

- **Final gate ricevuto e ratificato dalla nuova Chat Madre:** verdict `PASS`; `FASE 5.7 = CLOSED / FROZEN`.
- **Sistema path:** `OD_PATH_SYSTEM_OSM = FROZEN`.
- **Impedenza canonica:** `CANONICAL_ROUTE_IMPEDANCE = TIME_B5`; `DISTANCE = PATH_ATTRIBUTE`.
- **Pesi access-pair:** `PRODUCT_LAMBDA_PATH_WEIGHTS = FROZEN`, con `EXP_REL_300` come modello dei $\lambda$.
- **Dominio completo interno:** 215 comuni, 46.010 OD comunali ordinate, 9 access-pair per OD, 414.090 access-pair path.
- **Copertura:** 414.090/414.090 finite, 0 unreachable, 100%. Il dominio non è limitato alle 10.951 OD ISTAT positive.
- **Ricostruzione:** 100% dei path ricostruiti; zero failure di continuità, source/destination, forbidden transitions, transition count, time consistency e distance consistency.
- **Full F3 regression:** 98.559/98.559 `PASS`; errori massimi `9.095e-13 s` sul tempo, `9.022e-10 m` sulla distanza e `9.714e-17` sul pair weight.
- **Storage:** architettura `relational path metadata + CSR-like path sequences + B5 frozen backbone`; 598.707.601 transition slots.
- **Package finale:** `C:\Tesi\Tesi_QGIS\02_package\od_paths_osm_light\`; 9 file, circa 2,371 GB; atomic promotion `PASS`; SHA verification `PASS`.
- **Manifest:** `OSM_OD_PATHS_manifest_v01.json`, SHA-256 `9c3279c604685dbb8ebf52d18910658efc5fb7fe0ab1069d36d313476abb8fff`.
- **No GPKG path-edge massivo:** geometria ricostruibile `transition slot → B5 → edge_id → B2 → physical OSM segment`.
- **Confine di fase:** il package 5.7 riguarda soltanto le OD interne FVG e non deve essere modificato o rigenerato per introdurre zone esterne.
- **Stato issue:** `SYSTEMIC=0`, `BLOCKING=0`, `NETWORK AUDIT=STOP`.
- **STOP FASE 5.7.**
- **NEXT storico al 24/08/2026:** gateway / external demand; priorità successivamente superseduta dalla chiusura 5.8A e dal `NEXT` corrente 5.8B FVG-only.

## 24 agosto 2026 — Chiusura Fase 5.6 e freeze definitivo del backbone OSM

- **Fase 5.6 completata:** chiusa la costruzione e validazione della rete stradale light avviata dopo `SWITCH_OSM`.
- **Backbone frozen:** il routing B1--B5 è costruito sul PBF congelato `nord-est_2026-08-03.osm.pbf`.
- **Gate Gamma:** `E0=PASS`, `E1=PASS`, `E2=PASS`, `E3=PASS`.
- **Materializzazione Gamma:** 215 comuni, 645 accessi, 3 accessi/comune, 0 selection failures, 0 determinism failures, 0 separation violations, 0 topological independence failures.
- **E2:** 416.025/416.025 celle finite, 0 unreachable; 414.090 relazioni intercomunali ordinate.
- **Freeze accessi:** `Gamma_OSM = FROZEN` e `EXP_REL_300 = FROZEN`.
- **F1 B5 shadow regression:** `PASS`; 4 siti, 0 technical failures, 0 real B5 regressions.
- **F2 build fidelity/directional audit:** `PASS`; 10 campioni complessivi fra semantica way e turn restrictions, 0 pipeline discordances.
- **F3 regional time-optimal metric audit:** `PASS_NO_SYSTEMIC_SIGNAL`; 10.951 OD intercomunali ISTAT/TomTom e 98.559 access-pair, tutte le OD finite e 0 unreachable. Il rapporto mediano `D_OSM_TIME_PRODUCT_LAMBDA / KM_TOT` è 1,009302.
- **Classificazione F3:** 10.750 `EXPECTED_MODEL_DIFFERENCE`, 23 `ACCESS_EFFECT`, 178 `LOCAL_ROUTING_ANOMALY`, 0 `BOUNDARY_OR_DOMAIN`, 0 `POSSIBLE_SYSTEMIC`, 0 `UNRESOLVED`.
- **Nessun segnale sistemico:** `SYSTEMIC_CONNECTIVITY_ERROR=NO`, `SYSTEMIC_DISTANCE_INFLATION=NO`, `SYSTEMIC_ROUTING_BIAS=NO`.
- **Esito finale:** `G_OSM_operativo = FROZEN`, `Gamma_OSM = FROZEN`, `EXP_REL_300 = FROZEN`, `SYSTEMIC issues = 0`, `BLOCKING issues = 0`, `NETWORK AUDIT = STOP`.
- **Package backbone:** `C:\Tesi\Tesi_QGIS\02_package\grafo_operativo_osm`; `G_OSM_operativo_v01.gpkg` SHA-256 `f1d87245d1bc28f3ecab16e126514f8a3ab718b73ce7bd244db2f12628697ef3`; manifest finale SHA-256 `c4ea80c9c660f6a513b20400d0c3edb2f9e6ec10c3364f0a4b0beed91af55da3`.
- **Package Gamma:** `C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_osm_light\`; GPKG SHA-256 `b899a2e0e29d7ef366c42f4b68ef43a25b4ba1150052b6275770dd9ae23e1d3f`; CSV SHA-256 `a2ec905f84a2df2f566d670a7522ed583e4d9235664f77e3920f96e3fca1ebb5`; E2 matrix SHA-256 `b787665e7c4064cf6ec68e1905d8db5654a69aad1538919f1d0a92adafa006db`; intermunicipal pairs SHA-256 `07d371be745d37115186cd9cba74727f03de1b2d82a59b1f07764943d9e5ac85`; manifest Gamma SHA-256 `31ee2d78cd06c75cdf07014a8cecac4af2c58adeb26ec7285362ab4c02b11416`.
- **Decisione di arresto:** i 178 `LOCAL_ROUTING_ANOMALY` non giustificano una nuova campagna generale. La rete si riapre soltanto per un errore downstream dimostrato `BLOCKING` per OD assignment o FRLM.
- **Next storico al momento della chiusura 5.6:** impedenze/costi → shortest path OD. Questi blocchi sono stati successivamente completati nella Fase 5.7; la priorità gateway registrata al 24/08 è stata poi superseduta dalla decisione post-5.8A, che fissa come `NEXT` corrente la Fase 5.8B FVG-only.

## 21 agosto 2026

- **Gate OSM shadow CHIUSO:** 3 siti causali indipendenti e 1 controllo negativo valido. Esiti: `DATA_ERROR=1`, `GSFVG_DATA_MISSING=2`, `REPRESENTATION_ERROR=0`; OSM native correct 3/3; coverage advantage 2; controlli negativi peggiorati 0.
- **SHADOW-01 Cavazzo Carnico → Amaro:** `ID1 14897`, errore direzionale GSFVG correggibile con override; OSM nativamente corretto. Classificazione `DATA_ERROR`, non vantaggio strutturale.
- **SHADOW-02 Cercivento ↔ Paluzza:** collegamento Via dal Fiume assente anche nella sorgente GSFVG originale; OSM contiene nativamente la direttrice. `GSFVG_DATA_MISSING`, correction burden HIGH.
- **SHADOW-03 Ovaro ↔ Raveo:** Via Muina/SP35 assente anche da `GSFVG_IRDAT_FULL`; OSM contiene nativamente il collegamento. `GSFVG_DATA_MISSING`, correction burden HIGH.
- **Controllo negativo:** `ID1 15304`, Via Leonardo Andervolti. GSFVG e OSM entrambi funzionalmente corretti; nessuna regressione introdotta da OSM.
- **Decisione metodologica:** `REPEATED_GSFVG_COVERAGE_DEFICIT = YES`. Due missing-link indipendenti ad alto correction burden sono ritenuti evidenza sufficiente sul costo residuo anche senza `REPRESENTATION_ERROR` confermati.
- **Switch backbone:** approvato `SWITCH_OSM`. GSFVG resta benchmark storico e fonte istituzionale di confronto; stop alle campagne GSFVG estensive P1/P2/P3/P4/SP2 salvo necessità locale futura.
- **Fase 5.6:** nuova pipeline sul PBF congelato `nord-est_2026-08-03.osm.pbf`, con backbone routing B1--B5 e dominio turn-aware.
- **Domini di riferimento:** B3 giant SCC = 889.440 nodi; B5 mutual = 889.423 nodi.
- **Gamma OSM — rappresentazione strutturale:** 131.871 structural nodes, 167.033 topological segments; 757.552 shape-like nodes esclusi dai candidati Gamma ma mantenuti nel backbone.
- **Sensitivity cardinalità:** nearest-$K$ puro è ridondante; confermato $K=3$ con primary anchored, nessun topological segment incidente condiviso e separazione pairwise ≥100 m.
- **Candidate search:** top20 209/215, top25 214/215, top30 215/215; adozione di ricerca adattiva, non `top30` come parametro. Rank mediano 6, p95 16, massimo 29.
- **E0:** `v01 REJECTED`, `v02 SUPERSEDED`, `v03 PASS`, `v04 PASS`. Final audit: 10.750 candidate rows, 215 comuni, zero non-structural, zero senza incidence.
- **Stato Gamma:** `Gamma structural design = CLOSED`; **E1 ed E2 non ancora chiusi**, quindi `Gamma_OSM != FROZEN`.
- **Next:** E1 materializzazione 645 accessi + audit `EXP_REL_300`; E2 full Dijkstra 645-source sul B5 turn-aware. Freeze solo se `E1=PASS ∧ E2=PASS`.

## 20 agosto 2026

- **SP2 — segnali diagnostici:** 1.818 OD con $KM\_TOT < D_{\min}^{GSFVG}$ e shortlist di 125 OD; i conteggi non equivalgono a errori dimostrati.
- **Approvato Gate preliminare OSM shadow:** GSFVG resta baseline operativa durante il test; non è approvata alcuna migrazione a OSM.
- **Criterio prospettico:** confronto fra $T_{\mathrm{finish}}^{GSFVG}$ e $T_{\mathrm{switch+finish}}^{OSM}$; il lavoro passato è sunk cost ma informa il `manual correction burden`.
- **Time-box:** circa una giornata effettiva massima, con stop anticipato se l'evidenza è già sufficiente.
- **Unità sperimentale:** 3–5 siti causali fisici indipendenti, non singole OD.
- **Test locale:** controfattuale OSM a monte/a valle della causa, senza backbone OSM regionale, $\Gamma^{OSM}$ completo o seconda pipeline shortest-path.
- **Tassonomia causale:** `DATA_ERROR`, `REPRESENTATION_ERROR`, `GSFVG_DATA_MISSING`, `GSFVG_UNRESOLVED`, `OTHER`.
- **Ground truth:** OSM confrontato con evidenza indipendente; confronto binario con GSFVG non forzato quando il dato regionale è mancante o ambiguo.
- **Benchmark TomTom:** escluso l'uso di MAE/RMSE su `KM_TOT` come criterio generale di scelta del backbone.
- **Stop rule simmetriche:** `STOP OSM` se non emerge rapidamente vantaggio strutturale; `STOP audit GSFVG estensivo → switch feasibility` se emerge evidenza strutturale sufficiente.
- **Principio finale:** scegliere la soluzione che, da oggi, può essere resa sufficientemente affidabile prima.
## 14 agosto 2026

- **Fase 5.4 — SP0 graph and access regression:** ricostruiti 76.350 segmenti-parti da 76.349 feature, 152.700 endpoint e 61.947 nodi mediante `dwithin` 0,10 m + Union-Find. Regressione topologica esatta: 140.213 archi `DiGraph`, 10 WCC, giant WCC 61.914 nodi, 625 SCC, giant SCC 61.109 nodi.
- **Regressione accessi:** 645/645 `node_id` mappati, tutti distinti e nella giant SCC; errore massimo somma pesi $3{,}33\times10^{-16}$; errore coordinate $9{,}33\times10^{-10}$ m. SP0 superato.
- **Fase 5.4 — SP1 access-to-access:** 46.010 OD comunali ordinate × 9 combinazioni = 414.090 relazioni. Tutte 414.090/414.090 risultano raggiungibili.
- **Rappresentazione archi:** `MultiDiGraph` fisico con 140.668 directed edge instances; `DiGraph` ausiliario con 140.213 archi; 455 coppie ordinate parallele, molteplicità massima 2, 32 tie di costo minimo. Selezione minima + tie-break deterministico.
- **Dijkstra:** 645 single-source runs; confronto su tre sorgenti con NetworkX = differenza massima 0 m; errore massimo costo-path ricostruito $4{,}07\times10^{-10}$ m.
- **Uso della rete in SP1:** 61.852 directed edge instances appartenenti a $\mathcal A^{\mathrm{SP}}_{\mathrm{ACCESS}}$; 141.595.042 attraversamenti complessivi, coerenti con $\sum_aN_a^{\mathrm{ACCESS}}$.
- **Asimmetria:** mediana $\rho=1{,}004581$, p95 1,051639, massimo 4,060971; differenza assoluta massima andata/ritorno 27,154 km.
- **Sensibilità ai tre accessi:** spread delle nove alternative con mediana 0,924 km e p95 3,648 km.
- **MIN-PATH vs PRODUCT-LAMBDA:** MIN-PATH ha un minimo unico per 46.010/46.010 OD. PRODUCT-LAMBDA resta candidato; $D^{\mathrm{PL}}-D^{\mathrm{MIN}}$ ha mediana 0,464 km e p95 1,907 km.
- **Stato gate SP0/SP1:** computazionalmente SUPERATO. Resta aperta la scelta metodologica della rappresentazione comunale; prossimo passo = audit mirato delle code prima di congelare regola, shortest path comunali, $\mathcal A^{\mathrm{SP}}$ e $N_a^{\mathrm{OD}}$.

- **Gate strutturale degli accessi comunali light:** analizzati 20 candidati per ciascuno dei 215 comuni, per un totale di 4.300 nodi; 210/215 nearest-node appartengono già alla giant SCC e tutti i comuni dispongono di un candidato giant-SCC interno entro i primi 10.
- **Grafo usato per l'analisi accessi:** 61.947 nodi, 140.213 relazioni dirette, 625 SCC; giant SCC composta da 61.109 nodi. Il filtro giant-SCC viene adottato come requisito di ammissibilità, non come nuovo test indipendente di reachability.
- **Appartenenza territoriale:** 4.263/4.300 candidati risultano interni o entro 2 m dal proprio comune; nessun nearest-node e nessun primo candidato giant-SCC è esterno; tutti i 215 comuni hanno candidati giant-SCC interni nei top-20.
- **Sensibilità sulla numerosità:** la semplice scelta dei tre nearest ammissibili è risultata fortemente ridondante; 204/215 comuni condividono almeno un `ID1` fra due dei tre accessi. È stato quindi introdotto un vincolo esplicito di diversità topologica.
- **Regola anchored congelata:** $|\Gamma_o^{\mathrm L}|=3$ per 215/215 comuni; accesso primario = nodo ammissibile più vicino; secondari scelti nei top-20 con nessun `ID1` incidente condiviso e separazione pairwise ≥100 m.
- **Diagnostica anchored:** mediana extra-distanza 126,7 m, p95 394,6 m, massimo 711,5 m; separazione minima mediana 149,3 m, minimo osservato 100,5 m; massimo source rank utilizzato 19.
- **Pesi degli accessi:** testati $\tau=50,100,200,300,500$ m e pesi uniformi. $\tau=300$ m è il compromesso preferito: peso primario mediano 0,417, terziario mediano 0,268, $N_o^{\mathrm{eff}}$ mediano 2,894.
- **Materializzazione canonica completata:** prodotto `C:\Tesi\Tesi_QGIS\02_package\accessi_comunali_light\Gamma_L_comuni_fvg_v01.gpkg`, layer `Gamma_L_comuni_fvg_v01`, CSV gemello e manifest. Dataset: 215 comuni × 3 accessi = 645 record; `Gamma_L_metadata_v01` contiene 18 record di provenance.
- **Baseline sorgente accessi:** `GSFVG_operativo_direzionale_v02.gpkg`, SHA-256 `5afb71c500c2ff2cd779cee1601fb42354eadcf754ddc83ac2fc5fc481d7c02e`.
- **Checksum output:** GPKG `425dab4a127aeca2169911763f48238275d1dffe553d857a44c0fda773406179`; CSV `1a6d4d8075f84a9c98a39b4bde072ae15575a9a8dc5c02b39420fdb6cdd16386`.
- **Hard audit accessi:** 645/645 in giant SCC e nel proprio comune; primary nearest ammissibile 215/215; zero duplicati di `node_id` nello stesso comune; zero coppie <100 m; minimo 100,487 m; zero `ID1` incidenti condivisi; pesi positivi e somma 1 per 215/215 comuni; errori numerici massimi compatibili con precisione floating point; rilettura GPKG/metadata completa e `PRAGMA integrity_check = ok`.
- **Distribuzione distanze accessi:** mediana primary 130,0 m; mediana complessiva 204,3 m; p95 658,4 m; massimo 1.075,1 m. Non viene imposto un raggio assoluto rigido dal centroide.
- **Gate accessi comunali light:** CHIUSO. Sorgente operativa unica `Gamma_L_comuni_fvg_v01`; Run 1--4 mantenute soltanto come diagnostica/provenance.
- **Passo successivo:** shortest path metrici sulla baseline `GSFVG_operativo_direzionale_v02 + Gamma_L_comuni_fvg_v01`, con lunghezza geometrica come prima impedenza e preservazione degli archi paralleli tramite `MultiDiGraph` o trasformazione equivalente.

- **Primo ciclo di validazione funzionale P3/P4 completato.** Baseline Gate 4: 45.370/46.010 OD raggiungibili; stress test con esclusione dei 1.346 P3/P4: 43.893 raggiungibili, con perdita di 1.477 OD.
- **Attribuzione funzionale:** identificati sei archi con contributo individuale positivo alla riconnessione; analisi delle catene minime: 1.047 OD richiedono un solo P3/P4, 425 ne richiedono due e 5 ne richiedono tre. Rilevata la catena Vajont `25315 + 25414` su Via Colomber.
- **Revisione mirata 8/8:** 25315, 25414, 2762 e 21669 confermati `BIDIRECTIONAL`; 20894, 20974 e 51544 portati a `BWD_ONLY`; 67316 portato a `FWD_ONLY`.
- **Residuo P3/P4:** 1.338 archi non revisionati; stress test residuo = zero OD aggiuntive perse. Il gate P3/P4 sulla reachability è chiuso/convergente.
- **Diagnosi accessi comunali:** `G_iter1` presentava 44.946 OD raggiungibili, cioè 424 in meno del Gate 4. La causa è stata ricondotta agli accessi nearest-node di Sacile (node 49570) e Codroipo (node 37780), entrambi esterni alla giant SCC.
- **Sacile:** nearest originario 96,1 m; nearest giant-SCC node 48739 a 110,5 m (+14,4 m); sostituzione singola = +211 OD.
- **Codroipo:** nearest originario 61,6 m; nearest giant-SCC node 39946 a 98,1 m (≈+36,4 m); sostituzione singola = +212 OD.
- **Chiusura del ciclo:** sostituendo entrambi gli accessi la reachability torna esattamente a 45.370/46.010. Le 424 OD mancanti non erano dovute alle direzioni P3/P4.
- **Decisione metodologica:** `|Γ_o^L|=1` resta una baseline diagnostica ma la selezione del nodo non può basarsi sulla sola distanza geometrica. Il prossimo livello di validazione riguarda shortest path, distanze e tempi.

## 13 agosto 2026

- **Chiusura del consolidamento direzionale GSFVG:** costruita la copia persistente `GSFVG_operativo_direzionale_v01.gpkg`, che costituisce il confine tra review direzionale e futura costruzione del grafo computazionale.
- **Esiti finali P2:** 127 casi revisionati, di cui 65 `FWD_ONLY`, 46 `BWD_ONLY`, 16 `BIDIRECTIONAL`, zero `UNRESOLVED`; i 940 casi P2 fuori campione mantengono `TRIM_USAGE=0`.
- **Override manuali complessivi:** 879 `ID1` distinti, dati da 616 NULL + 136 P1 + 127 P2.
- **Distribuzione finale v01:** 11.256 `GSFVG_EXPLICIT`, 64.214 `GSFVG_DEFAULT`, 879 `MANUAL_REVIEW`; 9.371 `FWD_ONLY`, 2.657 `BWD_ONLY`, 64.321 `BIDIRECTIONAL`.
- **Audit source-output:** 76.349 `ID1`, FID sorgente e geometrie presenti; zero modifiche geometriche, zero mismatch sui 26 attributi originali, zero incoerenze direzionali. Corretto anche l'arrotondamento intermedio di `BEGIN`, `END` e `ORIG_LENGT` mediante serializzazione `Double` senza precisione nominale Shapefile.
- **Audit trail riproducibile:** `direction_overrides_v01` contiene gli 879 override; `build_metadata` registra input, conteggi, hash SHA-256, righe CSV, timestamp e informazioni di build. SHA-256 del GeoPackage v01: `24a699f71f5ad115afe2f9fa96c0f2b51334e022f18e5d3f47d3bc873cd0036a`.
- **Housekeeping QGIS:** materializzato `GSFVG_P2_review_display_127.gpkg` con verifica source-output 127/127; gruppo rinominato `02_P2_CHIUSO`; layer operativo collocato in `05_GRAFO_OPERATIVO`; progetto salvato con backup preventivi.
- **Separazione dei gate:** la v01 contiene soltanto la semantica direzionale. Accessi, velocità, tempi, impedenze/costi, eventuale riproiezione e accessi comunali devono essere applicati successivamente prima dell'audit funzionale del grafo computazionale.

- **Passaggio dalla revisione direzionale all'audit funzionale:** chiuse le revisioni manuali P1 e P2, viene sospesa la revisione esaustiva dei 1.346 archi P3/P4; la priorità passa alla verifica del comportamento complessivo del grafo.
- **Baseline funzionale $G^{\mathrm{base}}$:** `TRIM_USAGE=1/2` resta autorevole; i 616 NULL, P1 e i 127 casi P2 revisionati usano le decisioni manuali consolidate; P3/P4 non verificati mantengono temporaneamente `TRIM_USAGE=0`.
- **Razionalità conservativa:** P3/P4 vengono mantenuti bidirezionali nella baseline per evitare che sensi unici non verificati introducano artificialmente disconnessioni o deviazioni.
- **Revisione selettiva P3/P4:** le anomalie di reachability, componenti fortemente connesse, dead end, continuità delle arterie, shortest path e frequenza d'uso OD vengono retro-proiettate sugli archi; soltanto i casi con impatto materiale vengono riaperti manualmente.
- **Itinerari strategici:** l'audit comprende almeno Udine–Trieste, Udine–Pordenone, Trieste–Pordenone e Tolmezzo–Udine, affiancati all'analisi sistematica delle 46.010 OD direzionali.
- **Stress test OSM:** ammesso uno scenario $G^{\mathrm{stress}}_{\mathrm{OSM}}$ con applicazione più estesa delle indicazioni one-way P3/P4, esclusivamente per misurare la sensibilità di connettività e percorsi e non come validazione automatica.
- **Nuovo criterio decisionale:** la fase direzionale non richiede la ricostruzione esaustiva della micro-topologia urbana; è sufficiente dimostrare che le incertezze residue non alterano materialmente il routing necessario a shortest path, assegnazione OD e FRLM.

- **Formalizzazione della validazione funzionale pre-AON:** introdotta la Sezione 4.4.3, che definisce la validazione del grafo in termini di robustezza rispetto alle relazioni OD e non come certificazione esaustiva di ogni arco.
- **Dominio OD direzionale:** per i 215 comuni interni vengono considerate 46.010 coppie OD ordinate, escludendo gli intracomunali; andata e ritorno restano distinti perché la rete è diretta.
- **Audit conservativo di connettività:** formalizzata la distinzione tra $G^{\mathrm{full}}$ e $G^{\mathrm{safe}}$, con analisi di componenti deboli/forti, raggiungibilità direzionale e contributi di riconnessione $\Delta R_a$ ed eventualmente $\Delta F_a$.
- **Backbone funzionale dei shortest path:** definito $\mathcal A^{\mathrm{SP}}$ come unione degli archi dei cammini minimi e introdotti $N_a^{\mathrm{OD}}$ e $F_a^{\mathrm{OD}}$ per misurare l'importanza di ciascun arco prima dell'assegnazione.
- **Diagnostica con riferimenti esterni:** il confronto con tempi e distanze ISTAT/TomTom viene usato soltanto dopo verifica di compatibilità semantica e non costituisce un vincolo rigido. Le OD anomale vengono retro-proiettate sugli archi tramite $H_a$.
- **Prioritizzazione funzionale:** gli archi dubbi saranno ordinati combinando impatto sulla raggiungibilità, uso nei cammini, flusso, presenza nelle anomalie e rilevanza strategica; la prima applicazione può usare una gerarchia senza imporre un indice scalare unico.
- **Separazione pre/post assegnazione:** $\mathcal A^{\mathrm{SP}}$ è il sottografo funzionale costruito dai cammini minimi pre-AON; $\mathcal A^{\mathrm{used}}$ resta il sottografo post-AON delle sole OD con domanda positiva.
- **Criterio di arresto:** il grafo viene congelato quando le incertezze residue non alterano materialmente raggiungibilità, cammini minimi e indicatori aggregati; successive anomalie ANAS possono riaprire soltanto verifiche locali.

## 12 agosto 2026

- **Audit `TRIM_USAGE=0` — costruzione della shortlist:** individuati 3.349 archi GSFVG con `TRIM_USAGE=0` associati a one-way OSM con match `HIGH/STRICT`; 212 casi sono stati ricondotti a rotatorie e 588 a carreggiate o rami separati, lasciando 2.549 discordanti residue.
- **Stratificazione P1--P4:** i 2.549 casi residui sono stati suddivisi in P1=136 (`STRICT`, `DBPRIOR_ST=1`), P2=1.067 (`STRICT`, `DBPRIOR_ST` non compilato), P3=363 (`HIGH`, `DBPRIOR_ST=1`) e P4=983 (`HIGH`, `DBPRIOR_ST` non compilato).
- **Chiusura P1:** completata la revisione manuale esaustiva 136/136. Esiti: 102 `OSM_ONEWAY_CONFIRMED`, 13 `GSFVG_OK_BIDIRECTIONAL`, 20 `MODELLING_DIFFERENCE_GSFVG_OK`, 1 `ONEWAY_CONFIRMED_OSM_OPPOSITE`, 0 `UNRESOLVED`; direzioni finali 45 `FWD_ONLY`, 58 `BWD_ONLY`, 33 `BIDIRECTIONAL`.
- **Lezione metodologica P1:** 103/136 casi (75,7%) sono realmente monodirezionali, ma 33/136 (24,3%) devono restare bidirezionali; il segnale OSM strict è quindi informativo ma non trasferibile automaticamente al GSFVG, anche per differenze di granularità topologica.
- **Preparazione P2:** i 1.067 archi P2 sono stati organizzati in 775 cluster fisici, di cui 600 singleton e 175 multipli. Il clustering è mantenuto come supporto organizzativo, non come unità decisionale.
- **Regola P2:** adottata una validazione campionaria stratificata per classe stradale, direzione OSM, caratteristiche dell'elemento OSM e struttura del cluster; gli archi non verificati mantengono conservativamente `TRIM_USAGE=0`. Gli esiti guideranno il trattamento di P3 e P4.
- **Disegno campionario P2 consolidato:** predisposte 127 review, di cui 120 `INFERENTIAL_SAMPLE` e 7 `STRATEGIC_SUPPLEMENT` motorway/trunk. Le percentuali inferenziali saranno calcolate esclusivamente sui 120 casi inferenziali; i 7 strategici restano separati come controllo censuario delle arterie maggiormente rilevanti.
- **Benchmark Overpass API su P1:** testato un classificatore topologico sull'intera ground truth 136/136. Esiti: 43 `AUTO_ONEWAY_CANDIDATE`, 77 `AUTO_BIDIRECTIONAL_TOPOLOGY_CANDIDATE`, 16 `MANUAL_REVIEW_REQUIRED`; copertura teorica automatizzabile 88,24%.
- **Prestazioni insufficienti del classificatore:** precisione one-way 93,02%, precisione bidirectional 33,77%, accuratezza complessiva delle classi automatiche 55,00%; 3 falsi `AUTO_ONEWAY`, 51 falsi `AUTO_BIDIRECTIONAL`, 54 errori complessivi.
- **Decisione Overpass:** scartato l'uso come classificatore automatico o fonte di correzione di `TRIM_USAGE`. Overpass resta soltanto strumento diagnostico per way, nodi, tag e micro-topologia OSM; il layer di revisione P2 resta `P2_OSM_AS_GSFVG_DISPLAY` e il benchmark non modifica il campionamento né anticipa decisioni su P3/P4.
- **Analisi del precedente Tufaro:** integrata la tesi di dottorato di Teresa Tufaro come precedente metodologico per la trasformazione del GSFVG in un modello di rete stradale computazionale.
- **Conferma topologica indipendente:** il modello MRS parte da 76.350 segmenti, considera coincidenti estremi entro 0,1 m e, dopo la riduzione mediante `nodesR.f`, produce 29.695 nodi. La coincidenza delle 76.350 geometrie con l'audit corrente è considerata un forte indizio di continuità della base regionale, non una prova di identità delle versioni.
- **Separazione topologia--direzionalità:** dall'analisi di `conv.f` e `nodes.f` risulta che gli attributi direzionali non vengono trasferiti nel grafo computazionale e che le adiacenze sono costruite in entrambi i versi. Il precedente è quindi valido per la connettività, ma non giustifica una conversione automatica dei `TRIM_USAGE=0` a bidirezionali.
- **Baseline degli accessi comunali:** `com.f` associa ciascun comune al nodo della rete più prossimo; tale impostazione sarà utilizzata come baseline diagnostica $|\Gamma_o^{\mathrm L}|=1$ da confrontare con la formulazione multi-accesso pesata.
- **Confronto sui percorsi:** `per.f` utilizza una ricerca euristica con backtracking, rimozione casuale del 20% dei segmenti e fino a 10.000 tentativi; non viene adottato come algoritmo di routing. Restano confermati shortest path su costo temporale statico e all-or-nothing nella prima implementazione.
- **Cautela sui conteggi dei percorsi:** le quantità 6.397, 76.397 e «quasi 80.000» riportate in parti differenti della tesi di Tufaro sono trattate come incongruenza interna e non vengono utilizzate per validare quantitativamente il grafo corrente.

## 11 agosto 2026

- **Chiusura della revisione dei 616 `TRIM_USAGE = NULL`:** completata la verifica manuale dell'intera popolazione, articolata in 54 SC, 363 AS mainline e 199 AS residui. Esito finale: 557 `FWD_ONLY`, 1 `BWD_ONLY`, 58 `BIDIRECTIONAL`, 0 `UNRESOLVED`.
- **Falsificazione del tentativo automatico:** il controllo campionario sui 363 AS mainline ha individuato almeno un errore; l'attribuzione collettiva `FWD_ONLY` è stata quindi scartata e sostituita dalla revisione manuale completa, senza ripetere o aggiustare il campionamento.
- **Consolidamento dei 199 AS residui:** rifatta integralmente la revisione con CSV minimale `review_order; gsfvg_fid_original; ID1; TRIM_STR_C; DIR; ENTEXT; decisione`; ottenute 199 righe con chiavi univoche e corrispondenza esatta con `GSFVG_IRDAT_FULL`.
- **Audit globale:** confronto diretto dei tre CSV con il layer sorgente completo concluso con SC 54/54, AS mainline 363/363, AS residui 199/199, unione 616/616, zero sovrapposizioni, zero mancanti, zero estranei e zero `UNRESOLVED`.
- **Audit semantico degli attributi ufficiali:** censita la distribuzione `TRIM_USAGE` 64.477/8.704/2.552/616 per i valori 0/1/2/NULL e incrociati `TRIM_USAGE=0`, `DBPRIOR_ST`, `DBPRIOR_TI` e `DATA_FINE`; nessun elemento risulta `DBPRIOR_ST=2`, `DBPRIOR_TI=2` o con `DATA_FINE` valorizzato.
- **Cautela sui `TRIM_USAGE=0`:** i 64.477 archi non vengono convertiti automaticamente a bidirezionali; restano da verificare in particolare i 33.993 casi con `DBPRIOR_ST` e `DBPRIOR_TI` nulli e da trattare separatamente i 20 elementi `DBPRIOR_TI=3`.
- **Audit OSM ↔ GSFVG sui `TRIM_USAGE=1/2` concluso:** sui 11.256 archi monodirezionali espliciti risultano 7.456 `AGREEMENT_HIGH`, 3.009 `CONFLICT_HIGH`, 52 `AMBIGUOUS_MATCH`, 128 `AMBIGUOUS_DIRECTION` e 611 `INSUFFICIENT_MATCH`; tra i conflitti HIGH, 2.190 sono `OPPOSITE_ONEWAY`, 738 `GSFVG_ONEWAY_OSM_BIDIRECTIONAL` e 81 `GSFVG_ONEWAY_OSM_CLOSED`.
- **Validazione diagnostica del matching:** verificati manualmente 24 casi bilanciati nel regime `CLASSE=SC`, `ENTEXT=nd`, tre campioni validi, un solo `osm_id`, distanza media ≤2 m e allineamento ≤5°; esito 24/24 `MATCH_VALID`, 0 invalidi e 0 incerti. Il regime validato comprende 1.459 conflitti HIGH, pari al 48,49%, e 1.143 `OPPOSITE_ONEWAY`, pari al 52,19% della relativa classe.
- **Congelamento metodologico `TRIM_USAGE=1/2`:** `TRIM_USAGE=1` è assunto come `FWD_ONLY` e `TRIM_USAGE=2` come `BWD_ONLY`; GSFVG resta fonte primaria e OSM QA indipendente. Le discordanti non sono correzioni automatiche e non viene avviata una revisione manuale massiva dei 3.009 conflitti.
- **Tracciabilità futura:** nella copia operativa saranno mantenuti almeno `direction_source`, `direction_conflict_osm`, `direction_conflict_type` e `osm_match_confidence`; nessuna modifica al grafo operativo viene applicata in questa fase sulla base delle discordanti OSM.
- **Prossimo problema direzionale:** il blocco `TRIM_USAGE=1/2` è chiuso metodologicamente; l'analisi passa separatamente ai 64.477 archi con `TRIM_USAGE=0`.

## 10 agosto 2026

- **Controllo campionario dei 363 archi autostradali principali — impostazione storica poi falsificata l'11 agosto:** per il blocco con `TRIM_USAGE = NULL` già classificato `FWD_ONLY` era stata introdotta una validazione indipendente su un campione minimo di 81 archi.
- **Dimensionamento del campione:** il criterio adottato garantisce una probabilità superiore al 99% di intercettare almeno un errore se la quota reale di classificazioni errate è pari o superiore a circa il 5% nella popolazione finita di 363 archi.
- **Stratificazione:** gli 81 archi sono distribuiti rispetto alle otto combinazioni strada--direzione delle carreggiate principali A4, A23, A28 e A34, assicurando la rappresentazione di tutti i corridoi analizzati.
- **Regola di accettazione:** ogni arco campionato veniva verificato manualmente confrontando il verso di digitalizzazione GSFVG con la direzione di percorrenza della rete OSM validata; la discordanza effettivamente rilevata ha attivato la riapertura dell'intero blocco e il passaggio alla revisione manuale completa.

## 6 agosto 2026

- **Scelta del backbone topologico:** il grafo ufficiale GSFVG è stato selezionato come principale candidato per la rete light-duty; la struttura LRS regionale diventa il riferimento topologico, mentre OSM assume un ruolo complementare.
- **Audit GSFVG:** analizzate 76.350 parti lineari del file `GSFVG_IRDAT.shp`; 76.326 appartengono alla componente connessa principale, pari al 99,969%. Il risultato è stabile con tolleranze tra 0,10 e 1 metro.
- **Audit direzionale GSFVG:** `TRIM_USAGE` è assunto come campo ufficiale di percorribilità; `DIR`, `ENTEXT` e `GEOMETRY_R` restano attributi LRS non sufficienti, isolatamente, a dedurre il senso legale di marcia. Sono stati individuati 616 archi con `TRIM_USAGE` nullo.
- **Integrazione direzionale controllata:** per i 616 casi nulli è esclusa ogni inferenza automatica basata soltanto sulla geometria o sul nome della direzione; OSM potrà integrare il dato solo con matching geometrico adeguato, orientamento coerente e compatibilità di `oneway` e accesso.
- **Tracciabilità degli attributi:** i campi regionali originali non saranno sovrascritti; la copia operativa registrerà separatamente percorribilità forward e backward, fonte, confidenza e note di validazione.
- **Eccezioni topologiche:** le 24 feature esterne alla componente principale saranno verificate localmente; i 32 auto-anelli, connessi nel nodo di chiusura e privi di intersezioni intermedie, sono conservati come viabilità locale non essenziale per l'attraversamento.
- **Ruolo della rete OSM:** i layer già elaborati restano fonte complementare per verificare collegamenti e trasferire, previa validazione, velocità, sensi unici e restrizioni di accesso sul backbone GSFVG.
- **Integrità delle sorgenti:** `GSFVG_IRDAT.shp` e l'estratto OSM congelato non saranno modificati; riproiezioni, correzioni e nuovi attributi saranno applicati soltanto a copie operative versionate.
- **Chiusura metodologica del punto 5:** consolidate sui layer OSM complementari classi stradali e velocità di modello, direzionalità e regole di accesso, lunghezze ellissoidali validate, tempi statici forward e backward validati e costo della rete ordinaria uguale al tempo statico; resta da validarne il trasferimento sulla copia operativa GSFVG.
- **Avanzamento della roadmap:** il punto 6 diventa l'attività corrente, con costruzione degli insiemi di accesso comunali $\Gamma_o^{\mathrm L}$ e dei relativi pesi.
- **Accesso locale e `destination`:** consolidata la decisione di non attribuire ai versi con `routing_*_code=2` una penalità numerica generica, perché il vincolo descrive il ruolo ammissibile dell'arco e una penalità per geometria dipenderebbe dalla segmentazione OSM.
- **Regola operativa:** il codice 0 esclude il verso; il codice 1 identifica la rete core con costo temporale statico; il codice 2 esclude il verso dall'attraversamento ma lo conserva come possibile collegamento iniziale o finale.
- **Accessi zonali:** gli archi locali saranno trattati nella costruzione di $\Gamma_o^{\mathrm L}$ e non potranno essere utilizzati come scorciatoie intermedie.
- **Costo della rete core:** nella prima implementazione coincide con il tempo statico di percorrenza, senza penalità numeriche ulteriori non ancora giustificate.

## 5 agosto 2026

- **Sintesi storica della fase OSM:** selezionata e validata la base semantica OSM allora candidata alla rete light, conclusa la relativa normalizzazione del punto 4 e avviata la definizione delle velocità operative del punto 5; il 6 agosto GSFVG è stato successivamente scelto come backbone topologico principale.
- **Avanzamento GIS:** scelto l'estratto Geofabrik del 3 agosto 2026; costruita l'area FVG con buffer di 20 km; estratta e conservata la rete grezza; definite le classi light e separata la viabilità `service`; ottenute 112.466 geometrie singole; estratti i principali tag OSM.
- **Decisioni aggiunte nella fase OSM:** sorgente OSM congelata e derivati versionati; strade locali mantenute come possibili connettori; direzione rappresentata separando codice e stato; nessun controllo esaustivo dei valori nulli di `oneway`; validazione minima prima del routing e validazione per impatto dopo l'all-or-nothing; tutte le OD osservate positive vengono instradate. Tali regole restano disponibili per il trasferimento validato sul backbone GSFVG.
- **Correzione metodologica:** i casi `junction=circular` senza `oneway` sono stati prima isolati per audit, quindi le 22 geometrie osservate sono state riclassificate come monodirezionali nel verso della geometria; `junction=roundabout` e `highway=motorway` seguono l'implicazione documentata, salvo deroghe esplicite.
- **Censimento di `other_tags`:** analizzate 112.466 geometrie con zero errori di parsing; censite 446 chiavi residue, 312.530 occorrenze e 4.597 combinazioni distinte chiave--valore; corretta la terminologia tra occorrenze e combinazioni distinte.
- **Rettifica sull'accesso:** `vehicle=*` è presente su 80 geometrie; la gerarchia definitiva diventa `motorcar` → `motor_vehicle` → `vehicle` → `access`. Le regole condizionali e direzionali sono valutate come sovrapposizioni e i conteggi restano non additivi.
- **Rettifica sulla direzione:** rilevate 27 geometrie con `oneway:conditional`, delle quali 26 con alternanza stagionale e oraria; questa famiglia resta distinta da `alternating` e `reversible`.
- **Normalizzazione operativa:** ricostruito l'accesso secondo `motorcar` → `motor_vehicle` → `vehicle` → `access` → default implicito, con regole forward/backward separate; `open` è mappato su 1, `local_restricted` su 2 e `permission_restricted`, `authorized_only` e `prohibited` su 0, mantenendo le classi originali nei campi dedicati.
- **Audit finale della direzione:** verificate 118 geometrie speciali; `alternating` e `reversible` restano percorribili nei due versi con stato speciale, mentre i 22 `junction=circular` sono stati riclassificati nel verso della geometria. La partizione finale è 31.614 con `direction_code=1`, 80.851 con `direction_code=0` e 1 con `direction_code=-1`.
- **Verifica manuale:** il caso `reversible` di Via dei Bagni Nuova è un ponte a senso alternato regolato da semaforo.
- **Esiti finali per verso:** nel verso della geometria risultano 110.325 accessi ordinari, 500 locali, 1.640 chiusure per accesso e 1 per direzione; nel verso opposto 78.961 accessi ordinari, 424 locali, 1.467 chiusure per accesso e 31.614 per direzione. Ogni riga totalizza 112.466 geometrie.
- **Campi e layer:** introdotti anche `routing_fwd_closure_cause` e `routing_bwd_closure_cause`; il layer consolidato è `osm_rete_light_fvg_20km_routing_statico_validato` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg`.
- **Controlli finali:** non sono emersi valori nulli né incoerenze tra status, codici operativi e cause di chiusura; il layer validato costituisce la base per impedenze, velocità e tempi.
- **Primo grafo statico:** 58 regole condizionali di accesso e 26 di direzione sono risultate sintatticamente leggibili, con un caso sospetto per ciascuna famiglia; tali regole sono conservate ma non applicate senza una data e un'ora di riferimento. Il trattamento del codice 2, allora rinviato, è stato consolidato il 6 agosto come regola strutturale degli accessi zonali.
- **Normalizzazione OSM delle velocità:** creati `speed_fwd_obs_kmh` e `speed_bwd_obs_kmh` nel layer `osm_rete_light_fvg_20km_velocita_osm_normalizzate`, dando priorità ai valori specifici per verso e lasciando irrisolti i valori multipli o ambigui; il controllo non ha rilevato anomalie strutturali.
- **Separazione dato--modello:** i campi `speed_*_obs_kmh` rappresentano limiti ricavati da OSM, non velocità medie osservate sul campo; i campi `speed_*_model_kmh` rappresentano invece stime operative di routing e richiedono validazione e calibrazione.
- **Provenienza del benchmark:** la tabella completa delle velocità per classe è stata verificata nel profilo automobilistico ufficiale OSRM e fissata al tag riproducibile `v26.7.3`; rappresenta un benchmark di routing, non limiti legali o misure empiriche FVG.
- **Modello base delle velocità:** applicata per ciascun verso la regola $v^{\mathrm{model}}=\min(v^{\mathrm{obs}},v^{\mathrm{class}})$ quando è disponibile un valore OSM direzionale e il default OSRM di classe quando manca. Il layer `osm_rete_light_fvg_20km_velocita_modello_base` contiene i cinque campi di default, velocità forward/backward e relative fonti, valorizzati su tutte le 112.466 geometrie; intervallo 5--90 km/h, 69 differenze direzionali e nessuna anomalia.
- **Prossimo gate sulle impedenze:** non sono ancora stati calcolati `length_m`, tempi per verso, penalità o costo generalizzato e non è stato eseguito alcun shortest path o all-or-nothing. Il primo AON userà il costo temporale statico; velocità e tempi saranno controllati rispetto a ISTAT/TomTom e a itinerari campione.
- **Layer principali prodotti:** `osm_rete_light_fvg_20km_routing_statico_validato`, `osm_rete_light_fvg_20km_velocita_osm_normalizzate` e `osm_rete_light_fvg_20km_velocita_modello_base`.
- **Perimetro del censimento:** il filtro delle 191 chiavi è soltanto un richiamo; il layer lineare non comprende automaticamente relazioni di svolta e barriere puntuali.
- **Chiusura della fase 4:** il punto 4 è completato il 5 agosto 2026 sul piano della base semantica statica; la validazione non equivale a un controllo manuale esaustivo di ogni geometria e non comprende ancora relazioni di svolta, barriere puntuali o percorsi.
- **Limiti residui:** restano da completare grafo computazionale, controlli di connettività, restriction relation, impedenze temporali, penalità, costi e validazione dei percorsi.
- **Roadmap:** il punto 5 è in corso; dopo la prima assegnazione sarà applicato il ciclo di estrazione di $\mathcal A^{\mathrm{used}}$, controllo mirato, correzione e nuova assegnazione fino alla stabilizzazione.

## 4 agosto 2026

- **Sintesi:** consolidamento delle risposte alla checklist dei 22 punti e aggiornamento coordinato di domanda light, rete, dati, candidati, rete elettrica e FRLM.
- **Sezioni aggiornate:** stato iniziale, dati ISTAT/ANAS, perimetro FVG, zonizzazione, accessi al grafo, candidati, AFIR, matrice OD, assegnazione, connessione elettrica, dimensionamento, decisioni, questioni residue e roadmap.
- **Decisioni aggiunte:** giorno medio annuo; conversione minima dei pendolari con quota dei conducenti $0{,}711$, annualizzazione operativa su 220 giorni e flussi simmetrici di andata e ritorno; sole OD interzonali; focus corrente light; nessuna griglia uniforme; notazione osservato--stimato; matrice OD--link esplicita; transito esterno escluso; gateway dopo il grafo; accessi zonali multipli; compatibilità elettrica molti-a-molti; EV/HPC successivo; colonnine omogenee; AFIR provvisorio.
- **Questioni chiuse o riformulate:** chiusa la conversione minima ISTAT pendolari--veicoli e rimossi i relativi punti dall'elenco aperto; restano grafo, gateway, proxy, pesi ANAS, calibrazione, parametri FRLM, compatibilità elettrica e verifica normativa.
- **Contraddizioni risolte:** unità temporale, trattamento intrazonale, termine heavy basato sui conteggi, griglia fissa, notazione dei flussi, perimetro esterno, ruolo dei centroidi, sequenza grafo--gateway, connessione candidato--cabina e livello di dettaglio HPC.
- **Roadmap:** punto 2 completato il 4 agosto 2026; il punto 3, selezione e preparazione dei proxy territoriali per la componente non pendolare, è la prossima attività operativa.

---

# 25. Riferimenti principali

- `Modello_FRLM_Tesi.md`, bozza originaria del modello FRLM capacitato per il progetto.
- Kuby, M., Lim, S. (2005), *The flow-refueling location problem for alternative-fuel vehicles*, *Socio-Economic Planning Sciences*, 39(2), 125--145, [DOI 10.1016/j.seps.2004.03.001](https://doi.org/10.1016/j.seps.2004.03.001).
- De Padova, A., Schiera, D. S., Minuto, F. D., Lanzini, A. (2024), *Spatial MILP optimization framework for siting Hydrogen Refueling Stations in heavy-duty freight transport*, *International Journal of Hydrogen Energy*, 94, 669--686, [DOI 10.1016/j.ijhydene.2024.11.086](https://doi.org/10.1016/j.ijhydene.2024.11.086).
- Saadati, R., Saebi, J., Jafari-Nokandi, M. (2022), *Effect of uncertainties on siting and sizing of charging stations and renewable energy resources: A modified capacitated flow-refueling location model*, *Sustainable Energy, Grids and Networks*, 31, 100759, [DOI 10.1016/j.segan.2022.100759](https://doi.org/10.1016/j.segan.2022.100759).
- ISTAT (2025), *Matrice di pendolarismo per lavoro. Censimento permanente della popolazione e delle abitazioni 2021*, [pagina istituzionale](https://www.istat.it/notizia/matrice-di-pendolarismo-per-lavoro/).
- ISTAT (2019), *Regione Friuli-Venezia Giulia*, Tavola 3, [scheda regionale](https://www.istat.it/it/files/2020/05/06_Friuli-Venezia-Giulia_Scheda.pdf).
- ISTAT (2022), *Piano degli spostamenti casa-lavoro -- sedi romane*, [documento](https://www.istat.it/storage/trasparenza/19-altri-contenuti/mobilita-aziendale/2022/PSCL-sedi-romane.pdf).
- ISTAT (2022), *Piano degli spostamenti casa-lavoro -- Piemonte*, [documento](https://www.istat.it/storage/trasparenza/19-altri-contenuti/mobilita-aziendale/2022/PSCL-Piemonte.pdf).
- Parlamento europeo e Consiglio dell'Unione europea, Regolamento (UE) 2023/1804 sulla realizzazione di un'infrastruttura per i combustibili alternativi, [testo consolidato](https://eur-lex.europa.eu/legal-content/IT/TXT/?uri=CELEX:02023R1804-20260108).
- Regione Autonoma Friuli Venezia Giulia, *Rete viaria — Open Data FVG*, dataset GSFVG e dizionario degli attributi, [pagina istituzionale](https://www.dati.friuliveneziagiulia.it/Trasporti/Rete-viaria/4tby-g59n).
- Tufaro, T. (A.A. 2021/2022), *Valutazione del Rischio Sismico di un Modello di Rete Stradale per la Regione Friuli Venezia Giulia (NE)*, Tesi di Dottorato, XXXV ciclo; in particolare §2.2 e appendici con i programmi `conv.f`, `nodes.f`, `nodesR.f`, `com.f` e `per.f`.
- Geofabrik GmbH, *OpenStreetMap Data Extracts — Italy Nord-Est*, [pagina dell'estratto](https://download.geofabrik.de/europe/italy/nord-est.html).
- OpenStreetMap Foundation, *Copyright and License — Open Database License*, [informazioni sulla licenza](https://www.openstreetmap.org/copyright).
- Elaborazione propria in QGIS 3.40.0 sul file `GSFVG_IRDAT.shp`: audit geometrico, connettività degli estremi, componenti topologiche e auto-anelli.
- Elaborazione propria in QGIS 3.40.0 sugli attributi direzionali di `GSFVG_IRDAT.shp`: identificazione di `TRIM_USAGE` come campo primario e censimento di 616 archi con valore nullo.
- Elaborazione propria sulla revisione direzionale GSFVG dei 616 archi con `TRIM_USAGE = NULL`: diagnosi attributiva, tentativo automatico sui 363 AS mainline, controllo campionario con rilevazione di almeno un errore, successiva revisione manuale completa dei blocchi SC, AS mainline e AS residui e audit globale 616/616.
- Elaborazione propria sull'audit semantico GSFVG di `TRIM_USAGE`, `DBPRIOR_ST`, `DBPRIOR_TI` e `DATA_FINE` e sull'audit indipendente OSM ↔ GSFVG degli 11.256 archi con `TRIM_USAGE=1/2`, inclusa la validazione diagnostica manuale del matching su 24 casi nel regime geometrico robusto e il consolidamento della regola GSFVG-primary / OSM-QA.
- OpenStreetMap Wiki, *Key:oneway*, valori, restrizioni implicite e interpretazione per il routing, [documentazione](https://wiki.openstreetmap.org/wiki/Key:oneway).
- OpenStreetMap Wiki, *Tag:junction=circular*, caratteristiche e direzionalità delle intersezioni circolari, [documentazione](https://wiki.openstreetmap.org/wiki/Tag:junction%3Dcircular).
- OpenStreetMap Wiki (2026), *Key:access*, gerarchia delle restrizioni per modalità e prevalenza delle chiavi specifiche, revisione 3054035 del 25 giugno 2026, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:access&oldid=3054035).
- OpenStreetMap Wiki (2026), *Key:motor_vehicle*, ambito della restrizione per i veicoli a motore, revisione 2959485 del 2 marzo 2026, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:motor_vehicle&oldid=2959485).
- Censimento interno di `other_tags` sul layer `osm_rete_light_fvg_20km_tag_estratti`, eseguito il 5 agosto 2026 sull'estratto OSM del 3 agosto 2026; output completo con 4.597 combinazioni chiave--valore e relativi conteggi, nome e percorso permanenti da registrare.
- Normalizzazione interna delle regole di accesso e direzione, comunicata il 5 agosto 2026, applicata a valle del layer `osm_rete_light_fvg_20km_tag_estratti`: conteggi pre-audit per verso, controllo sintattico delle regole condizionali e verifiche di coerenza; prodotto intermedio `osm_rete_light_fvg_20km_routing_statico_operativo`.
- Audit interno finale della direzionalità e delle cause di chiusura, comunicato il 5 agosto 2026: verifica delle 118 geometrie speciali, riclassificazione dei 22 casi circolari, introduzione dei campi `closure_cause` e produzione di `osm_rete_light_fvg_20km_routing_statico_validato` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg`.
- Normalizzazione interna dei tag OSM di velocità, comunicata il 5 agosto 2026 nella nota `C:\Users\visen\.codex\attachments\0c83efe6-b138-458f-99d2-a218d7c1e325\pasted-text.txt`: definizione e controllo del layer `osm_rete_light_fvg_20km_velocita_osm_normalizzate`.
- Chiusura operativa della fase 4 e avanzamento del punto 5, comunicati il 5 agosto 2026 nella nota `C:\Users\visen\.codex\attachments\dcf81606-857d-4783-918a-3134de055493\pasted-text.txt`: conteggi finali di accesso e direzione, statistiche esatte delle velocità OSM, applicazione del benchmark OSRM mediante regola del minimo e produzione di `osm_rete_light_fvg_20km_velocita_modello_base`.
- Decisione metodologica interna sul trattamento degli archi ad accesso locale o `destination`, comunicata il 6 agosto 2026: esclusione del codice 2 dalla rete core di attraversamento e suo impiego limitato ai collegamenti iniziali o finali degli accessi zonali.
- Project OSRM (2026), *OSRM Backend -- Default car profile (`profiles/car.lua`)*, tabella `speeds.highway`, versione riproducibile v26.7.3, consultata il 5 agosto 2026; [profilo ufficiale](https://github.com/Project-OSRM/osrm-backend/blob/v26.7.3/profiles/car.lua).
- Project OSRM (2026), *OSRM Backend Documentation -- Profiles: Understanding speed, weight and rate*, versione v26.7.3, consultata il 5 agosto 2026; [documentazione ufficiale](https://github.com/Project-OSRM/osrm-backend/blob/v26.7.3/docs/profiles.md).
- OpenStreetMap Wiki (2025), *Key:maxspeed*, significato del limite massimo legale, unità, valori impliciti e tag specifici per verso, revisione 2936557 del 30 dicembre 2025, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:maxspeed&oldid=2936557).
- OpenStreetMap Wiki (2026), *Key:maxspeed:type*, tipologia e contesto del limite di velocità, revisione 2955947, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:maxspeed:type&oldid=2955947).
- OpenStreetMap Wiki (2026), *Default speed limits*, limiti impliciti e valori contestuali, inclusi i riferimenti per l'Italia, revisione 3061728 del 23 luglio 2026, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Default_speed_limits&oldid=3061728).
- OpenStreetMap Wiki (2026), *Conditional restrictions*, sintassi e precedenza delle restrizioni modali, direzionali e condizionali, revisione 3065730 del 4 agosto 2026, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Conditional_restrictions&oldid=3065730).
- OpenStreetMap Wiki (2026), *Node*, connessione tra ways e attraversamenti a quote differenti, revisione 2956443 del 19 febbraio 2026, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Node&oldid=2956443).
- OpenStreetMap Wiki, *Key:destination*, significato delle destinazioni indicate lungo una strada, [documentazione](https://wiki.openstreetmap.org/wiki/Key:destination).
- OpenStreetMap Wiki (2025), *Relation:restriction*, rappresentazione delle restrizioni di svolta, revisione 2821437 del 10 marzo 2025, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Relation:restriction&oldid=2821437).
- OpenStreetMap Wiki (2026), *Key:barrier*, barriere fisiche e controllo puntuale dell'accesso, revisione 2968970 del 22 marzo 2026, [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:barrier&oldid=2968970).

[^gsfvg-open-data]: Regione Autonoma Friuli Venezia Giulia, *Rete viaria — Open Data FVG*, dataset GSFVG e relativo dizionario degli attributi; [pagina istituzionale](https://www.dati.friuliveneziagiulia.it/Trasporti/Rete-viaria/4tby-g59n).

[^audit-gsfvg]: Elaborazione propria in QGIS 3.40.0 sul file originale `GSFVG_IRDAT.shp`, comunicata il 6 agosto 2026: audit geometrico e topologico su 76.350 parti lineari, 76.326 appartenenti alla componente connessa principale, 24 feature esterne e 32 auto-anelli; stabilità del risultato per tolleranze comprese tra 0,10 e 1 metro.

[^audit-trim-usage-gsfvg]: Elaborazione propria in QGIS 3.40.0 sul file originale `GSFVG_IRDAT.shp`, comunicata il 6 agosto 2026: audit attributivo delle informazioni direzionali, identificazione di `TRIM_USAGE` come campo ufficiale di percorribilità e censimento di 616 archi con valore nullo, prevalentemente autostradali e riconducibili alla medesima fonte interna del dataset.

[^audit-direzionale-completo-gsfvg]: Elaborazione propria consolidata l'11 agosto 2026 sulla popolazione completa dei 616 archi originariamente con `TRIM_USAGE = NULL`: 54 SC, 363 AS mainline e 199 AS residui; esito finale 557 `FWD_ONLY`, 1 `BWD_ONLY`, 58 `BIDIRECTIONAL`, 0 `UNRESOLVED`. Audit globale dei tre CSV rispetto a `GSFVG_IRDAT_FULL`: copertura 616/616, zero sovrapposizioni, zero mancanti e zero estranei. Il precedente tentativo automatico sui 363 AS mainline è stato abbandonato dopo che il controllo campionario ha rilevato almeno un errore.

[^audit-overture-20260811]: Elaborazione propria consolidata l'11 agosto 2026 sull'estratto Overture Maps Transportation comprendente FVG e intorno transfrontaliero: 246.223 feature automobilistiche e circa 37.574,7 km; 453.683 routing edge dotati di costo, con componente principale pari al 96,954%; 3.721 feature direzionalmente non risolte (1,511% delle feature; 3,179% della lunghezza); velocità esplicita utilizzabile sul 17,294% della rete; test su 247 casi GSFVG già validati con 186 concordi, 52 discordanti e 9 non risolti; 11/11 itinerari campione con cammino valido e nessun uso contromano, con un caso di deviazione anomala mantenuto come diagnostico; provenance dichiarata con OSM sul 99,739% delle feature e TomTom sullo 0,261%; 9.578 `prohibited_transitions` censite e non applicate nel routing preliminare.

[^tufaro2022]: Tufaro, T., *Valutazione del Rischio Sismico di un Modello di Rete Stradale per la Regione Friuli Venezia Giulia (NE)*, Tesi di Dottorato, XXXV ciclo, A.A. 2021/2022. Per la costruzione del MRS si vedano in particolare §2.2 (pp. 111--125) e le appendici con `conv.f`, `nodes.f`, `nodesR.f`, `com.f` e `per.f` (da p. 235). Il documento riporta 76.350 segmenti, soglia di coincidenza degli estremi pari a 0,1 m, 29.695 nodi ridotti, associazione comunale al nodo più prossimo e la procedura euristica di generazione dei percorsi. I conteggi finali dei percorsi risultano internamente non uniformi e non sono usati come benchmark quantitativo.

[^geofabrik-nord-est]: Geofabrik GmbH, *OpenStreetMap Data Extracts — Italy Nord-Est*; [pagina dell'estratto](https://download.geofabrik.de/europe/italy/nord-est.html).

[^osm-odbl]: OpenStreetMap Foundation, *Copyright and License*: attribuzione agli autori OpenStreetMap e licenza Open Database License; [informazioni ufficiali](https://www.openstreetmap.org/copyright).

[^kuby2005]: Kuby e Lim (2005), pp. 125--145, [doi:10.1016/j.seps.2004.03.001](https://doi.org/10.1016/j.seps.2004.03.001).

[^saadati2022]: Saadati, Saebi e Jafari-Nokandi (2022), art. 100759, [doi:10.1016/j.segan.2022.100759](https://doi.org/10.1016/j.segan.2022.100759).

[^istat2021]: ISTAT, *Matrice di pendolarismo per lavoro*, dati 2021: persone occupate che raggiungono il luogo abituale di lavoro almeno tre giorni alla settimana e rientrano giornalmente alla residenza; [pagina istituzionale](https://www.istat.it/notizia/matrice-di-pendolarismo-per-lavoro/).

[^istat-driver-fvg-2019]: ISTAT, *Regione Friuli-Venezia Giulia*, Tavola 3 (2019): quota regionale degli occupati che utilizzano l'auto privata come conducenti per recarsi al lavoro; [scheda regionale](https://www.istat.it/it/files/2020/05/06_Friuli-Venezia-Giulia_Scheda.pdf).

[^istat-pscl-roma-2022]: ISTAT, *Piano degli spostamenti casa-lavoro -- sedi romane* (2022): intervallo operativo di 200--220 giorni lavorativi annui; [documento](https://www.istat.it/storage/trasparenza/19-altri-contenuti/mobilita-aziendale/2022/PSCL-sedi-romane.pdf).

[^istat-pscl-piemonte-2022]: ISTAT, *Piano degli spostamenti casa-lavoro -- Piemonte* (2022): utilizzo operativo di 220 giorni lavorativi annui; [documento](https://www.istat.it/storage/trasparenza/19-altri-contenuti/mobilita-aziendale/2022/PSCL-Piemonte.pdf).

[^afir2023]: Regolamento (UE) 2023/1804, testo consolidato all'8 gennaio 2026; [EUR-Lex](https://eur-lex.europa.eu/legal-content/IT/TXT/?uri=CELEX:02023R1804-20260108).

[^depadova2024]: De Padova et al. (2024), pp. 669--686, [doi:10.1016/j.ijhydene.2024.11.086](https://doi.org/10.1016/j.ijhydene.2024.11.086).

[^osm-oneway]: OpenStreetMap Wiki, *Key:oneway*: `oneway=yes` segue il verso della geometria, `oneway=-1` il verso opposto; `junction=roundabout` e `highway=motorway` implicano normalmente `oneway=yes` salvo eccezioni esplicite; [documentazione](https://wiki.openstreetmap.org/wiki/Key:oneway).

[^osm-circular]: OpenStreetMap Wiki, *Tag:junction=circular*: le intersezioni circolari sono generalmente monodirezionali ma possono essere bidirezionali e dovrebbero riportare `oneway=yes/no`; [documentazione](https://wiki.openstreetmap.org/wiki/Tag:junction%3Dcircular).

[^osm-access-hierarchy]: OpenStreetMap Wiki, *Key:access*, revisione 3054035 del 25 giugno 2026, consultata il 5 agosto 2026: `access=*` è il livello generale della gerarchia e le chiavi specifiche di modalità possono sostituire il valore definito da una chiave genitrice; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:access&oldid=3054035).

[^osm-motor-vehicle]: OpenStreetMap Wiki, *Key:motor_vehicle*, revisione 2959485 del 2 marzo 2026, consultata il 5 agosto 2026: `motor_vehicle=*` riguarda tutti i veicoli a motore, mentre `motorcar=*` riguarda specificamente le automobili; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:motor_vehicle&oldid=2959485).

[^censimento-other-tags]: Elaborazione interna comunicata il 5 agosto 2026 sul layer `osm_rete_light_fvg_20km_tag_estratti`, riferito all'estratto `nord-est_2026-08-03.osm.pbf`. Il file completo contiene 4.597 combinazioni distinte chiave--valore con i relativi conteggi; nome, percorso e script di generazione devono ancora essere registrati stabilmente nei metadati del progetto.

[^normalizzazione-accesso-direzione]: Elaborazione interna comunicata il 5 agosto 2026 e integrata con la nota metodologica allegata `C:\Users\visen\.codex\attachments\8eead3d5-3a96-482a-a154-48af1d85bc88\pasted-text.txt` del 5 agosto 2026: ricostruzione gerarchica dell'accesso, separazione forward/backward, mappatura esaustiva delle cinque classi nei codici 0--2, audit dei tag condizionali, composizione pre-audit degli 80.873 casi `direction_code=0`, priorità diagnostica e conteggi completi sulle 112.466 geometrie. Il prodotto intermedio è `osm_rete_light_fvg_20km_routing_statico_operativo` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg`; i controlli riferiti non mostrano valori nulli, incoerenze tra status e codice o categorie inattese. L'audit successivo è documentato separatamente.[^audit-direzionalita-finale]

[^audit-direzionalita-finale]: Elaborazione interna comunicata il 5 agosto 2026 e consolidata nella nota `C:\Users\visen\.codex\attachments\dcf81606-857d-4783-918a-3134de055493\pasted-text.txt`: audit mirato delle 118 geometrie speciali, verifica manuale del caso `reversible` di Via dei Bagni Nuova, riclassificazione dei 22 `junction=circular` privi di `oneway`, introduzione e controllo di `routing_fwd_closure_cause` e `routing_bwd_closure_cause`. La partizione finale è 31.614 geometrie con `direction_code=1`, 80.851 con `direction_code=0` e 1 con `direction_code=-1`; i conteggi operativi backward finali sono 78.961 ordinari, 424 locali, 1.467 chiusi per accesso e 31.614 chiusi per direzione. Il layer validato è `osm_rete_light_fvg_20km_routing_statico_validato` nel GeoPackage `C:\Tesi\Tesi_QGIS\02_package\rete_stradale_osm_fvg.gpkg`; non sono stati riferiti valori nulli né incoerenze tra status, codici e cause di chiusura.

[^normalizzazione-velocita-osm]: Nota metodologica interna comunicata il 5 agosto 2026 e conservata in `C:\Users\visen\.codex\attachments\0c83efe6-b138-458f-99d2-a218d7c1e325\pasted-text.txt`. Documenta la prima normalizzazione dei tag OSM nei campi `speed_fwd_obs_kmh` e `speed_bwd_obs_kmh`, il layer `osm_rete_light_fvg_20km_velocita_osm_normalizzate` e l'assenza di anomalie strutturali. Le quantità allora riportate erano preliminari; i conteggi esatti e il modello successivamente applicato sono documentati nella chiusura della fase.[^velocita-modello-base]

[^velocita-modello-base]: Elaborazione interna consolidata il 5 agosto 2026 nella nota `C:\Users\visen\.codex\attachments\dcf81606-857d-4783-918a-3134de055493\pasted-text.txt`. I valori OSM risultano ricavabili per 27.484 geometrie forward e 27.490 backward, con intervallo 5--130 km/h, 93 differenze direzionali e 4 casi condizionali o variabili. Il layer `osm_rete_light_fvg_20km_velocita_modello_base` applica i default OSRM v26.7.3 e la regola del minimo per verso: tutti i cinque campi di modello sono valorizzati sulle 112.466 geometrie, le velocità ricadono nell'intervallo 5--90 km/h, le differenze direzionali sono 69 e i controlli non rilevano anomalie. Le fonti forward sono `class_default_missing` 84.979, `class_operational_cap` 17.091, `osm_limit_binding` 10.211, `osm_equals_class` 182 e `class_default_compound` 3; le fonti backward sono rispettivamente 84.973, 17.088, 10.220, 182 e 3.[^osrm-car-profile][^osrm-profile-docs]

[^decisione-accesso-locale]: Decisione metodologica interna comunicata il 6 agosto 2026: i versi con `routing_*_code=2` non sono modellati mediante una penalità fissa, poiché il vincolo riguarda l'ammissibilità dell'uso locale o terminale e una penalizzazione per segmento dipenderebbe dalla segmentazione OSM. Nella prima implementazione tali versi sono esclusi dalla rete core e ammessi soltanto nei collegamenti iniziali o finali costruiti mediante $\Gamma_o^{\mathrm L}$.

[^osrm-car-profile]: Project OSRM, *OSRM Backend -- Default car profile (`profiles/car.lua`)*, tag v26.7.3, consultato il 5 agosto 2026. La tabella `speeds.highway` riporta `motorway=90`, `motorway_link=45`, `trunk=85`, `trunk_link=40`, `primary=65`, `primary_link=30`, `secondary=55`, `secondary_link=25`, `tertiary=40`, `tertiary_link=20`, `unclassified=25`, `residential=25`, `living_street=10` e `service=15`; il profilo contiene inoltre cap e handler per superficie, `tracktype`, `smoothness`, `maxspeed` e altre condizioni; [versione riproducibile](https://github.com/Project-OSRM/osrm-backend/blob/v26.7.3/profiles/car.lua).

[^osrm-profile-docs]: Project OSRM, *OSRM Backend Documentation -- Profiles: Understanding speed, weight and rate*, tag v26.7.3, consultata il 5 agosto 2026. La documentazione distingue shortest e fastest routing, raccomanda velocità rappresentative dell'uso effettivo e specifica che `rate=speed` produce routing orientato al percorso più rapido; [versione riproducibile](https://github.com/Project-OSRM/osrm-backend/blob/v26.7.3/docs/profiles.md).

[^osm-maxspeed]: OpenStreetMap Wiki, *Key:maxspeed*, revisione 2936557 del 30 dicembre 2025, consultata il 5 agosto 2026: `maxspeed=*` rappresenta il limite massimo legale, è espresso in km/h in assenza di altra unità e può essere specificato per verso mediante `maxspeed:forward=*` e `maxspeed:backward=*`; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:maxspeed&oldid=2936557).

[^osm-maxspeed-type]: OpenStreetMap Wiki, *Key:maxspeed:type*, revisione 2955947, consultata il 5 agosto 2026: `maxspeed:type=*` descrive il modo o il contesto con cui il limite è entrato in vigore e può codificare limiti impliciti o di zona; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:maxspeed:type&oldid=2955947).

[^osm-default-speed-limits]: OpenStreetMap Wiki, *Default speed limits*, revisione 3061728 del 23 luglio 2026, consultata il 5 agosto 2026: inventario dei limiti impliciti per giurisdizione, con valori italiani fra cui 50 km/h in ambito urbano e 30 km/h per le zone corrispondenti; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Default_speed_limits&oldid=3061728).

[^osm-conditional]: OpenStreetMap Wiki, *Conditional restrictions*, revisione 3065730 del 4 agosto 2026, consultata il 5 agosto 2026: le restrizioni più specifiche per modalità prevalgono su quelle generali; a parità di modalità, le regole direzionali prevalgono sulle non direzionali e le condizionali sulle non condizionali; `forward` e `backward` dipendono dal verso della way; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Conditional_restrictions&oldid=3065730).

[^osm-nodes-crossings]: OpenStreetMap Wiki, *Node*, revisione 2956443 del 19 febbraio 2026, consultata il 5 agosto 2026: le ways che si intersecano alla stessa quota condividono un nodo, mentre quelle a quote differenti non devono connettersi e possono essere distinte mediante `layer=*`, `level=*`, `bridge=*` o attributi equivalenti; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Node&oldid=2956443).

[^osm-destination]: OpenStreetMap Wiki, *Key:destination*, consultata il 5 agosto 2026: `destination=*` identifica gli oggetti o luoghi raggiunti seguendo una strada e supporta l'annuncio delle destinazioni indicate; non coincide con `access=destination`; [documentazione](https://wiki.openstreetmap.org/wiki/Key:destination).

[^osm-turn-restrictions]: OpenStreetMap Wiki, *Relation:restriction*, revisione 2821437 del 10 marzo 2025, consultata il 5 agosto 2026: le restrizioni di svolta sono relazioni composte normalmente da membri `from`, `via` e `to`; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Relation:restriction&oldid=2821437).

[^osm-barriers]: OpenStreetMap Wiki, *Key:barrier*, revisione 2968970 del 22 marzo 2026, consultata il 5 agosto 2026: cancelli, dissuasori e altre barriere possono essere nodi collocati su una way e possiedono valori di accesso impliciti o espliciti rilevanti per il routing; [versione permanente](https://wiki.openstreetmap.org/w/index.php?title=Key:barrier&oldid=2968970).

